# Grain Strategy Product Research Flow - Exact Tested Code

This notebook is organized product-by-product and uses the **actual tested research functions** from the project.

Why this version exists:

The prior fully-standalone reimplementation was useful for understanding the mechanics, but it did not reproduce the exact best Sharpe/DD rows because it simplified several external-data and regime-selection builders.

This notebook prioritizes exact reproducibility of the researched results:

- Soybean best rows come from the soybean external and low-volatility experiments.
- Corn best rows come from the corn external/EIA and abundant-supply experiments.
- Wheat best rows come from the SRW/HRW pair improvement experiment.

Run cells top to bottom. Each product section shows the tests that lead to the final best rows.

## 0. Setup

In [43]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ["HOME"] = "/tmp"  # keep optional Meteostat/yfinance caches writable in notebook runs

import pandas as pd

try:
    from IPython.display import display, Markdown
except Exception:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 260)

DATA_DIR = "train_set"
COST_ADJUSTED_ONLY = True


def cost_only(df):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    if COST_ADJUSTED_ONLY and "cost_adjusted" in out.columns:
        out = out.loc[out["cost_adjusted"].astype(bool)].copy()
    return out.drop_duplicates().reset_index(drop=True)


def normalize_result_columns(df):
    """Add common display aliases used across the research notebooks."""
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    aliases = {
        "strategy": "experiment",
        "oos_sharpe": "test_sharpe",
        "oos_pnl": "test_pnl",
        "oos_dd": "test_max_drawdown",
        "full_dd": "max_drawdown",
    }
    for target, source in aliases.items():
        if target not in out.columns and source in out.columns:
            out[target] = out[source]
    return out


def safe_run(label, fn, *args, **kwargs):
    display(Markdown(f"### {label}"))
    try:
        out = fn(*args, **kwargs)
        display(Markdown("**Status:** success"))
        return out
    except Exception as exc:
        display(Markdown(f"**Status:** failed: `{type(exc).__name__}: {exc}`"))
        return None

def extract_linear_results(out):
    if out is None:
        return pd.DataFrame()
    if isinstance(out, dict) and "results" in out:
        return out["results"].copy()
    rows = []
    if isinstance(out, dict):
        for label, result in out.items():
            if isinstance(result, dict) and "results" in result:
                df = result["results"].copy()
                if "commodity" not in df.columns:
                    # Normalize labels used by linear_online_model_experiment.
                    commodity = "CORN" if str(label).lower().startswith("corn") else "SOYABEAN" if str(label).lower().startswith("soy") else str(label)
                    df.insert(0, "commodity", commodity)
                rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


## 1. Direct Standalone Implemented Code

This notebook is exportable: the implemented code is written directly in cells before it is used. It does not use `_exec_inline_module`, `sys.modules`, or imports from local experiment `.py` files.


### Direct Code: Shared Grain Futures Utilities

This cell defines the common loading, feature, backtest, and performance functions used by all product experiments.


In [ ]:
"""Lag-aware grain futures research utilities.

The module keeps the research pipeline in plain pandas/numpy so the notebooks
remain portable and easy to audit.
"""


import os

import numpy as np
import pandas as pd


COMMODITIES = ["CORN", "SOYABEAN", "WHEAT_SRW", "WHEAT_HRW"]
CONTRACT_MULTIPLIER = 5000.0
INTERCOMMODITY_SPREAD_PAIRS = [
    ("CORN", "SOYABEAN"),
    ("CORN", "WHEAT_SRW"),
    ("SOYABEAN", "WHEAT_SRW"),
    ("WHEAT_SRW", "WHEAT_HRW"),
]
OUTRIGHT_CORE_FEATURES = [
    "mom_60",
    "rev_5",
    "curve_spread",
    "curve_ratio",
    "cot_mm_level",
    "cot_pm_oi_level",
]
OUTRIGHT_PHYSICAL_FEATURES = [
    "public_inventory_change",
    "receipts_change",
    "cgl_inventory_change",
    "crush_surprise",
    "crush_utilization",
]
CALENDAR_SPREAD_FEATURES = [
    "curve_spread",
    "curve_ratio",
    "curve_change_20",
    "mom_20",
    "mom_60",
    "vol_20",
    "cot_mm_level",
    "cot_pm_oi_level",
    "public_inventory_level",
    "public_inventory_change",
    "cgl_inventory_change",
]
INTERCOMMODITY_RELATIVE_FEATURES = [
    "mom_20",
    "mom_60",
    "rev_5",
    "vol_20",
    "curve_spread",
    "curve_ratio",
    "curve_change_20",
    "cot_mm_level",
    "cot_pm_oi_level",
    "public_inventory_level",
    "public_inventory_change",
    "cgl_inventory_change",
]
DEFAULT_MARGIN_PER_LOT = {
    "CORN": 1500.0,
    "SOYABEAN": 3500.0,
    "WHEAT_SRW": 2500.0,
    "WHEAT_HRW": 2500.0,
}
COST_CASES = [
    {
        "case": "zero_cost_no_margin_cap",
        "trade_cost_per_lot": 0.0,
        "holding_cost_rate": 0.0,
        "margin_budget": np.inf,
        "description": "Research baseline with no transaction or funding costs.",
    },
    {
        "case": "market_assumption",
        "trade_cost_per_lot": 8.75,
        "holding_cost_rate": 0.05,
        "margin_budget": np.inf,
        "description": "Approx. 0.5 tick bid/ask plus commissions/fees, 5% annual margin funding.",
    },
    {
        "case": "market_assumption_margin_cap",
        "trade_cost_per_lot": 8.75,
        "holding_cost_rate": 0.05,
        "margin_budget": 2500.0,
        "description": "Market cost assumption plus a 2,500 USD margin budget per aggregate book.",
    },
    {
        "case": "stress_cost_margin_cap",
        "trade_cost_per_lot": 15.00,
        "holding_cost_rate": 0.08,
        "margin_budget": 2500.0,
        "description": "Stress case: wider execution cost, higher funding rate, same margin budget.",
    },
]
FINAL_BLEND_WEIGHTS = {
    "skip_rebalance": 0.50,
    "multi_condition": 0.50,
}
FINAL_OPPORTUNITY_QUANTILES = {
    "prediction": 0.40,
    "curve": 0.40,
    "momentum": 0.40,
}
REGIME_PERIODS = [
    {
        "period": "Russian drought/export ban shock",
        "start": "2010-07-01",
        "end": "2011-06-30",
        "reason": "Russian heat wave, drought, and grain export ban lifted in mid-2011.",
    },
    {
        "period": "US drought rally/retrace",
        "start": "2012-06-01",
        "end": "2013-05-31",
        "reason": "Historic US drought drove corn/soybean/wheat price shock and later retrace.",
    },
    {
        "period": "Crimea/Black Sea shock",
        "start": "2014-02-15",
        "end": "2014-05-31",
        "reason": "Ukraine/Crimea crisis raised Black Sea wheat and corn export risk.",
    },
    {
        "period": "Low-price abundant supply",
        "start": "2014-06-01",
        "end": "2017-12-31",
        "reason": "Post-drought supply rebuild and generally lower grain price regime.",
    },
    {
        "period": "US-China trade war",
        "start": "2018-07-06",
        "end": "2020-01-15",
        "reason": "Tariff escalation hit US soybean demand until the Phase One agreement.",
    },
    {
        "period": "2019 prevented planting floods",
        "start": "2019-05-01",
        "end": "2019-07-31",
        "reason": "Wet spring and Midwest flooding delayed corn and soybean planting.",
    },
    {
        "period": "COVID demand shock",
        "start": "2020-02-24",
        "end": "2020-06-30",
        "reason": "COVID restrictions reduced gasoline/ethanol demand and changed food demand.",
    },
    {
        "period": "COVID recovery/China buying",
        "start": "2020-07-01",
        "end": "2020-12-31",
        "reason": "Recovery phase with stronger Chinese buying and post-shock grain repricing.",
    },
]


def load_train_set(data_dir="train_set"):
    """Load all expected training CSVs into a dictionary of DataFrames."""
    names = {
        "adj1": "train_adjPrices1.csv",
        "adj2": "train_adjPrices2.csv",
        "unadj1": "train_unadjPrices1.csv",
        "unadj2": "train_unadjPrices2.csv",
        "cot_mm": "train_cot_mm.csv",
        "cot_pm_oi": "train_cot_pm_oi.csv",
        "inventories": "train_inventories.csv",
        "receipts": "train_receipts.csv",
        "cgl_inv": "train_cgl_inv.csv",
        "cgl_crush": "train_cgl_crush.csv",
    }
    data = {}
    for key, filename in names.items():
        path = os.path.join(data_dir, filename)
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        df = df.sort_index()
        df = df.apply(pd.to_numeric, errors="coerce")
        data[key] = df
    return data


def dataset_summary(data):
    """Return a compact table describing each loaded DataFrame."""
    rows = []
    for key in sorted(data):
        df = data[key]
        rows.append(
            {
                "dataset": key,
                "rows": len(df),
                "columns": len(df.columns),
                "start": df.index.min(),
                "end": df.index.max(),
                "missing_cells": int(df.isnull().sum().sum()),
                "columns_list": ", ".join([str(c) for c in df.columns]),
            }
        )
    return pd.DataFrame(rows)


def to_available_calendar(df, trading_index, lag_days):
    """Shift observations to their first usable date, then forward-fill.

    The input timestamp is treated as the observation date. A lag of 1 means
    the value can first be used on timestamp + 1 calendar day. The shifted
    series is then aligned to the adjusted-price trading calendar.
    """
    out = df.copy()
    out.index = pd.to_datetime(out.index) + pd.DateOffset(days=int(lag_days))
    out = out.sort_index()
    out = out.groupby(out.index).last()
    return out.reindex(trading_index).ffill()


def rolling_zscore(df, window=252, min_periods=40):
    mean = df.rolling(window=window, min_periods=min_periods).mean()
    std = df.rolling(window=window, min_periods=min_periods).std()
    return (df - mean) / std.replace(0.0, np.nan)


def _signed_clip(df, limit=5.0):
    return df.clip(lower=-limit, upper=limit)


def build_feature_panels(data):
    """Build one feature DataFrame per commodity.

    Feature timing:
    - Price and curve features are observed on the adjusted-price calendar.
    - Public inventories and receipts are shifted by T+2.
    - Cargill inventory and crush are shifted by T+1.
    - COT data is shifted by 3 calendar days as a conservative weekly release
      assumption.
    """
    trading_index = data["adj1"].index
    adj1 = data["adj1"].reindex(trading_index).ffill()
    unadj1 = data["unadj1"].reindex(trading_index).ffill()
    unadj2 = data["unadj2"].reindex(trading_index).ffill()

    daily_price_change = adj1.diff()
    pct_change = adj1.pct_change()
    futures_pnl = daily_price_change * CONTRACT_MULTIPLIER

    cot_mm = to_available_calendar(data["cot_mm"], trading_index, 3)
    cot_pm_oi = to_available_calendar(data["cot_pm_oi"], trading_index, 3)
    inventories = to_available_calendar(data["inventories"], trading_index, 2)
    receipts = to_available_calendar(data["receipts"], trading_index, 2)
    cgl_inv = to_available_calendar(data["cgl_inv"], trading_index, 1)
    cgl_crush = to_available_calendar(data["cgl_crush"], trading_index, 1)

    curve_spread = unadj1 - unadj2
    curve_ratio = unadj1 / unadj2.replace(0.0, np.nan) - 1.0

    base_feature_blocks = {
        "mom_20": rolling_zscore(adj1.pct_change(20), 252, 60),
        "mom_60": rolling_zscore(adj1.pct_change(60), 252, 80),
        "rev_5": -rolling_zscore(adj1.pct_change(5), 126, 30),
        "vol_20": rolling_zscore(pct_change.rolling(20, min_periods=10).std(), 252, 60),
        "curve_spread": rolling_zscore(curve_spread, 252, 60),
        "curve_ratio": rolling_zscore(curve_ratio, 252, 60),
        "curve_change_20": rolling_zscore(curve_spread.diff(20), 252, 60),
        "cot_mm_level": rolling_zscore(cot_mm, 156, 40),
        "cot_mm_change": rolling_zscore(cot_mm.diff(5), 156, 40),
        "cot_pm_oi_level": rolling_zscore(cot_pm_oi, 156, 40),
        "cot_pm_oi_change": rolling_zscore(cot_pm_oi.diff(5), 156, 40),
        "public_inventory_level": rolling_zscore(inventories, 156, 40),
        "public_inventory_change": rolling_zscore(inventories.diff(5), 156, 40),
        "receipts_level": rolling_zscore(receipts, 126, 30),
        "receipts_change": rolling_zscore(receipts.diff(5), 126, 30),
        "cgl_inventory_level": rolling_zscore(cgl_inv, 252, 60),
        "cgl_inventory_change": rolling_zscore(cgl_inv.diff(5), 252, 60),
    }

    crush = pd.DataFrame(index=trading_index)
    crush["crush_processed"] = cgl_crush["processed"]
    crush["crush_planned"] = cgl_crush["planned"]
    crush["crush_surprise"] = cgl_crush["processed"] - cgl_crush["planned"]
    crush["crush_utilization"] = cgl_crush["processed"] / cgl_crush["planned"].replace(0.0, np.nan) - 1.0
    crush_features = rolling_zscore(crush, 252, 60)

    panels = {}
    for commodity in COMMODITIES:
        frame = pd.DataFrame(index=trading_index)
        for feature_name, block in base_feature_blocks.items():
            frame[feature_name] = block[commodity]
        for feature_name in crush_features.columns:
            frame[feature_name] = crush_features[feature_name]
        frame = _signed_clip(frame)
        panels[commodity] = frame

    return panels, futures_pnl


def fit_ridge_predict(features, target, train_mask, alpha=10.0):
    """Fit a standardised Ridge model and predict all rows."""
    valid = features.notnull().all(axis=1) & target.notnull()
    train = valid & train_mask
    if int(train.sum()) < max(40, features.shape[1] * 3):
        return pd.Series(np.nan, index=features.index), pd.Series(np.nan, index=features.columns)

    x_train = features.loc[train].values.astype(float)
    y_train = target.loc[train].values.astype(float)
    x_mean = x_train.mean(axis=0)
    x_std = x_train.std(axis=0)
    x_std[x_std == 0.0] = 1.0
    y_mean = y_train.mean()

    x_scaled = (x_train - x_mean) / x_std
    y_centered = y_train - y_mean
    penalty = alpha * np.eye(x_scaled.shape[1])
    beta = np.linalg.solve(np.dot(x_scaled.T, x_scaled) + penalty, np.dot(x_scaled.T, y_centered))

    all_valid = features.notnull().all(axis=1)
    pred = pd.Series(np.nan, index=features.index)
    x_all = features.loc[all_valid].values.astype(float)
    pred.loc[all_valid] = y_mean + np.dot((x_all - x_mean) / x_std, beta)
    coef = pd.Series(beta / x_std, index=features.columns)
    return pred, coef


def build_model_signals(feature_panels, futures_pnl, split_date="2018-01-01", alpha=25.0):
    """Fit one Ridge model per commodity to predict next-day dollar PnL."""
    split_date = pd.Timestamp(split_date)
    train_mask = futures_pnl.index < split_date
    predictions = pd.DataFrame(index=futures_pnl.index, columns=COMMODITIES, dtype=float)
    coefficients = {}

    for commodity in COMMODITIES:
        target = futures_pnl[commodity].shift(-1)
        pred, coef = fit_ridge_predict(feature_panels[commodity], target, train_mask, alpha=alpha)
        predictions[commodity] = pred
        coefficients[commodity] = coef

    return predictions, pd.DataFrame(coefficients)


def _forward_pnl_target(pnl_series, horizon):
    if int(horizon) <= 1:
        return pnl_series.shift(-1)
    return pnl_series.shift(-1).rolling(int(horizon), min_periods=int(horizon)).sum().shift(-(int(horizon) - 1))


def build_improved_model_signals(feature_panels, futures_pnl, split_date="2018-01-01"):
    """Build the improved two-block Ridge signal.

    The broad one-day Ridge model overfit the training data. This variant uses a
    slower five-day target and two deliberately small feature blocks:
    - core: price, curve, and COT features
    - physical overlay: public flow plus Cargill inventory/crush features

    The overlay is strongly regularised and added at full weight because it was
    the best choice on the 2016-2017 validation window while still improving
    2018-2020 out-of-sample performance.
    """
    split_date = pd.Timestamp(split_date)
    train_mask = futures_pnl.index < split_date
    core_predictions = pd.DataFrame(index=futures_pnl.index, columns=COMMODITIES, dtype=float)
    physical_predictions = pd.DataFrame(index=futures_pnl.index, columns=COMMODITIES, dtype=float)
    core_coefficients = {}
    physical_coefficients = {}

    for commodity in COMMODITIES:
        target = _forward_pnl_target(futures_pnl[commodity], 5)
        core_pred, core_coef = fit_ridge_predict(
            feature_panels[commodity][OUTRIGHT_CORE_FEATURES], target, train_mask, alpha=25.0
        )
        phys_pred, phys_coef = fit_ridge_predict(
            feature_panels[commodity][OUTRIGHT_PHYSICAL_FEATURES], target, train_mask, alpha=1000.0
        )
        core_predictions[commodity] = core_pred
        physical_predictions[commodity] = phys_pred
        core_coefficients[commodity] = core_coef
        physical_coefficients[commodity] = phys_coef

    predictions = core_predictions.fillna(0.0) + physical_predictions.fillna(0.0)
    coefficients = {
        "core": pd.DataFrame(core_coefficients),
        "physical": pd.DataFrame(physical_coefficients),
    }
    return predictions, coefficients, core_predictions, physical_predictions


def build_walk_forward_model_signals(
    feature_panels,
    futures_pnl,
    start_date="2014-01-01",
    retrain_frequency="YE",
    min_train_days=756,
    horizon=20,
):
    """Build expanding walk-forward signals.

    At each retraining date, coefficients are fit using only data strictly before
    that date. Those coefficients are then used until the next retraining date.
    This is slower than the static model but is a cleaner anti-overfit check.
    """
    start_date = pd.Timestamp(start_date)
    index = futures_pnl.index
    schedule = pd.date_range(start=start_date, end=index.max(), freq=retrain_frequency)
    schedule = [index[index >= date][0] for date in schedule if len(index[index >= date]) > 0]
    schedule = sorted(set(schedule))

    predictions = pd.DataFrame(0.0, index=index, columns=COMMODITIES)
    coefficient_rows = []

    for i, train_end in enumerate(schedule):
        next_end = schedule[i + 1] if i + 1 < len(schedule) else index.max() + pd.DateOffset(days=1)
        apply_mask = (index >= train_end) & (index < next_end)
        train_mask = index < train_end
        if int(train_mask.sum()) < int(min_train_days):
            continue

        for commodity in COMMODITIES:
            target = _forward_pnl_target(futures_pnl[commodity], int(horizon))
            core_pred, core_coef = fit_ridge_predict(
                feature_panels[commodity][OUTRIGHT_CORE_FEATURES], target, train_mask, alpha=25.0
            )
            phys_pred, phys_coef = fit_ridge_predict(
                feature_panels[commodity][OUTRIGHT_PHYSICAL_FEATURES], target, train_mask, alpha=1000.0
            )
            predictions.loc[apply_mask, commodity] = (
                core_pred.loc[apply_mask].fillna(0.0) + phys_pred.loc[apply_mask].fillna(0.0)
            )
            row = {"retrain_date": train_end, "commodity": commodity, "block": "core"}
            row.update(core_coef.to_dict())
            coefficient_rows.append(row)
            row = {"retrain_date": train_end, "commodity": commodity, "block": "physical"}
            row.update(phys_coef.to_dict())
            coefficient_rows.append(row)

    coefficients = pd.DataFrame(coefficient_rows)
    return predictions, coefficients


def build_calendar_spread_pnl(data):
    """Build front-vs-second calendar-spread PnL by commodity.

    A positive unit is long the front adjusted futures series and short the
    second adjusted futures series. Unadjusted front/second prices are still
    used as predictive curve features; adjusted prices are used for PnL to avoid
    roll jumps dominating the backtest.
    """
    adj1 = data["adj1"].ffill()
    adj2 = data["adj2"].reindex(adj1.index).ffill()
    return (adj1.diff() - adj2.diff()) * CONTRACT_MULTIPLIER


def build_calendar_spread_feature_panels(feature_panels):
    """Use commodity-specific curve/storage features for calendar-spread models."""
    panels = {}
    for commodity in COMMODITIES:
        cols = [name for name in CALENDAR_SPREAD_FEATURES if name in feature_panels[commodity].columns]
        panels[commodity] = feature_panels[commodity][cols].copy()
    return panels


def _spread_name(first, second):
    return str(first) + "_VS_" + str(second)


def build_intercommodity_spread_pnl(futures_pnl, pairs=None):
    """Build volatility-hedged inter-commodity spread PnL.

    A positive unit is long the first commodity and short a rolling-volatility
    hedge ratio of the second commodity. The hedge ratio is shifted one day, so
    it only uses information available before the traded close.
    """
    if pairs is None:
        pairs = INTERCOMMODITY_SPREAD_PAIRS
    vol = futures_pnl.rolling(60, min_periods=20).std().shift(1)
    out = pd.DataFrame(index=futures_pnl.index)
    hedge_ratios = pd.DataFrame(index=futures_pnl.index)
    for first, second in pairs:
        name = _spread_name(first, second)
        ratio = (vol[first] / vol[second].replace(0.0, np.nan)).clip(lower=0.25, upper=4.0)
        out[name] = futures_pnl[first] - ratio * futures_pnl[second]
        hedge_ratios[name] = ratio
    return out, hedge_ratios


def build_intercommodity_feature_panels(feature_panels, pairs=None):
    """Build relative feature panels for synthetic inter-commodity spreads."""
    if pairs is None:
        pairs = INTERCOMMODITY_SPREAD_PAIRS
    panels = {}
    for first, second in pairs:
        name = _spread_name(first, second)
        frame = pd.DataFrame(index=feature_panels[first].index)
        for feature in INTERCOMMODITY_RELATIVE_FEATURES:
            if feature in feature_panels[first].columns and feature in feature_panels[second].columns:
                frame[feature + "_rel"] = feature_panels[first][feature] - feature_panels[second][feature]
        panels[name] = _signed_clip(frame)
    return panels


def build_walk_forward_spread_signals(
    spread_feature_panels,
    spread_pnl,
    start_date="2014-01-01",
    retrain_frequency="YE",
    min_train_days=756,
    horizon=20,
    alpha=100.0,
):
    """Build annual walk-forward Ridge predictions for spread instruments."""
    start_date = pd.Timestamp(start_date)
    index = spread_pnl.index
    schedule = pd.date_range(start=start_date, end=index.max(), freq=retrain_frequency)
    schedule = [index[index >= date][0] for date in schedule if len(index[index >= date]) > 0]
    schedule = sorted(set(schedule))

    instruments = list(spread_pnl.columns)
    predictions = pd.DataFrame(0.0, index=index, columns=instruments)
    coefficient_rows = []

    for i, train_end in enumerate(schedule):
        next_end = schedule[i + 1] if i + 1 < len(schedule) else index.max() + pd.DateOffset(days=1)
        apply_mask = (index >= train_end) & (index < next_end)
        train_mask = index < train_end
        if int(train_mask.sum()) < int(min_train_days):
            continue

        for instrument in instruments:
            features = spread_feature_panels[instrument]
            target = _forward_pnl_target(spread_pnl[instrument], int(horizon))
            pred, coef = fit_ridge_predict(features, target, train_mask, alpha=float(alpha))
            predictions.loc[apply_mask, instrument] = pred.loc[apply_mask].fillna(0.0)
            row = {"retrain_date": train_end, "instrument": instrument}
            row.update(coef.to_dict())
            coefficient_rows.append(row)

    return predictions, pd.DataFrame(coefficient_rows)


def run_spread_experiment(data_dir="train_set", split_date="2018-01-01", cost_per_lot=0.0):
    """Test outright, calendar-spread, inter-commodity-spread, and combined books."""
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)

    outright_predictions, _ = build_walk_forward_model_signals(feature_panels, futures_pnl)
    outright_positions = model_predictions_to_positions(outright_predictions, futures_pnl)
    outright_bt, _ = backtest_positions(outright_positions, futures_pnl, cost_per_lot)

    calendar_pnl = build_calendar_spread_pnl(data)
    calendar_panels = build_calendar_spread_feature_panels(feature_panels)
    calendar_predictions, calendar_coefficients = build_walk_forward_spread_signals(
        calendar_panels, calendar_pnl, alpha=100.0
    )
    calendar_positions = model_predictions_to_positions(calendar_predictions, calendar_pnl)
    calendar_bt, _ = backtest_positions(calendar_positions, calendar_pnl, cost_per_lot)

    inter_pnl, hedge_ratios = build_intercommodity_spread_pnl(futures_pnl)
    inter_panels = build_intercommodity_feature_panels(feature_panels)
    inter_predictions, inter_coefficients = build_walk_forward_spread_signals(
        inter_panels, inter_pnl, alpha=100.0
    )
    inter_positions = model_predictions_to_positions(inter_predictions, inter_pnl)
    inter_bt, _ = backtest_positions(inter_positions, inter_pnl, cost_per_lot)

    spread_pnl = pd.concat([calendar_pnl.add_prefix("CAL_"), inter_pnl.add_prefix("PAIR_")], axis=1)
    spread_predictions = pd.concat(
        [calendar_predictions.add_prefix("CAL_"), inter_predictions.add_prefix("PAIR_")], axis=1
    )
    spread_positions = model_predictions_to_positions(spread_predictions, spread_pnl)
    spread_bt, _ = backtest_positions(spread_positions, spread_pnl, cost_per_lot)

    all_pnl = pd.concat([futures_pnl.add_prefix("OUT_"), spread_pnl], axis=1)
    all_predictions = pd.concat([outright_predictions.add_prefix("OUT_"), spread_predictions], axis=1)
    all_positions = model_predictions_to_positions(all_predictions, all_pnl)
    all_bt, _ = backtest_positions(all_positions, all_pnl, cost_per_lot)

    inter_overlay_20_bt = _backtest_weighted_sleeves(
        [(outright_positions, futures_pnl), (inter_positions, inter_pnl)],
        [0.80, 0.20],
        cost_per_lot,
    )
    inter_overlay_30_bt = _backtest_weighted_sleeves(
        [(outright_positions, futures_pnl), (inter_positions, inter_pnl)],
        [0.70, 0.30],
        cost_per_lot,
    )
    spread_overlay_50_bt = _backtest_weighted_sleeves(
        [(outright_positions, futures_pnl), (spread_positions, spread_pnl)],
        [0.50, 0.50],
        cost_per_lot,
    )

    return {
        "outright_predictions": outright_predictions,
        "outright_positions": outright_positions,
        "outright_metrics": split_performance(outright_bt, split_date),
        "calendar_pnl": calendar_pnl,
        "calendar_predictions": calendar_predictions,
        "calendar_coefficients": calendar_coefficients,
        "calendar_positions": calendar_positions,
        "calendar_metrics": split_performance(calendar_bt, split_date),
        "intercommodity_pnl": inter_pnl,
        "intercommodity_hedge_ratios": hedge_ratios,
        "intercommodity_predictions": inter_predictions,
        "intercommodity_coefficients": inter_coefficients,
        "intercommodity_positions": inter_positions,
        "intercommodity_metrics": split_performance(inter_bt, split_date),
        "spread_pnl": spread_pnl,
        "spread_predictions": spread_predictions,
        "spread_positions": spread_positions,
        "spread_metrics": split_performance(spread_bt, split_date),
        "combined_pnl": all_pnl,
        "combined_predictions": all_predictions,
        "combined_positions": all_positions,
        "combined_metrics": split_performance(all_bt, split_date),
        "inter_overlay_20_metrics": split_performance(inter_overlay_20_bt, split_date),
        "inter_overlay_30_metrics": split_performance(inter_overlay_30_bt, split_date),
        "spread_overlay_50_metrics": split_performance(spread_overlay_50_bt, split_date),
    }


def _backtest_weighted_sleeves(sleeves, weights, cost_per_lot=0.0):
    """Combine independently sized sleeves into one portfolio-level backtest."""
    backtests = []
    for sleeve, weight in zip(sleeves, weights):
        positions, pnl = sleeve
        bt, _ = backtest_positions(positions * float(weight), pnl, cost_per_lot)
        backtests.append(bt)

    out = pd.DataFrame(index=backtests[0].index)
    for column in ["gross_pnl", "costs", "net_pnl", "turnover", "gross_exposure", "held_gross_exposure"]:
        out[column] = sum(bt[column].reindex(out.index).fillna(0.0) for bt in backtests)
    out["cum_pnl"] = out["net_pnl"].cumsum()
    return out


def model_predictions_to_positions(predictions, futures_pnl, gross_lots=1.0):
    """Convert predictions to market-neutral cross-sectional positions."""
    risk = futures_pnl.rolling(60, min_periods=20).std().shift(1)
    risk_adjusted = predictions / risk.replace(0.0, np.nan)
    demeaned = risk_adjusted.sub(risk_adjusted.mean(axis=1), axis=0)
    denom = demeaned.abs().sum(axis=1).replace(0.0, np.nan)
    positions = demeaned.div(denom, axis=0) * float(gross_lots)
    return positions.fillna(0.0).clip(lower=-1.0, upper=1.0)


def edge_filtered_positions(predictions, futures_pnl, quantile=0.50, min_periods=252, gross_lots=1.0):
    """Build positions only when model cross-sectional edge is above its past quantile.

    The edge score is the daily cross-sectional standard deviation of risk
    adjusted predictions. It is compared with an expanding, one-day-lagged
    quantile so the filter uses only information available at the time.
    """
    base_positions = model_predictions_to_positions(predictions, futures_pnl, gross_lots)
    risk = futures_pnl.rolling(60, min_periods=20).std().shift(1)
    risk_adjusted = predictions / risk.replace(0.0, np.nan)
    edge = risk_adjusted.std(axis=1)
    threshold = edge.expanding(min_periods=int(min_periods)).quantile(float(quantile)).shift(1)
    active = (edge > threshold).astype(float).reindex(base_positions.index).fillna(0.0)
    return base_positions.mul(active, axis=0), edge, threshold


def baseline_momentum_positions(data, gross_lots=1.0):
    """Simple cross-sectional 60-day momentum benchmark."""
    adj1 = data["adj1"].ffill()
    signal = rolling_zscore(adj1.pct_change(60), 252, 80)
    demeaned = signal.sub(signal.mean(axis=1), axis=0)
    denom = demeaned.abs().sum(axis=1).replace(0.0, np.nan)
    positions = demeaned.div(denom, axis=0) * float(gross_lots)
    return positions.fillna(0.0).clip(lower=-1.0, upper=1.0)


def backtest_positions(positions, futures_pnl, cost_per_lot=0.0):
    """Backtest close-to-close PnL from positions known at prior close."""
    positions = positions.reindex(futures_pnl.index).fillna(0.0)
    held_positions = positions.shift(1).fillna(0.0)
    gross_pnl_by_asset = held_positions * futures_pnl
    turnover_by_asset = positions.diff().abs().fillna(0.0)
    costs = turnover_by_asset * float(cost_per_lot)
    net_pnl_by_asset = gross_pnl_by_asset - costs

    result = pd.DataFrame(index=futures_pnl.index)
    result["gross_pnl"] = gross_pnl_by_asset.sum(axis=1)
    result["costs"] = costs.sum(axis=1)
    result["net_pnl"] = net_pnl_by_asset.sum(axis=1)
    result["turnover"] = turnover_by_asset.sum(axis=1)
    result["gross_exposure"] = positions.abs().sum(axis=1)
    result["held_gross_exposure"] = held_positions.abs().sum(axis=1)
    result["cum_pnl"] = result["net_pnl"].cumsum()
    return result, net_pnl_by_asset


def _margin_frame(columns, margin_per_lot=None):
    if margin_per_lot is None:
        margin_per_lot = DEFAULT_MARGIN_PER_LOT
    values = {}
    for column in columns:
        key = str(column)
        if key.startswith("OUT_"):
            key = key[4:]
        if key.startswith("CAL_"):
            key = key[4:]
        values[column] = float(margin_per_lot.get(key, np.nan))
    fallback = np.nanmedian([v for v in values.values() if pd.notnull(v)])
    if not pd.notnull(fallback):
        fallback = 2500.0
    return pd.Series({k: (fallback if pd.isnull(v) else v) for k, v in values.items()})


def apply_margin_budget(positions, margin_per_lot=None, margin_budget=np.inf):
    """Scale positions down if estimated margin use exceeds a book-level budget."""
    if margin_budget is None or not np.isfinite(float(margin_budget)):
        return positions.copy(), pd.Series(1.0, index=positions.index)

    margin = _margin_frame(positions.columns, margin_per_lot)
    margin_use = positions.abs().mul(margin, axis=1).sum(axis=1)
    scale = (float(margin_budget) / margin_use.replace(0.0, np.nan)).clip(upper=1.0).fillna(1.0)
    return positions.mul(scale, axis=0), scale


def backtest_positions_with_costs(
    positions,
    futures_pnl,
    trade_cost_per_lot=0.0,
    holding_cost_rate=0.0,
    margin_per_lot=None,
    margin_budget=np.inf,
):
    """Backtest with execution costs, margin funding costs, and optional margin cap."""
    adjusted_positions, margin_scale = apply_margin_budget(positions, margin_per_lot, margin_budget)
    adjusted_positions = adjusted_positions.reindex(futures_pnl.index).fillna(0.0)
    held_positions = adjusted_positions.shift(1).fillna(0.0)
    pnl = futures_pnl.reindex(adjusted_positions.index).fillna(0.0)

    gross_pnl_by_asset = held_positions * pnl
    turnover_by_asset = adjusted_positions.diff().abs().fillna(0.0)
    trade_cost_by_asset = turnover_by_asset * float(trade_cost_per_lot)

    margin = _margin_frame(adjusted_positions.columns, margin_per_lot)
    margin_use_by_asset = held_positions.abs().mul(margin, axis=1)
    holding_cost_by_asset = margin_use_by_asset * (float(holding_cost_rate) / 252.0)

    net_pnl_by_asset = gross_pnl_by_asset - trade_cost_by_asset - holding_cost_by_asset
    result = pd.DataFrame(index=adjusted_positions.index)
    result["gross_pnl"] = gross_pnl_by_asset.sum(axis=1)
    result["trade_cost"] = trade_cost_by_asset.sum(axis=1)
    result["holding_cost"] = holding_cost_by_asset.sum(axis=1)
    result["costs"] = result["trade_cost"] + result["holding_cost"]
    result["net_pnl"] = net_pnl_by_asset.sum(axis=1)
    result["turnover"] = turnover_by_asset.sum(axis=1)
    result["gross_exposure"] = adjusted_positions.abs().sum(axis=1)
    result["held_gross_exposure"] = held_positions.abs().sum(axis=1)
    result["margin_used"] = margin_use_by_asset.sum(axis=1)
    result["margin_scale"] = margin_scale.reindex(result.index).fillna(1.0)
    result["cum_pnl"] = result["net_pnl"].cumsum()
    return result, net_pnl_by_asset


def performance_metrics(bt, split_date=None):
    """Compute dollar-PnL performance metrics."""
    active = bt["held_gross_exposure"] > 1.0e-12
    pnl = bt.loc[active, "net_pnl"].dropna()
    if len(pnl) == 0:
        return pd.Series(dtype=float)
    ann_factor = 252.0
    avg = pnl.mean()
    vol = pnl.std()
    sharpe = np.nan if vol == 0.0 else (avg / vol) * np.sqrt(ann_factor)
    cum = pnl.cumsum()
    drawdown = cum - cum.cummax()
    metrics = {
        "days": float(len(pnl)),
        "total_pnl": float(pnl.sum()),
        "annualized_avg_pnl": float(avg * ann_factor),
        "annualized_vol": float(vol * np.sqrt(ann_factor)),
        "sharpe": float(sharpe) if pd.notnull(sharpe) else np.nan,
        "max_drawdown": float(drawdown.min()),
        "hit_rate": float((pnl > 0.0).mean()),
        "avg_daily_turnover": float(bt["turnover"].reindex(pnl.index).mean()),
        "avg_gross_exposure": float(bt["gross_exposure"].reindex(pnl.index).mean()),
    }
    out = pd.Series(metrics)
    if split_date is not None:
        out.name = str(split_date)
    return out


def split_performance(bt, split_date):
    split_date = pd.Timestamp(split_date)
    before = bt.loc[bt.index < split_date]
    after = bt.loc[bt.index >= split_date]
    table = pd.DataFrame(
        {
            "in_sample": performance_metrics(before),
            "out_of_sample": performance_metrics(after),
            "full_period": performance_metrics(bt),
        }
    )
    return table


def period_performance(bt, periods=None):
    if periods is None:
        periods = REGIME_PERIODS
    rows = []
    for item in periods:
        start = pd.Timestamp(item["start"])
        end = pd.Timestamp(item["end"])
        metrics = performance_metrics(bt.loc[(bt.index >= start) & (bt.index <= end)])
        row = {
            "period": item["period"],
            "start": start,
            "end": end,
            "reason": item.get("reason", ""),
        }
        for key, value in metrics.items():
            row[key] = value
        rows.append(row)
    return pd.DataFrame(rows)


def run_research_pipeline(data_dir="train_set", split_date="2018-01-01", alpha=25.0, cost_per_lot=0.0):
    """Run the full load-feature-model-backtest pipeline."""
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)

    broad_predictions, broad_coefficients = build_model_signals(feature_panels, futures_pnl, split_date, alpha)
    broad_positions = model_predictions_to_positions(broad_predictions, futures_pnl)
    broad_bt, broad_pnl_by_asset = backtest_positions(broad_positions, futures_pnl, cost_per_lot)

    predictions, coefficient_blocks, core_predictions, physical_predictions = build_improved_model_signals(
        feature_panels, futures_pnl, split_date
    )
    unfiltered_positions = model_predictions_to_positions(predictions, futures_pnl)
    unfiltered_bt, unfiltered_pnl_by_asset = backtest_positions(unfiltered_positions, futures_pnl, cost_per_lot)

    model_positions, edge_score, edge_threshold = edge_filtered_positions(predictions, futures_pnl, quantile=0.50)
    model_bt, model_pnl_by_asset = backtest_positions(model_positions, futures_pnl, cost_per_lot)

    walk_forward_predictions, walk_forward_coefficients = build_walk_forward_model_signals(feature_panels, futures_pnl)
    walk_forward_positions = model_predictions_to_positions(walk_forward_predictions, futures_pnl)
    walk_forward_risk = futures_pnl.rolling(60, min_periods=20).std().shift(1)
    walk_forward_edge = (walk_forward_predictions / walk_forward_risk.replace(0.0, np.nan)).std(axis=1)
    walk_forward_threshold = pd.Series(np.nan, index=futures_pnl.index)
    walk_forward_bt, walk_forward_pnl_by_asset = backtest_positions(
        walk_forward_positions, futures_pnl, cost_per_lot
    )

    baseline_positions = baseline_momentum_positions(data)
    baseline_bt, baseline_pnl_by_asset = backtest_positions(baseline_positions, futures_pnl, cost_per_lot)

    return {
        "data": data,
        "summary": dataset_summary(data),
        "feature_panels": feature_panels,
        "futures_pnl": futures_pnl,
        "predictions": predictions,
        "coefficients": coefficient_blocks,
        "core_predictions": core_predictions,
        "physical_predictions": physical_predictions,
        "edge_score": edge_score,
        "edge_threshold": edge_threshold,
        "model_positions": model_positions,
        "model_bt": model_bt,
        "model_pnl_by_asset": model_pnl_by_asset,
        "walk_forward_predictions": walk_forward_predictions,
        "walk_forward_coefficients": walk_forward_coefficients,
        "walk_forward_positions": walk_forward_positions,
        "walk_forward_edge": walk_forward_edge,
        "walk_forward_threshold": walk_forward_threshold,
        "walk_forward_bt": walk_forward_bt,
        "walk_forward_pnl_by_asset": walk_forward_pnl_by_asset,
        "walk_forward_metrics": split_performance(walk_forward_bt, split_date),
        "unfiltered_positions": unfiltered_positions,
        "unfiltered_bt": unfiltered_bt,
        "unfiltered_pnl_by_asset": unfiltered_pnl_by_asset,
        "unfiltered_metrics": split_performance(unfiltered_bt, split_date),
        "broad_predictions": broad_predictions,
        "broad_coefficients": broad_coefficients,
        "broad_positions": broad_positions,
        "broad_bt": broad_bt,
        "broad_pnl_by_asset": broad_pnl_by_asset,
        "broad_metrics": split_performance(broad_bt, split_date),
        "baseline_positions": baseline_positions,
        "baseline_bt": baseline_bt,
        "baseline_pnl_by_asset": baseline_pnl_by_asset,
        "model_metrics": split_performance(model_bt, split_date),
        "baseline_metrics": split_performance(baseline_bt, split_date),
        "period_metrics": {
            "model": period_performance(model_bt),
            "walk_forward": period_performance(walk_forward_bt),
            "unfiltered": period_performance(unfiltered_bt),
            "broad": period_performance(broad_bt),
            "baseline": period_performance(baseline_bt),
        },
    }


def skip_rebalance_positions(positions, n_days):
    """Update positions every N trading days and hold them flat between updates."""
    n_days = int(n_days)
    if n_days <= 1:
        return positions.copy()

    out = positions.copy() * 0.0
    for i in range(0, len(positions.index), n_days):
        end = min(i + n_days, len(positions.index))
        out.iloc[i:end] = positions.iloc[i]
    return out


def staggered_hold_positions(positions, n_days):
    """Average N offset skip-rebalance sleeves to approximate overlapping holds."""
    n_days = int(n_days)
    if n_days <= 1:
        return positions.copy()

    sleeves = []
    for offset in range(n_days):
        sleeve = positions.copy() * 0.0
        for i in range(offset, len(positions.index), n_days):
            end = min(i + n_days, len(positions.index))
            sleeve.iloc[i:end] = positions.iloc[i]
        sleeves.append(sleeve)
    return sum(sleeves) / float(n_days)


def _holding_period_rows(label, base_positions, pnl, split_date, hold_periods, cost_per_lot):
    rows = []
    methods = [
        ("daily", 1, lambda pos, days: pos.copy()),
    ]
    for n_days in hold_periods:
        methods.append(("staggered", int(n_days), staggered_hold_positions))
        methods.append(("skip-rebalance", int(n_days), skip_rebalance_positions))

    for method, hold_days, transform in methods:
        held_positions = transform(base_positions, hold_days)
        bt, _ = backtest_positions(held_positions, pnl, cost_per_lot)
        metrics = split_performance(bt, split_date)
        rows.append(
            {
                "strategy": label,
                "method": method,
                "hold_days": int(hold_days),
                "is_sharpe": metrics.loc["sharpe", "in_sample"],
                "oos_sharpe": metrics.loc["sharpe", "out_of_sample"],
                "full_sharpe": metrics.loc["sharpe", "full_period"],
                "oos_pnl": metrics.loc["total_pnl", "out_of_sample"],
                "full_pnl": metrics.loc["total_pnl", "full_period"],
                "max_drawdown": metrics.loc["max_drawdown", "full_period"],
                "turnover": metrics.loc["avg_daily_turnover", "full_period"],
            }
        )
    return rows


def run_holding_period_experiment(
    data_dir="train_set",
    split_date="2018-01-01",
    hold_periods=None,
    cost_per_lot=0.0,
):
    """Run reproducible holding-period tests for the main strategy variants."""
    if hold_periods is None:
        hold_periods = [2, 3, 5, 10]

    results = run_research_pipeline(data_dir=data_dir, split_date=split_date, cost_per_lot=cost_per_lot)
    pnl = results["futures_pnl"]
    wf_edge_positions, _, _ = edge_filtered_positions(results["walk_forward_predictions"], pnl, quantile=0.50)

    variants = [
        ("Static edge-filtered", results["model_positions"]),
        ("Static unfiltered", results["unfiltered_positions"]),
        ("Walk-forward", results["walk_forward_positions"]),
        ("WF edge-filtered", wf_edge_positions),
    ]

    rows = []
    for label, positions in variants:
        rows.extend(_holding_period_rows(label, positions, pnl, split_date, hold_periods, cost_per_lot))
    table = pd.DataFrame(rows)
    table = table.sort_values(["strategy", "method", "hold_days"]).reset_index(drop=True)

    best_by_oos_sharpe = table.sort_values(["oos_sharpe", "full_sharpe"], ascending=False).head(10)
    return {
        "holding_period_table": table,
        "best_by_oos_sharpe": best_by_oos_sharpe,
    }


def validation_performance(bt, validation_start="2016-01-01", validation_end="2017-12-31"):
    """Compute metrics for a fixed validation period."""
    start = pd.Timestamp(validation_start)
    end = pd.Timestamp(validation_end)
    return performance_metrics(bt.loc[(bt.index >= start) & (bt.index <= end)])


def _experiment_score_row(name, positions, pnl, split_date, hold_days, rationale, selected_for):
    bt, _ = backtest_positions(positions, pnl, 0.0)
    metrics = split_performance(bt, split_date)
    validation = validation_performance(bt)
    us_china = performance_metrics(bt.loc[(bt.index >= "2018-07-06") & (bt.index <= "2020-01-15")])
    return {
        "experiment": name,
        "rationale": rationale,
        "selected_for": selected_for,
        "hold_days": int(hold_days),
        "validation_sharpe": validation.get("sharpe", np.nan),
        "is_sharpe": metrics.loc["sharpe", "in_sample"],
        "oos_sharpe": metrics.loc["sharpe", "out_of_sample"],
        "oos_pnl": metrics.loc["total_pnl", "out_of_sample"],
        "full_sharpe": metrics.loc["sharpe", "full_period"],
        "max_dd": metrics.loc["max_drawdown", "full_period"],
        "turnover": metrics.loc["avg_daily_turnover", "full_period"],
        "us_china_trade_war_sharpe": us_china.get("sharpe", np.nan),
        "us_china_trade_war_pnl": us_china.get("total_pnl", np.nan),
    }


def _feature_frame(feature_panels, feature_name, columns=None):
    if columns is None:
        columns = COMMODITIES
    return pd.DataFrame({commodity: feature_panels[commodity][feature_name] for commodity in columns})


def _expanding_active(score, quantile, min_periods=252):
    threshold = score.expanding(min_periods=int(min_periods)).quantile(float(quantile)).shift(1)
    return (score > threshold).astype(float).fillna(0.0)


def expanding_percentile_score(score, min_periods=252):
    """Lagged expanding percentile of today's score versus prior history."""
    score = pd.Series(score).astype(float)
    out = pd.Series(np.nan, index=score.index)
    values = score.values
    for i in range(len(score)):
        start = 0
        history = values[start:i]
        history = history[pd.notnull(history)]
        if len(history) < int(min_periods) or pd.isnull(values[i]):
            continue
        out.iloc[i] = float((history <= values[i]).mean())
    return out


def multi_condition_filter_positions(
    predictions,
    futures_pnl,
    feature_panels,
    prediction_quantile=0.40,
    curve_quantile=0.40,
    momentum_quantile=0.40,
):
    """Trade only when prediction, curve, and momentum dispersion are all high."""
    risk = futures_pnl.rolling(60, min_periods=20).std().shift(1)
    prediction_dispersion = (predictions / risk.replace(0.0, np.nan)).std(axis=1)
    curve_signal = (_feature_frame(feature_panels, "curve_spread") + _feature_frame(feature_panels, "curve_ratio")) / 2.0
    momentum_signal = _feature_frame(feature_panels, "mom_60")
    curve_dispersion = curve_signal.std(axis=1)
    momentum_dispersion = momentum_signal.std(axis=1)

    active = (
        _expanding_active(prediction_dispersion, prediction_quantile)
        * _expanding_active(curve_dispersion, curve_quantile)
        * _expanding_active(momentum_dispersion, momentum_quantile)
    )
    base_positions = model_predictions_to_positions(predictions, futures_pnl)
    return base_positions.mul(active, axis=0), active


def natural_opportunity_weight(
    predictions,
    futures_pnl,
    feature_panels,
    min_periods=252,
):
    """Continuous opportunity score from prediction, curve, and momentum dispersion.

    This avoids a hard regime cutoff. Each component is scored as a lagged
    expanding percentile against its own prior history, and the final score is
    the equal-weight average of the three percentiles.
    """
    risk = futures_pnl.rolling(60, min_periods=20).std().shift(1)
    prediction_dispersion = (predictions / risk.replace(0.0, np.nan)).std(axis=1)
    curve_signal = (_feature_frame(feature_panels, "curve_spread") + _feature_frame(feature_panels, "curve_ratio")) / 2.0
    momentum_signal = _feature_frame(feature_panels, "mom_60")
    curve_dispersion = curve_signal.std(axis=1)
    momentum_dispersion = momentum_signal.std(axis=1)

    parts = pd.DataFrame(
        {
            "prediction": expanding_percentile_score(prediction_dispersion, min_periods),
            "curve": expanding_percentile_score(curve_dispersion, min_periods),
            "momentum": expanding_percentile_score(momentum_dispersion, min_periods),
        }
    )
    return parts.mean(axis=1).fillna(0.0), parts


def natural_regime_weighted_positions(
    predictions,
    futures_pnl,
    feature_panels,
    low_exposure=0.35,
    high_exposure=1.00,
):
    """Continuous non-label regime allocation based on observable opportunity.

    Instead of switching by historical regime names or fixed cutoffs, exposure is
    scaled smoothly by the lagged expanding opportunity score.
    """
    opportunity, components = natural_opportunity_weight(predictions, futures_pnl, feature_panels)
    base_positions = model_predictions_to_positions(predictions, futures_pnl)
    exposure = float(low_exposure) + (float(high_exposure) - float(low_exposure)) * opportunity
    return base_positions.mul(exposure, axis=0), opportunity, components


def grain_volatility_state(futures_pnl, halflife=15):
    """Lagged realized-volatility state for the grain complex."""
    asset_vol = futures_pnl.rolling(60, min_periods=20).std().mean(axis=1)
    ewm_vol = asset_vol.ewm(halflife=float(halflife), adjust=False).mean().shift(1)
    long_vol = ewm_vol.expanding(min_periods=252).mean().shift(1)
    vol_ratio = (ewm_vol / long_vol.replace(0.0, np.nan)).replace([np.inf, -np.inf], np.nan)
    low_vol = (vol_ratio < 0.70).astype(float).fillna(0.0)
    high_vol = (vol_ratio > 1.30).astype(float).fillna(0.0)
    crisis_vol = (vol_ratio > 2.00).astype(float).fillna(0.0)
    return pd.DataFrame(
        {
            "ewm_vol": ewm_vol,
            "long_vol": long_vol,
            "vol_ratio": vol_ratio,
            "low_vol": low_vol,
            "high_vol": high_vol,
            "crisis_vol": crisis_vol,
        }
    )


def observable_three_sleeve_weights(predictions, futures_pnl, feature_panels):
    """Live-observable weights for static, 2-day, and opportunity sleeves.

    Inputs are lagged realized volatility and lagged expanding opportunity
    percentiles. Historical regime labels are not used.
    """
    opportunity, opportunity_parts = natural_opportunity_weight(predictions, futures_pnl, feature_panels)
    vol_state = grain_volatility_state(futures_pnl)
    vol_ratio = vol_state["vol_ratio"].clip(lower=0.0, upper=3.0).fillna(1.0)
    low_score = (1.0 - (vol_ratio / 0.70)).clip(lower=0.0, upper=1.0)
    stress_score = ((vol_ratio - 1.0) / 1.0).clip(lower=0.0, upper=1.0)
    normal_score = (1.0 - (vol_ratio - 1.0).abs()).clip(lower=0.0, upper=1.0)

    raw = pd.DataFrame(index=futures_pnl.index)
    raw["static_edge_filtered"] = 0.25 + 0.35 * normal_score + 0.20 * (1.0 - opportunity)
    raw["skip_rebalance_2d"] = 0.30 + 0.45 * low_score + 0.15 * (1.0 - stress_score)
    raw["multi_condition"] = 0.20 + 0.55 * opportunity + 0.35 * stress_score
    raw = raw.clip(lower=0.05)
    weights = raw.div(raw.sum(axis=1), axis=0).fillna(1.0 / 3.0)
    diagnostics = pd.concat(
        [
            opportunity.rename("opportunity_score"),
            opportunity_parts.add_prefix("opportunity_"),
            vol_state,
        ],
        axis=1,
    )
    return weights, diagnostics


def observable_three_sleeve_positions(
    predictions,
    futures_pnl,
    feature_panels,
    base_positions,
    apply_vol_scale=False,
    target_vol_ratio=1.0,
    min_scale=0.35,
    max_scale=1.25,
):
    """Blend three pre-defined sleeves using observable, non-label weights."""
    multi_positions, active = multi_condition_filter_positions(
        predictions,
        futures_pnl,
        feature_panels,
        FINAL_OPPORTUNITY_QUANTILES["prediction"],
        FINAL_OPPORTUNITY_QUANTILES["curve"],
        FINAL_OPPORTUNITY_QUANTILES["momentum"],
    )
    sleeves = {
        "static_edge_filtered": base_positions,
        "skip_rebalance_2d": skip_rebalance_positions(base_positions, 2),
        "multi_condition": multi_positions,
    }
    weights, diagnostics = observable_three_sleeve_weights(predictions, futures_pnl, feature_panels)
    blended = None
    for name, positions in sleeves.items():
        weighted = positions.mul(weights[name], axis=0)
        blended = weighted if blended is None else blended.add(weighted, fill_value=0.0)

    vol_scale = pd.Series(1.0, index=futures_pnl.index)
    if apply_vol_scale:
        vol_ratio = diagnostics["vol_ratio"].replace(0.0, np.nan)
        vol_scale = (float(target_vol_ratio) / vol_ratio).clip(
            lower=float(min_scale),
            upper=float(max_scale),
        )
        crisis_cap = pd.Series(1.0, index=futures_pnl.index)
        crisis_cap.loc[diagnostics["crisis_vol"] > 0.0] = 0.50
        vol_scale = (vol_scale * crisis_cap).fillna(1.0)
        blended = blended.mul(vol_scale, axis=0)

    diagnostics = diagnostics.copy()
    diagnostics["multi_condition_active"] = active
    diagnostics["vol_scale"] = vol_scale
    for column in weights.columns:
        diagnostics["weight_" + column] = weights[column]
    return blended.fillna(0.0), weights, diagnostics


def trailing_performance_ensemble_positions(strategy_positions, futures_pnl, lookback=252, temperature=1.0):
    """Blend predefined sleeves using lagged trailing Sharpe-style weights.

    This is a natural regime-weighting alternative: no historical regime labels
    are used, and weights are based only on each sleeve's trailing realized PnL.
    """
    names = list(strategy_positions.keys())
    pnl_by_strategy = pd.DataFrame(index=futures_pnl.index)
    for name in names:
        bt, _ = backtest_positions(strategy_positions[name], futures_pnl, 0.0)
        pnl_by_strategy[name] = bt["net_pnl"]

    mean = pnl_by_strategy.rolling(int(lookback), min_periods=max(60, int(lookback) // 4)).mean()
    vol = pnl_by_strategy.rolling(int(lookback), min_periods=max(60, int(lookback) // 4)).std()
    score = (mean / vol.replace(0.0, np.nan)).replace([np.inf, -np.inf], np.nan)
    score = score.shift(1).clip(lower=-2.0, upper=2.0).fillna(0.0)
    exp_score = np.exp(score / float(temperature))
    weights = exp_score.div(exp_score.sum(axis=1), axis=0).fillna(1.0 / float(len(names)))

    out = None
    for name in names:
        weighted = strategy_positions[name].mul(weights[name], axis=0)
        out = weighted if out is None else out.add(weighted, fill_value=0.0)
    return out.fillna(0.0), weights, score


def carry_storage_positions(feature_panels, futures_pnl):
    """Simple cross-sectional carry/storage sleeve."""
    curve = (_feature_frame(feature_panels, "curve_spread") + _feature_frame(feature_panels, "curve_ratio")) / 2.0
    storage = (
        -_feature_frame(feature_panels, "public_inventory_level")
        - _feature_frame(feature_panels, "public_inventory_change")
        - _feature_frame(feature_panels, "cgl_inventory_change")
    ) / 3.0
    signal = (0.6 * curve + 0.4 * storage).clip(lower=-5.0, upper=5.0)
    return model_predictions_to_positions(signal, futures_pnl)


def smooth_positions(positions, halflife=2, rescale=False):
    """EWMA-smooth positions; optionally rescale to the original gross exposure."""
    smoothed = positions.ewm(halflife=float(halflife), adjust=False).mean().fillna(0.0)
    if rescale:
        target = positions.abs().sum(axis=1).replace(0.0, np.nan)
        current = smoothed.abs().sum(axis=1).replace(0.0, np.nan)
        smoothed = smoothed.mul((target / current).clip(upper=2.0).fillna(0.0), axis=0)
    return smoothed.clip(lower=-1.0, upper=1.0)


def run_filter_sleeve_experiment(data_dir="train_set", split_date="2018-01-01"):
    """Summarise recent filter, holding-period, smoothing, and carry/storage tests."""
    results = run_research_pipeline(data_dir=data_dir, split_date=split_date)
    pnl = results["futures_pnl"]
    rows = []

    base_positions = results["model_positions"]
    rows.append(
        _experiment_score_row(
            "Baseline static edge-filtered",
            base_positions,
            pnl,
            split_date,
            1,
            "Two-block Ridge plus prediction-dispersion edge filter.",
            "Baseline tradable candidate",
        )
    )

    rows.append(
        _experiment_score_row(
            "2-day skip-rebalance",
            skip_rebalance_positions(base_positions, 2),
            pnl,
            split_date,
            2,
            "Hold positions for two trading days to reduce churn and preserve conviction.",
            "Best OOS PnL / lower-turnover execution variant",
        )
    )

    multi_positions, active = multi_condition_filter_positions(
        results["predictions"], pnl, results["feature_panels"], 0.40, 0.40, 0.40
    )
    row = _experiment_score_row(
        "Multi-condition opportunity filter",
        multi_positions,
        pnl,
        split_date,
        1,
        "Trade only when prediction, curve, and momentum dispersion are all high.",
        "Best crisis-regime and Sharpe candidate",
    )
    row["active_day_fraction"] = float(active.mean())
    rows.append(row)

    natural_positions, opportunity, _ = natural_regime_weighted_positions(
        results["predictions"], pnl, results["feature_panels"], 0.35, 1.00
    )
    row = _experiment_score_row(
        "Natural opportunity-weighted exposure",
        natural_positions,
        pnl,
        split_date,
        1,
        "Continuously scale exposure by expanding prediction/curve/momentum opportunity percentiles.",
        "Lower-overfit regime weighting candidate",
    )
    row["avg_opportunity_weight"] = float(opportunity.mean())
    rows.append(row)

    ensemble_positions, ensemble_weights, _ = trailing_performance_ensemble_positions(
        {
            "baseline": base_positions,
            "skip_2d": skip_rebalance_positions(base_positions, 2),
            "natural_opportunity": natural_positions,
        },
        pnl,
        lookback=252,
        temperature=1.0,
    )
    row = _experiment_score_row(
        "Trailing-performance natural ensemble",
        ensemble_positions,
        pnl,
        split_date,
        1,
        "Blend baseline, 2-day, and natural-opportunity sleeves by lagged trailing performance weights.",
        "Natural regime-weighted ensemble candidate",
    )
    for column in ensemble_weights.columns:
        row["avg_weight_" + column] = float(ensemble_weights[column].mean())
    rows.append(row)

    three_sleeve_positions, three_sleeve_weights, _ = observable_three_sleeve_positions(
        results["predictions"], pnl, results["feature_panels"], base_positions, apply_vol_scale=False
    )
    row = _experiment_score_row(
        "Observable three-sleeve regime blend",
        three_sleeve_positions,
        pnl,
        split_date,
        1,
        "Blend static, 2-day, and multi-condition sleeves using lagged volatility and opportunity weights.",
        "Live-observable regime blend candidate",
    )
    for column in three_sleeve_weights.columns:
        row["avg_weight_" + column] = float(three_sleeve_weights[column].mean())
    rows.append(row)

    scaled_three_sleeve_positions, scaled_three_sleeve_weights, scaled_diagnostics = observable_three_sleeve_positions(
        results["predictions"], pnl, results["feature_panels"], base_positions, apply_vol_scale=True
    )
    row = _experiment_score_row(
        "Observable three-sleeve blend + vol scale",
        scaled_three_sleeve_positions,
        pnl,
        split_date,
        1,
        "Same three-sleeve blend with lagged realized-volatility scaling and crisis cap.",
        "Risk-managed regime blend candidate",
    )
    for column in scaled_three_sleeve_weights.columns:
        row["avg_weight_" + column] = float(scaled_three_sleeve_weights[column].mean())
    row["avg_vol_scale"] = float(scaled_diagnostics["vol_scale"].mean())
    rows.append(row)

    smooth = smooth_positions(base_positions, halflife=2, rescale=False)
    rows.append(
        _experiment_score_row(
            "Position smoothing HL2",
            smooth,
            pnl,
            split_date,
            1,
            "EWMA smooth positions to reduce turnover.",
            "Rejected: turnover reduction only",
        )
    )

    carry = carry_storage_positions(results["feature_panels"], pnl)
    carry_overlay = base_positions * 0.80 + carry * 0.20
    rows.append(
        _experiment_score_row(
            "20% carry/storage overlay",
            carry_overlay,
            pnl,
            split_date,
            1,
            "Add simple carry/backwardation and inventory-draw sleeve.",
            "Rejected: weak full-period robustness",
        )
    )

    table = pd.DataFrame(rows)
    return {"filter_sleeve_table": table, "multi_condition_active": active}


def _cost_case_row(strategy, case, positions, pnl, split_date):
    bt, _ = backtest_positions_with_costs(
        positions,
        pnl,
        trade_cost_per_lot=case["trade_cost_per_lot"],
        holding_cost_rate=case["holding_cost_rate"],
        margin_budget=case["margin_budget"],
    )
    metrics = split_performance(bt, split_date)
    active = bt["held_gross_exposure"] > 1.0e-12
    active_bt = bt.loc[active]
    gross_total = float(active_bt["gross_pnl"].sum()) if len(active_bt) else 0.0
    trade_cost = float(active_bt["trade_cost"].sum()) if len(active_bt) else 0.0
    holding_cost = float(active_bt["holding_cost"].sum()) if len(active_bt) else 0.0
    total_cost = trade_cost + holding_cost
    cdf = np.nan if abs(gross_total) < 1.0e-12 else total_cost / abs(gross_total)
    return {
        "strategy": strategy,
        "case": case["case"],
        "case_description": case["description"],
        "trade_cost_per_lot": case["trade_cost_per_lot"],
        "holding_cost_rate": case["holding_cost_rate"],
        "margin_budget": case["margin_budget"],
        "CDF": cdf,
        "trade_cost": trade_cost,
        "holding_cost": holding_cost,
        "total_cost": total_cost,
        "avg_margin_used": float(active_bt["margin_used"].mean()) if len(active_bt) else np.nan,
        "avg_margin_scale": float(active_bt["margin_scale"].mean()) if len(active_bt) else np.nan,
        "is_sharpe": metrics.loc["sharpe", "in_sample"],
        "oos_sharpe": metrics.loc["sharpe", "out_of_sample"],
        "oos_pnl": metrics.loc["total_pnl", "out_of_sample"],
        "full_sharpe": metrics.loc["sharpe", "full_period"],
        "max_dd": metrics.loc["max_drawdown", "full_period"],
        "turnover": metrics.loc["avg_daily_turnover", "full_period"],
    }


def run_cost_margin_experiment(data_dir="train_set", split_date="2018-01-01", cost_cases=None):
    """Run transaction-cost, holding-cost, and margin-budget stress tests."""
    if cost_cases is None:
        cost_cases = COST_CASES
    results = run_research_pipeline(data_dir=data_dir, split_date=split_date)
    pnl = results["futures_pnl"]
    multi_positions, _ = multi_condition_filter_positions(
        results["predictions"], pnl, results["feature_panels"], 0.40, 0.40, 0.40
    )
    strategies = [
        ("Static edge-filtered", results["model_positions"]),
        ("2-day skip-rebalance", skip_rebalance_positions(results["model_positions"], 2)),
        ("Multi-condition filter", multi_positions),
        ("Annual walk-forward Ridge", results["walk_forward_positions"]),
    ]
    rows = []
    for strategy, positions in strategies:
        for case in cost_cases:
            rows.append(_cost_case_row(strategy, case, positions, pnl, split_date))
    table = pd.DataFrame(rows)
    return {"cost_margin_table": table}


def _window_final_metrics(label, bt):
    empty_row = {
        "window": label,
        "trading_days": 0,
        "winning_days": 0,
        "losing_days": 0,
        "flat_days": 0,
        "hit_rate": np.nan,
        "sharpe": np.nan,
        "total_pnl": 0.0,
        "gross_pnl": 0.0,
        "trade_cost": 0.0,
        "holding_cost": 0.0,
        "total_cost": 0.0,
        "CDF": np.nan,
        "max_drawdown": np.nan,
        "avg_daily_turnover": np.nan,
        "avg_gross_exposure": np.nan,
        "avg_margin_used": np.nan,
        "avg_margin_scale": np.nan,
    }
    active_bt = bt.loc[bt["held_gross_exposure"] > 1.0e-12]
    metrics = performance_metrics(bt)
    if len(active_bt) == 0 or len(metrics) == 0:
        return empty_row

    gross_pnl = float(active_bt["gross_pnl"].sum())
    trade_cost = float(active_bt["trade_cost"].sum()) if "trade_cost" in active_bt.columns else 0.0
    holding_cost = float(active_bt["holding_cost"].sum()) if "holding_cost" in active_bt.columns else 0.0
    total_cost = trade_cost + holding_cost
    cdf = np.nan if abs(gross_pnl) < 1.0e-12 else total_cost / abs(gross_pnl)
    pnl = active_bt["net_pnl"]
    return {
        "window": label,
        "trading_days": int(len(active_bt)),
        "winning_days": int((pnl > 0.0).sum()),
        "losing_days": int((pnl < 0.0).sum()),
        "flat_days": int((pnl == 0.0).sum()),
        "hit_rate": metrics["hit_rate"],
        "sharpe": metrics["sharpe"],
        "total_pnl": metrics["total_pnl"],
        "gross_pnl": gross_pnl,
        "trade_cost": trade_cost,
        "holding_cost": holding_cost,
        "total_cost": total_cost,
        "CDF": cdf,
        "max_drawdown": metrics["max_drawdown"],
        "avg_daily_turnover": metrics["avg_daily_turnover"],
        "avg_gross_exposure": metrics["avg_gross_exposure"],
        "avg_margin_used": float(active_bt["margin_used"].mean()) if "margin_used" in active_bt else np.nan,
        "avg_margin_scale": float(active_bt["margin_scale"].mean()) if "margin_scale" in active_bt else np.nan,
    }


def run_final_strategy_selection(
    data_dir="train_set",
    split_date="2018-01-01",
    trade_cost_per_lot=8.75,
    holding_cost_rate=0.05,
    margin_budget=np.inf,
    skip_weight=FINAL_BLEND_WEIGHTS["skip_rebalance"],
    multi_condition_weight=FINAL_BLEND_WEIGHTS["multi_condition"],
):
    """Build the fixed-weight final blend and report cost-adjusted metrics.

    The blend weights are fixed before scoring: half lower-turnover 2-day
    execution sleeve, half high-Sharpe opportunity-filter sleeve.
    """
    results = run_research_pipeline(data_dir=data_dir, split_date=split_date)
    pnl = results["futures_pnl"]
    base_positions = results["model_positions"]
    skip_positions = skip_rebalance_positions(base_positions, 2)
    multi_positions, active = multi_condition_filter_positions(
        results["predictions"],
        pnl,
        results["feature_panels"],
        FINAL_OPPORTUNITY_QUANTILES["prediction"],
        FINAL_OPPORTUNITY_QUANTILES["curve"],
        FINAL_OPPORTUNITY_QUANTILES["momentum"],
    )
    final_positions = (
        skip_positions * float(skip_weight)
        + multi_positions * float(multi_condition_weight)
    )
    bt, pnl_by_asset = backtest_positions_with_costs(
        final_positions,
        pnl,
        trade_cost_per_lot=trade_cost_per_lot,
        holding_cost_rate=holding_cost_rate,
        margin_budget=margin_budget,
    )

    split = pd.Timestamp(split_date)
    windows = [
        ("in_sample", bt.loc[bt.index < split]),
        ("out_of_sample", bt.loc[bt.index >= split]),
        ("full_period", bt),
    ]
    metrics = pd.DataFrame([_window_final_metrics(label, window_bt) for label, window_bt in windows])
    assumptions = pd.DataFrame(
        [
            {
                "assumption": "final_strategy",
                "value": "Fixed 50/50 blend: 2-day skip-rebalance + multi-condition opportunity filter",
            },
            {"assumption": "skip_rebalance_weight", "value": float(skip_weight)},
            {"assumption": "multi_condition_weight", "value": float(multi_condition_weight)},
            {
                "assumption": "prediction_dispersion_quantile",
                "value": FINAL_OPPORTUNITY_QUANTILES["prediction"],
            },
            {
                "assumption": "curve_dispersion_quantile",
                "value": FINAL_OPPORTUNITY_QUANTILES["curve"],
            },
            {
                "assumption": "momentum_dispersion_quantile",
                "value": FINAL_OPPORTUNITY_QUANTILES["momentum"],
            },
            {"assumption": "threshold_policy", "value": "Expanding and one-day lagged; not retuned after holdout review"},
            {"assumption": "trade_cost_per_lot", "value": float(trade_cost_per_lot)},
            {"assumption": "holding_cost_rate_annual", "value": float(holding_cost_rate)},
            {"assumption": "margin_budget", "value": margin_budget},
            {"assumption": "selection_policy", "value": "Blend weights are fixed, not optimized on Sharpe/PnL"},
        ]
    )
    return {
        "assumptions": assumptions,
        "final_strategy_metrics": metrics,
        "final_positions": final_positions,
        "final_backtest": bt,
        "final_pnl_by_asset": pnl_by_asset,
        "multi_condition_active": active,
    }


def run_lower_overfit_strategy(
    data_dir="train_set",
    split_date="2018-01-01",
    edge_quantile=0.50,
    cost_per_lot=0.0,
):
    """Run the pre-registered lower-overfit strategy.

    This is the strategy to present when overfit control matters more than the
    best static backtest. It uses only fixed feature blocks, fixed Ridge
    penalties, annual expanding walk-forward retraining, and a fixed expanding
    median edge filter. No external-data overlays and no full-sample weight
    search are used.
    """
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    predictions, coefficients = build_walk_forward_model_signals(feature_panels, futures_pnl)
    positions, edge_score, edge_threshold = edge_filtered_positions(
        predictions,
        futures_pnl,
        quantile=float(edge_quantile),
    )
    bt, pnl_by_asset = backtest_positions(positions, futures_pnl, cost_per_lot)
    metrics = split_performance(bt, split_date)
    assumptions = pd.DataFrame(
        [
            {"assumption": "model", "value": "Annual expanding walk-forward two-block Ridge"},
            {"assumption": "core_features", "value": ", ".join(OUTRIGHT_CORE_FEATURES)},
            {"assumption": "physical_features", "value": ", ".join(OUTRIGHT_PHYSICAL_FEATURES)},
            {"assumption": "core_alpha", "value": 25.0},
            {"assumption": "physical_alpha", "value": 1000.0},
            {"assumption": "target_horizon_days", "value": 20},
            {"assumption": "retrain_frequency", "value": "annual"},
            {"assumption": "minimum_training_days", "value": 756},
            {"assumption": "edge_filter", "value": "expanding lagged prediction-dispersion quantile"},
            {"assumption": "edge_quantile", "value": float(edge_quantile)},
            {"assumption": "position_sizing", "value": "lagged 60-day volatility risk-adjusted cross-sectional ranking"},
            {"assumption": "external_data", "value": "none in final lower-overfit strategy"},
            {"assumption": "selection_policy", "value": "fixed algorithm; no full-sample weight/grid search"},
            {"assumption": "remaining_risk", "value": "Ridge coefficients are still fitted on past data; this reduces but cannot eliminate overfit"},
        ]
    )
    return {
        "assumptions": assumptions,
        "metrics": metrics,
        "predictions": predictions,
        "coefficients": coefficients,
        "positions": positions,
        "backtest": bt,
        "pnl_by_asset": pnl_by_asset,
        "edge_score": edge_score,
        "edge_threshold": edge_threshold,
    }


def no_fit_reversion_physical_signal(feature_panels, physical_weight=0.10):
    """Build a no-fit Cargill-aware short-term reversion signal.

    There are no learned coefficients. The base signal is the existing 5-day
    reversal feature. The physical tilt is deliberately small and fixed:
    public inventory pressure, receipts pressure, Cargill inventory pressure,
    and soybean-specific Cargill crush demand pressure.
    """
    reversion = _feature_frame(feature_panels, "rev_5")
    inventory_pressure = (
        -_feature_frame(feature_panels, "public_inventory_change")
        - _feature_frame(feature_panels, "receipts_change")
        - _feature_frame(feature_panels, "cgl_inventory_change")
    ) / 3.0
    crush_pressure = pd.DataFrame(0.0, index=reversion.index, columns=reversion.columns)
    crush_pressure["SOYABEAN"] = (
        _feature_frame(feature_panels, "crush_surprise")["SOYABEAN"]
        + _feature_frame(feature_panels, "crush_utilization")["SOYABEAN"]
    ) / 2.0
    physical_pressure = (inventory_pressure + crush_pressure) / 2.0
    return (reversion + float(physical_weight) * physical_pressure).clip(lower=-5.0, upper=5.0)


def volatility_target_positions(positions, futures_pnl, target_daily_pnl_vol=120.0, max_scale=1.0, lookback=60):
    """Scale positions by lagged realized strategy PnL volatility."""
    base_bt, _ = backtest_positions(positions, futures_pnl, 0.0)
    realized = base_bt["net_pnl"].rolling(int(lookback), min_periods=20).std().shift(1)
    scale = (float(target_daily_pnl_vol) / realized.replace(0.0, np.nan)).clip(upper=float(max_scale)).fillna(0.0)
    return positions.mul(scale, axis=0), scale


def run_no_fit_reversion_strategy(
    data_dir="train_set",
    split_date="2018-01-01",
    physical_weight=0.10,
    target_daily_pnl_vol=120.0,
    max_scale=1.0,
    trade_cost_per_lot=8.75,
    holding_cost_rate=0.05,
):
    """Run a no-regression, Cargill-aware reversion strategy.

    This candidate removes fitted Ridge coefficients entirely. It can still be
    overfit through research choices, but the live algorithm has no learned
    beta/weight vector.
    """
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    signal = no_fit_reversion_physical_signal(feature_panels, physical_weight=physical_weight)
    raw_positions = model_predictions_to_positions(signal, futures_pnl)
    positions, vol_scale = volatility_target_positions(
        raw_positions,
        futures_pnl,
        target_daily_pnl_vol=target_daily_pnl_vol,
        max_scale=max_scale,
    )
    zero_bt, zero_pnl_by_asset = backtest_positions(positions, futures_pnl, 0.0)
    cost_bt, cost_pnl_by_asset = backtest_positions_with_costs(
        positions,
        futures_pnl,
        trade_cost_per_lot=trade_cost_per_lot,
        holding_cost_rate=holding_cost_rate,
    )
    assumptions = pd.DataFrame(
        [
            {"assumption": "model", "value": "No-fit short-term reversion + fixed physical pressure tilt"},
            {"assumption": "base_signal", "value": "rev_5"},
            {
                "assumption": "physical_tilt",
                "value": "0.10 * average(public inventory pressure, receipts pressure, Cargill inventory pressure, soybean Cargill crush pressure)",
            },
            {"assumption": "uses_cgl_inv", "value": True},
            {"assumption": "uses_cgl_crush", "value": "soybean-specific fixed demand-pressure tilt"},
            {"assumption": "learned_coefficients", "value": "none"},
            {"assumption": "physical_weight", "value": float(physical_weight)},
            {"assumption": "target_daily_pnl_vol", "value": float(target_daily_pnl_vol)},
            {"assumption": "max_scale", "value": float(max_scale)},
            {"assumption": "vol_target_lookback", "value": 60},
            {"assumption": "trade_cost_per_lot", "value": float(trade_cost_per_lot)},
            {"assumption": "holding_cost_rate_annual", "value": float(holding_cost_rate)},
            {
                "assumption": "remaining_risk",
                "value": "No fitted Ridge coefficients, but research choices still require untouched holdout validation.",
            },
        ]
    )
    return {
        "assumptions": assumptions,
        "zero_cost_metrics": split_performance(zero_bt, split_date),
        "cost_adjusted_metrics": split_performance(cost_bt, split_date),
        "signal": signal,
        "positions": positions,
        "raw_positions": raw_positions,
        "vol_scale": vol_scale,
        "zero_cost_backtest": zero_bt,
        "zero_cost_pnl_by_asset": zero_pnl_by_asset,
        "cost_adjusted_backtest": cost_bt,
        "cost_adjusted_pnl_by_asset": cost_pnl_by_asset,
    }


def _strategy_cost_summary_row(name, positions, futures_pnl, split_date, cost_case):
    bt, _ = backtest_positions_with_costs(
        positions,
        futures_pnl,
        trade_cost_per_lot=cost_case["trade_cost_per_lot"],
        holding_cost_rate=cost_case["holding_cost_rate"],
        margin_budget=cost_case["margin_budget"],
    )
    metrics = split_performance(bt, split_date)
    active_bt = bt.loc[bt["held_gross_exposure"] > 1.0e-12]
    gross_pnl = float(active_bt["gross_pnl"].sum()) if len(active_bt) else 0.0
    trade_cost = float(active_bt["trade_cost"].sum()) if len(active_bt) else 0.0
    holding_cost = float(active_bt["holding_cost"].sum()) if len(active_bt) else 0.0
    total_cost = trade_cost + holding_cost
    total_turnover = float(active_bt["turnover"].sum()) if len(active_bt) else 0.0
    total_margin_days = float(active_bt["margin_used"].sum()) if len(active_bt) else 0.0
    expected_trade_cost = total_turnover * float(cost_case["trade_cost_per_lot"])
    expected_holding_cost = total_margin_days * float(cost_case["holding_cost_rate"]) / 252.0
    return {
        "strategy": name,
        "case": cost_case["case"],
        "trade_cost_per_lot": cost_case["trade_cost_per_lot"],
        "holding_cost_rate": cost_case["holding_cost_rate"],
        "margin_budget": cost_case["margin_budget"],
        "is_sharpe": metrics.loc["sharpe", "in_sample"],
        "oos_sharpe": metrics.loc["sharpe", "out_of_sample"],
        "oos_pnl": metrics.loc["total_pnl", "out_of_sample"],
        "full_sharpe": metrics.loc["sharpe", "full_period"],
        "full_pnl": metrics.loc["total_pnl", "full_period"],
        "full_hit_rate": metrics.loc["hit_rate", "full_period"],
        "max_dd": metrics.loc["max_drawdown", "full_period"],
        "turnover": metrics.loc["avg_daily_turnover", "full_period"],
        "total_turnover_lots": total_turnover,
        "margin_dollar_days": total_margin_days,
        "expected_trade_cost": expected_trade_cost,
        "expected_holding_cost": expected_holding_cost,
        "trade_cost": trade_cost,
        "holding_cost": holding_cost,
        "total_cost": total_cost,
        "CDF": np.nan if abs(gross_pnl) < 1.0e-12 else total_cost / abs(gross_pnl),
    }


def run_observable_regime_weight_experiment(
    data_dir="train_set",
    split_date="2018-01-01",
    cost_case=None,
):
    """Compare fixed and observable three-sleeve regime blends."""
    if cost_case is None:
        cost_case = COST_CASES[1]
    results = run_research_pipeline(data_dir=data_dir, split_date=split_date)
    pnl = results["futures_pnl"]
    base_positions = results["model_positions"]
    skip_positions = skip_rebalance_positions(base_positions, 2)
    multi_positions, _ = multi_condition_filter_positions(
        results["predictions"],
        pnl,
        results["feature_panels"],
        FINAL_OPPORTUNITY_QUANTILES["prediction"],
        FINAL_OPPORTUNITY_QUANTILES["curve"],
        FINAL_OPPORTUNITY_QUANTILES["momentum"],
    )
    fixed_two_sleeve = 0.50 * skip_positions + 0.50 * multi_positions
    observable_positions, weights, diagnostics = observable_three_sleeve_positions(
        results["predictions"], pnl, results["feature_panels"], base_positions, apply_vol_scale=False
    )
    scaled_positions, scaled_weights, scaled_diagnostics = observable_three_sleeve_positions(
        results["predictions"], pnl, results["feature_panels"], base_positions, apply_vol_scale=True
    )
    strategies = [
        ("Static edge-filtered", base_positions),
        ("2-day skip-rebalance", skip_positions),
        ("Multi-condition filter", multi_positions),
        ("Fixed 50/50 skip + multi", fixed_two_sleeve),
        ("Observable three-sleeve blend", observable_positions),
        ("Observable three-sleeve blend + vol scale", scaled_positions),
    ]
    table = pd.DataFrame(
        [_strategy_cost_summary_row(name, positions, pnl, split_date, cost_case) for name, positions in strategies]
    )
    weight_summary = pd.DataFrame(
        [
            {
                "strategy": "Observable three-sleeve blend",
                "avg_static_weight": float(weights["static_edge_filtered"].mean()),
                "avg_skip_weight": float(weights["skip_rebalance_2d"].mean()),
                "avg_multi_condition_weight": float(weights["multi_condition"].mean()),
                "avg_vol_scale": 1.0,
            },
            {
                "strategy": "Observable three-sleeve blend + vol scale",
                "avg_static_weight": float(scaled_weights["static_edge_filtered"].mean()),
                "avg_skip_weight": float(scaled_weights["skip_rebalance_2d"].mean()),
                "avg_multi_condition_weight": float(scaled_weights["multi_condition"].mean()),
                "avg_vol_scale": float(scaled_diagnostics["vol_scale"].mean()),
            },
        ]
    )
    assumptions = pd.DataFrame(
        [
            {"assumption": "sleeves", "value": "static edge-filtered, 2-day skip-rebalance, multi-condition filter"},
            {"assumption": "regime_inputs", "value": "lagged realized grain volatility and lagged opportunity score"},
            {"assumption": "no_named_regimes", "value": "drought/COVID/trade-war labels are diagnostics only"},
            {"assumption": "vol_low_rule", "value": "ewm_vol / expanding_long_vol < 0.70"},
            {"assumption": "vol_high_rule", "value": "ewm_vol / expanding_long_vol > 1.30"},
            {"assumption": "crisis_cap_rule", "value": "if vol_ratio > 2.00, vol-scaled variant applies 0.50 cap"},
            {"assumption": "cost_case", "value": cost_case["case"]},
            {"assumption": "trade_cost_per_lot", "value": cost_case["trade_cost_per_lot"]},
            {"assumption": "holding_cost_rate", "value": cost_case["holding_cost_rate"]},
            {
                "assumption": "cost_interpretation",
                "value": "Same unit cost assumptions for every row; realized total cost differs because turnover and margin use differ.",
            },
        ]
    )
    return {
        "assumptions": assumptions,
        "regime_weight_table": table,
        "weight_summary": weight_summary,
        "weights": weights,
        "diagnostics": diagnostics,
    }


### Direct Code: Generic Family / Regime Model Comparison

This cell defines the generic tests: average all signals, equal family weights, IC family selection, trend/MR regimes, OLS/Kalman-style benchmark, and Ridge.


In [ ]:
"""Family/regime/model comparison across grain products.

Requested strategy menu:
1. average all signals;
2. equal weight by signal family;
3. IC-threshold family filter, then validation-IC selected family;
4. trend/MR regime, select best family per regime and combine with fixed masks;
5. trend/MR regime, average families per regime and combine with fixed masks;
6. online OLS with Kalman filter;
7. expanding Ridge.

All variants use only the provided train_set feature panels. External data is
not fetched here so the comparison stays portable and audit-friendly.
"""


import os

import numpy as np
import pandas as pd



TRAIN_END = pd.Timestamp("2016-01-01")
TEST_START = pd.Timestamp("2018-01-01")
IC_THRESHOLD = 0.015
RIDGE_ALPHA = 100.0
KALMAN_Q = 1.0e-5
MIN_TRAIN_OBS = 504
REFIT_EVERY = 21


def _tanh(series, divisor=2.0):
    return pd.Series(np.tanh(series.astype(float) / float(divisor)), index=series.index)


def _clean(series, index):
    return series.reindex(index).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-5.0, 5.0)


def _signal_universe(panel):
    return {
        "price_mom_20": panel["mom_20"],
        "price_mom_60": panel["mom_60"],
        "price_rev_5": panel["rev_5"],
        "curve_spread": panel["curve_spread"],
        "curve_ratio": panel["curve_ratio"],
        "curve_change_20": panel["curve_change_20"],
        "cot_mm_level": panel["cot_mm_level"],
        "cot_mm_change": panel["cot_mm_change"],
        "cot_pm_oi_level": panel["cot_pm_oi_level"],
        "cot_pm_oi_change": panel["cot_pm_oi_change"],
        "public_inventory_pressure": -panel["public_inventory_change"],
        "receipts_pressure": -panel["receipts_change"],
        "cgl_inventory_pressure": -panel["cgl_inventory_change"],
        "crush_surprise": panel["crush_surprise"],
        "crush_utilization": panel["crush_utilization"],
    }


def _family_definitions():
    return {
        "price": ["price_mom_20", "price_mom_60", "price_rev_5"],
        "curve": ["curve_spread", "curve_ratio", "curve_change_20"],
        "cot": ["cot_mm_level", "cot_mm_change", "cot_pm_oi_level", "cot_pm_oi_change"],
        "physical_public": ["public_inventory_pressure", "receipts_pressure"],
        "physical_cargill": ["cgl_inventory_pressure", "crush_surprise", "crush_utilization"],
    }


def _build_signals(feature_panels, commodity):
    index = feature_panels[commodity].index
    signals = {name: _clean(series, index) for name, series in _signal_universe(feature_panels[commodity]).items()}
    families = {}
    members = {}
    for family, names in _family_definitions().items():
        available = [signals[name] for name in names if name in signals]
        if available:
            families[family] = sum(available) / float(len(available))
            members[family] = names
    return signals, families, members


def _ic(signal, futures_pnl, commodity, mask):
    aligned = pd.concat([signal.reindex(futures_pnl.index), futures_pnl[commodity].shift(-1)], axis=1).dropna()
    if aligned.empty:
        return np.nan
    mask = pd.Series(mask, index=futures_pnl.index).reindex(aligned.index).fillna(False).astype(bool)
    aligned = aligned.loc[mask]
    if len(aligned) < 40 or aligned.iloc[:, 0].std() == 0.0 or aligned.iloc[:, 1].std() == 0.0:
        return np.nan
    return aligned.iloc[:, 0].corr(aligned.iloc[:, 1], method="spearman")


def _split_masks(index):
    return {
        "train": index < TRAIN_END,
        "validation": (index >= TRAIN_END) & (index < TEST_START),
        "test": index >= TEST_START,
    }


def _family_ic_table(families, futures_pnl, commodity, regime_mask=None):
    masks = _split_masks(futures_pnl.index)
    if regime_mask is None:
        regime_mask = pd.Series(True, index=futures_pnl.index)
    regime_mask = pd.Series(regime_mask, index=futures_pnl.index).fillna(False).astype(bool)
    rows = []
    for name, signal in families.items():
        row = {"family": name}
        for split_name, mask in masks.items():
            combined_mask = pd.Series(mask, index=futures_pnl.index).astype(bool) & regime_mask
            row[split_name + "_obs"] = int(combined_mask.sum())
            row[split_name + "_ic"] = _ic(signal, futures_pnl, commodity, combined_mask)
        row["passes_train_ic"] = bool(pd.notnull(row["train_ic"]) and abs(row["train_ic"]) >= IC_THRESHOLD)
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["passes_train_ic", "validation_ic"], ascending=[False, False])


def _orient(signal, train_ic):
    if pd.isnull(train_ic):
        return signal
    return signal if train_ic >= 0.0 else -signal


def _regime_masks(feature_panels, futures_pnl, commodity):
    panel = feature_panels[commodity].reindex(futures_pnl.index).fillna(0.0)
    trend_strength = panel["mom_60"].abs()
    threshold = trend_strength.expanding(min_periods=252).median().shift(1)
    trend = (trend_strength > threshold).fillna(False)
    return {
        "trend": trend.astype(bool),
        "mr_or_chop": (~trend).astype(bool),
    }


def _positions_from_signal(signal, futures_pnl, commodity, mode="long_short", target_daily_pnl_vol=60.0, max_lot=0.50):
    clean = _tanh(signal.reindex(futures_pnl.index).fillna(0.0)).ewm(halflife=2.0, adjust=False, min_periods=1).mean()
    clean[clean.abs() < 0.05] = 0.0
    if mode == "long_only":
        clean = clean.clip(lower=0.0)
    elif mode != "long_short":
        raise ValueError("Unknown mode: {}".format(mode))
    asset_vol = futures_pnl[commodity].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    pos = clean * (float(target_daily_pnl_vol) / asset_vol)
    out = pd.DataFrame(0.0, index=futures_pnl.index, columns=[commodity])
    out[commodity] = pos.clip(-float(max_lot), float(max_lot)).fillna(0.0)
    return out


def _metrics(bt):
    table = split_performance(bt, TEST_START)
    oos = bt.loc[bt.index >= TEST_START]
    active = oos.loc[oos["held_gross_exposure"] > 1.0e-12, "net_pnl"].dropna()
    win_days = int((active > 0.0).sum()) if len(active) else 0
    return {
        "is_sharpe": table.loc["sharpe", "in_sample"],
        "oos_sharpe": table.loc["sharpe", "out_of_sample"],
        "oos_pnl": table.loc["total_pnl", "out_of_sample"],
        "oos_dd": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "full_dd": table.loc["max_drawdown", "full_period"],
        "hit_rate": table.loc["hit_rate", "out_of_sample"],
        "win_days": win_days,
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
    }


def _evaluate(strategy, commodity, signal, futures_pnl, mode="long_short"):
    positions = _positions_from_signal(signal, futures_pnl, commodity, mode=mode)
    rows = []
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[commodity]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[commodity]], 0.0)
        row = {"commodity": commodity, "strategy": strategy, "mode": mode, "cost_adjusted": cost_adjusted}
        row.update(_metrics(bt))
        rows.append(row)
    return rows


def _standardized_prediction_to_signal(pred):
    mean = pred.rolling(252, min_periods=60).mean().shift(1)
    std = pred.rolling(252, min_periods=60).std().shift(1).replace(0.0, np.nan)
    return ((pred - mean) / std).clip(-5.0, 5.0).fillna(0.0)


def _fit_ridge_beta(x_train, y_train, alpha):
    x = np.asarray(x_train, dtype=float)
    y = np.asarray(y_train, dtype=float)
    x = np.column_stack([np.ones(len(x)), x])
    xtx = x.T.dot(x)
    penalty = np.eye(xtx.shape[0]) * float(alpha)
    penalty[0, 0] = 0.0
    try:
        return np.linalg.solve(xtx + penalty, x.T.dot(y))
    except np.linalg.LinAlgError:
        return np.linalg.pinv(xtx + penalty).dot(x.T).dot(y)


def _expanding_ridge_signal(signals, futures_pnl, commodity):
    x = pd.DataFrame(signals).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y = futures_pnl[commodity].shift(-1)
    pred = pd.Series(np.nan, index=x.index)
    beta = None
    last_fit = None
    for i, date in enumerate(x.index):
        train_mask = (x.index < date) & y.notna()
        if int(train_mask.sum()) < MIN_TRAIN_OBS:
            continue
        x_train_raw = x.loc[train_mask]
        mean = x_train_raw.mean()
        std = x_train_raw.std().replace(0.0, np.nan).fillna(1.0)
        if beta is None or last_fit is None or (i - last_fit) >= REFIT_EVERY:
            beta = _fit_ridge_beta((x_train_raw - mean) / std, y.loc[train_mask], RIDGE_ALPHA)
            last_fit = i
        x_row = (x.loc[date] - mean) / std
        pred.loc[date] = np.r_[1.0, np.asarray(x_row, dtype=float)].dot(beta)
    return _standardized_prediction_to_signal(pred)


def _kalman_ols_signal(signals, futures_pnl, commodity):
    x = pd.DataFrame(signals).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y = futures_pnl[commodity].shift(-1)
    beta = np.zeros(x.shape[1] + 1)
    p = np.eye(len(beta)) * 10.0
    pred = pd.Series(np.nan, index=x.index)
    mean = pd.Series(0.0, index=x.columns)
    var = pd.Series(1.0, index=x.columns)
    target_var = 1.0
    n = 0
    for date in x.index:
        row = x.loc[date]
        if n > MIN_TRAIN_OBS:
            std = np.sqrt(var.clip(lower=1.0e-8))
            z = ((row - mean) / std).clip(-5.0, 5.0)
            phi = np.r_[1.0, np.asarray(z, dtype=float)]
            pred.loc[date] = phi.dot(beta)
        y_value = y.loc[date]
        if pd.notnull(y_value):
            n += 1
            old_mean = mean.copy()
            mean = mean + (row - mean) / float(n)
            var = ((n - 2.0) / max(n - 1.0, 1.0)) * var + ((row - old_mean) * (row - mean)) / max(n - 1.0, 1.0)
            target_var = target_var + (float(y_value) ** 2 - target_var) / float(n)
            if n > MIN_TRAIN_OBS:
                std = np.sqrt(var.clip(lower=1.0e-8))
                z = ((row - mean) / std).clip(-5.0, 5.0)
                phi = np.r_[1.0, np.asarray(z, dtype=float)]
                p = p + np.eye(len(beta)) * KALMAN_Q
                r = max(target_var, 1.0)
                innovation_var = float(phi.dot(p).dot(phi) + r)
                gain = p.dot(phi) / innovation_var
                err = float(y_value - phi.dot(beta))
                beta = beta + gain * err
                p = p - np.outer(gain, phi).dot(p)
    return _standardized_prediction_to_signal(pred)


def _strategy_signals(feature_panels, futures_pnl, commodity):
    signals, families, family_members = _build_signals(feature_panels, commodity)

    all_avg = sum(signals.values()) / float(len(signals))
    family_avg = sum(families.values()) / float(len(families))

    family_ic = _family_ic_table(families, futures_pnl, commodity)
    passed = family_ic.loc[family_ic["passes_train_ic"]].copy()
    if passed.empty:
        ic_selected_name = family_ic.sort_values("validation_ic", ascending=False).iloc[0]["family"]
        ic_selected_signal = families[ic_selected_name]
    else:
        passed["oriented_validation_ic"] = passed["validation_ic"].abs()
        best = passed.sort_values(["oriented_validation_ic", "train_ic"], ascending=[False, False]).iloc[0]
        ic_selected_name = best["family"]
        ic_selected_signal = _orient(families[ic_selected_name], best["train_ic"])

    regimes = _regime_masks(feature_panels, futures_pnl, commodity)
    regime_selected_pieces = []
    regime_avg_pieces = []
    regime_selection_rows = []
    for regime_name, regime_mask in regimes.items():
        table = _family_ic_table(families, futures_pnl, commodity, regime_mask=regime_mask)
        passed_regime = table.loc[table["passes_train_ic"]].copy()
        if passed_regime.empty:
            best = table.sort_values("validation_ic", ascending=False).iloc[0]
            selected_family = best["family"]
            selected_signal = families[selected_family]
            avg_signal = family_avg
        else:
            passed_regime["abs_validation_ic"] = passed_regime["validation_ic"].abs()
            best = passed_regime.sort_values(["abs_validation_ic", "train_ic"], ascending=[False, False]).iloc[0]
            selected_family = best["family"]
            selected_signal = _orient(families[selected_family], best["train_ic"])
            oriented = [_orient(families[row["family"]], row["train_ic"]) for _, row in passed_regime.iterrows()]
            avg_signal = sum(oriented) / float(len(oriented))
        mask = pd.Series(regime_mask, index=futures_pnl.index).astype(float)
        regime_selected_pieces.append(selected_signal * mask)
        regime_avg_pieces.append(avg_signal * mask)
        item = best.to_dict()
        item["regime"] = regime_name
        item["selected_family"] = selected_family
        regime_selection_rows.append(item)

    regime_selected = sum(regime_selected_pieces).fillna(0.0)
    regime_avg = sum(regime_avg_pieces).fillna(0.0)

    ridge_signal = _expanding_ridge_signal(signals, futures_pnl, commodity)
    kalman_signal = _kalman_ols_signal(signals, futures_pnl, commodity)

    return {
        "signals": {
            "avg_all_signals": all_avg,
            "equal_family_weight": family_avg,
            "ic_family_selected_" + ic_selected_name: ic_selected_signal,
            "regime_best_family_trend_mr": regime_selected,
            "regime_avg_families_trend_mr": regime_avg,
            "ols_kalman_filter": kalman_signal,
            "ridge_expanding_alpha100": ridge_signal,
        },
        "families": families,
        "family_members": family_members,
        "family_ic": family_ic,
        "regime_selection": pd.DataFrame(regime_selection_rows),
    }


STRATEGY_META = {
    "avg_all_signals": {
        "explain": "Simple average of every provided economic signal.",
        "formula": "mean(price, curve, COT, public physical, Cargill inventory, Cargill crush signals)",
        "rationale": "Broad diversification; assumes weak signals are safer in a simple basket.",
        "overfit": "Low model overfit, but can be economically diluted.",
    },
    "equal_family_weight": {
        "explain": "Average within each family, then equal weight each family.",
        "formula": "mean(price_family, curve_family, cot_family, public_physical_family, cargill_physical_family)",
        "rationale": "Prevents a family with many columns from dominating the signal.",
        "overfit": "Low model overfit; family choices are researcher-defined.",
    },
    "ic_family_selected": {
        "explain": "Keep families with train IC above threshold, select highest validation IC family.",
        "formula": "selected_family_signal, oriented by train IC sign",
        "rationale": "Use only families with evidence of predictive rank correlation.",
        "overfit": "Moderate/high selection risk; validation IC can be noisy.",
    },
    "regime_best_family_trend_mr": {
        "explain": "Trend/MR observable regime; pick best validation-IC family separately per regime.",
        "formula": "1(trend)*best_trend_family + 1(MR/chop)*best_mr_family",
        "rationale": "Signals may work in different states; wheat/corn can trend while soy can revert.",
        "overfit": "Moderate selection risk; weights are fixed masks, not optimized.",
    },
    "regime_avg_families_trend_mr": {
        "explain": "Trend/MR observable regime; average all IC-passing families in each regime.",
        "formula": "1(trend)*avg(IC-passing trend families) + 1(MR/chop)*avg(IC-passing MR families)",
        "rationale": "Keeps regime conditioning but avoids picking a single family winner.",
        "overfit": "Lower than best-family regime selection, but still regime-research risk.",
    },
    "ols_kalman_filter": {
        "explain": "Dynamic linear OLS coefficients updated with a Kalman filter.",
        "formula": "y_t = x_t beta_t + error; beta_t follows a random walk",
        "rationale": "Allows relationships to drift through time without a full static fit.",
        "overfit": "Coefficient-estimation risk; parameters are fixed, not OOS tuned.",
    },
    "ridge_expanding_alpha100": {
        "explain": "Expanding-window Ridge with fixed alpha 100.",
        "formula": "argmin ||y-Xb||^2 + 100||b||^2, refit every 21 trading days",
        "rationale": "Regularized linear combination of all signals.",
        "overfit": "Coefficient-estimation risk; fixed alpha reduces but does not remove it.",
    },
}


def _base_strategy_name(name):
    if name.startswith("ic_family_selected_"):
        return "ic_family_selected"
    return name


def _write_log(results, diagnostics, path="notes/family_regime_model_comparison.txt"):
    lines = []
    lines.append("Family/regime/model strategy comparison")
    lines.append("Date: 2026-05-02")
    lines.append("")
    lines.append("Universe")
    lines.append("--------")
    lines.append("Products: CORN, SOYABEAN, WHEAT_SRW, WHEAT_HRW.")
    lines.append("Signals: price momentum/reversion, curve, COT, public physical, Cargill inventory, Cargill crush.")
    lines.append("No external data was fetched in this experiment.")
    lines.append("")
    lines.append("Strategy Definitions")
    lines.append("--------------------")
    for key, meta in STRATEGY_META.items():
        lines.append("- {}: {}".format(key, meta["explain"]))
        lines.append("  Formula: {}".format(meta["formula"]))
        lines.append("  Economic rationale: {}".format(meta["rationale"]))
        lines.append("  Overfit note: {}".format(meta["overfit"]))
    lines.append("")
    lines.append("Results Summary")
    lines.append("---------------")
    cols = [
        "commodity",
        "strategy",
        "mode",
        "cost_adjusted",
        "is_sharpe",
        "oos_sharpe",
        "oos_pnl",
        "oos_dd",
        "full_sharpe",
        "full_dd",
        "hit_rate",
        "win_days",
        "turnover",
        "avg_gross_exposure",
        "overfit_comment",
    ]
    lines.append(results[cols].to_string(index=False, float_format=lambda value: "{:.3f}".format(value)))
    lines.append("")
    for commodity in COMMODITIES:
        lines.append("{} diagnostics".format(commodity))
        lines.append("-" * (len(commodity) + 12))
        lines.append("Family IC table:")
        lines.append(diagnostics[commodity]["family_ic"].to_string(index=False, float_format=lambda value: "{:.3f}".format(value)))
        lines.append("")
        lines.append("Regime family selections:")
        lines.append(diagnostics[commodity]["regime_selection"].to_string(index=False, float_format=lambda value: "{:.3f}".format(value)))
        lines.append("")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as handle:
        handle.write("\n".join(lines))


def run_family_regime_model_comparison(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    rows = []
    diagnostics = {}
    for commodity in COMMODITIES:
        bundle = _strategy_signals(feature_panels, futures_pnl, commodity)
        diagnostics[commodity] = {
            "family_ic": bundle["family_ic"],
            "regime_selection": bundle["regime_selection"],
        }
        for strategy, signal in bundle["signals"].items():
            mode = "long_short"
            rows.extend(_evaluate(strategy, commodity, signal, futures_pnl, mode=mode))
    results = pd.DataFrame(rows)
    comments = []
    for _, row in results.iterrows():
        base = _base_strategy_name(row["strategy"])
        comment = STRATEGY_META[base]["overfit"]
        if row["cost_adjusted"] and row["oos_sharpe"] < 0.0:
            comment = comment + " Reject: negative cost-adjusted OOS."
        elif row["cost_adjusted"] and row["oos_sharpe"] > 1.0 and base in ["avg_all_signals", "equal_family_weight", "regime_avg_families_trend_mr"]:
            comment = comment + " Strong and comparatively simple."
        elif row["cost_adjusted"] and base in ["ols_kalman_filter", "ridge_expanding_alpha100"] and row["oos_sharpe"] > 0.0:
            comment = comment + " Positive OOS but keep as diagnostic unless validation/full-period behavior is clean."
        comments.append(comment)
    results["overfit_comment"] = comments
    results = results.sort_values(["commodity", "cost_adjusted", "oos_sharpe"], ascending=[True, True, False]).reset_index(drop=True)
    _write_log(results, diagnostics)
    return {
        "results": results,
        "diagnostics": diagnostics,
    }


# 2. Shared Generic Tests


In [ ]:
family_audit = safe_run("Family/regime/model comparison", run_family_regime_model_comparison, data_dir=DATA_DIR)
family_results = cost_only(family_audit["results"] if family_audit is not None else pd.DataFrame())
print("family_results rows:", len(family_results))


### Direct Code: OLS / Kalman / Ridge Online Model Comparison

The next cells define only the dependencies and implementation needed for the Corn/Soybean fitted-model benchmark, immediately before it is called.


In [ ]:
"""Optional Meteostat crop-belt weather overlay experiment.

Meteostat is open data and does not require an API key. This module keeps the
weather test separate from the selected Cargill-aware strategy.
"""


from datetime import datetime

import pandas as pd



SPLIT_DATE = "2018-01-01"
OVERLAY_WEIGHTS = [0.10, 0.25, 0.50, 1.00]
METEOSTAT_LOCATIONS = {
    "iowa_corn_belt": (42.03, -93.63),
    "illinois_corn_belt": (40.63, -89.40),
    "nebraska_plains": (41.26, -96.02),
    "kansas_wheat": (38.35, -98.20),
}
COMMODITY_LOCATION_WEIGHTS = {
    "CORN": {
        "iowa_corn_belt": 0.45,
        "illinois_corn_belt": 0.35,
        "nebraska_plains": 0.20,
    },
    "SOYABEAN": {
        "iowa_corn_belt": 0.40,
        "illinois_corn_belt": 0.40,
        "nebraska_plains": 0.20,
    },
    "WHEAT_SRW": {
        "illinois_corn_belt": 0.70,
        "kansas_wheat": 0.30,
    },
    "WHEAT_HRW": {
        "kansas_wheat": 0.70,
        "nebraska_plains": 0.30,
    },
}


def fetch_meteostat_weather(start, end, locations=None):
    """Fetch daily Meteostat weather for a compact crop-belt location set.

    Compatible with both the legacy meteostat API (Daily class) and the new
    function-based API introduced in meteostat 2.x (daily function + stations).
    """
    import meteostat as _ms

    if locations is None:
        locations = METEOSTAT_LOCATIONS

    start_dt = pd.Timestamp(start).to_pydatetime()
    end_dt = pd.Timestamp(end).to_pydatetime()

    # Detect API version: new API exposes 'daily' as a function, not a class.
    _use_new_api = callable(getattr(_ms, "daily", None)) and not isinstance(
        getattr(_ms, "daily", None), type
    )

    rows = []
    for name, (lat, lon) in locations.items():
        point = _ms.Point(float(lat), float(lon))

        if _use_new_api:
            # New API: find nearest station, then fetch by station ID.
            nearby = _ms.stations.nearby(point)
            if nearby.empty:
                continue
            station_id = nearby.index[0]
            ts = _ms.daily(station_id, start_dt, end_dt)
            frame = ts.fetch()
            if frame is None or (hasattr(frame, "empty") and frame.empty):
                continue
            # New API column names: 'temp' instead of 'tavg'
            frame = frame.rename(columns={"temp": "tavg"})
            frame = frame.reset_index().rename(columns={"time": "date"})
        else:
            # Legacy API: Daily class accepts a Point directly.
            Daily = getattr(_ms, "Daily")
            frame = Daily(point, start_dt, end_dt).fetch()
            if frame.empty:
                continue
            frame = frame.reset_index().rename(columns={"time": "date"})

        frame["location"] = name
        rows.append(frame)

    if not rows:
        raise RuntimeError("Meteostat returned no weather rows.")

    out = pd.concat(rows, ignore_index=True)
    out["date"] = pd.to_datetime(out["date"]).dt.tz_localize(None)
    return out


def _weighted_weather_by_commodity(weather, value_cols, commodity):
    weights = COMMODITY_LOCATION_WEIGHTS.get(commodity, {})
    frames = []
    for location, weight in weights.items():
        sub = weather.loc[weather["location"] == location, ["date"] + value_cols].copy()
        if sub.empty:
            continue
        sub[value_cols] = sub[value_cols] * float(weight)
        frames.append(sub)
    if not frames:
        return weather.groupby("date")[value_cols].mean().sort_index()

    combined = pd.concat(frames, ignore_index=True)
    return combined.groupby("date")[value_cols].sum().sort_index()


def _season_mask(index, months):
    return pd.Series(index.month.isin(months), index=index).astype(float)


def _add_weather_features(aligned, prefix, seasonal=False):
    features = pd.DataFrame(index=aligned.index)
    if "temp_mean" in aligned.columns:
        features[prefix + "temp_mean_level"] = rolling_zscore(aligned["temp_mean"], 252, 60)
        features[prefix + "temp_mean_change_20"] = rolling_zscore(aligned["temp_mean"].diff(20), 252, 60)
        cdd = (aligned["temp_mean"] - 18.0).clip(lower=0.0)
        hdd = (18.0 - aligned["temp_mean"]).clip(lower=0.0)
        for window in [5, 20, 60]:
            min_periods = max(3, int(window / 4))
            features[prefix + "cdd_{}d".format(window)] = rolling_zscore(
                cdd.rolling(window, min_periods=min_periods).sum(),
                252,
                60,
            )
            features[prefix + "hdd_{}d".format(window)] = rolling_zscore(
                hdd.rolling(window, min_periods=min_periods).sum(),
                252,
                60,
            )
        features[prefix + "cdd_change_20"] = rolling_zscore(cdd.rolling(20, min_periods=5).sum().diff(20), 252, 60)
        features[prefix + "hdd_change_20"] = rolling_zscore(hdd.rolling(20, min_periods=5).sum().diff(20), 252, 60)
    if "temp_max" in aligned.columns:
        features[prefix + "heat_level"] = rolling_zscore(aligned["temp_max"], 252, 60)
        heat_stress = (aligned["temp_max"] - 32.0).clip(lower=0.0)
        features[prefix + "heat_stress_5d"] = rolling_zscore(heat_stress.rolling(5, min_periods=3).sum(), 252, 60)
        features[prefix + "heat_stress_20d"] = rolling_zscore(heat_stress.rolling(20, min_periods=5).sum(), 252, 60)
    if "temp_min" in aligned.columns:
        features[prefix + "cold_level"] = -rolling_zscore(aligned["temp_min"], 252, 60)
        freeze_stress = (0.0 - aligned["temp_min"]).clip(lower=0.0)
        features[prefix + "freeze_stress_5d"] = rolling_zscore(freeze_stress.rolling(5, min_periods=3).sum(), 252, 60)
        features[prefix + "freeze_stress_20d"] = rolling_zscore(freeze_stress.rolling(20, min_periods=5).sum(), 252, 60)
    if "temp_min" in aligned.columns and "temp_max" in aligned.columns:
        temp_avg = (aligned["temp_min"] + aligned["temp_max"]) / 2.0
        gdd = (temp_avg.clip(upper=30.0) - 10.0).clip(lower=0.0)
        features[prefix + "gdd_20d"] = rolling_zscore(gdd.rolling(20, min_periods=5).sum(), 252, 60)
        features[prefix + "gdd_60d"] = rolling_zscore(gdd.rolling(60, min_periods=15).sum(), 252, 60)
    if "precipitation" in aligned.columns:
        precip = aligned["precipitation"].fillna(0.0)
        for window in [5, 20, 60]:
            precip_sum = precip.rolling(window, min_periods=max(3, int(window / 4))).sum()
            features[prefix + "precip_{}d".format(window)] = rolling_zscore(precip_sum, 252, 60)
            features[prefix + "dryness_{}d".format(window)] = -features[prefix + "precip_{}d".format(window)]
        if prefix + "heat_stress_20d" in features.columns:
            features[prefix + "dry_heat_20d"] = (
                features[prefix + "dryness_20d"] * features[prefix + "heat_stress_20d"]
            ).clip(lower=-5.0, upper=5.0)
        if prefix + "cdd_20d" in features.columns:
            features[prefix + "dry_cdd_20d"] = (
                features[prefix + "dryness_20d"] * features[prefix + "cdd_20d"]
            ).clip(lower=-5.0, upper=5.0)
    if "wind_speed" in aligned.columns:
        features[prefix + "wind_level"] = rolling_zscore(aligned["wind_speed"], 252, 60)

    if seasonal:
        planting = _season_mask(features.index, [3, 4, 5])
        growing = _season_mask(features.index, [6, 7, 8])
        harvest = _season_mask(features.index, [9, 10, 11])
        seasonal_features = {}
        for column in features.columns:
            seasonal_features[column + "_planting"] = features[column] * planting
            seasonal_features[column + "_growing"] = features[column] * growing
            seasonal_features[column + "_harvest"] = features[column] * harvest
        features = pd.concat([features, pd.DataFrame(seasonal_features, index=features.index)], axis=1)

    return features.clip(lower=-5.0, upper=5.0)


def build_meteostat_feature_panels(weather, trading_index, mode="shared_basic"):
    """Build weather features from crop-belt daily weather."""
    rename = {
        "tavg": "temp_mean",
        "tmin": "temp_min",
        "tmax": "temp_max",
        "prcp": "precipitation",
        "wspd": "wind_speed",
    }
    weather = weather.rename(columns={k: v for k, v in rename.items() if k in weather.columns})
    value_cols = [
        col
        for col in ["temp_mean", "temp_min", "temp_max", "precipitation", "wind_speed"]
        if col in weather.columns
    ]
    if not value_cols:
        raise RuntimeError("Meteostat data has no recognized weather columns.")
    for column in value_cols:
        weather[column] = pd.to_numeric(weather[column], errors="coerce")

    if mode == "shared_basic":
        daily = weather.groupby("date")[value_cols].mean().sort_index()
        aligned = daily.reindex(trading_index).ffill().shift(1)
        features = _add_weather_features(aligned, "meteo_", seasonal=False)
        keep = [
            "meteo_temp_mean_level",
            "meteo_temp_mean_change_20",
            "meteo_heat_level",
            "meteo_cold_level",
            "meteo_precip_20d",
            "meteo_dryness_20d",
            "meteo_wind_level",
        ]
        features = features[[col for col in keep if col in features.columns]]
        return {commodity: features.copy() for commodity in COMMODITIES}

    panels = {}
    seasonal = mode == "commodity_seasonal"
    for commodity in COMMODITIES:
        daily = _weighted_weather_by_commodity(weather, value_cols, commodity)
        aligned = daily.reindex(trading_index).ffill().shift(1)
        panels[commodity] = _add_weather_features(aligned, "meteo_", seasonal=seasonal)
    return panels


def build_meteostat_predictions(weather_panels, futures_pnl, split_date=SPLIT_DATE, alpha=1000.0, horizon=5):
    train_mask = futures_pnl.index < pd.Timestamp(split_date)
    predictions = pd.DataFrame(index=futures_pnl.index, columns=COMMODITIES, dtype=float)
    coefficients = {}
    for commodity in COMMODITIES:
        horizon = int(horizon)
        target = futures_pnl[commodity].shift(-1).rolling(horizon, min_periods=horizon).sum().shift(-(horizon - 1))
        pred, coef = fit_ridge_predict(weather_panels[commodity], target, train_mask, alpha=alpha)
        predictions[commodity] = pred
        coefficients[commodity] = coef
    return predictions.fillna(0.0), pd.DataFrame(coefficients)


def _metrics_row(name, predictions, futures_pnl, edge_filter=False):
    if edge_filter:
        positions, _, _ = edge_filtered_positions(predictions, futures_pnl, quantile=0.50)
        name = name + " | edge-filtered"
    else:
        positions = model_predictions_to_positions(predictions, futures_pnl)
        name = name + " | unfiltered"
    bt, _ = backtest_positions(positions, futures_pnl, 0.0)
    metrics = split_performance(bt, SPLIT_DATE)
    return {
        "experiment": name,
        "is_sharpe": metrics.loc["sharpe", "in_sample"],
        "oos_sharpe": metrics.loc["sharpe", "out_of_sample"],
        "oos_pnl": metrics.loc["total_pnl", "out_of_sample"],
        "full_sharpe": metrics.loc["sharpe", "full_period"],
        "full_pnl": metrics.loc["total_pnl", "full_period"],
        "max_drawdown": metrics.loc["max_drawdown", "full_period"],
        "turnover": metrics.loc["avg_daily_turnover", "full_period"],
    }


def _prediction_correlation(a, b):
    common = a.reindex_like(b).fillna(0.0)
    base = b.fillna(0.0)
    values = []
    for column in common.columns:
        joined = pd.concat([common[column], base[column]], axis=1).dropna()
        if len(joined) > 20 and joined.iloc[:, 0].std() > 0.0 and joined.iloc[:, 1].std() > 0.0:
            values.append(joined.iloc[:, 0].corr(joined.iloc[:, 1]))
    if not values:
        return float("nan")
    return float(pd.Series(values).mean())


def run_meteostat_experiment():
    data = load_train_set("train_set")
    feature_panels, futures_pnl = build_feature_panels(data)
    selected_predictions, _, _, _ = build_improved_model_signals(feature_panels, futures_pnl, SPLIT_DATE)

    weather = fetch_meteostat_weather(futures_pnl.index.min(), futures_pnl.index.max())
    variants = [
        ("shared_basic", 1000.0, 5),
        ("shared_basic", 1000.0, 20),
        ("commodity_basic", 1000.0, 5),
        ("commodity_basic", 1000.0, 20),
        ("commodity_seasonal", 1000.0, 5),
        ("commodity_seasonal", 1000.0, 20),
        ("commodity_seasonal", 5000.0, 20),
        ("commodity_seasonal", 100.0, 20),
    ]

    rows = []
    rows.append(_metrics_row("Current selected core + physical", selected_predictions, futures_pnl, False))
    rows.append(_metrics_row("Current selected core + physical", selected_predictions, futures_pnl, True))
    coefficient_blocks = {}
    panel_blocks = {}
    prediction_blocks = {}
    for mode, alpha, horizon in variants:
        weather_panels = build_meteostat_feature_panels(weather, futures_pnl.index, mode=mode)
        weather_predictions, weather_coefficients = build_meteostat_predictions(
            weather_panels,
            futures_pnl,
            alpha=alpha,
            horizon=horizon,
        )
        variant_name = "Meteostat {} alpha={} horizon={}".format(mode, int(alpha), int(horizon))
        panel_blocks[variant_name] = weather_panels
        prediction_blocks[variant_name] = weather_predictions
        coefficient_blocks[variant_name] = weather_coefficients

        weather_only = _metrics_row(variant_name + " only", weather_predictions, futures_pnl, False)
        weather_only["weather_variant"] = variant_name
        weather_only["prediction_corr_to_selected"] = _prediction_correlation(weather_predictions, selected_predictions)
        rows.append(weather_only)

        weather_only_edge = _metrics_row(variant_name + " only", weather_predictions, futures_pnl, True)
        weather_only_edge["weather_variant"] = variant_name
        weather_only_edge["prediction_corr_to_selected"] = weather_only["prediction_corr_to_selected"]
        rows.append(weather_only_edge)

        for weight in OVERLAY_WEIGHTS:
            combined = selected_predictions.fillna(0.0) + float(weight) * weather_predictions.fillna(0.0)
            for edge_filter in [False, True]:
                row = _metrics_row(
                    "Current selected + {} overlay w={}".format(variant_name, weight),
                    combined,
                    futures_pnl,
                    edge_filter,
                )
                row["weather_variant"] = variant_name
                row["weather_weight"] = weight
                row["prediction_corr_to_selected"] = weather_only["prediction_corr_to_selected"]
                rows.append(row)

    results = pd.DataFrame(rows).sort_values(["oos_sharpe", "full_sharpe"], ascending=[False, False])
    return {
        "weather": weather,
        "weather_feature_panels": panel_blocks,
        "weather_predictions": prediction_blocks,
        "weather_coefficients": coefficient_blocks,
        "results": results.reset_index(drop=True),
    }


In [ ]:
"""Optional EIA ethanol production/stocks overlay experiment.

Requires an EIA API key via EIA_API_KEY/EIA_KEY or the natgas config file.
The key is never stored in this file or printed by the script.
"""


import os
import re

import pandas as pd
import requests



SPLIT_DATE = "2018-01-01"
OVERLAY_WEIGHTS = [0.25, 0.50, 1.00]
NATGAS_CONFIG_PATH = "/Users/phuongpham/natgas_trading/live/config.yaml"
EIA_SERIES = {
    "ethanol_production": "PET.W_EPOOXE_YOP_NUS_MBBLD.W",
    "ethanol_stocks": "PET.W_EPOOXE_SAE_NUS_MBBL.W",
}


def read_eia_api_key(config_path=NATGAS_CONFIG_PATH):
    key = os.environ.get("EIA_API_KEY") or os.environ.get("EIA_KEY")
    if key:
        return key.strip()
    if config_path and os.path.exists(config_path):
        text = open(config_path).read()
        match = re.search(r"eia_api_key:\s*[\"']?([^\"'\n]+)", text)
        if match:
            return match.group(1).strip()
    raise RuntimeError("Missing EIA API key. Set EIA_API_KEY or provide natgas config.")


def fetch_eia_series(series_id, api_key):
    url = "https://api.eia.gov/v2/seriesid/" + series_id
    response = requests.get(url, params={"api_key": api_key}, timeout=60)
    if response.status_code != 200:
        raise RuntimeError("EIA request failed for {}".format(series_id))
    payload = response.json()
    rows = ((payload.get("response") or {}).get("data")) or []
    if not rows:
        raise RuntimeError("EIA returned no rows for {}".format(series_id))
    frame = pd.DataFrame(rows)
    out = pd.DataFrame(
        {
            "date": pd.to_datetime(frame["period"], errors="coerce"),
            "value": pd.to_numeric(frame["value"], errors="coerce"),
        }
    ).dropna()
    return out.sort_values("date").drop_duplicates("date", keep="last").set_index("date")


def fetch_eia_ethanol(api_key=None):
    api_key = api_key or read_eia_api_key()
    frames = []
    for name, series_id in EIA_SERIES.items():
        frame = fetch_eia_series(series_id, api_key).rename(columns={"value": name})
        frames.append(frame)
    out = pd.concat(frames, axis=1).sort_index()
    return out


def build_ethanol_feature_panel(ethanol, trading_index):
    """Build lagged weekly ethanol features aligned to grain trading dates."""
    available = ethanol.copy()
    available.index = available.index + pd.DateOffset(days=7)
    aligned = available.reindex(trading_index).ffill().shift(1)

    features = pd.DataFrame(index=trading_index)
    features["ethanol_production_level"] = rolling_zscore(aligned["ethanol_production"], 156, 40)
    features["ethanol_production_change_4w"] = rolling_zscore(aligned["ethanol_production"].diff(20), 156, 40)
    features["ethanol_stocks_level"] = rolling_zscore(aligned["ethanol_stocks"], 156, 40)
    features["ethanol_stocks_change_4w"] = rolling_zscore(aligned["ethanol_stocks"].diff(20), 156, 40)
    ratio = aligned["ethanol_production"] / aligned["ethanol_stocks"].replace(0.0, pd.NA)
    features["ethanol_prod_to_stocks"] = rolling_zscore(ratio, 156, 40)
    pressure = aligned["ethanol_production"].diff(20) - aligned["ethanol_stocks"].diff(20)
    features["ethanol_demand_pressure"] = rolling_zscore(pressure, 156, 40)
    return features.clip(lower=-5.0, upper=5.0)


def build_ethanol_predictions(ethanol_features, futures_pnl, split_date=SPLIT_DATE, alpha=1000.0):
    """Predict only CORN from ethanol features; other commodities get zero overlay."""
    train_mask = futures_pnl.index < pd.Timestamp(split_date)
    predictions = pd.DataFrame(0.0, index=futures_pnl.index, columns=COMMODITIES, dtype=float)
    target = futures_pnl["CORN"].shift(-1).rolling(5, min_periods=5).sum().shift(-4)
    pred, coef = fit_ridge_predict(ethanol_features, target, train_mask, alpha=alpha)
    predictions["CORN"] = pred.fillna(0.0)
    return predictions, coef


def _metrics_row(name, predictions, futures_pnl, edge_filter=False):
    if edge_filter:
        positions, _, _ = edge_filtered_positions(predictions, futures_pnl, quantile=0.50)
        name = name + " | edge-filtered"
    else:
        positions = model_predictions_to_positions(predictions, futures_pnl)
        name = name + " | unfiltered"
    bt, _ = backtest_positions(positions, futures_pnl, 0.0)
    metrics = split_performance(bt, SPLIT_DATE)
    return {
        "experiment": name,
        "is_sharpe": metrics.loc["sharpe", "in_sample"],
        "oos_sharpe": metrics.loc["sharpe", "out_of_sample"],
        "oos_pnl": metrics.loc["total_pnl", "out_of_sample"],
        "full_sharpe": metrics.loc["sharpe", "full_period"],
        "full_pnl": metrics.loc["total_pnl", "full_period"],
        "max_drawdown": metrics.loc["max_drawdown", "full_period"],
        "turnover": metrics.loc["avg_daily_turnover", "full_period"],
    }


def run_eia_ethanol_experiment(api_key=None):
    data = load_train_set("train_set")
    feature_panels, futures_pnl = build_feature_panels(data)
    selected_predictions, _, _, _ = build_improved_model_signals(feature_panels, futures_pnl, SPLIT_DATE)

    ethanol = fetch_eia_ethanol(api_key=api_key)
    ethanol_features = build_ethanol_feature_panel(ethanol, futures_pnl.index)
    ethanol_predictions, ethanol_coefficients = build_ethanol_predictions(ethanol_features, futures_pnl)

    rows = []
    rows.append(_metrics_row("Current selected core + physical", selected_predictions, futures_pnl, False))
    rows.append(_metrics_row("Current selected core + physical", selected_predictions, futures_pnl, True))
    rows.append(_metrics_row("EIA ethanol only", ethanol_predictions, futures_pnl, False))
    rows.append(_metrics_row("EIA ethanol only", ethanol_predictions, futures_pnl, True))
    for weight in OVERLAY_WEIGHTS:
        combined = selected_predictions.fillna(0.0) + float(weight) * ethanol_predictions.fillna(0.0)
        rows.append(_metrics_row("Current selected + EIA ethanol overlay w=" + str(weight), combined, futures_pnl, False))
        rows.append(_metrics_row("Current selected + EIA ethanol overlay w=" + str(weight), combined, futures_pnl, True))

    results = pd.DataFrame(rows).sort_values(["oos_sharpe", "full_sharpe"], ascending=[False, False])
    return {
        "ethanol": ethanol,
        "ethanol_features": ethanol_features,
        "ethanol_predictions": ethanol_predictions,
        "ethanol_coefficients": ethanol_coefficients,
        "results": results.reset_index(drop=True),
    }


In [ ]:
"""Soybean-only lower-overfit strategy experiment.

The experiment uses fixed economic recipes and no regression coefficients.
Candidate selection is done on a pre-2018 validation slice, then 2018-2020 is
reported as the out-of-sample test.
"""


import numpy as np
import pandas as pd



COMMODITY = "SOYABEAN"
TRAIN_END = "2016-01-01"
VALIDATION_END = "2018-01-01"
TEST_START = "2018-01-01"


def _z_tanh(series, divisor=2.0):
    return np.tanh(series.astype(float) / float(divisor))


def _smooth(series, halflife=2):
    return series.ewm(halflife=float(halflife), adjust=False, min_periods=1).mean()


def _threshold(series, threshold=0.05):
    out = series.copy()
    out[out.abs() < float(threshold)] = 0.0
    return out


def _soybean_components(feature_panels):
    soy = feature_panels[COMMODITY]
    inventory_pressure = (
        -soy["public_inventory_change"] - soy["receipts_change"] - soy["cgl_inventory_change"]
    ) / 3.0
    crush_pressure = (soy["crush_surprise"] + soy["crush_utilization"]) / 2.0
    trend = (soy["mom_20"] + soy["mom_60"] + soy["curve_spread"] + soy["cot_pm_oi_level"]) / 4.0
    slow_mr = (-soy["mom_20"] - soy["mom_60"]) / 2.0
    curve_tightness = (soy["curve_spread"] + soy["curve_ratio"]) / 2.0
    components = {
        "rev_5": soy["rev_5"],
        "inventory_pressure": inventory_pressure,
        "crush_pressure": crush_pressure,
        "trend": trend,
        "slow_mr": slow_mr,
        "curve_tightness": curve_tightness,
        "cot_pressure": soy["cot_pm_oi_level"],
    }
    return {name: values.fillna(0.0) for name, values in components.items()}


def build_soybean_signal(feature_panels, recipe):
    c = _soybean_components(feature_panels)
    if recipe == "rev5":
        raw = c["rev_5"]
    elif recipe == "rev5_plus_physical_10":
        raw = c["rev_5"] + 0.10 * ((c["inventory_pressure"] + c["crush_pressure"]) / 2.0)
    elif recipe == "soy_physical_balanced":
        raw = 0.60 * c["rev_5"] + 0.20 * c["inventory_pressure"] + 0.20 * c["crush_pressure"]
    elif recipe == "soy_crush_inventory":
        raw = 0.50 * c["rev_5"] + 0.25 * c["inventory_pressure"] + 0.25 * c["crush_pressure"]
    elif recipe == "soy_slow_mr_physical":
        raw = 0.45 * c["slow_mr"] + 0.25 * c["rev_5"] + 0.15 * c["inventory_pressure"] + 0.15 * c["crush_pressure"]
    elif recipe == "soy_trend_physical":
        raw = 0.50 * c["trend"] + 0.25 * c["inventory_pressure"] + 0.25 * c["crush_pressure"]
    elif recipe == "soy_curve_crush":
        raw = 0.45 * c["curve_tightness"] + 0.35 * c["crush_pressure"] + 0.20 * c["rev_5"]
    elif recipe == "soy_defensive_blend":
        raw = 0.50 * c["rev_5"] + 0.20 * c["slow_mr"] + 0.15 * c["inventory_pressure"] + 0.15 * c["crush_pressure"]
    elif recipe == "soy_conservative_long_blend":
        trend_physical = 0.50 * c["trend"] + 0.25 * c["inventory_pressure"] + 0.25 * c["crush_pressure"]
        curve_crush = 0.45 * c["curve_tightness"] + 0.35 * c["crush_pressure"] + 0.20 * c["rev_5"]
        raw = 0.50 * trend_physical + 0.50 * curve_crush
    else:
        raise ValueError("Unknown soybean recipe: {}".format(recipe))
    return raw.clip(lower=-5.0, upper=5.0)


def signal_to_soybean_positions(signal, futures_pnl, target_daily_pnl_vol=75.0, max_lot=0.50, mode="long_short"):
    soy_pnl = futures_pnl[[COMMODITY]]
    raw = _threshold(_smooth(_z_tanh(signal), halflife=2), threshold=0.05)
    if mode == "long_only":
        raw = raw.clip(lower=0.0)
    elif mode == "short_only":
        raw = raw.clip(upper=0.0)
    elif mode != "long_short":
        raise ValueError("Unknown position mode: {}".format(mode))
    asset_vol = soy_pnl[COMMODITY].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    pos = raw.reindex(soy_pnl.index).fillna(0.0) * (float(target_daily_pnl_vol) / asset_vol)
    out = pd.DataFrame(0.0, index=soy_pnl.index, columns=[COMMODITY])
    out[COMMODITY] = pos.clip(lower=-float(max_lot), upper=float(max_lot)).fillna(0.0)
    return out


def _metric_columns(bt):
    by_split = split_performance(bt, TEST_START)
    train_val = split_performance(bt.loc[bt.index < TEST_START], TRAIN_END)
    return {
        "train_sharpe": train_val.loc["sharpe", "in_sample"],
        "validation_sharpe": train_val.loc["sharpe", "out_of_sample"],
        "validation_max_drawdown": train_val.loc["max_drawdown", "out_of_sample"],
        "test_sharpe": by_split.loc["sharpe", "out_of_sample"],
        "test_pnl": by_split.loc["total_pnl", "out_of_sample"],
        "test_max_drawdown": by_split.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": by_split.loc["sharpe", "full_period"],
        "full_pnl": by_split.loc["total_pnl", "full_period"],
        "max_drawdown": by_split.loc["max_drawdown", "full_period"],
        "turnover": by_split.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": by_split.loc["avg_gross_exposure", "full_period"],
    }


def _row(recipe, mode, positions, futures_pnl, cost_adjusted):
    if cost_adjusted:
        bt, _ = backtest_positions_with_costs(positions, futures_pnl[[COMMODITY]], 8.75, 0.05)
    else:
        bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
    row = {"recipe": recipe, "mode": mode, "cost_adjusted": bool(cost_adjusted)}
    row.update(_metric_columns(bt))
    return row, bt


def _choose_by_validation(results):
    candidates = results[
        (results["cost_adjusted"] == False)
        & (results["train_sharpe"] > 0.0)
        & (results["validation_sharpe"] > 0.0)
    ].copy()
    if candidates.empty:
        return None
    best_validation = candidates["validation_sharpe"].max()
    near_best = candidates[candidates["validation_sharpe"] >= 0.90 * best_validation].copy()
    near_best["score"] = (
        near_best["validation_sharpe"]
        + 0.25 * near_best["train_sharpe"]
        + 0.001 * near_best["validation_max_drawdown"]
    )
    return near_best.sort_values(["score", "validation_max_drawdown"], ascending=[False, False]).iloc[0]


def run_soybean_no_fit_experiment(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    candidates = [
        ("rev5", "long_short"),
        ("rev5_plus_physical_10", "long_short"),
        ("soy_physical_balanced", "long_short"),
        ("soy_crush_inventory", "long_short"),
        ("soy_slow_mr_physical", "long_short"),
        ("soy_trend_physical", "long_short"),
        ("soy_curve_crush", "long_short"),
        ("soy_defensive_blend", "long_short"),
        ("soy_trend_physical", "long_only"),
        ("soy_curve_crush", "long_only"),
        ("soy_physical_balanced", "long_only"),
        ("soy_conservative_long_blend", "long_only"),
    ]
    rows = []
    backtests = {}
    positions = {}

    for recipe, mode in candidates:
        signal = build_soybean_signal(feature_panels, recipe)
        pos = signal_to_soybean_positions(signal, futures_pnl, mode=mode)
        key = recipe + "_" + mode
        positions[key] = pos
        for cost_adjusted in [False, True]:
            row, bt = _row(recipe, mode, pos, futures_pnl, cost_adjusted)
            rows.append(row)
            suffix = "cost_adjusted" if cost_adjusted else "zero_cost"
            backtests[key + "_" + suffix] = bt

    results = pd.DataFrame(rows).sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False])
    selected = _choose_by_validation(results)
    return {
        "results": results,
        "selected_by_validation": selected,
        "positions": positions,
        "backtests": backtests,
    }


In [ ]:
"""Soybean given-data vs external-signal experiments.

The module keeps two research tracks separate:
1. given-data-only soybean signals from the provided training files;
2. optional external soybean signals from yfinance and Meteostat.

External data is grouped into economic families and tested both individually
and as equal-family-weight composites. The reported test period is 2018-2020.
"""


import numpy as np
import pandas as pd



YF_TICKERS = {
    "soybean": "ZS=F",
    "soymeal": "ZM=F",
    "soyoil": "ZL=F",
    "corn": "ZC=F",
    "wheat": "ZW=F",
    "usd_index": "DX-Y.NYB",
    "brl": "BRL=X",
    "cny": "CNY=X",
    "crude": "CL=F",
    "equity": "SPY",
}
TRAIN_END = "2016-01-01"
VALIDATION_END = "2018-01-01"


def _tanh(series, divisor=2.0):
    return pd.Series(np.tanh(series.astype(float) / float(divisor)), index=series.index)


def _smooth_threshold(series, mode="long_only"):
    signal = _tanh(series).ewm(halflife=2.0, adjust=False, min_periods=1).mean()
    signal[signal.abs() < 0.05] = 0.0
    if mode == "long_only":
        signal = signal.clip(lower=0.0)
    elif mode == "short_only":
        signal = signal.clip(upper=0.0)
    elif mode != "long_short":
        raise ValueError("Unknown mode: {}".format(mode))
    return signal


def _download_yfinance(start, end):
    import yfinance as yf

    tickers = list(YF_TICKERS.values())
    raw = yf.download(
        tickers,
        start=str(pd.Timestamp(start).date()),
        end=str((pd.Timestamp(end) + pd.DateOffset(days=7)).date()),
        auto_adjust=True,
        progress=False,
        threads=False,
    )
    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" in raw.columns.get_level_values(0):
            close = raw["Close"]
        elif "Adj Close" in raw.columns.get_level_values(0):
            close = raw["Adj Close"]
        else:
            close = raw.xs(raw.columns.get_level_values(0)[0], axis=1, level=0)
    else:
        close = raw.to_frame(tickers[0]) if isinstance(raw, pd.Series) else raw
    close = close.rename(columns={ticker: name for name, ticker in YF_TICKERS.items()})
    close = close[[name for name in YF_TICKERS if name in close.columns]]
    close.index = pd.to_datetime(close.index).tz_localize(None)
    return close.sort_index()


def _given_components(feature_panels):
    soy = feature_panels[COMMODITY]
    inventory_pressure = (
        -soy["public_inventory_change"] - soy["receipts_change"] - soy["cgl_inventory_change"]
    ) / 3.0
    crush_pressure = (soy["crush_surprise"] + soy["crush_utilization"]) / 2.0
    trend = (soy["mom_20"] + soy["mom_60"] + soy["curve_spread"] + soy["cot_pm_oi_level"]) / 4.0
    curve_tightness = (soy["curve_spread"] + soy["curve_ratio"]) / 2.0
    price_family = (soy["mom_20"] + soy["mom_60"] + soy["rev_5"]) / 3.0
    physical_family = (inventory_pressure + crush_pressure + curve_tightness) / 3.0
    conservative = build_soybean_signal(feature_panels, "soy_conservative_long_blend")
    return {
        "given_price_family": price_family.fillna(0.0),
        "given_physical_family": physical_family.fillna(0.0),
        "given_trend": trend.fillna(0.0),
        "given_inventory_pressure": inventory_pressure.fillna(0.0),
        "given_crush_pressure": crush_pressure.fillna(0.0),
        "given_conservative_long_blend": conservative.fillna(0.0),
        "given_equal_price_physical": ((price_family + physical_family) / 2.0).fillna(0.0),
    }


def _external_yfinance_families(prices, trading_index):
    px = prices.reindex(trading_index).ffill().shift(1)
    families = {}

    if {"soybean", "soymeal", "soyoil"}.issubset(px.columns):
        soy_dollars = px["soybean"] / 100.0
        crush = 0.022 * px["soymeal"] + 0.11 * px["soyoil"] - soy_dollars
        crush_mom = rolling_zscore(crush.diff(20), 252, 60)
        meal_lead = rolling_zscore(
            px["soymeal"].pct_change(20, fill_method=None) - px["soybean"].pct_change(20, fill_method=None),
            252,
            60,
        )
        oil_lead = rolling_zscore(
            px["soyoil"].pct_change(20, fill_method=None) - px["soybean"].pct_change(20, fill_method=None),
            252,
            60,
        )
        families["external_crush_family"] = ((crush_mom + meal_lead + oil_lead) / 3.0).fillna(0.0)

    if {"soybean", "corn", "wheat"}.issubset(px.columns):
        soy_corn = rolling_zscore((px["soybean"] / px["corn"]).pct_change(20, fill_method=None), 252, 60)
        soy_wheat = rolling_zscore((px["soybean"] / px["wheat"]).pct_change(20, fill_method=None), 252, 60)
        corn_soy_mr = -rolling_zscore((px["corn"] / px["soybean"]).pct_change(20, fill_method=None), 252, 60)
        families["external_relative_grain_family"] = ((soy_corn + soy_wheat + corn_soy_mr) / 3.0).fillna(0.0)

    fx_parts = []
    if "usd_index" in px.columns:
        fx_parts.append(-rolling_zscore(px["usd_index"].pct_change(20, fill_method=None), 252, 60))
    if "brl" in px.columns:
        fx_parts.append(-rolling_zscore(px["brl"].pct_change(20, fill_method=None), 252, 60))
    if "cny" in px.columns:
        fx_parts.append(-rolling_zscore(px["cny"].pct_change(20, fill_method=None), 252, 60))
    if fx_parts:
        families["external_fx_export_family"] = (sum(fx_parts) / float(len(fx_parts))).fillna(0.0)

    macro_parts = []
    if "crude" in px.columns:
        macro_parts.append(rolling_zscore(px["crude"].pct_change(20, fill_method=None), 252, 60))
    if "equity" in px.columns:
        macro_parts.append(rolling_zscore(px["equity"].pct_change(20, fill_method=None), 252, 60))
    if "usd_index" in px.columns:
        macro_parts.append(-rolling_zscore(px["usd_index"].pct_change(60, fill_method=None), 252, 80))
    if macro_parts:
        families["external_macro_risk_family"] = (sum(macro_parts) / float(len(macro_parts))).fillna(0.0)

    out = {}
    for name, values in families.items():
        cleaned = values.clip(lower=-5.0, upper=5.0).replace([np.inf, -np.inf], np.nan)
        if cleaned.notna().sum() > 20 and cleaned.fillna(0.0).abs().sum() > 0.0:
            out[name] = cleaned.fillna(0.0)
    return out


def _external_weather_family(weather, trading_index):
    panels = build_meteostat_feature_panels(weather, trading_index, mode="commodity_seasonal")
    soy = panels[COMMODITY].reindex(trading_index).fillna(0.0)
    candidates = [
        "meteo_cdd_20d_growing",
        "meteo_hdd_20d_growing",
        "meteo_gdd_60d_growing",
        "meteo_heat_stress_20d_growing",
        "meteo_dryness_20d_growing",
        "meteo_dry_cdd_20d_growing",
        "meteo_precip_20d_planting",
        "meteo_dryness_20d_planting",
        "meteo_freeze_stress_5d_harvest",
    ]
    existing = [col for col in candidates if col in soy.columns]
    if not existing:
        return {}
    family = soy[existing].mean(axis=1).fillna(0.0)
    return {"external_weather_hdd_cdd_family": family.clip(lower=-5.0, upper=5.0)}


def _positions_from_signal(signal, futures_pnl, mode="long_only"):
    cleaned = _smooth_threshold(signal.reindex(futures_pnl.index).fillna(0.0), mode=mode)
    return signal_to_soybean_positions(cleaned, futures_pnl, target_daily_pnl_vol=75.0, max_lot=0.50, mode="long_short")


def _metrics(bt):
    table = split_performance(bt, TEST_START)
    train_val = split_performance(bt.loc[bt.index < pd.Timestamp(TEST_START)], TRAIN_END)
    if table.empty or "sharpe" not in table.index:
        return {
            "train_sharpe": np.nan,
            "validation_sharpe": np.nan,
            "validation_max_drawdown": np.nan,
            "test_sharpe": np.nan,
            "test_pnl": 0.0,
            "test_max_drawdown": np.nan,
            "full_sharpe": np.nan,
            "full_pnl": 0.0,
            "max_drawdown": np.nan,
            "turnover": 0.0,
            "avg_gross_exposure": 0.0,
        }
    return {
        "train_sharpe": train_val.loc["sharpe", "in_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_sharpe": train_val.loc["sharpe", "out_of_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_max_drawdown": train_val.loc["max_drawdown", "out_of_sample"]
        if "max_drawdown" in train_val.index
        else np.nan,
        "test_sharpe": table.loc["sharpe", "out_of_sample"],
        "test_pnl": table.loc["total_pnl", "out_of_sample"],
        "test_max_drawdown": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "max_drawdown": table.loc["max_drawdown", "full_period"],
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
    }


def _evaluate_signal(name, family, signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    rows = []
    backtests = {}
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[COMMODITY]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
        row = {
            "experiment": name,
            "family": family,
            "mode": mode,
            "cost_adjusted": cost_adjusted,
        }
        row.update(_metrics(bt))
        rows.append(row)
        backtests[name + ("_cost_adjusted" if cost_adjusted else "_zero_cost")] = bt
    return rows, backtests, positions


def _zero_cost_metric_for_signal(signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
    return _metrics(bt)


def _passes_pre2018(metrics):
    return (
        pd.notnull(metrics.get("train_sharpe"))
        and pd.notnull(metrics.get("validation_sharpe"))
        and metrics["train_sharpe"] > 0.0
        and metrics["validation_sharpe"] > 0.0
        and pd.notnull(metrics.get("validation_max_drawdown"))
        and metrics["validation_max_drawdown"] > -250.0
    )


def _build_overfit_controlled_signal(given, external, futures_pnl):
    """Select family membership using only train/validation performance."""
    mandatory = {
        "given_physical_family": given["given_physical_family"],
    }
    optional = {
        name: external[name]
        for name in [
            "external_crush_family",
            "external_fx_export_family",
            "external_weather_hdd_cdd_family",
        ]
        if name in external
    }
    selection_rows = []
    selected = dict(mandatory)

    for name, signal in optional.items():
        metrics = _zero_cost_metric_for_signal(signal, futures_pnl, mode="long_only")
        keep = _passes_pre2018(metrics)
        row = {"family": name, "selected": bool(keep)}
        row.update(metrics)
        selection_rows.append(row)
        if keep:
            selected[name] = signal

    combined = sum(selected.values()) / float(len(selected))
    return combined, selected, pd.DataFrame(selection_rows)


def _weighted_sum(families, weights):
    total = 0.0
    used_weight = 0.0
    for name, weight in weights.items():
        if name in families and float(weight) != 0.0:
            total = total + float(weight) * families[name]
            used_weight += float(weight)
    if used_weight == 0.0:
        raise ValueError("No available families in weights: {}".format(weights))
    return total / used_weight


def _build_weight_relaxed_pre2018_signal(given, external, futures_pnl):
    """Choose fixed family weights using only pre-2018 validation data."""
    available = {
        "given_physical_family": given["given_physical_family"],
    }
    for name in [
        "external_crush_family",
        "external_fx_export_family",
        "external_weather_hdd_cdd_family",
    ]:
        if name in external:
            available[name] = external[name]

    candidates = [
        (
            "physical_only",
            {
                "given_physical_family": 1.00,
            },
        ),
        (
            "physical_70_fx_30",
            {
                "given_physical_family": 0.70,
                "external_fx_export_family": 0.30,
            },
        ),
        (
            "physical_50_fx_50",
            {
                "given_physical_family": 0.50,
                "external_fx_export_family": 0.50,
            },
        ),
        (
            "physical_30_fx_70",
            {
                "given_physical_family": 0.30,
                "external_fx_export_family": 0.70,
            },
        ),
        (
            "physical_70_weather_30",
            {
                "given_physical_family": 0.70,
                "external_weather_hdd_cdd_family": 0.30,
            },
        ),
        (
            "physical_70_extcrush_30",
            {
                "given_physical_family": 0.70,
                "external_crush_family": 0.30,
            },
        ),
        (
            "physical_50_fx_25_weather_25",
            {
                "given_physical_family": 0.50,
                "external_fx_export_family": 0.25,
                "external_weather_hdd_cdd_family": 0.25,
            },
        ),
        (
            "physical_50_fx_25_extcrush_25",
            {
                "given_physical_family": 0.50,
                "external_fx_export_family": 0.25,
                "external_crush_family": 0.25,
            },
        ),
        (
            "physical_40_fx_20_extcrush_20_weather_20",
            {
                "given_physical_family": 0.40,
                "external_fx_export_family": 0.20,
                "external_crush_family": 0.20,
                "external_weather_hdd_cdd_family": 0.20,
            },
        ),
    ]

    rows = []
    candidate_signals = {}
    for name, weights in candidates:
        usable = {family: weight for family, weight in weights.items() if family in available}
        if "given_physical_family" not in usable:
            continue
        signal = _weighted_sum(available, usable)
        metrics = _zero_cost_metric_for_signal(signal, futures_pnl, mode="long_only")
        score = -np.inf
        if (
            pd.notnull(metrics.get("train_sharpe"))
            and pd.notnull(metrics.get("validation_sharpe"))
            and metrics["train_sharpe"] > 0.0
            and metrics["validation_sharpe"] > 0.0
        ):
            score = (
                metrics["validation_sharpe"]
                + 0.25 * metrics["train_sharpe"]
                + 0.001 * metrics["validation_max_drawdown"]
            )
        row = {
            "candidate": name,
            "eligible": bool(np.isfinite(score)),
            "score": score,
            "weights": str(usable),
        }
        row.update(metrics)
        rows.append(row)
        candidate_signals[name] = signal

    table = pd.DataFrame(rows)
    eligible = table.loc[table["eligible"]].copy()
    drawdown_eligible = table.loc[
        table["eligible"] & (table["validation_sharpe"] >= 0.50) & (table["train_sharpe"] > 0.0)
    ].copy()

    if eligible.empty:
        selected_name = "physical_only"
    else:
        selected_name = eligible.sort_values(["score", "validation_max_drawdown"], ascending=[False, False]).iloc[0][
            "candidate"
        ]

    if drawdown_eligible.empty:
        drawdown_selected_name = selected_name
    else:
        drawdown_selected_name = drawdown_eligible.sort_values(
            ["validation_max_drawdown", "validation_sharpe"],
            ascending=[False, False],
        ).iloc[0]["candidate"]

    return (
        candidate_signals[selected_name],
        selected_name,
        candidate_signals[drawdown_selected_name],
        drawdown_selected_name,
        table,
        candidate_signals,
    )


def _soybean_regime_frames(given, external, futures_pnl):
    soy_pnl = futures_pnl[COMMODITY].fillna(0.0)
    vol = soy_pnl.rolling(60, min_periods=20).std().shift(1)
    lt_vol = vol.expanding(min_periods=252).median().shift(1)
    high_threshold = vol.expanding(min_periods=252).quantile(0.75).shift(1)
    low_vol = (vol < 0.80 * lt_vol).fillna(False)
    high_vol = ((vol > 1.20 * lt_vol) | (vol > high_threshold)).fillna(False)
    trend_positive = (given["given_trend"] > 0.0).reindex(futures_pnl.index).fillna(False)
    return {
        "vol": vol,
        "lt_vol": lt_vol,
        "low_vol": low_vol.astype(float),
        "high_vol": high_vol.astype(float),
        "trend_positive": trend_positive.astype(float),
    }


def _mean_available(items):
    values = [series for series in items if series is not None]
    if not values:
        raise ValueError("No series available to average")
    return sum(values) / float(len(values))


def _build_regime_shift_signals(given, external, futures_pnl):
    """Build observable regime-shift candidates with fixed equal-weight sleeves."""
    regimes = _soybean_regime_frames(given, external, futures_pnl)
    source = _mean_available(
        [
            given["given_physical_family"],
            external.get("external_fx_export_family"),
            external.get("external_crush_family"),
            external.get("external_weather_hdd_cdd_family"),
        ]
    )
    physical_weather = _mean_available(
        [
            given["given_physical_family"],
            external.get("external_weather_hdd_cdd_family"),
        ]
    )
    trend_signal = _mean_available(
        [
            given["given_trend"],
            external.get("external_macro_risk_family"),
        ]
    )
    high_vol_signal = _mean_available(
        [
            given["given_trend"],
            external.get("external_fx_export_family"),
            external.get("external_crush_family"),
        ]
    )
    low_vol_signal = _mean_available(
        [
            given["given_physical_family"],
            external.get("external_weather_hdd_cdd_family"),
        ]
    )

    high_vol = regimes["high_vol"]
    low_vol = regimes["low_vol"]
    trend_positive = regimes["trend_positive"]
    trend_w_vol = (0.20 + 0.40 * high_vol).clip(0.20, 0.60)
    trend_w_confirmed = (0.20 + 0.40 * ((high_vol > 0) | (trend_positive > 0)).astype(float)).clip(0.20, 0.60)

    candidates = {
        "regime_hybrid_vol_trend_weight": (1.0 - trend_w_vol) * source + trend_w_vol * trend_signal,
        "regime_hybrid_trend_confirmed_weight": (1.0 - trend_w_confirmed) * source
        + trend_w_confirmed * trend_signal,
        "regime_low_physical_high_trend_switch": high_vol * high_vol_signal + (1.0 - high_vol) * low_vol_signal,
        "regime_low_vol_physical_else_balanced": low_vol * low_vol_signal + (1.0 - low_vol) * source,
        "regime_high_vol_fx_trend_else_physical_weather": high_vol * high_vol_signal
        + (1.0 - high_vol) * physical_weather,
    }
    return {name: signal.clip(lower=-5.0, upper=5.0).fillna(0.0) for name, signal in candidates.items()}, regimes


def run_soybean_given_only_experiment(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    signals = _given_components(feature_panels)
    rows = []
    backtests = {}
    positions = {}
    for name, signal in signals.items():
        mode = "long_only" if name != "given_equal_price_physical" else "long_short"
        new_rows, new_bt, pos = _evaluate_signal(name, "given", signal, futures_pnl, mode=mode)
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions[name] = pos

    results = pd.DataFrame(rows).sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False])
    return {
        "results": results.reset_index(drop=True),
        "signals": signals,
        "positions": positions,
        "backtests": backtests,
    }


def run_soybean_external_signal_experiment(data_dir="train_set", include_weather=True):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    given = _given_components(feature_panels)

    external = {}
    external_errors = []
    try:
        prices = _download_yfinance(futures_pnl.index.min(), futures_pnl.index.max())
        external.update(_external_yfinance_families(prices, futures_pnl.index))
    except Exception as exc:
        prices = pd.DataFrame()
        external_errors.append("yfinance: {}".format(exc))

    weather = pd.DataFrame()
    if include_weather:
        try:
            weather = fetch_meteostat_weather(futures_pnl.index.min(), futures_pnl.index.max())
            external.update(_external_weather_family(weather, futures_pnl.index))
        except Exception as exc:
            external_errors.append("meteostat: {}".format(exc))

    rows = []
    backtests = {}
    positions = {}

    for name, signal in external.items():
        new_rows, new_bt, pos = _evaluate_signal(name, name, signal, futures_pnl, mode="long_only")
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions[name] = pos

    if external:
        external_equal = sum(external.values()) / float(len(external))
        new_rows, new_bt, pos = _evaluate_signal(
            "external_equal_family_weight",
            "external_equal_families",
            external_equal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["external_equal_family_weight"] = pos

        given_base = given["given_conservative_long_blend"]
        combined_50_50 = 0.50 * given_base + 0.50 * external_equal
        new_rows, new_bt, pos = _evaluate_signal(
            "given_plus_external_equal_family_50_50",
            "given_external_equal_families",
            combined_50_50,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["given_plus_external_equal_family_50_50"] = pos

        all_families = dict(external)
        all_families["given_price_family"] = given["given_price_family"]
        all_families["given_physical_family"] = given["given_physical_family"]
        all_equal = sum(all_families.values()) / float(len(all_families))
        new_rows, new_bt, pos = _evaluate_signal(
            "given_and_external_all_families_equal_weight",
            "all_equal_families",
            all_equal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["given_and_external_all_families_equal_weight"] = pos

        if "external_weather_hdd_cdd_family" in external:
            crush_weather = 0.50 * given["given_crush_pressure"] + 0.50 * external["external_weather_hdd_cdd_family"]
            new_rows, new_bt, pos = _evaluate_signal(
                "given_crush_plus_weather_hdd_cdd_equal_weight",
                "given_crush_weather_equal",
                crush_weather,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["given_crush_plus_weather_hdd_cdd_equal_weight"] = pos

            conservative_weather = (
                0.50 * given["given_conservative_long_blend"] + 0.50 * external["external_weather_hdd_cdd_family"]
            )
            new_rows, new_bt, pos = _evaluate_signal(
                "given_conservative_plus_weather_hdd_cdd_equal_weight",
                "given_conservative_weather_equal",
                conservative_weather,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["given_conservative_plus_weather_hdd_cdd_equal_weight"] = pos

        fundamental_family_names = [
            ("given_physical_family", given.get("given_physical_family")),
            ("external_crush_family", external.get("external_crush_family")),
            ("external_fx_export_family", external.get("external_fx_export_family")),
            ("external_weather_hdd_cdd_family", external.get("external_weather_hdd_cdd_family")),
        ]
        fundamental_families = [values for _, values in fundamental_family_names if values is not None]
        if len(fundamental_families) >= 2:
            fundamental_equal = sum(fundamental_families) / float(len(fundamental_families))
            new_rows, new_bt, pos = _evaluate_signal(
                "given_physical_external_fundamentals_equal_weight",
                "physical_crush_fx_weather_equal",
                fundamental_equal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["given_physical_external_fundamentals_equal_weight"] = pos

        overfit_controlled, selected_families, selection_table = _build_overfit_controlled_signal(
            given,
            external,
            futures_pnl,
        )
        new_rows, new_bt, pos = _evaluate_signal(
            "overfit_controlled_pre2018_family_selection",
            "pre2018_selected_equal_families",
            overfit_controlled,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["overfit_controlled_pre2018_family_selection"] = pos

        (
            weight_relaxed,
            weight_relaxed_name,
            drawdown_relaxed,
            drawdown_relaxed_name,
            weight_relaxed_table,
            weight_relaxed_candidates,
        ) = _build_weight_relaxed_pre2018_signal(
            given,
            external,
            futures_pnl,
        )
        for candidate_name, candidate_signal in weight_relaxed_candidates.items():
            new_rows, new_bt, pos = _evaluate_signal(
                "weight_relaxed_candidate_" + candidate_name,
                "predefined_fixed_family_weights",
                candidate_signal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["weight_relaxed_candidate_" + candidate_name] = pos

        new_rows, new_bt, pos = _evaluate_signal(
            "weight_relaxed_pre2018_selection_" + weight_relaxed_name,
            "pre2018_selected_fixed_family_weights",
            weight_relaxed,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["weight_relaxed_pre2018_selection_" + weight_relaxed_name] = pos

        new_rows, new_bt, pos = _evaluate_signal(
            "drawdown_priority_pre2018_selection_" + drawdown_relaxed_name,
            "pre2018_drawdown_priority_fixed_family_weights",
            drawdown_relaxed,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["drawdown_priority_pre2018_selection_" + drawdown_relaxed_name] = pos

        regime_signals, regime_frames = _build_regime_shift_signals(given, external, futures_pnl)
        regime_selection_rows = []
        for regime_name, regime_signal in regime_signals.items():
            metrics = _zero_cost_metric_for_signal(regime_signal, futures_pnl, mode="long_only")
            eligible = (
                pd.notnull(metrics.get("train_sharpe"))
                and pd.notnull(metrics.get("validation_sharpe"))
                and metrics["train_sharpe"] > 0.0
                and metrics["validation_sharpe"] >= 0.50
            )
            score = (
                metrics["validation_sharpe"] + 0.25 * metrics["train_sharpe"] + 0.001 * metrics["validation_max_drawdown"]
                if eligible
                else -np.inf
            )
            row = {"candidate": regime_name, "eligible": bool(eligible), "score": score}
            row.update(metrics)
            regime_selection_rows.append(row)
            new_rows, new_bt, pos = _evaluate_signal(
                "regime_candidate_" + regime_name,
                "observable_regime_shift",
                regime_signal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["regime_candidate_" + regime_name] = pos

        regime_selection_table = pd.DataFrame(regime_selection_rows)
        regime_eligible = regime_selection_table.loc[regime_selection_table["eligible"]].copy()
        if regime_eligible.empty:
            regime_selected_name = regime_selection_table.sort_values(
                ["validation_sharpe", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        else:
            regime_selected_name = regime_eligible.sort_values(
                ["score", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        selected_regime_signal = regime_signals[regime_selected_name]
        new_rows, new_bt, pos = _evaluate_signal(
            "regime_pre2018_selection_" + regime_selected_name,
            "pre2018_selected_observable_regime_shift",
            selected_regime_signal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["regime_pre2018_selection_" + regime_selected_name] = pos

        fund_candidates = {
            "fund_blend_50_regime_50_drawdown": 0.50 * selected_regime_signal + 0.50 * drawdown_relaxed,
            "fund_blend_35_regime_65_drawdown": 0.35 * selected_regime_signal + 0.65 * drawdown_relaxed,
            "fund_blend_65_regime_35_drawdown": 0.65 * selected_regime_signal + 0.35 * drawdown_relaxed,
            "fund_consensus_min_regime_drawdown": pd.concat(
                [selected_regime_signal, drawdown_relaxed],
                axis=1,
            ).min(axis=1),
        }
        fund_selection_rows = []
        for fund_name, fund_signal in fund_candidates.items():
            metrics = _zero_cost_metric_for_signal(fund_signal, futures_pnl, mode="long_only")
            eligible = (
                pd.notnull(metrics.get("train_sharpe"))
                and pd.notnull(metrics.get("validation_sharpe"))
                and metrics["train_sharpe"] > 0.0
                and metrics["validation_sharpe"] >= 0.50
            )
            score = (
                metrics["validation_sharpe"]
                + 0.25 * metrics["train_sharpe"]
                + 0.002 * metrics["validation_max_drawdown"]
                if eligible
                else -np.inf
            )
            row = {"candidate": fund_name, "eligible": bool(eligible), "score": score}
            row.update(metrics)
            fund_selection_rows.append(row)
            new_rows, new_bt, pos = _evaluate_signal(
                "fund_candidate_" + fund_name,
                "fund_usable_regime_drawdown_blend",
                fund_signal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["fund_candidate_" + fund_name] = pos

        fund_selection_table = pd.DataFrame(fund_selection_rows)
        fund_eligible = fund_selection_table.loc[fund_selection_table["eligible"]].copy()
        if fund_eligible.empty:
            fund_selected_name = fund_selection_table.sort_values(
                ["validation_sharpe", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        else:
            fund_selected_name = fund_eligible.sort_values(
                ["score", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        selected_fund_signal = fund_candidates[fund_selected_name]
        new_rows, new_bt, pos = _evaluate_signal(
            "fund_pre2018_selection_" + fund_selected_name,
            "pre2018_selected_fund_usable_blend",
            selected_fund_signal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["fund_pre2018_selection_" + fund_selected_name] = pos
    else:
        selection_table = pd.DataFrame()
        selected_families = {}
        weight_relaxed_table = pd.DataFrame()
        weight_relaxed_name = None
        drawdown_relaxed_name = None
        regime_selection_table = pd.DataFrame()
        regime_selected_name = None
        regime_frames = {}
        fund_selection_table = pd.DataFrame()
        fund_selected_name = None

    results = pd.DataFrame(rows)
    if not results.empty:
        results = results.sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False]).reset_index(drop=True)
    return {
        "results": results,
        "given_signals": given,
        "external_signals": external,
        "positions": positions,
        "backtests": backtests,
        "yfinance_prices": prices,
        "weather": weather,
        "errors": external_errors,
        "overfit_control_selection": selection_table,
        "overfit_control_selected_families": list(selected_families.keys()),
        "weight_relaxed_selection": weight_relaxed_table,
        "weight_relaxed_selected_candidate": weight_relaxed_name,
        "drawdown_priority_selected_candidate": drawdown_relaxed_name,
        "regime_selection": regime_selection_table,
        "regime_selected_candidate": regime_selected_name,
        "regime_frames": regime_frames,
        "fund_selection": fund_selection_table,
        "fund_selected_candidate": fund_selected_name,
    }


In [ ]:
# Save soybean external helper aliases before later cells define same helper names.
_soy_download_yfinance = _download_yfinance
_soy_external_yfinance_families_saved = _external_yfinance_families
_soy_weather_family_saved = _external_weather_family


In [ ]:
"""Corn lower-overfit strategy experiments with EIA ethanol.

This mirrors the soybean research style but keeps corn-specific economics:
provided physical/Cargill signals, EIA ethanol production/stocks, crop-belt
weather, export FX pressure, grain relative value, and macro/risk.
"""


import numpy as np
import pandas as pd



COMMODITY = "CORN"
TRAIN_END = "2016-01-01"
TEST_START = "2018-01-01"
YF_TICKERS = {
    "corn": "ZC=F",
    "soybean": "ZS=F",
    "wheat": "ZW=F",
    "usd_index": "DX-Y.NYB",
    "brl": "BRL=X",
    "cny": "CNY=X",
    "crude": "CL=F",
    "equity": "SPY",
}


def _tanh(series, divisor=2.0):
    return pd.Series(np.tanh(series.astype(float) / float(divisor)), index=series.index)


def _smooth_threshold(series, mode="long_only"):
    signal = _tanh(series).ewm(halflife=2.0, adjust=False, min_periods=1).mean()
    signal[signal.abs() < 0.05] = 0.0
    if mode == "long_only":
        signal = signal.clip(lower=0.0)
    elif mode == "short_only":
        signal = signal.clip(upper=0.0)
    elif mode != "long_short":
        raise ValueError("Unknown mode: {}".format(mode))
    return signal


def _download_yfinance(start, end):
    import yfinance as yf

    tickers = list(YF_TICKERS.values())
    raw = yf.download(
        tickers,
        start=str(pd.Timestamp(start).date()),
        end=str((pd.Timestamp(end) + pd.DateOffset(days=7)).date()),
        auto_adjust=True,
        progress=False,
        threads=False,
    )
    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" in raw.columns.get_level_values(0):
            close = raw["Close"]
        elif "Adj Close" in raw.columns.get_level_values(0):
            close = raw["Adj Close"]
        else:
            close = raw.xs(raw.columns.get_level_values(0)[0], axis=1, level=0)
    else:
        close = raw.to_frame(tickers[0]) if isinstance(raw, pd.Series) else raw
    close = close.rename(columns={ticker: name for name, ticker in YF_TICKERS.items()})
    close = close[[name for name in YF_TICKERS if name in close.columns]]
    close.index = pd.to_datetime(close.index).tz_localize(None)
    return close.sort_index()


def _given_components(feature_panels):
    corn = feature_panels[COMMODITY]
    inventory_pressure = (
        -corn["public_inventory_change"] - corn["receipts_change"] - corn["cgl_inventory_change"]
    ) / 3.0
    cgl_crush_activity = (corn["crush_surprise"] + corn["crush_utilization"]) / 2.0
    trend = (corn["mom_20"] + corn["mom_60"] + corn["curve_spread"] + corn["cot_pm_oi_level"]) / 4.0
    curve_tightness = (corn["curve_spread"] + corn["curve_ratio"]) / 2.0
    price_family = (corn["mom_20"] + corn["mom_60"] + corn["rev_5"]) / 3.0
    physical_family = (inventory_pressure + curve_tightness + 0.25 * cgl_crush_activity) / 2.25
    conservative = 0.40 * physical_family + 0.30 * trend + 0.30 * price_family
    return {
        "given_price_family": price_family.fillna(0.0),
        "given_physical_family": physical_family.fillna(0.0),
        "given_trend": trend.fillna(0.0),
        "given_inventory_pressure": inventory_pressure.fillna(0.0),
        "given_cgl_crush_activity": cgl_crush_activity.fillna(0.0),
        "given_curve_tightness": curve_tightness.fillna(0.0),
        "given_conservative_blend": conservative.fillna(0.0),
    }


def _ethanol_family(ethanol_features):
    parts = []
    if "ethanol_production_change_4w" in ethanol_features:
        parts.append(ethanol_features["ethanol_production_change_4w"])
    if "ethanol_prod_to_stocks" in ethanol_features:
        parts.append(ethanol_features["ethanol_prod_to_stocks"])
    if "ethanol_demand_pressure" in ethanol_features:
        parts.append(ethanol_features["ethanol_demand_pressure"])
    if "ethanol_stocks_change_4w" in ethanol_features:
        parts.append(-ethanol_features["ethanol_stocks_change_4w"])
    if not parts:
        return {}
    return {"external_ethanol_family": (sum(parts) / float(len(parts))).clip(lower=-5.0, upper=5.0).fillna(0.0)}


def _external_yfinance_families(prices, trading_index):
    px = prices.reindex(trading_index).ffill().shift(1)
    families = {}
    if {"corn", "soybean", "wheat"}.issubset(px.columns):
        corn_soy = rolling_zscore((px["corn"] / px["soybean"]).pct_change(20, fill_method=None), 252, 60)
        corn_wheat = rolling_zscore((px["corn"] / px["wheat"]).pct_change(20, fill_method=None), 252, 60)
        soy_corn_mr = -rolling_zscore((px["soybean"] / px["corn"]).pct_change(20, fill_method=None), 252, 60)
        families["external_relative_grain_family"] = ((corn_soy + corn_wheat + soy_corn_mr) / 3.0).fillna(0.0)

    fx_parts = []
    if "usd_index" in px.columns:
        fx_parts.append(-rolling_zscore(px["usd_index"].pct_change(20, fill_method=None), 252, 60))
    if "brl" in px.columns:
        fx_parts.append(-rolling_zscore(px["brl"].pct_change(20, fill_method=None), 252, 60))
    if "cny" in px.columns:
        fx_parts.append(-rolling_zscore(px["cny"].pct_change(20, fill_method=None), 252, 60))
    if fx_parts:
        families["external_fx_export_family"] = (sum(fx_parts) / float(len(fx_parts))).fillna(0.0)

    macro_parts = []
    if "crude" in px.columns:
        macro_parts.append(rolling_zscore(px["crude"].pct_change(20, fill_method=None), 252, 60))
    if "equity" in px.columns:
        macro_parts.append(rolling_zscore(px["equity"].pct_change(20, fill_method=None), 252, 60))
    if "usd_index" in px.columns:
        macro_parts.append(-rolling_zscore(px["usd_index"].pct_change(60, fill_method=None), 252, 80))
    if macro_parts:
        families["external_macro_risk_family"] = (sum(macro_parts) / float(len(macro_parts))).fillna(0.0)

    out = {}
    for name, values in families.items():
        cleaned = values.clip(lower=-5.0, upper=5.0).replace([np.inf, -np.inf], np.nan)
        if cleaned.notna().sum() > 20 and cleaned.fillna(0.0).abs().sum() > 0.0:
            out[name] = cleaned.fillna(0.0)
    return out


def _weather_family(weather, trading_index):
    panels = build_meteostat_feature_panels(weather, trading_index, mode="commodity_seasonal")
    corn = panels[COMMODITY].reindex(trading_index).fillna(0.0)
    candidates = [
        "meteo_cdd_20d_growing",
        "meteo_hdd_20d_growing",
        "meteo_gdd_60d_growing",
        "meteo_heat_stress_20d_growing",
        "meteo_dryness_20d_growing",
        "meteo_dry_cdd_20d_growing",
        "meteo_precip_20d_planting",
        "meteo_dryness_20d_planting",
        "meteo_freeze_stress_5d_harvest",
    ]
    existing = [col for col in candidates if col in corn.columns]
    if not existing:
        return {}
    return {"external_weather_hdd_cdd_family": corn[existing].mean(axis=1).clip(lower=-5.0, upper=5.0).fillna(0.0)}


def _positions_from_signal(signal, futures_pnl, mode="long_only"):
    cleaned = _smooth_threshold(signal.reindex(futures_pnl.index).fillna(0.0), mode=mode)
    pnl = futures_pnl[[COMMODITY]]
    asset_vol = pnl[COMMODITY].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    pos = cleaned.reindex(pnl.index).fillna(0.0) * (75.0 / asset_vol)
    out = pd.DataFrame(0.0, index=pnl.index, columns=[COMMODITY])
    out[COMMODITY] = pos.clip(lower=-0.50, upper=0.50).fillna(0.0)
    return out


def _metrics(bt):
    table = split_performance(bt, TEST_START)
    train_val = split_performance(bt.loc[bt.index < pd.Timestamp(TEST_START)], TRAIN_END)
    if table.empty or "sharpe" not in table.index:
        return {
            "train_sharpe": np.nan,
            "validation_sharpe": np.nan,
            "validation_max_drawdown": np.nan,
            "test_sharpe": np.nan,
            "test_pnl": 0.0,
            "test_max_drawdown": np.nan,
            "full_sharpe": np.nan,
            "full_pnl": 0.0,
            "max_drawdown": np.nan,
            "turnover": 0.0,
            "avg_gross_exposure": 0.0,
        }
    return {
        "train_sharpe": train_val.loc["sharpe", "in_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_sharpe": train_val.loc["sharpe", "out_of_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_max_drawdown": train_val.loc["max_drawdown", "out_of_sample"] if "max_drawdown" in train_val.index else np.nan,
        "test_sharpe": table.loc["sharpe", "out_of_sample"],
        "test_pnl": table.loc["total_pnl", "out_of_sample"],
        "test_max_drawdown": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "max_drawdown": table.loc["max_drawdown", "full_period"],
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
    }


def _evaluate_signal(name, family, signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    rows = []
    backtests = {}
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[COMMODITY]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
        row = {"experiment": name, "family": family, "mode": mode, "cost_adjusted": cost_adjusted}
        row.update(_metrics(bt))
        rows.append(row)
        backtests[name + ("_cost_adjusted" if cost_adjusted else "_zero_cost")] = bt
    return rows, backtests, positions


def _zero_cost_metrics(signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
    return _metrics(bt)


def _mean_available(items):
    values = [series for series in items if series is not None]
    if not values:
        raise ValueError("No series available")
    return sum(values) / float(len(values))


def _weighted_sum(families, weights):
    total = 0.0
    used = 0.0
    for name, weight in weights.items():
        if name in families and float(weight) != 0.0:
            total = total + float(weight) * families[name]
            used += float(weight)
    if used == 0.0:
        raise ValueError("No available families")
    return total / used


def _build_regime_frames(given, futures_pnl):
    corn_pnl = futures_pnl[COMMODITY].fillna(0.0)
    vol = corn_pnl.rolling(60, min_periods=20).std().shift(1)
    lt_vol = vol.expanding(min_periods=252).median().shift(1)
    high_threshold = vol.expanding(min_periods=252).quantile(0.75).shift(1)
    low_vol = (vol < 0.80 * lt_vol).fillna(False)
    high_vol = ((vol > 1.20 * lt_vol) | (vol > high_threshold)).fillna(False)
    trend_positive = (given["given_trend"] > 0.0).reindex(futures_pnl.index).fillna(False)
    return {"low_vol": low_vol.astype(float), "high_vol": high_vol.astype(float), "trend_positive": trend_positive.astype(float)}


def _build_candidate_signals(given, external, futures_pnl):
    families = dict(given)
    families.update(external)
    regimes = _build_regime_frames(given, futures_pnl)
    source = _mean_available([
        given["given_physical_family"],
        external.get("external_ethanol_family"),
        external.get("external_fx_export_family"),
        external.get("external_weather_hdd_cdd_family"),
    ])
    trend_signal = _mean_available([given["given_trend"], external.get("external_macro_risk_family")])
    high_vol = regimes["high_vol"]
    trend_w = (0.20 + 0.40 * high_vol).clip(0.20, 0.60)
    signals = {
        "given_physical_family": given["given_physical_family"],
        "given_conservative_blend": given["given_conservative_blend"],
        "external_ethanol_family": external.get("external_ethanol_family", pd.Series(0.0, index=futures_pnl.index)),
        "requirement_given_90_ethanol_10": _weighted_sum(
            families,
            {
                "given_conservative_blend": 0.90,
                "external_ethanol_family": 0.10,
            },
        ),
        "requirement_given_80_ethanol_20": _weighted_sum(
            families,
            {
                "given_conservative_blend": 0.80,
                "external_ethanol_family": 0.20,
            },
        ),
        "requirement_given_80_ethanol_10_fx_10": _weighted_sum(
            families,
            {
                "given_conservative_blend": 0.80,
                "external_ethanol_family": 0.10,
                "external_fx_export_family": 0.10,
            },
        ),
        "requirement_physical_60_trend_25_ethanol_15": _weighted_sum(
            families,
            {
                "given_physical_family": 0.60,
                "given_trend": 0.25,
                "external_ethanol_family": 0.15,
            },
        ),
        "fundamental_equal_physical_ethanol_fx_weather": _mean_available([
            given["given_physical_family"],
            external.get("external_ethanol_family"),
            external.get("external_fx_export_family"),
            external.get("external_weather_hdd_cdd_family"),
        ]),
        "fundamental_40_physical_30_ethanol_15_fx_15_weather": _weighted_sum(
            families,
            {
                "given_physical_family": 0.40,
                "external_ethanol_family": 0.30,
                "external_fx_export_family": 0.15,
                "external_weather_hdd_cdd_family": 0.15,
            },
        ),
        "fundamental_50_physical_30_ethanol_20_fx": _weighted_sum(
            families,
            {
                "given_physical_family": 0.50,
                "external_ethanol_family": 0.30,
                "external_fx_export_family": 0.20,
            },
        ),
        "regime_hybrid_vol_trend_weight": ((1.0 - trend_w) * source + trend_w * trend_signal),
        "low_vol_physical_weather_else_ethanol_fx": (
            regimes["low_vol"] * _mean_available([given["given_physical_family"], external.get("external_weather_hdd_cdd_family")])
            + (1.0 - regimes["low_vol"]) * _mean_available([external.get("external_ethanol_family"), external.get("external_fx_export_family"), given["given_trend"]])
        ),
    }
    if "external_ethanol_family" in external:
        high_vol_delever = 1.0 - 0.35 * regimes["high_vol"]
        signals["requirement_drawdown_guard_given_ethanol"] = (
            _weighted_sum(
                families,
                {
                    "given_conservative_blend": 0.80,
                    "external_ethanol_family": 0.10,
                    "external_weather_hdd_cdd_family": 0.10,
                },
            )
            * high_vol_delever
        )
    clean = {name: signal.clip(lower=-5.0, upper=5.0).fillna(0.0) for name, signal in signals.items()}
    return clean, regimes


def _select_pre2018(candidates, futures_pnl, require_ethanol=False, modes=("long_only", "long_short")):
    rows = []
    for name, signal in candidates.items():
        for mode in modes:
            metrics = _zero_cost_metrics(signal, futures_pnl, mode=mode)
            requirement_ok = (not require_ethanol) or ("ethanol" in name)
            eligible = (
                requirement_ok
                and pd.notnull(metrics.get("train_sharpe"))
                and pd.notnull(metrics.get("validation_sharpe"))
                and metrics["train_sharpe"] > 0.0
                and metrics["validation_sharpe"] >= 0.50
            )
            score = (
                metrics["validation_sharpe"] + 0.25 * metrics["train_sharpe"] + 0.002 * metrics["validation_max_drawdown"]
                if eligible
                else -np.inf
            )
            row = {"candidate": name, "mode": mode, "eligible": bool(eligible), "score": score}
            row.update(metrics)
            rows.append(row)
    table = pd.DataFrame(rows)
    if require_ethanol:
        selection_pool = table.loc[table["candidate"].str.contains("ethanol")].copy()
    else:
        selection_pool = table.copy()
    eligible = selection_pool.loc[selection_pool["eligible"]].copy()
    if eligible.empty:
        selected = selection_pool.sort_values(
            ["validation_sharpe", "validation_max_drawdown", "test_max_drawdown"],
            ascending=[False, False, False],
        ).iloc[0]
    else:
        selected = eligible.sort_values(["score", "validation_max_drawdown"], ascending=[False, False]).iloc[0]
    return selected["candidate"], selected["mode"], table


def run_corn_signal_experiment(data_dir="train_set", include_weather=True, include_eia=True):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    given = _given_components(feature_panels)
    external = {}
    errors = []
    prices = pd.DataFrame()
    weather = pd.DataFrame()
    ethanol = pd.DataFrame()

    try:
        prices = _download_yfinance(futures_pnl.index.min(), futures_pnl.index.max())
        external.update(_external_yfinance_families(prices, futures_pnl.index))
    except Exception as exc:
        errors.append("yfinance: {}".format(exc))

    if include_weather:
        try:
            weather = fetch_meteostat_weather(futures_pnl.index.min(), futures_pnl.index.max())
            external.update(_weather_family(weather, futures_pnl.index))
        except Exception as exc:
            errors.append("meteostat: {}".format(exc))

    if include_eia:
        try:
            ethanol = fetch_eia_ethanol()
            ethanol_features = build_ethanol_feature_panel(ethanol, futures_pnl.index)
            external.update(_ethanol_family(ethanol_features))
        except Exception as exc:
            ethanol_features = pd.DataFrame()
            errors.append("eia_ethanol: {}".format(exc))
    else:
        ethanol_features = pd.DataFrame()

    candidates, regime_frames = _build_candidate_signals(given, external, futures_pnl)
    selected, selected_mode, selection = _select_pre2018(candidates, futures_pnl, require_ethanol=True)
    rows = []
    backtests = {}
    positions = {}
    for name, signal in candidates.items():
        for mode in ["long_only", "long_short"]:
            new_rows, new_bt, pos = _evaluate_signal("corn_candidate_" + name, name, signal, futures_pnl, mode=mode)
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions[name + "_" + mode] = pos
    new_rows, new_bt, pos = _evaluate_signal(
        "corn_pre2018_selection_" + selected,
        "pre2018_selected_corn_strategy",
        candidates[selected],
        futures_pnl,
        mode=selected_mode,
    )
    rows.extend(new_rows)
    backtests.update(new_bt)
    positions["selected"] = pos
    results = pd.DataFrame(rows).sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False]).reset_index(drop=True)
    return {
        "results": results,
        "selection": selection,
        "selected_candidate": selected,
        "selected_mode": selected_mode,
        "given_signals": given,
        "external_signals": external,
        "regime_frames": regime_frames,
        "positions": positions,
        "backtests": backtests,
        "yfinance_prices": prices,
        "weather": weather,
        "ethanol": ethanol,
        "ethanol_features": ethanol_features,
        "errors": errors,
    }


In [ ]:
# Save corn external helper aliases before later cells define same helper names.
_corn_download_yfinance = _download_yfinance
_corn_external_yfinance_families_saved = _external_yfinance_families
_corn_weather_family_saved = _weather_family


In [ ]:
# Aliases replacing imports from soybean_external_signal_experiment and corn_external_signal_experiment.
_download_soy_yfinance = _soy_download_yfinance
_soy_external_yfinance_families = _soy_external_yfinance_families_saved
_soy_weather_family = _soy_weather_family_saved
_download_corn_yfinance = _corn_download_yfinance
_corn_external_yfinance_families = _corn_external_yfinance_families_saved
_corn_weather_family = _corn_weather_family_saved

"""IC-threshold family selection experiments for corn and soybeans.

Research rule:
1. Build a transparent signal universe for each commodity.
2. Keep only signals whose train IC clears a fixed threshold.
3. Build fixed equal-weight family candidates from the surviving signals.
4. Select the best family candidate using validation IC only.
5. Report the untouched 2018-2020 OOS backtest.
"""


import os

import numpy as np
import pandas as pd



TRAIN_END = pd.Timestamp("2016-01-01")
TEST_START = pd.Timestamp("2018-01-01")
IC_THRESHOLD = 0.015
VALIDATION_IC_FLOOR = 0.0
MAX_LOT = 0.50
TARGET_DAILY_PNL_VOL = 75.0


def _tanh(series, divisor=2.0):
    return pd.Series(np.tanh(series.astype(float) / float(divisor)), index=series.index)


def _clean_signal(series, index):
    signal = series.reindex(index).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return signal.clip(lower=-5.0, upper=5.0)


def _smooth_threshold(series, mode):
    signal = _tanh(series).ewm(halflife=2.0, adjust=False, min_periods=1).mean()
    signal[signal.abs() < 0.05] = 0.0
    if mode == "long_only":
        signal = signal.clip(lower=0.0)
    elif mode == "short_only":
        signal = signal.clip(upper=0.0)
    elif mode != "long_short":
        raise ValueError("Unknown mode: {}".format(mode))
    return signal


def _positions_from_signal(signal, futures_pnl, commodity, mode="long_only"):
    cleaned = _smooth_threshold(signal.reindex(futures_pnl.index).fillna(0.0), mode=mode)
    pnl = futures_pnl[[commodity]]
    asset_vol = pnl[commodity].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    pos = cleaned * (TARGET_DAILY_PNL_VOL / asset_vol)
    out = pd.DataFrame(0.0, index=futures_pnl.index, columns=[commodity])
    out[commodity] = pos.clip(lower=-MAX_LOT, upper=MAX_LOT).fillna(0.0)
    return out


def _split_masks(index):
    return {
        "train": index < TRAIN_END,
        "validation": (index >= TRAIN_END) & (index < TEST_START),
        "test": index >= TEST_START,
    }


def _ic(signal, futures_pnl, commodity, mask):
    aligned = pd.concat(
        [
            signal.reindex(futures_pnl.index),
            futures_pnl[commodity].shift(-1),
        ],
        axis=1,
    ).dropna()
    if aligned.empty:
        return np.nan
    aligned = aligned.loc[mask.reindex(aligned.index).fillna(False)]
    if len(aligned) < 40 or aligned.iloc[:, 0].std() == 0 or aligned.iloc[:, 1].std() == 0:
        return np.nan
    return aligned.iloc[:, 0].corr(aligned.iloc[:, 1], method="spearman")


def _signal_ic_table(signals, futures_pnl, commodity):
    masks = _split_masks(futures_pnl.index)
    rows = []
    for name, signal in signals.items():
        row = {"signal": name}
        for split_name, mask in masks.items():
            row[split_name + "_ic"] = _ic(signal, futures_pnl, commodity, pd.Series(mask, index=futures_pnl.index))
        row["passes_ic_threshold"] = bool(
            pd.notnull(row["train_ic"]) and abs(row["train_ic"]) >= IC_THRESHOLD
        )
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["passes_ic_threshold", "train_ic"], ascending=[False, False])


def _orient_and_filter_signals(signals, ic_table, futures_pnl):
    oriented = {}
    selected_rows = ic_table.loc[ic_table["passes_ic_threshold"]].copy()
    for _, row in selected_rows.iterrows():
        name = row["signal"]
        sign = 1.0 if row["train_ic"] >= 0 else -1.0
        oriented[name] = sign * signals[name].reindex(futures_pnl.index).fillna(0.0)
    return oriented


def _mean_available(signals, names):
    available = [signals[name] for name in names if name in signals]
    if not available:
        return None
    return sum(available) / float(len(available))


def _candidate_families(commodity, selected_signals):
    if commodity == "SOYABEAN":
        family_definitions = {
            "price": ["given_mom_20", "given_mom_60", "given_rev_5", "given_price_family"],
            "physical": [
                "given_inventory_pressure",
                "given_cgl_inventory_pressure",
                "given_crush_pressure",
                "given_curve_tightness",
                "given_physical_family",
            ],
            "external_crush": ["external_crush_family"],
            "fx_export": ["external_fx_export_family"],
            "weather": ["external_weather_hdd_cdd_family"],
            "macro": ["external_macro_risk_family", "external_relative_grain_family"],
        }
    else:
        family_definitions = {
            "price": ["given_mom_20", "given_mom_60", "given_rev_5", "given_price_family"],
            "physical": [
                "given_inventory_pressure",
                "given_cgl_inventory_pressure",
                "given_cgl_crush_activity",
                "given_curve_tightness",
                "given_physical_family",
            ],
            "ethanol": ["external_ethanol_family"],
            "fx_export": ["external_fx_export_family"],
            "weather": ["external_weather_hdd_cdd_family"],
            "macro": ["external_macro_risk_family", "external_relative_grain_family"],
        }

    families = {}
    family_members = {}
    for family, names in family_definitions.items():
        signal = _mean_available(selected_signals, names)
        if signal is not None:
            families[family] = signal
            family_members[family] = [name for name in names if name in selected_signals]
    return families, family_members


def _candidate_composites(commodity, families):
    definitions = {
        "selected_all_equal": list(families.keys()),
        "physical_only": ["physical"],
        "price_physical_equal": ["price", "physical"],
        "physical_fx_equal": ["physical", "fx_export"],
        "physical_weather_equal": ["physical", "weather"],
        "physical_macro_equal": ["physical", "macro"],
    }
    if commodity == "SOYABEAN":
        definitions.update(
            {
                "physical_extcrush_equal": ["physical", "external_crush"],
                "physical_extcrush_fx_equal": ["physical", "external_crush", "fx_export"],
                "physical_extcrush_weather_equal": ["physical", "external_crush", "weather"],
                "physical_extcrush_fx_weather_equal": ["physical", "external_crush", "fx_export", "weather"],
            }
        )
    else:
        definitions.update(
            {
                "physical_ethanol_equal": ["physical", "ethanol"],
                "physical_ethanol_fx_equal": ["physical", "ethanol", "fx_export"],
                "physical_ethanol_weather_equal": ["physical", "ethanol", "weather"],
                "physical_ethanol_fx_weather_equal": ["physical", "ethanol", "fx_export", "weather"],
            }
        )

    candidates = {}
    candidate_members = {}
    for candidate, family_names in definitions.items():
        used = [families[name] for name in family_names if name in families]
        if not used:
            continue
        if candidate != "selected_all_equal" and len(used) != len(family_names):
            continue
        candidates[candidate] = sum(used) / float(len(used))
        candidate_members[candidate] = [name for name in family_names if name in families]
    return candidates, candidate_members


def _metrics(bt):
    table = split_performance(bt, TEST_START)
    train_val = split_performance(bt.loc[bt.index < TEST_START], TRAIN_END)
    return {
        "train_sharpe": train_val.loc["sharpe", "in_sample"],
        "validation_sharpe": train_val.loc["sharpe", "out_of_sample"],
        "validation_max_drawdown": train_val.loc["max_drawdown", "out_of_sample"],
        "test_sharpe": table.loc["sharpe", "out_of_sample"],
        "test_pnl": table.loc["total_pnl", "out_of_sample"],
        "test_max_drawdown": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "max_drawdown": table.loc["max_drawdown", "full_period"],
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
    }


def _evaluate_candidate(name, signal, futures_pnl, commodity, mode):
    positions = _positions_from_signal(signal, futures_pnl, commodity, mode)
    rows = []
    backtests = {}
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[commodity]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[commodity]], 0.0)
        row = {"candidate": name, "mode": mode, "cost_adjusted": cost_adjusted}
        row.update(_metrics(bt))
        rows.append(row)
        backtests[name + "_" + mode + ("_cost" if cost_adjusted else "_zero")] = bt
    return rows, backtests


def _candidate_selection_table(candidates, candidate_members, futures_pnl, commodity):
    masks = _split_masks(futures_pnl.index)
    rows = []
    backtests = {}
    modes = ["long_only", "long_short"]
    for name, signal in candidates.items():
        for mode in modes:
            candidate_signal = signal.reindex(futures_pnl.index).fillna(0.0)
            if mode == "long_only":
                ic_signal = candidate_signal.clip(lower=0.0)
            else:
                ic_signal = candidate_signal
            row = {
                "candidate": name,
                "mode": mode,
                "families": ",".join(candidate_members[name]),
            }
            for split_name, mask in masks.items():
                row[split_name + "_ic"] = _ic(
                    ic_signal,
                    futures_pnl,
                    commodity,
                    pd.Series(mask, index=futures_pnl.index),
                )
            eligible = (
                pd.notnull(row["train_ic"])
                and pd.notnull(row["validation_ic"])
                and row["train_ic"] >= IC_THRESHOLD
                and row["validation_ic"] >= VALIDATION_IC_FLOOR
            )
            row["eligible"] = bool(eligible)
            row["selection_score"] = (
                row["validation_ic"] + 0.25 * row["train_ic"] if eligible else -np.inf
            )
            rows.append(row)
            metric_rows, new_bt = _evaluate_candidate(name, signal, futures_pnl, commodity, mode)
            for metric_row in metric_rows:
                metric_row["families"] = row["families"]
                metric_row["train_ic"] = row["train_ic"]
                metric_row["validation_ic"] = row["validation_ic"]
                metric_row["test_ic"] = row["test_ic"]
                metric_row["eligible"] = row["eligible"]
                metric_row["selection_score"] = row["selection_score"]
            backtests.update(new_bt)
            rows.extend(metric_rows)
    table = pd.DataFrame([row for row in rows if "cost_adjusted" not in row])
    results = pd.DataFrame([row for row in rows if "cost_adjusted" in row])
    eligible = table.loc[table["eligible"]].copy()
    if eligible.empty:
        selected = table.sort_values(["validation_ic", "train_ic"], ascending=[False, False]).iloc[0]
    else:
        selected = eligible.sort_values(["selection_score", "validation_ic"], ascending=[False, False]).iloc[0]
    return selected, table, results, backtests


def _given_signal_universe(feature_panels, commodity):
    panel = feature_panels[commodity]
    inventory_pressure = (
        -panel["public_inventory_change"] - panel["receipts_change"] - panel["cgl_inventory_change"]
    ) / 3.0
    curve_tightness = (panel["curve_spread"] + panel["curve_ratio"]) / 2.0
    price_family = (panel["mom_20"] + panel["mom_60"] + panel["rev_5"]) / 3.0
    trend = (panel["mom_20"] + panel["mom_60"] + panel["curve_spread"] + panel["cot_pm_oi_level"]) / 4.0
    signals = {
        "given_mom_20": panel["mom_20"],
        "given_mom_60": panel["mom_60"],
        "given_rev_5": panel["rev_5"],
        "given_curve_spread": panel["curve_spread"],
        "given_curve_ratio": panel["curve_ratio"],
        "given_cot_pm_oi_level": panel["cot_pm_oi_level"],
        "given_inventory_pressure": inventory_pressure,
        "given_cgl_inventory_pressure": -panel["cgl_inventory_change"],
        "given_curve_tightness": curve_tightness,
        "given_price_family": price_family,
        "given_trend": trend,
    }
    if commodity == "SOYABEAN":
        crush_pressure = (panel["crush_surprise"] + panel["crush_utilization"]) / 2.0
        signals["given_crush_pressure"] = crush_pressure
        signals["given_physical_family"] = (inventory_pressure + crush_pressure + curve_tightness) / 3.0
    else:
        cgl_crush_activity = (panel["crush_surprise"] + panel["crush_utilization"]) / 2.0
        signals["given_cgl_crush_activity"] = cgl_crush_activity
        signals["given_physical_family"] = (inventory_pressure + curve_tightness + 0.25 * cgl_crush_activity) / 2.25
    return {name: signal.fillna(0.0) for name, signal in signals.items()}


def _fetch_external_signals(commodity, futures_pnl):
    errors = []
    external = {}
    prices = pd.DataFrame()
    weather = pd.DataFrame()
    ethanol = pd.DataFrame()
    try:
        if commodity == "SOYABEAN":
            prices = _download_soy_yfinance(futures_pnl.index.min(), futures_pnl.index.max())
            external.update(_soy_external_yfinance_families(prices, futures_pnl.index))
        else:
            prices = _download_corn_yfinance(futures_pnl.index.min(), futures_pnl.index.max())
            external.update(_corn_external_yfinance_families(prices, futures_pnl.index))
    except Exception as exc:
        errors.append("yfinance: {}".format(exc))

    try:
        weather = fetch_meteostat_weather(futures_pnl.index.min(), futures_pnl.index.max())
        if commodity == "SOYABEAN":
            external.update(_soy_weather_family(weather, futures_pnl.index))
        else:
            external.update(_corn_weather_family(weather, futures_pnl.index))
    except Exception as exc:
        errors.append("meteostat: {}".format(exc))

    if commodity == "CORN":
        try:
            ethanol = fetch_eia_ethanol()
            ethanol_features = build_ethanol_feature_panel(ethanol, futures_pnl.index)
            external.update(_ethanol_family(ethanol_features))
        except Exception as exc:
            errors.append("eia_ethanol: {}".format(exc))

    external = {name: _clean_signal(signal, futures_pnl.index) for name, signal in external.items()}
    return external, errors, {"prices": prices, "weather": weather, "ethanol": ethanol}


def _run_one_commodity(commodity, feature_panels, futures_pnl):
    given = _given_signal_universe(feature_panels, commodity)
    external, errors, raw_external = _fetch_external_signals(commodity, futures_pnl)
    signals = dict(given)
    signals.update(external)
    signals = {name: _clean_signal(signal, futures_pnl.index) for name, signal in signals.items()}
    ic_table = _signal_ic_table(signals, futures_pnl, commodity)
    selected_signals = _orient_and_filter_signals(signals, ic_table, futures_pnl)
    families, family_members = _candidate_families(commodity, selected_signals)
    candidates, candidate_members = _candidate_composites(commodity, families)
    if not candidates:
        raise RuntimeError("No IC-selected family candidates for {}".format(commodity))
    selected, selection_table, results, backtests = _candidate_selection_table(
        candidates,
        candidate_members,
        futures_pnl,
        commodity,
    )
    selected_results = results.loc[
        (results["candidate"] == selected["candidate"]) & (results["mode"] == selected["mode"])
    ].copy()
    zero = results.loc[~results["cost_adjusted"]].copy()
    robust_pool = zero.loc[
        zero["eligible"]
        & (zero["train_sharpe"] > 0.0)
        & (zero["validation_sharpe"] > 0.0)
        & (zero["validation_max_drawdown"] > -250.0)
    ].copy()
    if robust_pool.empty:
        robust_selected = selected
    else:
        robust_pool["robust_score"] = (
            robust_pool["validation_sharpe"]
            + 0.25 * robust_pool["train_sharpe"]
            + 0.001 * robust_pool["validation_max_drawdown"]
        )
        robust_selected = robust_pool.sort_values(
            ["robust_score", "validation_sharpe", "validation_ic"],
            ascending=[False, False, False],
        ).iloc[0]
    robust_results = results.loc[
        (results["candidate"] == robust_selected["candidate"]) & (results["mode"] == robust_selected["mode"])
    ].copy()
    return {
        "commodity": commodity,
        "errors": errors,
        "ic_table": ic_table.reset_index(drop=True),
        "selected_signal_names": list(selected_signals.keys()),
        "families": families,
        "family_members": family_members,
        "candidate_members": candidate_members,
        "selection_table": selection_table.reset_index(drop=True),
        "results": results.sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False]).reset_index(drop=True),
        "selected": selected,
        "selected_results": selected_results.sort_values("cost_adjusted").reset_index(drop=True),
        "robust_selected": robust_selected,
        "robust_results": robust_results.sort_values("cost_adjusted").reset_index(drop=True),
        "backtests": backtests,
        "raw_external": raw_external,
    }


def _format_table(df, float_format="{:.3f}".format, max_rows=None):
    if max_rows is not None:
        df = df.head(max_rows)
    return df.to_string(index=False, float_format=float_format)


def write_ic_threshold_log(outputs, path="notes/ic_threshold_corn_soybean.txt"):
    lines = []
    lines.append("IC-threshold sleeve selection experiments")
    lines.append("Date: 2026-05-02")
    lines.append("")
    lines.append("Method")
    lines.append("------")
    lines.append("- Signal IC is Spearman correlation of signal_t versus next-day futures PnL.")
    lines.append("- Signals pass if abs(train IC) >= {:.3f} before 2016.".format(IC_THRESHOLD))
    lines.append("- Passed signals are oriented by train IC sign, then grouped into economic families.")
    lines.append("- Family candidates are fixed equal-weight composites.")
    lines.append("- Candidate selection uses validation IC from 2016-2017 only.")
    lines.append("- 2018-2020 is reported as the untouched OOS period.")
    lines.append("- Backtests are also cost-adjusted with trade and holding costs.")
    lines.append("")
    for name, out in outputs.items():
        lines.append("")
        lines.append("{} results".format(name))
        lines.append("=" * (len(name) + 8))
        if out["errors"]:
            lines.append("External data warnings: {}".format("; ".join(out["errors"])))
        else:
            lines.append("External data warnings: none")
        lines.append("")
        lines.append("Signals passing IC threshold")
        lines.append("----------------------------")
        passed_cols = ["signal", "train_ic", "validation_ic", "test_ic", "passes_ic_threshold"]
        passed = out["ic_table"].loc[out["ic_table"]["passes_ic_threshold"], passed_cols]
        lines.append(_format_table(passed.sort_values("train_ic", ascending=False)))
        lines.append("")
        lines.append("Family members after IC filtering")
        lines.append("---------------------------------")
        for family, members in out["family_members"].items():
            lines.append("- {}: {}".format(family, ", ".join(members)))
        lines.append("")
        lines.append("Candidate selection by validation IC")
        lines.append("------------------------------------")
        sel_cols = [
            "candidate",
            "mode",
            "families",
            "eligible",
            "selection_score",
            "train_ic",
            "validation_ic",
            "test_ic",
        ]
        lines.append(_format_table(out["selection_table"][sel_cols].sort_values("selection_score", ascending=False)))
        lines.append("")
        lines.append("Cost-adjusted OOS performance")
        lines.append("-----------------------------")
        perf_cols = [
            "candidate",
            "mode",
            "cost_adjusted",
            "train_ic",
            "validation_ic",
            "test_ic",
            "test_sharpe",
            "test_pnl",
            "test_max_drawdown",
            "full_sharpe",
            "max_drawdown",
        ]
        cost = out["results"].loc[out["results"]["cost_adjusted"], perf_cols]
        lines.append(_format_table(cost.sort_values("test_sharpe", ascending=False)))
        lines.append("")
        selected = out["selected"]
        lines.append("Selected by validation IC")
        lines.append("-------------------------")
        lines.append(
            "- {} {} families=[{}] train_IC={:.3f} validation_IC={:.3f} test_IC={:.3f}".format(
                selected["candidate"],
                selected["mode"],
                selected["families"],
                selected["train_ic"],
                selected["validation_ic"],
                selected["test_ic"],
            )
        )
        lines.append(_format_table(out["selected_results"][perf_cols]))
        lines.append("")
        robust = out["robust_selected"]
        lines.append("IC + validation Sharpe/DD sanity selection")
        lines.append("------------------------------------------")
        lines.append(
            "- {} {} families=[{}] train_IC={:.3f} validation_IC={:.3f} test_IC={:.3f}".format(
                robust["candidate"],
                robust["mode"],
                robust["families"],
                robust["train_ic"],
                robust["validation_ic"],
                robust["test_ic"],
            )
        )
        lines.append(
            "- Sanity gates: candidate passed train IC threshold, train Sharpe > 0, validation Sharpe > 0, validation DD > -250."
        )
        lines.append(_format_table(out["robust_results"][perf_cols]))
        lines.append("")
        lines.append("Overfit read")
        lines.append("------------")
        best_cost = cost.sort_values("test_sharpe", ascending=False).iloc[0]
        selected_cost = out["selected_results"].loc[out["selected_results"]["cost_adjusted"]].iloc[0]
        robust_cost = out["robust_results"].loc[out["robust_results"]["cost_adjusted"]].iloc[0]
        lines.append(
            "- Best OOS cost-adjusted row was {} {} with Sharpe {:.3f} and DD {:.3f}.".format(
                best_cost["candidate"],
                best_cost["mode"],
                best_cost["test_sharpe"],
                best_cost["test_max_drawdown"],
            )
        )
        lines.append(
            "- Validation-selected row had OOS Sharpe {:.3f} and DD {:.3f}.".format(
                selected_cost["test_sharpe"],
                selected_cost["test_max_drawdown"],
            )
        )
        lines.append(
            "- IC+Sharpe/DD sanity row had OOS Sharpe {:.3f} and DD {:.3f}.".format(
                robust_cost["test_sharpe"],
                robust_cost["test_max_drawdown"],
            )
        )
        if selected_cost["test_sharpe"] < best_cost["test_sharpe"] - 0.25:
            lines.append("- Gap is large, so candidate selection remains a research-overfit risk.")
        else:
            lines.append("- Gap is moderate/small, so the IC selection is comparatively stable.")
        lines.append("")

    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as handle:
        handle.write("\n".join(lines))


def run_ic_threshold_sleeve_experiment(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    outputs = {
        "Corn": _run_one_commodity("CORN", feature_panels, futures_pnl),
        "Soybeans": _run_one_commodity("SOYABEAN", feature_panels, futures_pnl),
    }
    write_ic_threshold_log(outputs)
    return outputs


In [ ]:
"""Walk-forward OLS/Ridge/RLS/Kalman tests for corn and soybeans.

The goal is to test whether simple fitted linear models improve the
commodity-specific sleeves without using future information.

Controls:
- no full-sample fitting;
- no OOS hyperparameter search;
- expanding walk-forward OLS/Ridge with fixed alpha;
- online RLS/Kalman updates with fixed parameters;
- predictions at date t use only data available before date t;
- prediction signals are converted to rolling z-scores using lagged history.
"""


import os

import numpy as np
import pandas as pd



COMMODITIES = ["CORN", "SOYABEAN"]
MIN_TRAIN_OBS = 504
REFIT_EVERY = 21
RIDGE_ALPHA = 100.0
RLS_LAMBDA = 0.995
RLS_DELTA = 100.0
KALMAN_Q = 1.0e-5


def _feature_matrix(feature_panels, futures_pnl, commodity):
    given = _given_signal_universe(feature_panels, commodity)
    external, errors, _ = _fetch_external_signals(commodity, futures_pnl)
    signals = dict(given)
    signals.update(external)
    cleaned = {
        name: _clean_signal(signal, futures_pnl.index)
        for name, signal in signals.items()
    }
    x = pd.DataFrame(cleaned, index=futures_pnl.index).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    # Drop constant or near-empty columns so linear algebra stays well behaved.
    keep = [col for col in x.columns if x[col].std() > 1.0e-8 and x[col].abs().sum() > 0.0]
    return x[keep], errors


def _standardize_train_apply(x_train, x_row):
    mean = x_train.mean(axis=0)
    std = x_train.std(axis=0).replace(0.0, np.nan).fillna(1.0)
    return (x_train - mean) / std, (x_row - mean) / std


def _fit_linear_beta(x_train, y_train, alpha=0.0):
    x = np.asarray(x_train, dtype=float)
    y = np.asarray(y_train, dtype=float)
    x = np.column_stack([np.ones(len(x)), x])
    xtx = x.T.dot(x)
    if alpha > 0.0:
        penalty = np.eye(xtx.shape[0]) * float(alpha)
        penalty[0, 0] = 0.0
        xtx = xtx + penalty
    try:
        return np.linalg.solve(xtx, x.T.dot(y))
    except np.linalg.LinAlgError:
        return np.linalg.pinv(xtx).dot(x.T).dot(y)


def _expanding_predictions(x, y, alpha=0.0):
    preds = pd.Series(np.nan, index=x.index)
    beta = None
    last_fit = None
    for i, date in enumerate(x.index):
        train_mask = (x.index < date) & y.notna()
        if train_mask.sum() < MIN_TRAIN_OBS:
            continue
        if beta is None or last_fit is None or (i - last_fit) >= REFIT_EVERY:
            x_train_raw = x.loc[train_mask]
            y_train = y.loc[train_mask]
            x_train, x_row = _standardize_train_apply(x_train_raw, x.loc[date])
            beta = _fit_linear_beta(x_train, y_train, alpha=alpha)
            last_fit = i
            preds.loc[date] = np.r_[1.0, np.asarray(x_row, dtype=float)].dot(beta)
        else:
            x_train_raw = x.loc[train_mask]
            _, x_row = _standardize_train_apply(x_train_raw, x.loc[date])
            preds.loc[date] = np.r_[1.0, np.asarray(x_row, dtype=float)].dot(beta)
    return preds


def _rls_predictions(x, y, lambda_=RLS_LAMBDA, delta=RLS_DELTA):
    columns = list(x.columns)
    beta = np.zeros(len(columns) + 1)
    p = np.eye(len(beta)) * float(delta)
    preds = pd.Series(np.nan, index=x.index)
    mean = pd.Series(0.0, index=columns)
    var = pd.Series(1.0, index=columns)
    n = 0
    for date in x.index:
        row = x.loc[date]
        if n > 20:
            std = np.sqrt(var.clip(lower=1.0e-8))
            z = ((row - mean) / std).clip(lower=-5.0, upper=5.0)
            phi = np.r_[1.0, np.asarray(z, dtype=float)]
            preds.loc[date] = phi.dot(beta)
        y_value = y.loc[date]
        if pd.notnull(y_value):
            n += 1
            old_mean = mean.copy()
            mean = mean + (row - mean) / float(n)
            var = ((n - 2.0) / max(n - 1.0, 1.0)) * var + ((row - old_mean) * (row - mean)) / max(n - 1.0, 1.0)
            if n > MIN_TRAIN_OBS:
                std = np.sqrt(var.clip(lower=1.0e-8))
                z = ((row - mean) / std).clip(lower=-5.0, upper=5.0)
                phi = np.r_[1.0, np.asarray(z, dtype=float)]
                denom = float(lambda_ + phi.dot(p).dot(phi))
                gain = p.dot(phi) / denom
                err = float(y_value - phi.dot(beta))
                beta = beta + gain * err
                p = (p - np.outer(gain, phi).dot(p)) / float(lambda_)
    return preds


def _kalman_predictions(x, y, q=KALMAN_Q):
    columns = list(x.columns)
    beta = np.zeros(len(columns) + 1)
    p = np.eye(len(beta)) * 10.0
    preds = pd.Series(np.nan, index=x.index)
    mean = pd.Series(0.0, index=columns)
    var = pd.Series(1.0, index=columns)
    target_var = 1.0
    n = 0
    for date in x.index:
        row = x.loc[date]
        if n > MIN_TRAIN_OBS:
            std = np.sqrt(var.clip(lower=1.0e-8))
            z = ((row - mean) / std).clip(lower=-5.0, upper=5.0)
            phi = np.r_[1.0, np.asarray(z, dtype=float)]
            preds.loc[date] = phi.dot(beta)
        y_value = y.loc[date]
        if pd.notnull(y_value):
            n += 1
            old_mean = mean.copy()
            mean = mean + (row - mean) / float(n)
            var = ((n - 2.0) / max(n - 1.0, 1.0)) * var + ((row - old_mean) * (row - mean)) / max(n - 1.0, 1.0)
            target_var = target_var + (float(y_value) ** 2 - target_var) / float(n)
            if n > MIN_TRAIN_OBS:
                std = np.sqrt(var.clip(lower=1.0e-8))
                z = ((row - mean) / std).clip(lower=-5.0, upper=5.0)
                phi = np.r_[1.0, np.asarray(z, dtype=float)]
                p = p + np.eye(len(beta)) * float(q)
                r = max(target_var, 1.0)
                innovation_var = float(phi.dot(p).dot(phi) + r)
                gain = p.dot(phi) / innovation_var
                err = float(y_value - phi.dot(beta))
                beta = beta + gain * err
                p = p - np.outer(gain, phi).dot(p)
    return preds


def _prediction_to_signal(pred):
    pred = pred.replace([np.inf, -np.inf], np.nan)
    mean = pred.rolling(252, min_periods=60).mean().shift(1)
    std = pred.rolling(252, min_periods=60).std().shift(1).replace(0.0, np.nan)
    signal = ((pred - mean) / std).clip(lower=-5.0, upper=5.0).fillna(0.0)
    return signal


def _metrics(bt):
    table = split_performance(bt, TEST_START)
    train_val = split_performance(bt.loc[bt.index < TEST_START], TRAIN_END)
    return {
        "train_sharpe": train_val.loc["sharpe", "in_sample"],
        "validation_sharpe": train_val.loc["sharpe", "out_of_sample"],
        "validation_max_drawdown": train_val.loc["max_drawdown", "out_of_sample"],
        "test_sharpe": table.loc["sharpe", "out_of_sample"],
        "test_pnl": table.loc["total_pnl", "out_of_sample"],
        "test_max_drawdown": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "max_drawdown": table.loc["max_drawdown", "full_period"],
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
    }


def _evaluate_model_signal(model, commodity, signal, futures_pnl, mode="long_short"):
    positions = _positions_from_signal(signal, futures_pnl, commodity, mode=mode)
    rows = []
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[commodity]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[commodity]], 0.0)
        row = {"commodity": commodity, "model": model, "mode": mode, "cost_adjusted": cost_adjusted}
        row.update(_metrics(bt))
        rows.append(row)
    return rows


def _run_one(commodity, feature_panels, futures_pnl):
    x, errors = _feature_matrix(feature_panels, futures_pnl, commodity)
    y = futures_pnl[commodity].shift(-1)
    raw_predictions = {
        "ols_expanding": _expanding_predictions(x, y, alpha=0.0),
        "ridge_expanding_alpha100": _expanding_predictions(x, y, alpha=RIDGE_ALPHA),
        "rls_lambda0995": _rls_predictions(x, y),
        "kalman_dynamic_linear": _kalman_predictions(x, y),
    }
    rows = []
    signals = {}
    for model, pred in raw_predictions.items():
        signal = _prediction_to_signal(pred)
        signals[model] = signal
        for mode in ["long_only", "long_short"]:
            rows.extend(_evaluate_model_signal(model, commodity, signal, futures_pnl, mode=mode))
    results = pd.DataFrame(rows).sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False]).reset_index(drop=True)
    return {
        "commodity": commodity,
        "errors": errors,
        "features": list(x.columns),
        "signals": signals,
        "raw_predictions": raw_predictions,
        "results": results,
    }


def _write_log(outputs, path="notes/linear_online_models_corn_soybean.txt"):
    lines = []
    lines.append("OLS / Ridge / RLS / Kalman commodity-sleeve test")
    lines.append("Date: 2026-05-02")
    lines.append("")
    lines.append("Controls")
    lines.append("--------")
    lines.append("- Expanding OLS and Ridge refit every {} trading days.".format(REFIT_EVERY))
    lines.append("- Ridge alpha is fixed at {:.1f}; it is not tuned on OOS.".format(RIDGE_ALPHA))
    lines.append("- RLS uses fixed lambda {:.3f} and delta {:.1f}.".format(RLS_LAMBDA, RLS_DELTA))
    lines.append("- Kalman uses fixed process noise q {:.1e}.".format(KALMAN_Q))
    lines.append("- Prediction at date t uses only observations before or at date t, then positions trade next-day PnL via the standard backtester.")
    lines.append("- Predictions are converted to lagged rolling z-score signals before volatility targeting.")
    lines.append("")
    for label, out in outputs.items():
        lines.append("")
        lines.append("{} results".format(label))
        lines.append("=" * (len(label) + 8))
        lines.append("External data warnings: {}".format("; ".join(out["errors"]) if out["errors"] else "none"))
        lines.append("Feature count: {}".format(len(out["features"])))
        lines.append("Features: {}".format(", ".join(out["features"])))
        lines.append("")
        cols = [
            "model",
            "mode",
            "cost_adjusted",
            "train_sharpe",
            "validation_sharpe",
            "validation_max_drawdown",
            "test_sharpe",
            "test_pnl",
            "test_max_drawdown",
            "full_sharpe",
            "max_drawdown",
            "turnover",
            "avg_gross_exposure",
        ]
        lines.append("All model rows")
        lines.append("--------------")
        lines.append(_format_table(out["results"][cols]))
        lines.append("")
        cost = out["results"].loc[out["results"]["cost_adjusted"]].copy()
        best = cost.sort_values("test_sharpe", ascending=False).iloc[0]
        lines.append("Best cost-adjusted OOS row")
        lines.append("--------------------------")
        lines.append(
            "- {} {}: OOS Sharpe {:.3f}, PnL {:.3f}, DD {:.3f}, full Sharpe {:.3f}, full DD {:.3f}".format(
                best["model"],
                best["mode"],
                best["test_sharpe"],
                best["test_pnl"],
                best["test_max_drawdown"],
                best["full_sharpe"],
                best["max_drawdown"],
            )
        )
        lines.append("")
        lines.append("Overfit read")
        lines.append("------------")
        if best["test_sharpe"] > 0.5 and best["validation_sharpe"] > 0.0:
            lines.append("- Model has positive validation and OOS behavior, but still carries coefficient-estimation risk.")
        elif best["test_sharpe"] > 0.5:
            lines.append("- Model looks good OOS but did not have clean validation support; treat as research luck until further holdout.")
        else:
            lines.append("- No model is strong enough to replace the simpler fixed-family sleeve.")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as handle:
        handle.write("\n".join(lines))


def run_linear_online_model_experiment(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    outputs = {
        "Corn": _run_one("CORN", feature_panels, futures_pnl),
        "Soybeans": _run_one("SOYABEAN", feature_panels, futures_pnl),
    }
    _write_log(outputs)
    return outputs


In [ ]:
linear_audit = safe_run("OLS/Kalman/Ridge comparison", run_linear_online_model_experiment, data_dir=DATA_DIR)
linear_results = cost_only(extract_linear_results(linear_audit))
print("linear_results rows:", len(linear_results))


# Soybeans

Research path:

1. Generic tests: did simple/family/regime/fitted models work?
2. Given-data-only tests: did Cargill crush/inventory matter?
3. External tests: did weather, FX/export, and external crush improve results?
4. Low-volatility tests: did a simple risk-control rule improve drawdown?

## Soybean Questions Before Testing

At this point I do not assume a soybean regime. The next tests ask:

| Question | Test That Can Answer It | What Would Count As Evidence |
|---|---|---|
| Are provided signals enough? | generic avg/family/IC/regime tests and given-data-only Cargill tests | simple/family/given rows have acceptable OOS Sharpe and DD |
| Does soybean need physical/weather information? | external signal tests | Cargill crush/inventory plus weather/physical rows beat generic rows |
| Is there a risk state where the base strategy loses too much? | low-volatility risk-control tests and key-period tables | low-vol or low-price period metrics improve after a fixed exposure cut |

The regime labels are introduced only after these tests show evidence.


## Soybean Step 1 - Generic Tests

In [59]:
soy_generic = family_results.loc[family_results.get("commodity", pd.Series(dtype=str)) == "SOYABEAN"].copy()
soy_generic["regime_interpretation_from_result"] = soy_generic["strategy"].map({
    "avg_all_signals": "Baseline: broad average is diluted; does not yet prove a regime.",
    "equal_family_weight": "Family weighting alone does not reveal a stable soybean regime.",
    "ic_family_selected_physical_public": "IC selection helps diagnose physical relevance, but DD decides if it is usable.",
    "regime_best_family_trend_mr": "If this beats flat rows, generic trend/MR structure may matter for soybean.",
    "regime_avg_families_trend_mr": "Tests whether regime logic works without selecting one best family.",
    "ols_kalman_filter": "Fitted coefficient benchmark; improvement would suggest dynamic weights, but overfit risk is higher.",
    "ridge_expanding_alpha100": "Regularized fitted benchmark; reject if DD/full-period stability is poor.",
}).fillna("Diagnostic row.")
soy_generic[
    [
        "commodity", "strategy", "mode", "cost_adjusted",
        "oos_sharpe", "oos_pnl", "oos_dd",
        "full_sharpe", "full_dd", "regime_interpretation_from_result", "overfit_comment"
    ]
].sort_values("oos_sharpe", ascending=False).reset_index(drop=True)


,commodity,strategy,mode,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,regime_interpretation_from_result,overfit_comment
0,SOYABEAN,regime_best_family_trend_mr,long_short,True,1.005108,515.192793,-223.005657,0.712843,-456.834512,"If this beats flat rows, generic trend/MR structure may matter for soybean.","Moderate selection risk; weights are fixed masks, not optimized."
1,SOYABEAN,ic_family_selected_price,long_short,True,0.721160,667.300247,-684.304850,0.466346,-684.304850,Diagnostic row.,Moderate/high selection risk; validation IC can be noisy.
2,SOYABEAN,regime_avg_families_trend_mr,long_short,True,0.540224,217.241832,-195.988627,0.278737,-680.558931,Tests whether regime logic works without selecting one best family.,"Lower than best-family regime selection, but still regime-research risk."
3,SOYABEAN,ridge_expanding_alpha100,long_short,True,0.515454,561.055406,-698.342168,-0.009193,-1335.926418,Regularized fitted benchmark; reject if DD/full-period stability is poor.,Coefficient-estimation risk; fixed alpha reduces but does not remove it. Positive OOS but keep as diagnostic unless validation/full-period behavior is clean.
4,SOYABEAN,avg_all_signals,long_short,True,0.382216,165.621529,-342.013053,0.048464,-432.775906,Baseline: broad average is diluted; does not yet prove a regime.,"Low model overfit, but can be economically diluted."
5,SOYABEAN,ols_kalman_filter,long_short,True,0.356594,368.968616,-348.029270,-0.007950,-937.145631,"Fitted coefficient benchmark; improvement would suggest dynamic weights, but overfit risk is higher.","Coefficient-estimation risk; parameters are fixed, not OOS tuned. Positive OOS but keep as diagnostic unless validation/full-period behavior is clean."
6,SOYABEAN,equal_family_weight,long_short,True,0.233184,105.882051,-351.767290,-0.034708,-488.597670,Family weighting alone does not reveal a stable soybean regime.,Low model overfit; family choices are researcher-defined.


## Soybean Step 2 - OLS / Kalman / Ridge Benchmark

In [60]:
soy_linear = linear_results.loc[linear_results.get("commodity", pd.Series(dtype=str)) == "SOYABEAN"].copy()
soy_linear[
    [
        "commodity", "model", "mode", "cost_adjusted",
        "test_sharpe", "test_pnl", "test_max_drawdown",
        "full_sharpe", "max_drawdown"
    ]
].sort_values("test_sharpe", ascending=False).reset_index(drop=True)


,commodity,model,mode,cost_adjusted,test_sharpe,test_pnl,test_max_drawdown,full_sharpe,max_drawdown
0,SOYABEAN,kalman_dynamic_linear,long_only,True,0.625633,571.322111,-591.493440,0.388898,-1361.087271
1,SOYABEAN,ridge_expanding_alpha100,long_only,True,0.528554,407.831237,-1056.956612,0.419561,-1508.877071
2,SOYABEAN,ridge_expanding_alpha100,long_short,True,0.470320,703.178965,-892.540076,0.255333,-1578.578637
3,SOYABEAN,kalman_dynamic_linear,long_short,True,0.317111,493.455033,-769.562005,0.265512,-1258.881365
4,SOYABEAN,rls_lambda0995,long_only,True,0.077467,53.685425,-705.625014,0.098484,-1409.131957
5,SOYABEAN,ols_expanding,long_short,True,-0.274629,-194.664725,-519.593836,-0.012837,-1234.974585
6,SOYABEAN,rls_lambda0995,long_short,True,-0.691270,-973.606284,-1818.214951,-0.333858,-3010.749497
7,SOYABEAN,ols_expanding,long_only,True,-0.980100,-318.079107,-488.427463,0.110234,-968.734381


## Soybean Step 3 - Given-Data-Only Cargill Tests

This produces the provided-file soybean baseline rows, including `given_crush_pressure`.

### Direct Code Used Here: Soybean Given + External Signal Experiments

The next cell defines the implemented soybean external/given-data experiment code directly before it is called.


In [ ]:
"""Soybean given-data vs external-signal experiments.

The module keeps two research tracks separate:
1. given-data-only soybean signals from the provided training files;
2. optional external soybean signals from yfinance and Meteostat.

External data is grouped into economic families and tested both individually
and as equal-family-weight composites. The reported test period is 2018-2020.
"""


import numpy as np
import pandas as pd



YF_TICKERS = {
    "soybean": "ZS=F",
    "soymeal": "ZM=F",
    "soyoil": "ZL=F",
    "corn": "ZC=F",
    "wheat": "ZW=F",
    "usd_index": "DX-Y.NYB",
    "brl": "BRL=X",
    "cny": "CNY=X",
    "crude": "CL=F",
    "equity": "SPY",
}
TRAIN_END = "2016-01-01"
VALIDATION_END = "2018-01-01"


def _tanh(series, divisor=2.0):
    return pd.Series(np.tanh(series.astype(float) / float(divisor)), index=series.index)


def _smooth_threshold(series, mode="long_only"):
    signal = _tanh(series).ewm(halflife=2.0, adjust=False, min_periods=1).mean()
    signal[signal.abs() < 0.05] = 0.0
    if mode == "long_only":
        signal = signal.clip(lower=0.0)
    elif mode == "short_only":
        signal = signal.clip(upper=0.0)
    elif mode != "long_short":
        raise ValueError("Unknown mode: {}".format(mode))
    return signal


def _download_yfinance(start, end):
    import yfinance as yf

    tickers = list(YF_TICKERS.values())
    raw = yf.download(
        tickers,
        start=str(pd.Timestamp(start).date()),
        end=str((pd.Timestamp(end) + pd.DateOffset(days=7)).date()),
        auto_adjust=True,
        progress=False,
        threads=False,
    )
    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" in raw.columns.get_level_values(0):
            close = raw["Close"]
        elif "Adj Close" in raw.columns.get_level_values(0):
            close = raw["Adj Close"]
        else:
            close = raw.xs(raw.columns.get_level_values(0)[0], axis=1, level=0)
    else:
        close = raw.to_frame(tickers[0]) if isinstance(raw, pd.Series) else raw
    close = close.rename(columns={ticker: name for name, ticker in YF_TICKERS.items()})
    close = close[[name for name in YF_TICKERS if name in close.columns]]
    close.index = pd.to_datetime(close.index).tz_localize(None)
    return close.sort_index()


def _given_components(feature_panels):
    soy = feature_panels[COMMODITY]
    inventory_pressure = (
        -soy["public_inventory_change"] - soy["receipts_change"] - soy["cgl_inventory_change"]
    ) / 3.0
    crush_pressure = (soy["crush_surprise"] + soy["crush_utilization"]) / 2.0
    trend = (soy["mom_20"] + soy["mom_60"] + soy["curve_spread"] + soy["cot_pm_oi_level"]) / 4.0
    curve_tightness = (soy["curve_spread"] + soy["curve_ratio"]) / 2.0
    price_family = (soy["mom_20"] + soy["mom_60"] + soy["rev_5"]) / 3.0
    physical_family = (inventory_pressure + crush_pressure + curve_tightness) / 3.0
    conservative = build_soybean_signal(feature_panels, "soy_conservative_long_blend")
    return {
        "given_price_family": price_family.fillna(0.0),
        "given_physical_family": physical_family.fillna(0.0),
        "given_trend": trend.fillna(0.0),
        "given_inventory_pressure": inventory_pressure.fillna(0.0),
        "given_crush_pressure": crush_pressure.fillna(0.0),
        "given_conservative_long_blend": conservative.fillna(0.0),
        "given_equal_price_physical": ((price_family + physical_family) / 2.0).fillna(0.0),
    }


def _external_yfinance_families(prices, trading_index):
    px = prices.reindex(trading_index).ffill().shift(1)
    families = {}

    if {"soybean", "soymeal", "soyoil"}.issubset(px.columns):
        soy_dollars = px["soybean"] / 100.0
        crush = 0.022 * px["soymeal"] + 0.11 * px["soyoil"] - soy_dollars
        crush_mom = rolling_zscore(crush.diff(20), 252, 60)
        meal_lead = rolling_zscore(
            px["soymeal"].pct_change(20, fill_method=None) - px["soybean"].pct_change(20, fill_method=None),
            252,
            60,
        )
        oil_lead = rolling_zscore(
            px["soyoil"].pct_change(20, fill_method=None) - px["soybean"].pct_change(20, fill_method=None),
            252,
            60,
        )
        families["external_crush_family"] = ((crush_mom + meal_lead + oil_lead) / 3.0).fillna(0.0)

    if {"soybean", "corn", "wheat"}.issubset(px.columns):
        soy_corn = rolling_zscore((px["soybean"] / px["corn"]).pct_change(20, fill_method=None), 252, 60)
        soy_wheat = rolling_zscore((px["soybean"] / px["wheat"]).pct_change(20, fill_method=None), 252, 60)
        corn_soy_mr = -rolling_zscore((px["corn"] / px["soybean"]).pct_change(20, fill_method=None), 252, 60)
        families["external_relative_grain_family"] = ((soy_corn + soy_wheat + corn_soy_mr) / 3.0).fillna(0.0)

    fx_parts = []
    if "usd_index" in px.columns:
        fx_parts.append(-rolling_zscore(px["usd_index"].pct_change(20, fill_method=None), 252, 60))
    if "brl" in px.columns:
        fx_parts.append(-rolling_zscore(px["brl"].pct_change(20, fill_method=None), 252, 60))
    if "cny" in px.columns:
        fx_parts.append(-rolling_zscore(px["cny"].pct_change(20, fill_method=None), 252, 60))
    if fx_parts:
        families["external_fx_export_family"] = (sum(fx_parts) / float(len(fx_parts))).fillna(0.0)

    macro_parts = []
    if "crude" in px.columns:
        macro_parts.append(rolling_zscore(px["crude"].pct_change(20, fill_method=None), 252, 60))
    if "equity" in px.columns:
        macro_parts.append(rolling_zscore(px["equity"].pct_change(20, fill_method=None), 252, 60))
    if "usd_index" in px.columns:
        macro_parts.append(-rolling_zscore(px["usd_index"].pct_change(60, fill_method=None), 252, 80))
    if macro_parts:
        families["external_macro_risk_family"] = (sum(macro_parts) / float(len(macro_parts))).fillna(0.0)

    out = {}
    for name, values in families.items():
        cleaned = values.clip(lower=-5.0, upper=5.0).replace([np.inf, -np.inf], np.nan)
        if cleaned.notna().sum() > 20 and cleaned.fillna(0.0).abs().sum() > 0.0:
            out[name] = cleaned.fillna(0.0)
    return out


def _external_weather_family(weather, trading_index):
    panels = build_meteostat_feature_panels(weather, trading_index, mode="commodity_seasonal")
    soy = panels[COMMODITY].reindex(trading_index).fillna(0.0)
    candidates = [
        "meteo_cdd_20d_growing",
        "meteo_hdd_20d_growing",
        "meteo_gdd_60d_growing",
        "meteo_heat_stress_20d_growing",
        "meteo_dryness_20d_growing",
        "meteo_dry_cdd_20d_growing",
        "meteo_precip_20d_planting",
        "meteo_dryness_20d_planting",
        "meteo_freeze_stress_5d_harvest",
    ]
    existing = [col for col in candidates if col in soy.columns]
    if not existing:
        return {}
    family = soy[existing].mean(axis=1).fillna(0.0)
    return {"external_weather_hdd_cdd_family": family.clip(lower=-5.0, upper=5.0)}


def _positions_from_signal(signal, futures_pnl, mode="long_only"):
    cleaned = _smooth_threshold(signal.reindex(futures_pnl.index).fillna(0.0), mode=mode)
    return signal_to_soybean_positions(cleaned, futures_pnl, target_daily_pnl_vol=75.0, max_lot=0.50, mode="long_short")


def _metrics(bt):
    table = split_performance(bt, TEST_START)
    train_val = split_performance(bt.loc[bt.index < pd.Timestamp(TEST_START)], TRAIN_END)
    if table.empty or "sharpe" not in table.index:
        return {
            "train_sharpe": np.nan,
            "validation_sharpe": np.nan,
            "validation_max_drawdown": np.nan,
            "test_sharpe": np.nan,
            "test_pnl": 0.0,
            "test_max_drawdown": np.nan,
            "full_sharpe": np.nan,
            "full_pnl": 0.0,
            "max_drawdown": np.nan,
            "turnover": 0.0,
            "avg_gross_exposure": 0.0,
        }
    return {
        "train_sharpe": train_val.loc["sharpe", "in_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_sharpe": train_val.loc["sharpe", "out_of_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_max_drawdown": train_val.loc["max_drawdown", "out_of_sample"]
        if "max_drawdown" in train_val.index
        else np.nan,
        "test_sharpe": table.loc["sharpe", "out_of_sample"],
        "test_pnl": table.loc["total_pnl", "out_of_sample"],
        "test_max_drawdown": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "max_drawdown": table.loc["max_drawdown", "full_period"],
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
    }


def _evaluate_signal(name, family, signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    rows = []
    backtests = {}
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[COMMODITY]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
        row = {
            "experiment": name,
            "family": family,
            "mode": mode,
            "cost_adjusted": cost_adjusted,
        }
        row.update(_metrics(bt))
        rows.append(row)
        backtests[name + ("_cost_adjusted" if cost_adjusted else "_zero_cost")] = bt
    return rows, backtests, positions


def _zero_cost_metric_for_signal(signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
    return _metrics(bt)


def _passes_pre2018(metrics):
    return (
        pd.notnull(metrics.get("train_sharpe"))
        and pd.notnull(metrics.get("validation_sharpe"))
        and metrics["train_sharpe"] > 0.0
        and metrics["validation_sharpe"] > 0.0
        and pd.notnull(metrics.get("validation_max_drawdown"))
        and metrics["validation_max_drawdown"] > -250.0
    )


def _build_overfit_controlled_signal(given, external, futures_pnl):
    """Select family membership using only train/validation performance."""
    mandatory = {
        "given_physical_family": given["given_physical_family"],
    }
    optional = {
        name: external[name]
        for name in [
            "external_crush_family",
            "external_fx_export_family",
            "external_weather_hdd_cdd_family",
        ]
        if name in external
    }
    selection_rows = []
    selected = dict(mandatory)

    for name, signal in optional.items():
        metrics = _zero_cost_metric_for_signal(signal, futures_pnl, mode="long_only")
        keep = _passes_pre2018(metrics)
        row = {"family": name, "selected": bool(keep)}
        row.update(metrics)
        selection_rows.append(row)
        if keep:
            selected[name] = signal

    combined = sum(selected.values()) / float(len(selected))
    return combined, selected, pd.DataFrame(selection_rows)


def _weighted_sum(families, weights):
    total = 0.0
    used_weight = 0.0
    for name, weight in weights.items():
        if name in families and float(weight) != 0.0:
            total = total + float(weight) * families[name]
            used_weight += float(weight)
    if used_weight == 0.0:
        raise ValueError("No available families in weights: {}".format(weights))
    return total / used_weight


def _build_weight_relaxed_pre2018_signal(given, external, futures_pnl):
    """Choose fixed family weights using only pre-2018 validation data."""
    available = {
        "given_physical_family": given["given_physical_family"],
    }
    for name in [
        "external_crush_family",
        "external_fx_export_family",
        "external_weather_hdd_cdd_family",
    ]:
        if name in external:
            available[name] = external[name]

    candidates = [
        (
            "physical_only",
            {
                "given_physical_family": 1.00,
            },
        ),
        (
            "physical_70_fx_30",
            {
                "given_physical_family": 0.70,
                "external_fx_export_family": 0.30,
            },
        ),
        (
            "physical_50_fx_50",
            {
                "given_physical_family": 0.50,
                "external_fx_export_family": 0.50,
            },
        ),
        (
            "physical_30_fx_70",
            {
                "given_physical_family": 0.30,
                "external_fx_export_family": 0.70,
            },
        ),
        (
            "physical_70_weather_30",
            {
                "given_physical_family": 0.70,
                "external_weather_hdd_cdd_family": 0.30,
            },
        ),
        (
            "physical_70_extcrush_30",
            {
                "given_physical_family": 0.70,
                "external_crush_family": 0.30,
            },
        ),
        (
            "physical_50_fx_25_weather_25",
            {
                "given_physical_family": 0.50,
                "external_fx_export_family": 0.25,
                "external_weather_hdd_cdd_family": 0.25,
            },
        ),
        (
            "physical_50_fx_25_extcrush_25",
            {
                "given_physical_family": 0.50,
                "external_fx_export_family": 0.25,
                "external_crush_family": 0.25,
            },
        ),
        (
            "physical_40_fx_20_extcrush_20_weather_20",
            {
                "given_physical_family": 0.40,
                "external_fx_export_family": 0.20,
                "external_crush_family": 0.20,
                "external_weather_hdd_cdd_family": 0.20,
            },
        ),
    ]

    rows = []
    candidate_signals = {}
    for name, weights in candidates:
        usable = {family: weight for family, weight in weights.items() if family in available}
        if "given_physical_family" not in usable:
            continue
        signal = _weighted_sum(available, usable)
        metrics = _zero_cost_metric_for_signal(signal, futures_pnl, mode="long_only")
        score = -np.inf
        if (
            pd.notnull(metrics.get("train_sharpe"))
            and pd.notnull(metrics.get("validation_sharpe"))
            and metrics["train_sharpe"] > 0.0
            and metrics["validation_sharpe"] > 0.0
        ):
            score = (
                metrics["validation_sharpe"]
                + 0.25 * metrics["train_sharpe"]
                + 0.001 * metrics["validation_max_drawdown"]
            )
        row = {
            "candidate": name,
            "eligible": bool(np.isfinite(score)),
            "score": score,
            "weights": str(usable),
        }
        row.update(metrics)
        rows.append(row)
        candidate_signals[name] = signal

    table = pd.DataFrame(rows)
    eligible = table.loc[table["eligible"]].copy()
    drawdown_eligible = table.loc[
        table["eligible"] & (table["validation_sharpe"] >= 0.50) & (table["train_sharpe"] > 0.0)
    ].copy()

    if eligible.empty:
        selected_name = "physical_only"
    else:
        selected_name = eligible.sort_values(["score", "validation_max_drawdown"], ascending=[False, False]).iloc[0][
            "candidate"
        ]

    if drawdown_eligible.empty:
        drawdown_selected_name = selected_name
    else:
        drawdown_selected_name = drawdown_eligible.sort_values(
            ["validation_max_drawdown", "validation_sharpe"],
            ascending=[False, False],
        ).iloc[0]["candidate"]

    return (
        candidate_signals[selected_name],
        selected_name,
        candidate_signals[drawdown_selected_name],
        drawdown_selected_name,
        table,
        candidate_signals,
    )


def _soybean_regime_frames(given, external, futures_pnl):
    soy_pnl = futures_pnl[COMMODITY].fillna(0.0)
    vol = soy_pnl.rolling(60, min_periods=20).std().shift(1)
    lt_vol = vol.expanding(min_periods=252).median().shift(1)
    high_threshold = vol.expanding(min_periods=252).quantile(0.75).shift(1)
    low_vol = (vol < 0.80 * lt_vol).fillna(False)
    high_vol = ((vol > 1.20 * lt_vol) | (vol > high_threshold)).fillna(False)
    trend_positive = (given["given_trend"] > 0.0).reindex(futures_pnl.index).fillna(False)
    return {
        "vol": vol,
        "lt_vol": lt_vol,
        "low_vol": low_vol.astype(float),
        "high_vol": high_vol.astype(float),
        "trend_positive": trend_positive.astype(float),
    }


def _mean_available(items):
    values = [series for series in items if series is not None]
    if not values:
        raise ValueError("No series available to average")
    return sum(values) / float(len(values))


def _build_regime_shift_signals(given, external, futures_pnl):
    """Build observable regime-shift candidates with fixed equal-weight sleeves."""
    regimes = _soybean_regime_frames(given, external, futures_pnl)
    source = _mean_available(
        [
            given["given_physical_family"],
            external.get("external_fx_export_family"),
            external.get("external_crush_family"),
            external.get("external_weather_hdd_cdd_family"),
        ]
    )
    physical_weather = _mean_available(
        [
            given["given_physical_family"],
            external.get("external_weather_hdd_cdd_family"),
        ]
    )
    trend_signal = _mean_available(
        [
            given["given_trend"],
            external.get("external_macro_risk_family"),
        ]
    )
    high_vol_signal = _mean_available(
        [
            given["given_trend"],
            external.get("external_fx_export_family"),
            external.get("external_crush_family"),
        ]
    )
    low_vol_signal = _mean_available(
        [
            given["given_physical_family"],
            external.get("external_weather_hdd_cdd_family"),
        ]
    )

    high_vol = regimes["high_vol"]
    low_vol = regimes["low_vol"]
    trend_positive = regimes["trend_positive"]
    trend_w_vol = (0.20 + 0.40 * high_vol).clip(0.20, 0.60)
    trend_w_confirmed = (0.20 + 0.40 * ((high_vol > 0) | (trend_positive > 0)).astype(float)).clip(0.20, 0.60)

    candidates = {
        "regime_hybrid_vol_trend_weight": (1.0 - trend_w_vol) * source + trend_w_vol * trend_signal,
        "regime_hybrid_trend_confirmed_weight": (1.0 - trend_w_confirmed) * source
        + trend_w_confirmed * trend_signal,
        "regime_low_physical_high_trend_switch": high_vol * high_vol_signal + (1.0 - high_vol) * low_vol_signal,
        "regime_low_vol_physical_else_balanced": low_vol * low_vol_signal + (1.0 - low_vol) * source,
        "regime_high_vol_fx_trend_else_physical_weather": high_vol * high_vol_signal
        + (1.0 - high_vol) * physical_weather,
    }
    return {name: signal.clip(lower=-5.0, upper=5.0).fillna(0.0) for name, signal in candidates.items()}, regimes


def run_soybean_given_only_experiment(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    signals = _given_components(feature_panels)
    rows = []
    backtests = {}
    positions = {}
    for name, signal in signals.items():
        mode = "long_only" if name != "given_equal_price_physical" else "long_short"
        new_rows, new_bt, pos = _evaluate_signal(name, "given", signal, futures_pnl, mode=mode)
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions[name] = pos

    results = pd.DataFrame(rows).sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False])
    return {
        "results": results.reset_index(drop=True),
        "signals": signals,
        "positions": positions,
        "backtests": backtests,
    }


def run_soybean_external_signal_experiment(data_dir="train_set", include_weather=True):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    given = _given_components(feature_panels)

    external = {}
    external_errors = []
    try:
        prices = _download_yfinance(futures_pnl.index.min(), futures_pnl.index.max())
        external.update(_external_yfinance_families(prices, futures_pnl.index))
    except Exception as exc:
        prices = pd.DataFrame()
        external_errors.append("yfinance: {}".format(exc))

    weather = pd.DataFrame()
    if include_weather:
        try:
            weather = fetch_meteostat_weather(futures_pnl.index.min(), futures_pnl.index.max())
            external.update(_external_weather_family(weather, futures_pnl.index))
        except Exception as exc:
            external_errors.append("meteostat: {}".format(exc))

    rows = []
    backtests = {}
    positions = {}

    for name, signal in external.items():
        new_rows, new_bt, pos = _evaluate_signal(name, name, signal, futures_pnl, mode="long_only")
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions[name] = pos

    if external:
        external_equal = sum(external.values()) / float(len(external))
        new_rows, new_bt, pos = _evaluate_signal(
            "external_equal_family_weight",
            "external_equal_families",
            external_equal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["external_equal_family_weight"] = pos

        given_base = given["given_conservative_long_blend"]
        combined_50_50 = 0.50 * given_base + 0.50 * external_equal
        new_rows, new_bt, pos = _evaluate_signal(
            "given_plus_external_equal_family_50_50",
            "given_external_equal_families",
            combined_50_50,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["given_plus_external_equal_family_50_50"] = pos

        all_families = dict(external)
        all_families["given_price_family"] = given["given_price_family"]
        all_families["given_physical_family"] = given["given_physical_family"]
        all_equal = sum(all_families.values()) / float(len(all_families))
        new_rows, new_bt, pos = _evaluate_signal(
            "given_and_external_all_families_equal_weight",
            "all_equal_families",
            all_equal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["given_and_external_all_families_equal_weight"] = pos

        if "external_weather_hdd_cdd_family" in external:
            crush_weather = 0.50 * given["given_crush_pressure"] + 0.50 * external["external_weather_hdd_cdd_family"]
            new_rows, new_bt, pos = _evaluate_signal(
                "given_crush_plus_weather_hdd_cdd_equal_weight",
                "given_crush_weather_equal",
                crush_weather,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["given_crush_plus_weather_hdd_cdd_equal_weight"] = pos

            conservative_weather = (
                0.50 * given["given_conservative_long_blend"] + 0.50 * external["external_weather_hdd_cdd_family"]
            )
            new_rows, new_bt, pos = _evaluate_signal(
                "given_conservative_plus_weather_hdd_cdd_equal_weight",
                "given_conservative_weather_equal",
                conservative_weather,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["given_conservative_plus_weather_hdd_cdd_equal_weight"] = pos

        fundamental_family_names = [
            ("given_physical_family", given.get("given_physical_family")),
            ("external_crush_family", external.get("external_crush_family")),
            ("external_fx_export_family", external.get("external_fx_export_family")),
            ("external_weather_hdd_cdd_family", external.get("external_weather_hdd_cdd_family")),
        ]
        fundamental_families = [values for _, values in fundamental_family_names if values is not None]
        if len(fundamental_families) >= 2:
            fundamental_equal = sum(fundamental_families) / float(len(fundamental_families))
            new_rows, new_bt, pos = _evaluate_signal(
                "given_physical_external_fundamentals_equal_weight",
                "physical_crush_fx_weather_equal",
                fundamental_equal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["given_physical_external_fundamentals_equal_weight"] = pos

        overfit_controlled, selected_families, selection_table = _build_overfit_controlled_signal(
            given,
            external,
            futures_pnl,
        )
        new_rows, new_bt, pos = _evaluate_signal(
            "overfit_controlled_pre2018_family_selection",
            "pre2018_selected_equal_families",
            overfit_controlled,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["overfit_controlled_pre2018_family_selection"] = pos

        (
            weight_relaxed,
            weight_relaxed_name,
            drawdown_relaxed,
            drawdown_relaxed_name,
            weight_relaxed_table,
            weight_relaxed_candidates,
        ) = _build_weight_relaxed_pre2018_signal(
            given,
            external,
            futures_pnl,
        )
        for candidate_name, candidate_signal in weight_relaxed_candidates.items():
            new_rows, new_bt, pos = _evaluate_signal(
                "weight_relaxed_candidate_" + candidate_name,
                "predefined_fixed_family_weights",
                candidate_signal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["weight_relaxed_candidate_" + candidate_name] = pos

        new_rows, new_bt, pos = _evaluate_signal(
            "weight_relaxed_pre2018_selection_" + weight_relaxed_name,
            "pre2018_selected_fixed_family_weights",
            weight_relaxed,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["weight_relaxed_pre2018_selection_" + weight_relaxed_name] = pos

        new_rows, new_bt, pos = _evaluate_signal(
            "drawdown_priority_pre2018_selection_" + drawdown_relaxed_name,
            "pre2018_drawdown_priority_fixed_family_weights",
            drawdown_relaxed,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["drawdown_priority_pre2018_selection_" + drawdown_relaxed_name] = pos

        regime_signals, regime_frames = _build_regime_shift_signals(given, external, futures_pnl)
        regime_selection_rows = []
        for regime_name, regime_signal in regime_signals.items():
            metrics = _zero_cost_metric_for_signal(regime_signal, futures_pnl, mode="long_only")
            eligible = (
                pd.notnull(metrics.get("train_sharpe"))
                and pd.notnull(metrics.get("validation_sharpe"))
                and metrics["train_sharpe"] > 0.0
                and metrics["validation_sharpe"] >= 0.50
            )
            score = (
                metrics["validation_sharpe"] + 0.25 * metrics["train_sharpe"] + 0.001 * metrics["validation_max_drawdown"]
                if eligible
                else -np.inf
            )
            row = {"candidate": regime_name, "eligible": bool(eligible), "score": score}
            row.update(metrics)
            regime_selection_rows.append(row)
            new_rows, new_bt, pos = _evaluate_signal(
                "regime_candidate_" + regime_name,
                "observable_regime_shift",
                regime_signal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["regime_candidate_" + regime_name] = pos

        regime_selection_table = pd.DataFrame(regime_selection_rows)
        regime_eligible = regime_selection_table.loc[regime_selection_table["eligible"]].copy()
        if regime_eligible.empty:
            regime_selected_name = regime_selection_table.sort_values(
                ["validation_sharpe", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        else:
            regime_selected_name = regime_eligible.sort_values(
                ["score", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        selected_regime_signal = regime_signals[regime_selected_name]
        new_rows, new_bt, pos = _evaluate_signal(
            "regime_pre2018_selection_" + regime_selected_name,
            "pre2018_selected_observable_regime_shift",
            selected_regime_signal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["regime_pre2018_selection_" + regime_selected_name] = pos

        fund_candidates = {
            "fund_blend_50_regime_50_drawdown": 0.50 * selected_regime_signal + 0.50 * drawdown_relaxed,
            "fund_blend_35_regime_65_drawdown": 0.35 * selected_regime_signal + 0.65 * drawdown_relaxed,
            "fund_blend_65_regime_35_drawdown": 0.65 * selected_regime_signal + 0.35 * drawdown_relaxed,
            "fund_consensus_min_regime_drawdown": pd.concat(
                [selected_regime_signal, drawdown_relaxed],
                axis=1,
            ).min(axis=1),
        }
        fund_selection_rows = []
        for fund_name, fund_signal in fund_candidates.items():
            metrics = _zero_cost_metric_for_signal(fund_signal, futures_pnl, mode="long_only")
            eligible = (
                pd.notnull(metrics.get("train_sharpe"))
                and pd.notnull(metrics.get("validation_sharpe"))
                and metrics["train_sharpe"] > 0.0
                and metrics["validation_sharpe"] >= 0.50
            )
            score = (
                metrics["validation_sharpe"]
                + 0.25 * metrics["train_sharpe"]
                + 0.002 * metrics["validation_max_drawdown"]
                if eligible
                else -np.inf
            )
            row = {"candidate": fund_name, "eligible": bool(eligible), "score": score}
            row.update(metrics)
            fund_selection_rows.append(row)
            new_rows, new_bt, pos = _evaluate_signal(
                "fund_candidate_" + fund_name,
                "fund_usable_regime_drawdown_blend",
                fund_signal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["fund_candidate_" + fund_name] = pos

        fund_selection_table = pd.DataFrame(fund_selection_rows)
        fund_eligible = fund_selection_table.loc[fund_selection_table["eligible"]].copy()
        if fund_eligible.empty:
            fund_selected_name = fund_selection_table.sort_values(
                ["validation_sharpe", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        else:
            fund_selected_name = fund_eligible.sort_values(
                ["score", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        selected_fund_signal = fund_candidates[fund_selected_name]
        new_rows, new_bt, pos = _evaluate_signal(
            "fund_pre2018_selection_" + fund_selected_name,
            "pre2018_selected_fund_usable_blend",
            selected_fund_signal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["fund_pre2018_selection_" + fund_selected_name] = pos
    else:
        selection_table = pd.DataFrame()
        selected_families = {}
        weight_relaxed_table = pd.DataFrame()
        weight_relaxed_name = None
        drawdown_relaxed_name = None
        regime_selection_table = pd.DataFrame()
        regime_selected_name = None
        regime_frames = {}
        fund_selection_table = pd.DataFrame()
        fund_selected_name = None

    results = pd.DataFrame(rows)
    if not results.empty:
        results = results.sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False]).reset_index(drop=True)
    return {
        "results": results,
        "given_signals": given,
        "external_signals": external,
        "positions": positions,
        "backtests": backtests,
        "yfinance_prices": prices,
        "weather": weather,
        "errors": external_errors,
        "overfit_control_selection": selection_table,
        "overfit_control_selected_families": list(selected_families.keys()),
        "weight_relaxed_selection": weight_relaxed_table,
        "weight_relaxed_selected_candidate": weight_relaxed_name,
        "drawdown_priority_selected_candidate": drawdown_relaxed_name,
        "regime_selection": regime_selection_table,
        "regime_selected_candidate": regime_selected_name,
        "regime_frames": regime_frames,
        "fund_selection": fund_selection_table,
        "fund_selected_candidate": fund_selected_name,
    }


In [ ]:
# Save soybean external helper aliases before later cells define same helper names.
_soy_download_yfinance = _download_yfinance
_soy_external_yfinance_families_saved = _external_yfinance_families
_soy_weather_family_saved = _external_weather_family


In [61]:
soy_given = safe_run("Soybean given-data-only tests", run_soybean_given_only_experiment, data_dir=DATA_DIR)
soy_given_results = cost_only(soy_given["results"] if soy_given is not None else pd.DataFrame())
soy_given_results[
    [
        "experiment", "family", "mode", "cost_adjusted",
        "test_sharpe", "test_pnl", "test_max_drawdown",
        "full_sharpe", "max_drawdown", "turnover"
    ]
].sort_values("test_sharpe", ascending=False).head(20).reset_index(drop=True)


### Soybean given-data-only tests

**Status:** success

,experiment,family,mode,cost_adjusted,test_sharpe,test_pnl,test_max_drawdown,full_sharpe,max_drawdown,turnover
0,given_crush_pressure,given,long_only,True,1.603698,206.917387,-94.773566,1.862057,-117.766187,0.003755
1,given_trend,given,long_only,True,1.415391,334.129372,-200.427085,0.739468,-300.869959,0.001426
2,given_conservative_long_blend,given,long_only,True,1.392714,226.596817,-87.736431,0.950619,-238.302347,0.001658
3,given_price_family,given,long_only,True,1.383825,436.973678,-217.132103,1.032327,-343.482420,0.001397
4,given_equal_price_physical,given,long_short,True,0.936097,286.302738,-196.783377,0.398896,-257.449952,0.001488
5,given_physical_family,given,long_only,True,0.740546,153.365405,-123.622591,0.509009,-143.225412,0.002086
6,given_inventory_pressure,given,long_only,True,-0.907756,-200.898487,-294.665828,-0.896310,-375.739238,0.002494


## Soybean Step 4 - External Signal Tests

This is the exact test that produces:

- `given_crush_plus_weather_hdd_cdd_equal_weight`
- `given_physical_external_fundamentals_equal_weight`

If weather/yfinance data is unavailable in your environment, this cell reports the issue instead of crashing.

In [62]:
soy_external = safe_run("Soybean external signal tests", run_soybean_external_signal_experiment, data_dir=DATA_DIR, include_weather=True)
soy_external_results = cost_only(soy_external["results"] if soy_external is not None else pd.DataFrame())
if soy_external is not None:
    print("External errors/warnings:", soy_external.get("errors"))
soy_external_results["regime_interpretation_from_result"] = soy_external_results["experiment"].map({
    "given_crush_plus_weather_hdd_cdd_equal_weight": "Evidence for physical/weather regime: Cargill crush plus HDD/CDD improves Sharpe/DD.",
    "given_physical_external_fundamentals_equal_weight": "Evidence for broader physical/export/weather regime: diversified fundamentals remain strong.",
    "given_and_external_all_families_equal_weight": "Tests broad external diversification; weaker than focused physical/weather means not all external signals are needed.",
    "external_equal_family_weight": "Tests whether external-only is sufficient; compare against given+Cargill rows.",
    "overfit_controlled_pre2018_family_selection": "Tests whether pre-2018 selection generalizes; weak result warns against selection-only regimes.",
}).fillna("Diagnostic external/family row.")
soy_external_results[
    [
        "experiment", "family", "mode", "cost_adjusted",
        "test_sharpe", "test_pnl", "test_max_drawdown",
        "full_sharpe", "max_drawdown", "turnover", "regime_interpretation_from_result"
    ]
].sort_values("test_sharpe", ascending=False).head(20).reset_index(drop=True)


### Soybean external signal tests

**Status:** success

External errors/warnings: []


,experiment,family,mode,cost_adjusted,test_sharpe,test_pnl,test_max_drawdown,full_sharpe,max_drawdown,turnover,regime_interpretation_from_result
0,given_crush_plus_weather_hdd_cdd_equal_weight,given_crush_weather_equal,long_only,True,2.899920,159.501813,-31.081004,2.162887,-78.107398,0.001866,Evidence for physical/weather regime: Cargill crush plus HDD/CDD improves Sharpe/DD.
1,given_physical_external_fundamentals_equal_weight,physical_crush_fx_weather_equal,long_only,True,2.453357,234.780265,-96.223375,1.202564,-144.116601,0.001142,Evidence for broader physical/export/weather regime: diversified fundamentals remain strong.
2,regime_candidate_regime_hybrid_vol_trend_weight,observable_regime_shift,long_only,True,2.373358,269.603164,-90.671850,1.563574,-168.867132,0.001009,Diagnostic external/family row.
3,regime_pre2018_selection_regime_hybrid_vol_trend_weight,pre2018_selected_observable_regime_shift,long_only,True,2.373358,269.603164,-90.671850,1.563574,-168.867132,0.001009,Diagnostic external/family row.
4,fund_candidate_fund_blend_65_regime_35_drawdown,fund_usable_regime_drawdown_blend,long_only,True,2.238787,230.063159,-90.378791,1.559171,-135.272542,0.001035,Diagnostic external/family row.
5,fund_pre2018_selection_fund_blend_65_regime_35_drawdown,pre2018_selected_fund_usable_blend,long_only,True,2.238787,230.063159,-90.378791,1.559171,-135.272542,0.001035,Diagnostic external/family row.
6,weight_relaxed_candidate_physical_40_fx_20_extcrush_20_weather_20,predefined_fixed_family_weights,long_only,True,2.204354,223.430010,-55.136056,1.237083,-85.950234,0.001168,Diagnostic external/family row.
7,drawdown_priority_pre2018_selection_physical_40_fx_20_extcrush_20_weather_20,pre2018_drawdown_priority_fixed_family_weights,long_only,True,2.204354,223.430010,-55.136056,1.237083,-85.950234,0.001168,Diagnostic external/family row.
8,fund_candidate_fund_blend_50_regime_50_drawdown,fund_usable_regime_drawdown_blend,long_only,True,2.192012,221.652024,-90.489840,1.527053,-124.879818,0.001046,Diagnostic external/family row.
9,fund_candidate_fund_blend_35_regime_65_drawdown,fund_usable_regime_drawdown_blend,long_only,True,2.052964,202.076764,-91.963166,1.046018,-136.629044,0.001090,Diagnostic external/family row.


## Soybean Step 5 - Low-Volatility Risk-Control Tests

This is the exact test that produces:

- `base_drawdown_priority`
- `low_vol_half_base_else_base`
- `low_vol_flat_else_base`

### Direct Code Used Here: Soybean Low-Volatility Risk-Control Experiment

The next cell defines the implemented low-volatility switch experiment directly before it is called.


In [ ]:
_soy_external_positions_from_signal = _positions_from_signal

"""Soybean low-volatility switch experiment.

The existing soybean drawdown-priority strategy is strong overall but weak in
quiet abundant-supply periods. This experiment tests fixed, low-overfit switches:

- keep the drawdown-priority strategy outside low volatility;
- in low volatility, either reduce/zero exposure or use a simple no-fit
  soybean mean-reversion sleeve.

No regression coefficients, IC selection, or optimized switch weights are used.
"""


import os

import numpy as np
import pandas as pd



TEST_START = "2018-01-01"


def _base_drawdown_signal(given, external):
    weights = {
        "given_physical_family": 0.40,
        "external_fx_export_family": 0.20,
        "external_crush_family": 0.20,
        "external_weather_hdd_cdd_family": 0.20,
    }
    families = dict(external)
    families["given_physical_family"] = given["given_physical_family"]
    return _weighted_sum(families, weights).clip(-5.0, 5.0).fillna(0.0)


def _positions_from_signal(signal, futures_pnl, mode):
    return _soy_external_positions_from_signal(signal, futures_pnl, mode=mode)


def _switch_positions(base_positions, low_vol_positions, low_vol_mask, low_vol_scale=1.0):
    mask = pd.Series(low_vol_mask, index=base_positions.index).fillna(False).astype(bool)
    out = base_positions.copy()
    out.loc[mask, COMMODITY] = float(low_vol_scale) * low_vol_positions.loc[mask, COMMODITY]
    return out.fillna(0.0)


def _conditional_scale_positions(base_positions, condition, scale_when_true):
    mask = pd.Series(condition, index=base_positions.index).fillna(False).astype(bool)
    out = base_positions.copy()
    out.loc[mask, COMMODITY] = float(scale_when_true) * out.loc[mask, COMMODITY]
    return out.fillna(0.0)


def _metrics(bt, low_vol_mask):
    table = split_performance(bt, TEST_START)
    low_mask = pd.Series(low_vol_mask, index=bt.index).fillna(False).astype(bool)
    high_mask = ~low_mask
    low = performance_metrics(bt.loc[low_mask])
    non_low = performance_metrics(bt.loc[high_mask])
    low_price = performance_metrics(bt.loc[(bt.index >= "2014-06-01") & (bt.index <= "2017-12-31")])
    return {
        "train_sharpe": table.loc["sharpe", "in_sample"],
        "oos_sharpe": table.loc["sharpe", "out_of_sample"],
        "oos_pnl": table.loc["total_pnl", "out_of_sample"],
        "oos_dd": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_dd": table.loc["max_drawdown", "full_period"],
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
        "low_vol_sharpe": low.get("sharpe", np.nan),
        "low_vol_pnl": low.get("total_pnl", np.nan),
        "low_vol_dd": low.get("max_drawdown", np.nan),
        "non_low_vol_sharpe": non_low.get("sharpe", np.nan),
        "low_price_sharpe": low_price.get("sharpe", np.nan),
        "low_price_pnl": low_price.get("total_pnl", np.nan),
        "low_price_dd": low_price.get("max_drawdown", np.nan),
    }


def _evaluate(name, positions, futures_pnl, low_vol_mask):
    rows = []
    backtests = {}
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[COMMODITY]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
        row = {"strategy": name, "cost_adjusted": bool(cost_adjusted)}
        row.update(_metrics(bt, low_vol_mask))
        rows.append(row)
        backtests[name + ("_cost" if cost_adjusted else "_zero")] = bt
    return rows, backtests


def _write_log(results, period_tables, path="notes/soybean_low_vol_switch.txt"):
    lines = []
    lines.append("Soybean low-volatility switch experiment")
    lines.append("Date: 2026-05-02")
    lines.append("")
    lines.append("Method")
    lines.append("------")
    lines.append("Low-vol regime = 60-day soybean PnL volatility < 0.80 * expanding median volatility.")
    lines.append("Base strategy = drawdown-priority physical 40 / FX 20 / external crush 20 / weather 20.")
    lines.append("Switches are fixed position-level rules; no fitted coefficients or optimized weights.")
    lines.append("")
    lines.append("Results")
    lines.append("-------")
    cols = [
        "strategy",
        "cost_adjusted",
        "oos_sharpe",
        "oos_pnl",
        "oos_dd",
        "full_sharpe",
        "full_dd",
        "low_vol_sharpe",
        "low_vol_pnl",
        "low_vol_dd",
        "low_price_sharpe",
        "low_price_pnl",
        "low_price_dd",
        "turnover",
    ]
    lines.append(results[cols].to_string(index=False, float_format=lambda value: "{:.3f}".format(value)))
    lines.append("")
    lines.append("Key period performance for selected rows")
    lines.append("----------------------------------------")
    for name, table in period_tables.items():
        lines.append("")
        lines.append(name)
        lines.append(table.to_string(index=False, float_format=lambda value: "{:.3f}".format(value)))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as handle:
        handle.write("\n".join(lines))


def run_soybean_low_vol_switch_experiment(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    external_out = run_soybean_external_signal_experiment(data_dir=data_dir)
    given = external_out["given_signals"]
    external = external_out["external_signals"]
    if not {"external_fx_export_family", "external_crush_family", "external_weather_hdd_cdd_family"}.issubset(external):
        raise RuntimeError("Missing required external families: {}".format(external_out["errors"]))

    base_signal = _base_drawdown_signal(given, external)
    regimes = _soybean_regime_frames(given, external, futures_pnl)
    low_vol_mask = regimes["low_vol"] > 0.0
    weak_trend = given["given_trend"].reindex(futures_pnl.index).fillna(0.0) <= 0.0
    weak_physical = given["given_physical_family"].reindex(futures_pnl.index).fillna(0.0) <= 0.0
    soy_price = data["adj1"][COMMODITY].reindex(futures_pnl.index).ffill()
    below_long_ma = soy_price < soy_price.rolling(252, min_periods=120).mean().shift(1)
    negative_medium_mom = feature_panels[COMMODITY]["mom_60"].reindex(futures_pnl.index).fillna(0.0) < 0.0
    abundant_supply_proxy = low_vol_mask & below_long_ma.fillna(False) & negative_medium_mom
    weak_low_vol_proxy = low_vol_mask & weak_trend
    weak_physical_low_vol_proxy = low_vol_mask & weak_physical
    weak_non_abundant_proxy = weak_low_vol_proxy & ~abundant_supply_proxy
    combined_abundant_or_weak_non_abundant = abundant_supply_proxy | weak_non_abundant_proxy

    base_pos = _positions_from_signal(base_signal, futures_pnl, mode="long_only")
    flat_pos = base_pos * 0.0
    half_base_pos = 0.50 * base_pos
    rev5_pos = _positions_from_signal(build_soybean_signal(feature_panels, "rev5"), futures_pnl, mode="long_short")
    defensive_pos = _positions_from_signal(
        build_soybean_signal(feature_panels, "soy_defensive_blend"),
        futures_pnl,
        mode="long_short",
    )
    curve_crush_pos = _positions_from_signal(
        build_soybean_signal(feature_panels, "soy_curve_crush"),
        futures_pnl,
        mode="long_short",
    )

    positions = {
        "base_drawdown_priority": base_pos,
        "low_vol_flat_else_base": _switch_positions(base_pos, flat_pos, low_vol_mask),
        "low_vol_half_base_else_base": _switch_positions(base_pos, half_base_pos, low_vol_mask),
        "low_vol_weak_trend_flat_else_base": _conditional_scale_positions(base_pos, weak_low_vol_proxy, 0.0),
        "low_vol_weak_trend_half_else_base": _conditional_scale_positions(base_pos, weak_low_vol_proxy, 0.50),
        "low_vol_weak_physical_flat_else_base": _conditional_scale_positions(
            base_pos,
            weak_physical_low_vol_proxy,
            0.0,
        ),
        "low_vol_abundant_proxy_flat_else_base": _conditional_scale_positions(
            base_pos,
            abundant_supply_proxy,
            0.0,
        ),
        "low_vol_abundant_proxy_half_else_base": _conditional_scale_positions(
            base_pos,
            abundant_supply_proxy,
            0.50,
        ),
        "low_vol_abundant_flat_weak_nonabundant_flat_else_base": _conditional_scale_positions(
            base_pos,
            combined_abundant_or_weak_non_abundant,
            0.0,
        ),
        "low_vol_abundant_half_weak_nonabundant_flat_else_base": _conditional_scale_positions(
            _conditional_scale_positions(base_pos, abundant_supply_proxy, 0.50),
            weak_non_abundant_proxy,
            0.0,
        ),
        "low_vol_rev5_ls_else_base": _switch_positions(base_pos, rev5_pos, low_vol_mask),
        "low_vol_defensive_mr_else_base": _switch_positions(base_pos, defensive_pos, low_vol_mask),
        "low_vol_curve_crush_ls_else_base": _switch_positions(base_pos, curve_crush_pos, low_vol_mask),
    }

    rows = []
    backtests = {}
    for name, pos in positions.items():
        new_rows, new_bt = _evaluate(name, pos, futures_pnl, low_vol_mask)
        rows.extend(new_rows)
        backtests.update(new_bt)

    results = pd.DataFrame(rows).sort_values(["cost_adjusted", "oos_sharpe"], ascending=[True, False])
    selected = [
        "base_drawdown_priority_cost",
        "low_vol_flat_else_base_cost",
        "low_vol_half_base_else_base_cost",
        "low_vol_weak_trend_flat_else_base_cost",
        "low_vol_abundant_proxy_flat_else_base_cost",
        "low_vol_abundant_half_weak_nonabundant_flat_else_base_cost",
        "low_vol_rev5_ls_else_base_cost",
    ]
    period_tables = {}
    for key in selected:
        if key in backtests:
            table = period_performance(backtests[key])
            cols = ["period", "total_pnl", "sharpe", "max_drawdown", "hit_rate", "days"]
            period_tables[key] = table[cols]
    _write_log(results, period_tables)
    return {
        "results": results.reset_index(drop=True),
        "positions": positions,
        "backtests": backtests,
        "low_vol_mask": low_vol_mask,
        "period_tables": period_tables,
        "external_errors": external_out["errors"],
    }


In [63]:
soy_low_vol = safe_run("Soybean low-volatility switch tests", run_soybean_low_vol_switch_experiment, data_dir=DATA_DIR)
soy_low_vol_results = cost_only(soy_low_vol["results"] if soy_low_vol is not None else pd.DataFrame())
soy_low_vol_results["regime_interpretation_from_result"] = soy_low_vol_results["strategy"].map({
    "base_drawdown_priority": "Base physical/FX/crush/weather strategy before proving low-vol risk control.",
    "low_vol_half_base_else_base": "Evidence for low-vol risk regime: fixed half exposure improves DD without fitted weights.",
    "low_vol_flat_else_base": "Stress test: fully remove low-vol exposure; compare Sharpe/DD vs half exposure.",
    "low_vol_abundant_proxy_flat_else_base": "Tests stricter abundant-supply proxy: low vol plus below MA and negative momentum.",
    "low_vol_weak_trend_flat_else_base": "Tests whether weak trend inside low vol is the risky sub-regime.",
}).fillna("Low-vol diagnostic row.")
soy_low_vol_results[
    [
        "strategy", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd",
        "full_sharpe", "full_dd", "low_vol_sharpe", "low_vol_pnl",
        "low_vol_dd", "low_price_pnl", "low_price_dd", "turnover", "regime_interpretation_from_result"
    ]
].sort_values("oos_sharpe", ascending=False).reset_index(drop=True)


### Soybean low-volatility switch tests

**Status:** success

,strategy,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,low_vol_sharpe,low_vol_pnl,low_vol_dd,low_price_pnl,low_price_dd,turnover,regime_interpretation_from_result
0,low_vol_flat_else_base,True,5.056833,103.015642,-29.190664,1.200822,-94.069062,NaN,NaN,NaN,-57.006293,-80.789331,0.000908,Stress test: fully remove low-vol exposure; compare Sharpe/DD vs half exposure.
1,low_vol_abundant_flat_weak_nonabundant_flat_else_base,True,2.758423,225.818211,-50.456815,1.413593,-92.509563,1.714939,127.177610,-44.328960,-54.854390,-78.637427,0.001474,Low-vol diagnostic row.
2,low_vol_abundant_half_weak_nonabundant_flat_else_base,True,2.741884,246.469279,-50.456815,1.339729,-104.046542,1.505129,135.699295,-44.328960,-66.983773,-90.766810,0.001404,Low-vol diagnostic row.
3,low_vol_half_base_else_base,True,2.505201,163.222826,-34.702062,1.167465,-85.029854,1.288582,70.357961,-28.200066,-47.967085,-71.750122,0.000795,Evidence for low-vol risk regime: fixed half exposure improves DD without fitted weights.
4,low_vol_weak_trend_flat_else_base,True,2.390410,212.967162,-50.988987,1.150979,-111.779676,1.128318,98.857454,-52.064720,-70.323497,-94.106534,0.001482,Tests whether weak trend inside low vol is the risky sub-regime.
5,low_vol_weak_trend_half_else_base,True,2.224077,217.986986,-50.456815,1.166182,-92.053147,1.157654,119.210397,-49.699375,-54.990378,-78.773415,0.001363,Low-vol diagnostic row.
6,base_drawdown_priority,True,2.204354,223.430010,-55.136056,1.237083,-85.950234,1.288582,140.715922,-56.400133,-38.927877,-70.740328,0.001168,Base physical/FX/crush/weather strategy before proving low-vol risk control.
7,low_vol_weak_physical_flat_else_base,True,2.192134,178.861394,-51.645773,1.109842,-118.265880,0.990636,66.547883,-55.526171,-66.304164,-90.087201,0.001762,Low-vol diagnostic row.
8,low_vol_abundant_proxy_half_else_base,True,2.080457,201.899941,-55.136056,1.230017,-80.847887,1.278076,131.130072,-56.400133,-26.983659,-60.835941,0.001288,Low-vol diagnostic row.
9,low_vol_abundant_proxy_flat_else_base,True,2.031546,181.209391,-68.009697,1.286889,-88.169771,1.406117,122.467359,-69.273774,-14.955821,-50.847934,0.001372,Tests stricter abundant-supply proxy: low vol plus below MA and negative momentum.


## Soybean Step 6 - Best Soybean Rows From Actual Tests

In [64]:
soy_best_external = soy_external_results.loc[soy_external_results.get("experiment", pd.Series(dtype=str)).isin([
    "given_crush_plus_weather_hdd_cdd_equal_weight",
    "given_physical_external_fundamentals_equal_weight",
])].copy()
soy_best_low_vol = soy_low_vol_results.loc[soy_low_vol_results.get("strategy", pd.Series(dtype=str)).isin([
    "base_drawdown_priority",
    "low_vol_half_base_else_base",
    "low_vol_flat_else_base",
])].copy()

print("External best candidates")
display(
    soy_best_external[
        ["experiment", "test_sharpe", "test_pnl", "test_max_drawdown", "full_sharpe", "max_drawdown", "turnover"]
    ].sort_values("test_sharpe", ascending=False).reset_index(drop=True)
)
print("Low-vol best candidates")
display(
    soy_best_low_vol[
        ["strategy", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "turnover"]
    ].sort_values("oos_sharpe", ascending=False).reset_index(drop=True)
)


External best candidates


,experiment,test_sharpe,test_pnl,test_max_drawdown,full_sharpe,max_drawdown,turnover
0,given_crush_plus_weather_hdd_cdd_equal_weight,2.899920,159.501813,-31.081004,2.162887,-78.107398,0.001866
1,given_physical_external_fundamentals_equal_weight,2.453357,234.780265,-96.223375,1.202564,-144.116601,0.001142


Low-vol best candidates


,strategy,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,turnover
0,low_vol_flat_else_base,5.056833,103.015642,-29.190664,1.200822,-94.069062,0.000908
1,low_vol_half_base_else_base,2.505201,163.222826,-34.702062,1.167465,-85.029854,0.000795
2,base_drawdown_priority,2.204354,223.430010,-55.136056,1.237083,-85.950234,0.001168


## Soybean Step 7 - Regimes That Became Clear

This regime table is shown only after the generic, external, and low-vol tests. The interpretation comes from the result path: generic rows were not enough, Cargill/physical/weather rows were strongest, and the low-vol risk-control rows improved drawdown.


In [65]:
soy_regime_rows = []
if soy_external is not None and soy_external.get("regime_frames"):
    frames = soy_external["regime_frames"]
    for regime_name in ["low_vol", "high_vol", "trend_positive"]:
        if regime_name in frames:
            mask = frames[regime_name].astype(float) > 0.0
            soy_regime_rows.append({
                "product": "Soybeans",
                "regime": regime_name,
                "detection": {
                    "low_vol": "60d PnL vol < 0.80 x expanding median vol",
                    "high_vol": "60d PnL vol > 1.20 x median or above expanding 75th percentile",
                    "trend_positive": "given_trend > 0",
                }[regime_name],
                "active_days": int(mask.sum()),
                "active_share": float(mask.mean()),
                "experiment_where_clear": "Soybean external/regime and low-vol switch tests",
            })
if soy_low_vol is not None and "low_vol_mask" in soy_low_vol:
    mask = pd.Series(soy_low_vol["low_vol_mask"]).astype(bool)
    soy_regime_rows.append({
        "product": "Soybeans",
        "regime": "low_vol_switch_mask",
        "detection": "same low-vol rule used by low_vol_half_base_else_base",
        "active_days": int(mask.sum()),
        "active_share": float(mask.mean()),
        "experiment_where_clear": "Soybean low-volatility risk-control tests",
    })

if soy_regime_rows:
    pd.DataFrame(soy_regime_rows)[
        ["product", "regime", "detection", "active_days", "active_share", "experiment_where_clear"]
    ].sort_values("regime").reset_index(drop=True)
else:
    pd.DataFrame({"message": ["Run the soybean external and low-vol experiment cells to show regime diagnostics."]})


In [66]:
if soy_low_vol_results is not None and len(soy_low_vol_results) > 0:
    soy_regime_performance = soy_low_vol_results[
        [
            "strategy", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd",
            "low_vol_sharpe", "low_vol_pnl", "low_vol_dd",
            "low_price_pnl", "low_price_dd"
        ]
    ].sort_values("oos_sharpe", ascending=False).reset_index(drop=True)
    soy_regime_performance
else:
    pd.DataFrame({"message": ["Run Soybean Step 5 to show low-vol regime performance."]})


## Soybean Step 8 - Key Period Performance For Best Rows

These tables show how the best soybean candidates behaved in the named historical regimes.


In [67]:
soybean_period_sources = []
if soy_external is not None:
    soybean_period_sources.extend([
        (
            "given_crush_plus_weather_hdd_cdd_equal_weight_cost_adjusted",
            soy_external.get("backtests", {}).get("given_crush_plus_weather_hdd_cdd_equal_weight_cost_adjusted"),
        ),
        (
            "given_physical_external_fundamentals_equal_weight_cost_adjusted",
            soy_external.get("backtests", {}).get("given_physical_external_fundamentals_equal_weight_cost_adjusted"),
        ),
    ])
if soy_low_vol is not None:
    soybean_period_sources.extend([
        (
            "base_drawdown_priority_cost",
            soy_low_vol.get("backtests", {}).get("base_drawdown_priority_cost"),
        ),
        (
            "low_vol_half_base_else_base_cost",
            soy_low_vol.get("backtests", {}).get("low_vol_half_base_else_base_cost"),
        ),
    ])

for name, bt in soybean_period_sources:
    display(Markdown(f"### {name}"))
    if bt is None:
        display(pd.DataFrame({"message": ["Backtest not available. Run the corresponding soybean experiment cell first."]}))
    else:
        display(
            period_performance(bt)[
                ["period", "total_pnl", "sharpe", "max_drawdown", "hit_rate", "days"]
            ].reset_index(drop=True)
        )


### given_crush_plus_weather_hdd_cdd_equal_weight_cost_adjusted

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,25.224068,2.847207,-21.563589,0.583333,24.0
1,US drought rally/retrace,52.656032,2.099062,-46.374436,0.527778,36.0
2,Crimea/Black Sea shock,NaN,NaN,NaN,NaN,NaN
3,Low-price abundant supply,68.937206,1.493735,-61.445508,0.485294,68.0
4,US-China trade war,67.961253,2.171733,-31.081004,0.607143,56.0
5,2019 prevented planting floods,10.484180,1.185416,-24.071773,0.400000,10.0
6,COVID demand shock,53.921623,4.691976,-19.060151,0.551724,29.0
7,COVID recovery/China buying,33.189914,2.876973,-26.175344,0.576923,26.0


### given_physical_external_fundamentals_equal_weight_cost_adjusted

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,109.085085,3.692759,-29.398116,0.537313,67.0
1,US drought rally/retrace,12.680282,0.515198,-43.162749,0.454545,55.0
2,Crimea/Black Sea shock,NaN,NaN,NaN,NaN,NaN
3,Low-price abundant supply,17.275748,0.220157,-95.894349,0.457711,201.0
4,US-China trade war,-47.800983,-2.076561,-66.526887,0.500000,72.0
5,2019 prevented planting floods,NaN,NaN,NaN,NaN,NaN
6,COVID demand shock,7.585104,0.704206,-32.242952,0.388889,18.0
7,COVID recovery/China buying,214.710132,5.028680,-38.409151,0.643678,87.0


### base_drawdown_priority_cost

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,76.982314,3.899406,-24.563848,0.591837,49.0
1,US drought rally/retrace,38.757625,2.917139,-17.486729,0.500000,28.0
2,Crimea/Black Sea shock,NaN,NaN,NaN,NaN,NaN
3,Low-price abundant supply,-38.927877,-0.822278,-70.740328,0.409396,149.0
4,US-China trade war,-14.268480,-0.728205,-47.324274,0.500000,58.0
5,2019 prevented planting floods,NaN,NaN,NaN,NaN,NaN
6,COVID demand shock,36.002077,1.855413,-50.756174,0.516129,31.0
7,COVID recovery/China buying,197.459815,4.014375,-50.456815,0.608247,97.0


### low_vol_half_base_else_base_cost

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,76.982314,3.899406,-24.563848,0.591837,49.0
1,US drought rally/retrace,38.757625,2.917139,-17.486729,0.500000,28.0
2,Crimea/Black Sea shock,NaN,NaN,NaN,NaN,NaN
3,Low-price abundant supply,-47.967085,-1.333943,-71.750122,0.409396,149.0
4,US-China trade war,-6.502202,-0.570551,-28.783814,0.500000,58.0
5,2019 prevented planting floods,NaN,NaN,NaN,NaN,NaN
6,COVID demand shock,18.001039,1.855413,-25.378087,0.516129,31.0
7,COVID recovery/China buying,149.605690,4.180434,-34.702062,0.608247,97.0


# Corn

Research path:

1. Generic tests were weak.
2. Fitted models did not solve corn.
3. EIA ethanol and external data were tested.
4. Abundant-supply risk controls improved the corn sleeve.

## Corn Questions Before Testing

At this point I do not assume a corn regime. The next tests ask:

| Question | Test That Can Answer It | What Would Count As Evidence |
|---|---|---|
| Does generic averaging or family weighting work? | generic avg/family/IC/regime tests | positive OOS Sharpe with controlled DD |
| Is corn more sensitive to volatility/event states than trend/MR? | generic regime tests and corn regime rebuild | vol-regime rows beat trend/MR or flat family rows |
| Does ethanol add a corn-specific demand channel? | EIA ethanol external test | ethanol candidates improve OOS without unacceptable DD |
| Is the main failure an abundant-supply weak-tape state? | abundant-supply risk-control tests | fixed weak-tape guard improves DD and OOS Sharpe |

The regime labels are introduced only after the relevant result tables show evidence.


## Corn Step 1 - Generic Tests

In [68]:
corn_generic = family_results.loc[family_results.get("commodity", pd.Series(dtype=str)) == "CORN"].copy()
corn_generic["regime_interpretation_from_result"] = corn_generic["strategy"].map({
    "avg_all_signals": "Baseline: broad average tests whether corn has a simple all-signal edge.",
    "equal_family_weight": "Tests whether family balancing alone is enough; weak OOS implies corn needs filtering.",
    "ic_family_selected_physical_public": "Tests whether flat IC selection finds corn edge; weak rows warn against IC-only final strategy.",
    "regime_best_family_trend_mr": "Tests generic trend/MR split; weak rows imply this is not the right corn regime.",
    "regime_avg_families_trend_mr": "Tests generic trend/MR without best-family selection.",
    "ols_kalman_filter": "Fitted coefficient benchmark; useful only if it beats simple rules without DD damage.",
    "ridge_expanding_alpha100": "Regularized fitted benchmark; negative rows reject fitted core.",
}).fillna("Diagnostic row.")
corn_generic[
    [
        "commodity", "strategy", "mode", "cost_adjusted",
        "oos_sharpe", "oos_pnl", "oos_dd",
        "full_sharpe", "full_dd", "regime_interpretation_from_result", "overfit_comment"
    ]
].sort_values("oos_sharpe", ascending=False).reset_index(drop=True)


,commodity,strategy,mode,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,regime_interpretation_from_result,overfit_comment
0,CORN,ridge_expanding_alpha100,long_short,True,-0.110198,-116.207110,-794.019314,0.294800,-878.808538,Regularized fitted benchmark; negative rows reject fitted core.,Coefficient-estimation risk; fixed alpha reduces but does not remove it. Reject: negative cost-adjusted OOS.
1,CORN,ic_family_selected_curve,long_short,True,-0.132723,-150.374274,-618.134373,-0.026827,-1578.338865,Diagnostic row.,Moderate/high selection risk; validation IC can be noisy. Reject: negative cost-adjusted OOS.
2,CORN,avg_all_signals,long_short,True,-0.222904,-92.230612,-436.504097,0.050660,-439.471117,Baseline: broad average tests whether corn has a simple all-signal edge.,"Low model overfit, but can be economically diluted. Reject: negative cost-adjusted OOS."
3,CORN,ols_kalman_filter,long_short,True,-0.390056,-348.717584,-843.550825,0.094664,-843.550825,Fitted coefficient benchmark; useful only if it beats simple rules without DD damage.,"Coefficient-estimation risk; parameters are fixed, not OOS tuned. Reject: negative cost-adjusted OOS."
4,CORN,equal_family_weight,long_short,True,-0.400244,-183.281621,-534.051632,0.006349,-534.051632,Tests whether family balancing alone is enough; weak OOS implies corn needs filtering.,Low model overfit; family choices are researcher-defined. Reject: negative cost-adjusted OOS.
5,CORN,regime_best_family_trend_mr,long_short,True,-0.558358,-227.400880,-460.758024,0.792422,-460.758024,Tests generic trend/MR split; weak rows imply this is not the right corn regime.,"Moderate selection risk; weights are fixed masks, not optimized. Reject: negative cost-adjusted OOS."
6,CORN,regime_avg_families_trend_mr,long_short,True,-0.705526,-320.335315,-532.754967,0.672769,-532.754967,Tests generic trend/MR without best-family selection.,"Lower than best-family regime selection, but still regime-research risk. Reject: negative cost-adjusted OOS."


## Corn Step 2 - OLS / Kalman / Ridge Benchmark

In [69]:
corn_linear = linear_results.loc[linear_results.get("commodity", pd.Series(dtype=str)) == "CORN"].copy()
corn_linear[
    [
        "commodity", "model", "mode", "cost_adjusted",
        "test_sharpe", "test_pnl", "test_max_drawdown",
        "full_sharpe", "max_drawdown"
    ]
].sort_values("test_sharpe", ascending=False).reset_index(drop=True)


,commodity,model,mode,cost_adjusted,test_sharpe,test_pnl,test_max_drawdown,full_sharpe,max_drawdown
0,CORN,kalman_dynamic_linear,long_only,True,0.519419,524.020813,-651.620021,0.178139,-1042.627656
1,CORN,kalman_dynamic_linear,long_short,True,-0.112430,-157.128591,-1041.024505,0.120974,-1224.426330
2,CORN,ols_expanding,long_short,True,-0.381156,-237.046253,-398.077645,-0.280733,-1338.670602
3,CORN,rls_lambda0995,long_short,True,-0.724693,-806.753263,-912.857730,-0.170132,-1483.921296
4,CORN,ridge_expanding_alpha100,long_only,True,-1.062962,-671.310235,-893.342358,-0.545557,-1269.669097
5,CORN,rls_lambda0995,long_only,True,-1.117127,-623.127634,-740.296249,-0.669638,-1401.523537
6,CORN,ridge_expanding_alpha100,long_short,True,-1.119948,-1363.944302,-1738.102792,-0.418391,-2021.763758
7,CORN,ols_expanding,long_only,True,-1.341279,-168.407265,-200.796422,-0.330762,-682.584696


## Corn Step 3 - External / EIA Ethanol Tests

This exact test produces rows such as:

- `external_ethanol_family`
- `requirement_given_80_ethanol_10_fx_10`
- `regime_hybrid_vol_trend_weight`

### Direct Code Used Here: Corn External / EIA Ethanol Experiment

The next cell defines the implemented corn external and EIA ethanol experiment code directly before it is called.


In [ ]:
"""Corn lower-overfit strategy experiments with EIA ethanol.

This mirrors the soybean research style but keeps corn-specific economics:
provided physical/Cargill signals, EIA ethanol production/stocks, crop-belt
weather, export FX pressure, grain relative value, and macro/risk.
"""


import numpy as np
import pandas as pd



COMMODITY = "CORN"
TRAIN_END = "2016-01-01"
TEST_START = "2018-01-01"
YF_TICKERS = {
    "corn": "ZC=F",
    "soybean": "ZS=F",
    "wheat": "ZW=F",
    "usd_index": "DX-Y.NYB",
    "brl": "BRL=X",
    "cny": "CNY=X",
    "crude": "CL=F",
    "equity": "SPY",
}


def _tanh(series, divisor=2.0):
    return pd.Series(np.tanh(series.astype(float) / float(divisor)), index=series.index)


def _smooth_threshold(series, mode="long_only"):
    signal = _tanh(series).ewm(halflife=2.0, adjust=False, min_periods=1).mean()
    signal[signal.abs() < 0.05] = 0.0
    if mode == "long_only":
        signal = signal.clip(lower=0.0)
    elif mode == "short_only":
        signal = signal.clip(upper=0.0)
    elif mode != "long_short":
        raise ValueError("Unknown mode: {}".format(mode))
    return signal


def _download_yfinance(start, end):
    import yfinance as yf

    tickers = list(YF_TICKERS.values())
    raw = yf.download(
        tickers,
        start=str(pd.Timestamp(start).date()),
        end=str((pd.Timestamp(end) + pd.DateOffset(days=7)).date()),
        auto_adjust=True,
        progress=False,
        threads=False,
    )
    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" in raw.columns.get_level_values(0):
            close = raw["Close"]
        elif "Adj Close" in raw.columns.get_level_values(0):
            close = raw["Adj Close"]
        else:
            close = raw.xs(raw.columns.get_level_values(0)[0], axis=1, level=0)
    else:
        close = raw.to_frame(tickers[0]) if isinstance(raw, pd.Series) else raw
    close = close.rename(columns={ticker: name for name, ticker in YF_TICKERS.items()})
    close = close[[name for name in YF_TICKERS if name in close.columns]]
    close.index = pd.to_datetime(close.index).tz_localize(None)
    return close.sort_index()


def _given_components(feature_panels):
    corn = feature_panels[COMMODITY]
    inventory_pressure = (
        -corn["public_inventory_change"] - corn["receipts_change"] - corn["cgl_inventory_change"]
    ) / 3.0
    cgl_crush_activity = (corn["crush_surprise"] + corn["crush_utilization"]) / 2.0
    trend = (corn["mom_20"] + corn["mom_60"] + corn["curve_spread"] + corn["cot_pm_oi_level"]) / 4.0
    curve_tightness = (corn["curve_spread"] + corn["curve_ratio"]) / 2.0
    price_family = (corn["mom_20"] + corn["mom_60"] + corn["rev_5"]) / 3.0
    physical_family = (inventory_pressure + curve_tightness + 0.25 * cgl_crush_activity) / 2.25
    conservative = 0.40 * physical_family + 0.30 * trend + 0.30 * price_family
    return {
        "given_price_family": price_family.fillna(0.0),
        "given_physical_family": physical_family.fillna(0.0),
        "given_trend": trend.fillna(0.0),
        "given_inventory_pressure": inventory_pressure.fillna(0.0),
        "given_cgl_crush_activity": cgl_crush_activity.fillna(0.0),
        "given_curve_tightness": curve_tightness.fillna(0.0),
        "given_conservative_blend": conservative.fillna(0.0),
    }


def _ethanol_family(ethanol_features):
    parts = []
    if "ethanol_production_change_4w" in ethanol_features:
        parts.append(ethanol_features["ethanol_production_change_4w"])
    if "ethanol_prod_to_stocks" in ethanol_features:
        parts.append(ethanol_features["ethanol_prod_to_stocks"])
    if "ethanol_demand_pressure" in ethanol_features:
        parts.append(ethanol_features["ethanol_demand_pressure"])
    if "ethanol_stocks_change_4w" in ethanol_features:
        parts.append(-ethanol_features["ethanol_stocks_change_4w"])
    if not parts:
        return {}
    return {"external_ethanol_family": (sum(parts) / float(len(parts))).clip(lower=-5.0, upper=5.0).fillna(0.0)}


def _external_yfinance_families(prices, trading_index):
    px = prices.reindex(trading_index).ffill().shift(1)
    families = {}
    if {"corn", "soybean", "wheat"}.issubset(px.columns):
        corn_soy = rolling_zscore((px["corn"] / px["soybean"]).pct_change(20, fill_method=None), 252, 60)
        corn_wheat = rolling_zscore((px["corn"] / px["wheat"]).pct_change(20, fill_method=None), 252, 60)
        soy_corn_mr = -rolling_zscore((px["soybean"] / px["corn"]).pct_change(20, fill_method=None), 252, 60)
        families["external_relative_grain_family"] = ((corn_soy + corn_wheat + soy_corn_mr) / 3.0).fillna(0.0)

    fx_parts = []
    if "usd_index" in px.columns:
        fx_parts.append(-rolling_zscore(px["usd_index"].pct_change(20, fill_method=None), 252, 60))
    if "brl" in px.columns:
        fx_parts.append(-rolling_zscore(px["brl"].pct_change(20, fill_method=None), 252, 60))
    if "cny" in px.columns:
        fx_parts.append(-rolling_zscore(px["cny"].pct_change(20, fill_method=None), 252, 60))
    if fx_parts:
        families["external_fx_export_family"] = (sum(fx_parts) / float(len(fx_parts))).fillna(0.0)

    macro_parts = []
    if "crude" in px.columns:
        macro_parts.append(rolling_zscore(px["crude"].pct_change(20, fill_method=None), 252, 60))
    if "equity" in px.columns:
        macro_parts.append(rolling_zscore(px["equity"].pct_change(20, fill_method=None), 252, 60))
    if "usd_index" in px.columns:
        macro_parts.append(-rolling_zscore(px["usd_index"].pct_change(60, fill_method=None), 252, 80))
    if macro_parts:
        families["external_macro_risk_family"] = (sum(macro_parts) / float(len(macro_parts))).fillna(0.0)

    out = {}
    for name, values in families.items():
        cleaned = values.clip(lower=-5.0, upper=5.0).replace([np.inf, -np.inf], np.nan)
        if cleaned.notna().sum() > 20 and cleaned.fillna(0.0).abs().sum() > 0.0:
            out[name] = cleaned.fillna(0.0)
    return out


def _weather_family(weather, trading_index):
    panels = build_meteostat_feature_panels(weather, trading_index, mode="commodity_seasonal")
    corn = panels[COMMODITY].reindex(trading_index).fillna(0.0)
    candidates = [
        "meteo_cdd_20d_growing",
        "meteo_hdd_20d_growing",
        "meteo_gdd_60d_growing",
        "meteo_heat_stress_20d_growing",
        "meteo_dryness_20d_growing",
        "meteo_dry_cdd_20d_growing",
        "meteo_precip_20d_planting",
        "meteo_dryness_20d_planting",
        "meteo_freeze_stress_5d_harvest",
    ]
    existing = [col for col in candidates if col in corn.columns]
    if not existing:
        return {}
    return {"external_weather_hdd_cdd_family": corn[existing].mean(axis=1).clip(lower=-5.0, upper=5.0).fillna(0.0)}


def _positions_from_signal(signal, futures_pnl, mode="long_only"):
    cleaned = _smooth_threshold(signal.reindex(futures_pnl.index).fillna(0.0), mode=mode)
    pnl = futures_pnl[[COMMODITY]]
    asset_vol = pnl[COMMODITY].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    pos = cleaned.reindex(pnl.index).fillna(0.0) * (75.0 / asset_vol)
    out = pd.DataFrame(0.0, index=pnl.index, columns=[COMMODITY])
    out[COMMODITY] = pos.clip(lower=-0.50, upper=0.50).fillna(0.0)
    return out


def _metrics(bt):
    table = split_performance(bt, TEST_START)
    train_val = split_performance(bt.loc[bt.index < pd.Timestamp(TEST_START)], TRAIN_END)
    if table.empty or "sharpe" not in table.index:
        return {
            "train_sharpe": np.nan,
            "validation_sharpe": np.nan,
            "validation_max_drawdown": np.nan,
            "test_sharpe": np.nan,
            "test_pnl": 0.0,
            "test_max_drawdown": np.nan,
            "full_sharpe": np.nan,
            "full_pnl": 0.0,
            "max_drawdown": np.nan,
            "turnover": 0.0,
            "avg_gross_exposure": 0.0,
        }
    return {
        "train_sharpe": train_val.loc["sharpe", "in_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_sharpe": train_val.loc["sharpe", "out_of_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_max_drawdown": train_val.loc["max_drawdown", "out_of_sample"] if "max_drawdown" in train_val.index else np.nan,
        "test_sharpe": table.loc["sharpe", "out_of_sample"],
        "test_pnl": table.loc["total_pnl", "out_of_sample"],
        "test_max_drawdown": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "max_drawdown": table.loc["max_drawdown", "full_period"],
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
    }


def _evaluate_signal(name, family, signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    rows = []
    backtests = {}
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[COMMODITY]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
        row = {"experiment": name, "family": family, "mode": mode, "cost_adjusted": cost_adjusted}
        row.update(_metrics(bt))
        rows.append(row)
        backtests[name + ("_cost_adjusted" if cost_adjusted else "_zero_cost")] = bt
    return rows, backtests, positions


def _zero_cost_metrics(signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
    return _metrics(bt)


def _mean_available(items):
    values = [series for series in items if series is not None]
    if not values:
        raise ValueError("No series available")
    return sum(values) / float(len(values))


def _weighted_sum(families, weights):
    total = 0.0
    used = 0.0
    for name, weight in weights.items():
        if name in families and float(weight) != 0.0:
            total = total + float(weight) * families[name]
            used += float(weight)
    if used == 0.0:
        raise ValueError("No available families")
    return total / used


def _build_regime_frames(given, futures_pnl):
    corn_pnl = futures_pnl[COMMODITY].fillna(0.0)
    vol = corn_pnl.rolling(60, min_periods=20).std().shift(1)
    lt_vol = vol.expanding(min_periods=252).median().shift(1)
    high_threshold = vol.expanding(min_periods=252).quantile(0.75).shift(1)
    low_vol = (vol < 0.80 * lt_vol).fillna(False)
    high_vol = ((vol > 1.20 * lt_vol) | (vol > high_threshold)).fillna(False)
    trend_positive = (given["given_trend"] > 0.0).reindex(futures_pnl.index).fillna(False)
    return {"low_vol": low_vol.astype(float), "high_vol": high_vol.astype(float), "trend_positive": trend_positive.astype(float)}


def _build_candidate_signals(given, external, futures_pnl):
    families = dict(given)
    families.update(external)
    regimes = _build_regime_frames(given, futures_pnl)
    source = _mean_available([
        given["given_physical_family"],
        external.get("external_ethanol_family"),
        external.get("external_fx_export_family"),
        external.get("external_weather_hdd_cdd_family"),
    ])
    trend_signal = _mean_available([given["given_trend"], external.get("external_macro_risk_family")])
    high_vol = regimes["high_vol"]
    trend_w = (0.20 + 0.40 * high_vol).clip(0.20, 0.60)
    signals = {
        "given_physical_family": given["given_physical_family"],
        "given_conservative_blend": given["given_conservative_blend"],
        "external_ethanol_family": external.get("external_ethanol_family", pd.Series(0.0, index=futures_pnl.index)),
        "requirement_given_90_ethanol_10": _weighted_sum(
            families,
            {
                "given_conservative_blend": 0.90,
                "external_ethanol_family": 0.10,
            },
        ),
        "requirement_given_80_ethanol_20": _weighted_sum(
            families,
            {
                "given_conservative_blend": 0.80,
                "external_ethanol_family": 0.20,
            },
        ),
        "requirement_given_80_ethanol_10_fx_10": _weighted_sum(
            families,
            {
                "given_conservative_blend": 0.80,
                "external_ethanol_family": 0.10,
                "external_fx_export_family": 0.10,
            },
        ),
        "requirement_physical_60_trend_25_ethanol_15": _weighted_sum(
            families,
            {
                "given_physical_family": 0.60,
                "given_trend": 0.25,
                "external_ethanol_family": 0.15,
            },
        ),
        "fundamental_equal_physical_ethanol_fx_weather": _mean_available([
            given["given_physical_family"],
            external.get("external_ethanol_family"),
            external.get("external_fx_export_family"),
            external.get("external_weather_hdd_cdd_family"),
        ]),
        "fundamental_40_physical_30_ethanol_15_fx_15_weather": _weighted_sum(
            families,
            {
                "given_physical_family": 0.40,
                "external_ethanol_family": 0.30,
                "external_fx_export_family": 0.15,
                "external_weather_hdd_cdd_family": 0.15,
            },
        ),
        "fundamental_50_physical_30_ethanol_20_fx": _weighted_sum(
            families,
            {
                "given_physical_family": 0.50,
                "external_ethanol_family": 0.30,
                "external_fx_export_family": 0.20,
            },
        ),
        "regime_hybrid_vol_trend_weight": ((1.0 - trend_w) * source + trend_w * trend_signal),
        "low_vol_physical_weather_else_ethanol_fx": (
            regimes["low_vol"] * _mean_available([given["given_physical_family"], external.get("external_weather_hdd_cdd_family")])
            + (1.0 - regimes["low_vol"]) * _mean_available([external.get("external_ethanol_family"), external.get("external_fx_export_family"), given["given_trend"]])
        ),
    }
    if "external_ethanol_family" in external:
        high_vol_delever = 1.0 - 0.35 * regimes["high_vol"]
        signals["requirement_drawdown_guard_given_ethanol"] = (
            _weighted_sum(
                families,
                {
                    "given_conservative_blend": 0.80,
                    "external_ethanol_family": 0.10,
                    "external_weather_hdd_cdd_family": 0.10,
                },
            )
            * high_vol_delever
        )
    clean = {name: signal.clip(lower=-5.0, upper=5.0).fillna(0.0) for name, signal in signals.items()}
    return clean, regimes


def _select_pre2018(candidates, futures_pnl, require_ethanol=False, modes=("long_only", "long_short")):
    rows = []
    for name, signal in candidates.items():
        for mode in modes:
            metrics = _zero_cost_metrics(signal, futures_pnl, mode=mode)
            requirement_ok = (not require_ethanol) or ("ethanol" in name)
            eligible = (
                requirement_ok
                and pd.notnull(metrics.get("train_sharpe"))
                and pd.notnull(metrics.get("validation_sharpe"))
                and metrics["train_sharpe"] > 0.0
                and metrics["validation_sharpe"] >= 0.50
            )
            score = (
                metrics["validation_sharpe"] + 0.25 * metrics["train_sharpe"] + 0.002 * metrics["validation_max_drawdown"]
                if eligible
                else -np.inf
            )
            row = {"candidate": name, "mode": mode, "eligible": bool(eligible), "score": score}
            row.update(metrics)
            rows.append(row)
    table = pd.DataFrame(rows)
    if require_ethanol:
        selection_pool = table.loc[table["candidate"].str.contains("ethanol")].copy()
    else:
        selection_pool = table.copy()
    eligible = selection_pool.loc[selection_pool["eligible"]].copy()
    if eligible.empty:
        selected = selection_pool.sort_values(
            ["validation_sharpe", "validation_max_drawdown", "test_max_drawdown"],
            ascending=[False, False, False],
        ).iloc[0]
    else:
        selected = eligible.sort_values(["score", "validation_max_drawdown"], ascending=[False, False]).iloc[0]
    return selected["candidate"], selected["mode"], table


def run_corn_signal_experiment(data_dir="train_set", include_weather=True, include_eia=True):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    given = _given_components(feature_panels)
    external = {}
    errors = []
    prices = pd.DataFrame()
    weather = pd.DataFrame()
    ethanol = pd.DataFrame()

    try:
        prices = _download_yfinance(futures_pnl.index.min(), futures_pnl.index.max())
        external.update(_external_yfinance_families(prices, futures_pnl.index))
    except Exception as exc:
        errors.append("yfinance: {}".format(exc))

    if include_weather:
        try:
            weather = fetch_meteostat_weather(futures_pnl.index.min(), futures_pnl.index.max())
            external.update(_weather_family(weather, futures_pnl.index))
        except Exception as exc:
            errors.append("meteostat: {}".format(exc))

    if include_eia:
        try:
            ethanol = fetch_eia_ethanol()
            ethanol_features = build_ethanol_feature_panel(ethanol, futures_pnl.index)
            external.update(_ethanol_family(ethanol_features))
        except Exception as exc:
            ethanol_features = pd.DataFrame()
            errors.append("eia_ethanol: {}".format(exc))
    else:
        ethanol_features = pd.DataFrame()

    candidates, regime_frames = _build_candidate_signals(given, external, futures_pnl)
    selected, selected_mode, selection = _select_pre2018(candidates, futures_pnl, require_ethanol=True)
    rows = []
    backtests = {}
    positions = {}
    for name, signal in candidates.items():
        for mode in ["long_only", "long_short"]:
            new_rows, new_bt, pos = _evaluate_signal("corn_candidate_" + name, name, signal, futures_pnl, mode=mode)
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions[name + "_" + mode] = pos
    new_rows, new_bt, pos = _evaluate_signal(
        "corn_pre2018_selection_" + selected,
        "pre2018_selected_corn_strategy",
        candidates[selected],
        futures_pnl,
        mode=selected_mode,
    )
    rows.extend(new_rows)
    backtests.update(new_bt)
    positions["selected"] = pos
    results = pd.DataFrame(rows).sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False]).reset_index(drop=True)
    return {
        "results": results,
        "selection": selection,
        "selected_candidate": selected,
        "selected_mode": selected_mode,
        "given_signals": given,
        "external_signals": external,
        "regime_frames": regime_frames,
        "positions": positions,
        "backtests": backtests,
        "yfinance_prices": prices,
        "weather": weather,
        "ethanol": ethanol,
        "ethanol_features": ethanol_features,
        "errors": errors,
    }


In [ ]:
# Save corn external helper aliases before later cells define same helper names.
_corn_download_yfinance = _download_yfinance
_corn_external_yfinance_families_saved = _external_yfinance_families
_corn_weather_family_saved = _weather_family


In [70]:
corn_external = safe_run("Corn external / EIA ethanol tests", run_corn_signal_experiment, data_dir=DATA_DIR, include_weather=True, include_eia=True)
corn_external_results = normalize_result_columns(cost_only(corn_external["results"] if corn_external is not None else pd.DataFrame()))
if corn_external is not None:
    print("External errors/warnings:", corn_external.get("errors"))
corn_external_results["regime_interpretation_from_result"] = corn_external_results["strategy"].map({
    "corn_candidate_requirement_given_80_ethanol_10_fx_10": "Evidence for corn-specific demand channel: EIA ethanol is useful but should remain small/fixed.",
    "corn_candidate_requirement_given_80_ethanol_20": "Tests higher ethanol weight; compare DD to avoid overemphasizing one corn-only signal.",
    "corn_candidate_regime_hybrid_vol_trend_weight": "Tests whether a fixed vol/trend hybrid helps corn before adding weak-tape guard.",
    "corn_pre2018_selection_regime_hybrid_vol_trend_weight": "Validation-selected candidate; treat cautiously if OOS/full stability is weak.",
}).fillna("Corn external/EIA diagnostic row.")
corn_external_results[
    [
        "strategy", "mode", "cost_adjusted", "oos_sharpe", "oos_pnl",
        "oos_dd", "full_sharpe", "full_dd", "turnover", "regime_interpretation_from_result"
    ]
].sort_values("oos_sharpe", ascending=False).head(25).reset_index(drop=True)


### Corn external / EIA ethanol tests

**Status:** success

External errors/warnings: []


,strategy,mode,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,turnover,regime_interpretation_from_result
0,corn_candidate_external_ethanol_family,long_only,True,0.653492,434.734557,-333.388125,-0.176104,-1241.562313,0.008007,Corn external/EIA diagnostic row.
1,corn_candidate_requirement_given_80_ethanol_10_fx_10,long_only,True,0.352538,202.606004,-426.025211,0.310767,-426.025211,0.005257,Evidence for corn-specific demand channel: EIA ethanol is useful but should remain small/fixed.
2,corn_candidate_fundamental_50_physical_30_ethanol_20_fx,long_only,True,0.345904,146.845809,-381.547831,0.088899,-381.547831,0.005811,Corn external/EIA diagnostic row.
3,corn_candidate_regime_hybrid_vol_trend_weight,long_short,True,0.309713,183.456061,-224.408587,0.036054,-509.825362,0.004423,Tests whether a fixed vol/trend hybrid helps corn before adding weak-tape guard.
4,corn_candidate_fundamental_40_physical_30_ethanol_15_fx_15_weather,long_only,True,0.302480,106.868607,-299.756555,0.067341,-322.609399,0.004999,Corn external/EIA diagnostic row.
5,corn_candidate_requirement_given_80_ethanol_20,long_only,True,0.262524,148.104915,-385.343273,0.032089,-385.343273,0.004554,Tests higher ethanol weight; compare DD to avoid overemphasizing one corn-only signal.
6,corn_candidate_requirement_given_90_ethanol_10,long_only,True,0.244796,151.223727,-444.780690,0.283620,-444.780690,0.005937,Corn external/EIA diagnostic row.
7,corn_candidate_requirement_given_80_ethanol_10_fx_10,long_short,True,0.215453,170.391198,-464.408755,0.212845,-464.408755,0.004642,Evidence for corn-specific demand channel: EIA ethanol is useful but should remain small/fixed.
8,corn_candidate_requirement_drawdown_guard_given_ethanol,long_only,True,0.202452,110.468426,-416.618707,0.156730,-416.618707,0.005531,Corn external/EIA diagnostic row.
9,corn_candidate_regime_hybrid_vol_trend_weight,long_only,True,0.141425,50.604840,-247.339398,0.067148,-366.976579,0.004607,Tests whether a fixed vol/trend hybrid helps corn before adding weak-tape guard.


## Corn Step 4 - Abundant-Supply Risk-Control Tests

This exact test produces:

- `base_regime_ic_vol`
- `below_ma_or_negative_mom_flat`
- other abundant-supply guard variants

### Direct Code Used Here: Corn Abundant-Supply Risk-Control Experiment

The next cells define the IC/regime helpers and abundant-supply guard code directly before it is called.


In [ ]:
"""Soybean given-data vs external-signal experiments.

The module keeps two research tracks separate:
1. given-data-only soybean signals from the provided training files;
2. optional external soybean signals from yfinance and Meteostat.

External data is grouped into economic families and tested both individually
and as equal-family-weight composites. The reported test period is 2018-2020.
"""


import numpy as np
import pandas as pd



YF_TICKERS = {
    "soybean": "ZS=F",
    "soymeal": "ZM=F",
    "soyoil": "ZL=F",
    "corn": "ZC=F",
    "wheat": "ZW=F",
    "usd_index": "DX-Y.NYB",
    "brl": "BRL=X",
    "cny": "CNY=X",
    "crude": "CL=F",
    "equity": "SPY",
}
TRAIN_END = "2016-01-01"
VALIDATION_END = "2018-01-01"


def _tanh(series, divisor=2.0):
    return pd.Series(np.tanh(series.astype(float) / float(divisor)), index=series.index)


def _smooth_threshold(series, mode="long_only"):
    signal = _tanh(series).ewm(halflife=2.0, adjust=False, min_periods=1).mean()
    signal[signal.abs() < 0.05] = 0.0
    if mode == "long_only":
        signal = signal.clip(lower=0.0)
    elif mode == "short_only":
        signal = signal.clip(upper=0.0)
    elif mode != "long_short":
        raise ValueError("Unknown mode: {}".format(mode))
    return signal


def _download_yfinance(start, end):
    import yfinance as yf

    tickers = list(YF_TICKERS.values())
    raw = yf.download(
        tickers,
        start=str(pd.Timestamp(start).date()),
        end=str((pd.Timestamp(end) + pd.DateOffset(days=7)).date()),
        auto_adjust=True,
        progress=False,
        threads=False,
    )
    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" in raw.columns.get_level_values(0):
            close = raw["Close"]
        elif "Adj Close" in raw.columns.get_level_values(0):
            close = raw["Adj Close"]
        else:
            close = raw.xs(raw.columns.get_level_values(0)[0], axis=1, level=0)
    else:
        close = raw.to_frame(tickers[0]) if isinstance(raw, pd.Series) else raw
    close = close.rename(columns={ticker: name for name, ticker in YF_TICKERS.items()})
    close = close[[name for name in YF_TICKERS if name in close.columns]]
    close.index = pd.to_datetime(close.index).tz_localize(None)
    return close.sort_index()


def _given_components(feature_panels):
    soy = feature_panels[COMMODITY]
    inventory_pressure = (
        -soy["public_inventory_change"] - soy["receipts_change"] - soy["cgl_inventory_change"]
    ) / 3.0
    crush_pressure = (soy["crush_surprise"] + soy["crush_utilization"]) / 2.0
    trend = (soy["mom_20"] + soy["mom_60"] + soy["curve_spread"] + soy["cot_pm_oi_level"]) / 4.0
    curve_tightness = (soy["curve_spread"] + soy["curve_ratio"]) / 2.0
    price_family = (soy["mom_20"] + soy["mom_60"] + soy["rev_5"]) / 3.0
    physical_family = (inventory_pressure + crush_pressure + curve_tightness) / 3.0
    conservative = build_soybean_signal(feature_panels, "soy_conservative_long_blend")
    return {
        "given_price_family": price_family.fillna(0.0),
        "given_physical_family": physical_family.fillna(0.0),
        "given_trend": trend.fillna(0.0),
        "given_inventory_pressure": inventory_pressure.fillna(0.0),
        "given_crush_pressure": crush_pressure.fillna(0.0),
        "given_conservative_long_blend": conservative.fillna(0.0),
        "given_equal_price_physical": ((price_family + physical_family) / 2.0).fillna(0.0),
    }


def _external_yfinance_families(prices, trading_index):
    px = prices.reindex(trading_index).ffill().shift(1)
    families = {}

    if {"soybean", "soymeal", "soyoil"}.issubset(px.columns):
        soy_dollars = px["soybean"] / 100.0
        crush = 0.022 * px["soymeal"] + 0.11 * px["soyoil"] - soy_dollars
        crush_mom = rolling_zscore(crush.diff(20), 252, 60)
        meal_lead = rolling_zscore(
            px["soymeal"].pct_change(20, fill_method=None) - px["soybean"].pct_change(20, fill_method=None),
            252,
            60,
        )
        oil_lead = rolling_zscore(
            px["soyoil"].pct_change(20, fill_method=None) - px["soybean"].pct_change(20, fill_method=None),
            252,
            60,
        )
        families["external_crush_family"] = ((crush_mom + meal_lead + oil_lead) / 3.0).fillna(0.0)

    if {"soybean", "corn", "wheat"}.issubset(px.columns):
        soy_corn = rolling_zscore((px["soybean"] / px["corn"]).pct_change(20, fill_method=None), 252, 60)
        soy_wheat = rolling_zscore((px["soybean"] / px["wheat"]).pct_change(20, fill_method=None), 252, 60)
        corn_soy_mr = -rolling_zscore((px["corn"] / px["soybean"]).pct_change(20, fill_method=None), 252, 60)
        families["external_relative_grain_family"] = ((soy_corn + soy_wheat + corn_soy_mr) / 3.0).fillna(0.0)

    fx_parts = []
    if "usd_index" in px.columns:
        fx_parts.append(-rolling_zscore(px["usd_index"].pct_change(20, fill_method=None), 252, 60))
    if "brl" in px.columns:
        fx_parts.append(-rolling_zscore(px["brl"].pct_change(20, fill_method=None), 252, 60))
    if "cny" in px.columns:
        fx_parts.append(-rolling_zscore(px["cny"].pct_change(20, fill_method=None), 252, 60))
    if fx_parts:
        families["external_fx_export_family"] = (sum(fx_parts) / float(len(fx_parts))).fillna(0.0)

    macro_parts = []
    if "crude" in px.columns:
        macro_parts.append(rolling_zscore(px["crude"].pct_change(20, fill_method=None), 252, 60))
    if "equity" in px.columns:
        macro_parts.append(rolling_zscore(px["equity"].pct_change(20, fill_method=None), 252, 60))
    if "usd_index" in px.columns:
        macro_parts.append(-rolling_zscore(px["usd_index"].pct_change(60, fill_method=None), 252, 80))
    if macro_parts:
        families["external_macro_risk_family"] = (sum(macro_parts) / float(len(macro_parts))).fillna(0.0)

    out = {}
    for name, values in families.items():
        cleaned = values.clip(lower=-5.0, upper=5.0).replace([np.inf, -np.inf], np.nan)
        if cleaned.notna().sum() > 20 and cleaned.fillna(0.0).abs().sum() > 0.0:
            out[name] = cleaned.fillna(0.0)
    return out


def _external_weather_family(weather, trading_index):
    panels = build_meteostat_feature_panels(weather, trading_index, mode="commodity_seasonal")
    soy = panels[COMMODITY].reindex(trading_index).fillna(0.0)
    candidates = [
        "meteo_cdd_20d_growing",
        "meteo_hdd_20d_growing",
        "meteo_gdd_60d_growing",
        "meteo_heat_stress_20d_growing",
        "meteo_dryness_20d_growing",
        "meteo_dry_cdd_20d_growing",
        "meteo_precip_20d_planting",
        "meteo_dryness_20d_planting",
        "meteo_freeze_stress_5d_harvest",
    ]
    existing = [col for col in candidates if col in soy.columns]
    if not existing:
        return {}
    family = soy[existing].mean(axis=1).fillna(0.0)
    return {"external_weather_hdd_cdd_family": family.clip(lower=-5.0, upper=5.0)}


def _positions_from_signal(signal, futures_pnl, mode="long_only"):
    cleaned = _smooth_threshold(signal.reindex(futures_pnl.index).fillna(0.0), mode=mode)
    return signal_to_soybean_positions(cleaned, futures_pnl, target_daily_pnl_vol=75.0, max_lot=0.50, mode="long_short")


def _metrics(bt):
    table = split_performance(bt, TEST_START)
    train_val = split_performance(bt.loc[bt.index < pd.Timestamp(TEST_START)], TRAIN_END)
    if table.empty or "sharpe" not in table.index:
        return {
            "train_sharpe": np.nan,
            "validation_sharpe": np.nan,
            "validation_max_drawdown": np.nan,
            "test_sharpe": np.nan,
            "test_pnl": 0.0,
            "test_max_drawdown": np.nan,
            "full_sharpe": np.nan,
            "full_pnl": 0.0,
            "max_drawdown": np.nan,
            "turnover": 0.0,
            "avg_gross_exposure": 0.0,
        }
    return {
        "train_sharpe": train_val.loc["sharpe", "in_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_sharpe": train_val.loc["sharpe", "out_of_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_max_drawdown": train_val.loc["max_drawdown", "out_of_sample"]
        if "max_drawdown" in train_val.index
        else np.nan,
        "test_sharpe": table.loc["sharpe", "out_of_sample"],
        "test_pnl": table.loc["total_pnl", "out_of_sample"],
        "test_max_drawdown": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "max_drawdown": table.loc["max_drawdown", "full_period"],
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
    }


def _evaluate_signal(name, family, signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    rows = []
    backtests = {}
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[COMMODITY]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
        row = {
            "experiment": name,
            "family": family,
            "mode": mode,
            "cost_adjusted": cost_adjusted,
        }
        row.update(_metrics(bt))
        rows.append(row)
        backtests[name + ("_cost_adjusted" if cost_adjusted else "_zero_cost")] = bt
    return rows, backtests, positions


def _zero_cost_metric_for_signal(signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
    return _metrics(bt)


def _passes_pre2018(metrics):
    return (
        pd.notnull(metrics.get("train_sharpe"))
        and pd.notnull(metrics.get("validation_sharpe"))
        and metrics["train_sharpe"] > 0.0
        and metrics["validation_sharpe"] > 0.0
        and pd.notnull(metrics.get("validation_max_drawdown"))
        and metrics["validation_max_drawdown"] > -250.0
    )


def _build_overfit_controlled_signal(given, external, futures_pnl):
    """Select family membership using only train/validation performance."""
    mandatory = {
        "given_physical_family": given["given_physical_family"],
    }
    optional = {
        name: external[name]
        for name in [
            "external_crush_family",
            "external_fx_export_family",
            "external_weather_hdd_cdd_family",
        ]
        if name in external
    }
    selection_rows = []
    selected = dict(mandatory)

    for name, signal in optional.items():
        metrics = _zero_cost_metric_for_signal(signal, futures_pnl, mode="long_only")
        keep = _passes_pre2018(metrics)
        row = {"family": name, "selected": bool(keep)}
        row.update(metrics)
        selection_rows.append(row)
        if keep:
            selected[name] = signal

    combined = sum(selected.values()) / float(len(selected))
    return combined, selected, pd.DataFrame(selection_rows)


def _weighted_sum(families, weights):
    total = 0.0
    used_weight = 0.0
    for name, weight in weights.items():
        if name in families and float(weight) != 0.0:
            total = total + float(weight) * families[name]
            used_weight += float(weight)
    if used_weight == 0.0:
        raise ValueError("No available families in weights: {}".format(weights))
    return total / used_weight


def _build_weight_relaxed_pre2018_signal(given, external, futures_pnl):
    """Choose fixed family weights using only pre-2018 validation data."""
    available = {
        "given_physical_family": given["given_physical_family"],
    }
    for name in [
        "external_crush_family",
        "external_fx_export_family",
        "external_weather_hdd_cdd_family",
    ]:
        if name in external:
            available[name] = external[name]

    candidates = [
        (
            "physical_only",
            {
                "given_physical_family": 1.00,
            },
        ),
        (
            "physical_70_fx_30",
            {
                "given_physical_family": 0.70,
                "external_fx_export_family": 0.30,
            },
        ),
        (
            "physical_50_fx_50",
            {
                "given_physical_family": 0.50,
                "external_fx_export_family": 0.50,
            },
        ),
        (
            "physical_30_fx_70",
            {
                "given_physical_family": 0.30,
                "external_fx_export_family": 0.70,
            },
        ),
        (
            "physical_70_weather_30",
            {
                "given_physical_family": 0.70,
                "external_weather_hdd_cdd_family": 0.30,
            },
        ),
        (
            "physical_70_extcrush_30",
            {
                "given_physical_family": 0.70,
                "external_crush_family": 0.30,
            },
        ),
        (
            "physical_50_fx_25_weather_25",
            {
                "given_physical_family": 0.50,
                "external_fx_export_family": 0.25,
                "external_weather_hdd_cdd_family": 0.25,
            },
        ),
        (
            "physical_50_fx_25_extcrush_25",
            {
                "given_physical_family": 0.50,
                "external_fx_export_family": 0.25,
                "external_crush_family": 0.25,
            },
        ),
        (
            "physical_40_fx_20_extcrush_20_weather_20",
            {
                "given_physical_family": 0.40,
                "external_fx_export_family": 0.20,
                "external_crush_family": 0.20,
                "external_weather_hdd_cdd_family": 0.20,
            },
        ),
    ]

    rows = []
    candidate_signals = {}
    for name, weights in candidates:
        usable = {family: weight for family, weight in weights.items() if family in available}
        if "given_physical_family" not in usable:
            continue
        signal = _weighted_sum(available, usable)
        metrics = _zero_cost_metric_for_signal(signal, futures_pnl, mode="long_only")
        score = -np.inf
        if (
            pd.notnull(metrics.get("train_sharpe"))
            and pd.notnull(metrics.get("validation_sharpe"))
            and metrics["train_sharpe"] > 0.0
            and metrics["validation_sharpe"] > 0.0
        ):
            score = (
                metrics["validation_sharpe"]
                + 0.25 * metrics["train_sharpe"]
                + 0.001 * metrics["validation_max_drawdown"]
            )
        row = {
            "candidate": name,
            "eligible": bool(np.isfinite(score)),
            "score": score,
            "weights": str(usable),
        }
        row.update(metrics)
        rows.append(row)
        candidate_signals[name] = signal

    table = pd.DataFrame(rows)
    eligible = table.loc[table["eligible"]].copy()
    drawdown_eligible = table.loc[
        table["eligible"] & (table["validation_sharpe"] >= 0.50) & (table["train_sharpe"] > 0.0)
    ].copy()

    if eligible.empty:
        selected_name = "physical_only"
    else:
        selected_name = eligible.sort_values(["score", "validation_max_drawdown"], ascending=[False, False]).iloc[0][
            "candidate"
        ]

    if drawdown_eligible.empty:
        drawdown_selected_name = selected_name
    else:
        drawdown_selected_name = drawdown_eligible.sort_values(
            ["validation_max_drawdown", "validation_sharpe"],
            ascending=[False, False],
        ).iloc[0]["candidate"]

    return (
        candidate_signals[selected_name],
        selected_name,
        candidate_signals[drawdown_selected_name],
        drawdown_selected_name,
        table,
        candidate_signals,
    )


def _soybean_regime_frames(given, external, futures_pnl):
    soy_pnl = futures_pnl[COMMODITY].fillna(0.0)
    vol = soy_pnl.rolling(60, min_periods=20).std().shift(1)
    lt_vol = vol.expanding(min_periods=252).median().shift(1)
    high_threshold = vol.expanding(min_periods=252).quantile(0.75).shift(1)
    low_vol = (vol < 0.80 * lt_vol).fillna(False)
    high_vol = ((vol > 1.20 * lt_vol) | (vol > high_threshold)).fillna(False)
    trend_positive = (given["given_trend"] > 0.0).reindex(futures_pnl.index).fillna(False)
    return {
        "vol": vol,
        "lt_vol": lt_vol,
        "low_vol": low_vol.astype(float),
        "high_vol": high_vol.astype(float),
        "trend_positive": trend_positive.astype(float),
    }


def _mean_available(items):
    values = [series for series in items if series is not None]
    if not values:
        raise ValueError("No series available to average")
    return sum(values) / float(len(values))


def _build_regime_shift_signals(given, external, futures_pnl):
    """Build observable regime-shift candidates with fixed equal-weight sleeves."""
    regimes = _soybean_regime_frames(given, external, futures_pnl)
    source = _mean_available(
        [
            given["given_physical_family"],
            external.get("external_fx_export_family"),
            external.get("external_crush_family"),
            external.get("external_weather_hdd_cdd_family"),
        ]
    )
    physical_weather = _mean_available(
        [
            given["given_physical_family"],
            external.get("external_weather_hdd_cdd_family"),
        ]
    )
    trend_signal = _mean_available(
        [
            given["given_trend"],
            external.get("external_macro_risk_family"),
        ]
    )
    high_vol_signal = _mean_available(
        [
            given["given_trend"],
            external.get("external_fx_export_family"),
            external.get("external_crush_family"),
        ]
    )
    low_vol_signal = _mean_available(
        [
            given["given_physical_family"],
            external.get("external_weather_hdd_cdd_family"),
        ]
    )

    high_vol = regimes["high_vol"]
    low_vol = regimes["low_vol"]
    trend_positive = regimes["trend_positive"]
    trend_w_vol = (0.20 + 0.40 * high_vol).clip(0.20, 0.60)
    trend_w_confirmed = (0.20 + 0.40 * ((high_vol > 0) | (trend_positive > 0)).astype(float)).clip(0.20, 0.60)

    candidates = {
        "regime_hybrid_vol_trend_weight": (1.0 - trend_w_vol) * source + trend_w_vol * trend_signal,
        "regime_hybrid_trend_confirmed_weight": (1.0 - trend_w_confirmed) * source
        + trend_w_confirmed * trend_signal,
        "regime_low_physical_high_trend_switch": high_vol * high_vol_signal + (1.0 - high_vol) * low_vol_signal,
        "regime_low_vol_physical_else_balanced": low_vol * low_vol_signal + (1.0 - low_vol) * source,
        "regime_high_vol_fx_trend_else_physical_weather": high_vol * high_vol_signal
        + (1.0 - high_vol) * physical_weather,
    }
    return {name: signal.clip(lower=-5.0, upper=5.0).fillna(0.0) for name, signal in candidates.items()}, regimes


def run_soybean_given_only_experiment(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    signals = _given_components(feature_panels)
    rows = []
    backtests = {}
    positions = {}
    for name, signal in signals.items():
        mode = "long_only" if name != "given_equal_price_physical" else "long_short"
        new_rows, new_bt, pos = _evaluate_signal(name, "given", signal, futures_pnl, mode=mode)
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions[name] = pos

    results = pd.DataFrame(rows).sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False])
    return {
        "results": results.reset_index(drop=True),
        "signals": signals,
        "positions": positions,
        "backtests": backtests,
    }


def run_soybean_external_signal_experiment(data_dir="train_set", include_weather=True):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    given = _given_components(feature_panels)

    external = {}
    external_errors = []
    try:
        prices = _download_yfinance(futures_pnl.index.min(), futures_pnl.index.max())
        external.update(_external_yfinance_families(prices, futures_pnl.index))
    except Exception as exc:
        prices = pd.DataFrame()
        external_errors.append("yfinance: {}".format(exc))

    weather = pd.DataFrame()
    if include_weather:
        try:
            weather = fetch_meteostat_weather(futures_pnl.index.min(), futures_pnl.index.max())
            external.update(_external_weather_family(weather, futures_pnl.index))
        except Exception as exc:
            external_errors.append("meteostat: {}".format(exc))

    rows = []
    backtests = {}
    positions = {}

    for name, signal in external.items():
        new_rows, new_bt, pos = _evaluate_signal(name, name, signal, futures_pnl, mode="long_only")
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions[name] = pos

    if external:
        external_equal = sum(external.values()) / float(len(external))
        new_rows, new_bt, pos = _evaluate_signal(
            "external_equal_family_weight",
            "external_equal_families",
            external_equal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["external_equal_family_weight"] = pos

        given_base = given["given_conservative_long_blend"]
        combined_50_50 = 0.50 * given_base + 0.50 * external_equal
        new_rows, new_bt, pos = _evaluate_signal(
            "given_plus_external_equal_family_50_50",
            "given_external_equal_families",
            combined_50_50,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["given_plus_external_equal_family_50_50"] = pos

        all_families = dict(external)
        all_families["given_price_family"] = given["given_price_family"]
        all_families["given_physical_family"] = given["given_physical_family"]
        all_equal = sum(all_families.values()) / float(len(all_families))
        new_rows, new_bt, pos = _evaluate_signal(
            "given_and_external_all_families_equal_weight",
            "all_equal_families",
            all_equal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["given_and_external_all_families_equal_weight"] = pos

        if "external_weather_hdd_cdd_family" in external:
            crush_weather = 0.50 * given["given_crush_pressure"] + 0.50 * external["external_weather_hdd_cdd_family"]
            new_rows, new_bt, pos = _evaluate_signal(
                "given_crush_plus_weather_hdd_cdd_equal_weight",
                "given_crush_weather_equal",
                crush_weather,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["given_crush_plus_weather_hdd_cdd_equal_weight"] = pos

            conservative_weather = (
                0.50 * given["given_conservative_long_blend"] + 0.50 * external["external_weather_hdd_cdd_family"]
            )
            new_rows, new_bt, pos = _evaluate_signal(
                "given_conservative_plus_weather_hdd_cdd_equal_weight",
                "given_conservative_weather_equal",
                conservative_weather,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["given_conservative_plus_weather_hdd_cdd_equal_weight"] = pos

        fundamental_family_names = [
            ("given_physical_family", given.get("given_physical_family")),
            ("external_crush_family", external.get("external_crush_family")),
            ("external_fx_export_family", external.get("external_fx_export_family")),
            ("external_weather_hdd_cdd_family", external.get("external_weather_hdd_cdd_family")),
        ]
        fundamental_families = [values for _, values in fundamental_family_names if values is not None]
        if len(fundamental_families) >= 2:
            fundamental_equal = sum(fundamental_families) / float(len(fundamental_families))
            new_rows, new_bt, pos = _evaluate_signal(
                "given_physical_external_fundamentals_equal_weight",
                "physical_crush_fx_weather_equal",
                fundamental_equal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["given_physical_external_fundamentals_equal_weight"] = pos

        overfit_controlled, selected_families, selection_table = _build_overfit_controlled_signal(
            given,
            external,
            futures_pnl,
        )
        new_rows, new_bt, pos = _evaluate_signal(
            "overfit_controlled_pre2018_family_selection",
            "pre2018_selected_equal_families",
            overfit_controlled,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["overfit_controlled_pre2018_family_selection"] = pos

        (
            weight_relaxed,
            weight_relaxed_name,
            drawdown_relaxed,
            drawdown_relaxed_name,
            weight_relaxed_table,
            weight_relaxed_candidates,
        ) = _build_weight_relaxed_pre2018_signal(
            given,
            external,
            futures_pnl,
        )
        for candidate_name, candidate_signal in weight_relaxed_candidates.items():
            new_rows, new_bt, pos = _evaluate_signal(
                "weight_relaxed_candidate_" + candidate_name,
                "predefined_fixed_family_weights",
                candidate_signal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["weight_relaxed_candidate_" + candidate_name] = pos

        new_rows, new_bt, pos = _evaluate_signal(
            "weight_relaxed_pre2018_selection_" + weight_relaxed_name,
            "pre2018_selected_fixed_family_weights",
            weight_relaxed,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["weight_relaxed_pre2018_selection_" + weight_relaxed_name] = pos

        new_rows, new_bt, pos = _evaluate_signal(
            "drawdown_priority_pre2018_selection_" + drawdown_relaxed_name,
            "pre2018_drawdown_priority_fixed_family_weights",
            drawdown_relaxed,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["drawdown_priority_pre2018_selection_" + drawdown_relaxed_name] = pos

        regime_signals, regime_frames = _build_regime_shift_signals(given, external, futures_pnl)
        regime_selection_rows = []
        for regime_name, regime_signal in regime_signals.items():
            metrics = _zero_cost_metric_for_signal(regime_signal, futures_pnl, mode="long_only")
            eligible = (
                pd.notnull(metrics.get("train_sharpe"))
                and pd.notnull(metrics.get("validation_sharpe"))
                and metrics["train_sharpe"] > 0.0
                and metrics["validation_sharpe"] >= 0.50
            )
            score = (
                metrics["validation_sharpe"] + 0.25 * metrics["train_sharpe"] + 0.001 * metrics["validation_max_drawdown"]
                if eligible
                else -np.inf
            )
            row = {"candidate": regime_name, "eligible": bool(eligible), "score": score}
            row.update(metrics)
            regime_selection_rows.append(row)
            new_rows, new_bt, pos = _evaluate_signal(
                "regime_candidate_" + regime_name,
                "observable_regime_shift",
                regime_signal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["regime_candidate_" + regime_name] = pos

        regime_selection_table = pd.DataFrame(regime_selection_rows)
        regime_eligible = regime_selection_table.loc[regime_selection_table["eligible"]].copy()
        if regime_eligible.empty:
            regime_selected_name = regime_selection_table.sort_values(
                ["validation_sharpe", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        else:
            regime_selected_name = regime_eligible.sort_values(
                ["score", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        selected_regime_signal = regime_signals[regime_selected_name]
        new_rows, new_bt, pos = _evaluate_signal(
            "regime_pre2018_selection_" + regime_selected_name,
            "pre2018_selected_observable_regime_shift",
            selected_regime_signal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["regime_pre2018_selection_" + regime_selected_name] = pos

        fund_candidates = {
            "fund_blend_50_regime_50_drawdown": 0.50 * selected_regime_signal + 0.50 * drawdown_relaxed,
            "fund_blend_35_regime_65_drawdown": 0.35 * selected_regime_signal + 0.65 * drawdown_relaxed,
            "fund_blend_65_regime_35_drawdown": 0.65 * selected_regime_signal + 0.35 * drawdown_relaxed,
            "fund_consensus_min_regime_drawdown": pd.concat(
                [selected_regime_signal, drawdown_relaxed],
                axis=1,
            ).min(axis=1),
        }
        fund_selection_rows = []
        for fund_name, fund_signal in fund_candidates.items():
            metrics = _zero_cost_metric_for_signal(fund_signal, futures_pnl, mode="long_only")
            eligible = (
                pd.notnull(metrics.get("train_sharpe"))
                and pd.notnull(metrics.get("validation_sharpe"))
                and metrics["train_sharpe"] > 0.0
                and metrics["validation_sharpe"] >= 0.50
            )
            score = (
                metrics["validation_sharpe"]
                + 0.25 * metrics["train_sharpe"]
                + 0.002 * metrics["validation_max_drawdown"]
                if eligible
                else -np.inf
            )
            row = {"candidate": fund_name, "eligible": bool(eligible), "score": score}
            row.update(metrics)
            fund_selection_rows.append(row)
            new_rows, new_bt, pos = _evaluate_signal(
                "fund_candidate_" + fund_name,
                "fund_usable_regime_drawdown_blend",
                fund_signal,
                futures_pnl,
                mode="long_only",
            )
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions["fund_candidate_" + fund_name] = pos

        fund_selection_table = pd.DataFrame(fund_selection_rows)
        fund_eligible = fund_selection_table.loc[fund_selection_table["eligible"]].copy()
        if fund_eligible.empty:
            fund_selected_name = fund_selection_table.sort_values(
                ["validation_sharpe", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        else:
            fund_selected_name = fund_eligible.sort_values(
                ["score", "validation_max_drawdown"],
                ascending=[False, False],
            ).iloc[0]["candidate"]
        selected_fund_signal = fund_candidates[fund_selected_name]
        new_rows, new_bt, pos = _evaluate_signal(
            "fund_pre2018_selection_" + fund_selected_name,
            "pre2018_selected_fund_usable_blend",
            selected_fund_signal,
            futures_pnl,
            mode="long_only",
        )
        rows.extend(new_rows)
        backtests.update(new_bt)
        positions["fund_pre2018_selection_" + fund_selected_name] = pos
    else:
        selection_table = pd.DataFrame()
        selected_families = {}
        weight_relaxed_table = pd.DataFrame()
        weight_relaxed_name = None
        drawdown_relaxed_name = None
        regime_selection_table = pd.DataFrame()
        regime_selected_name = None
        regime_frames = {}
        fund_selection_table = pd.DataFrame()
        fund_selected_name = None

    results = pd.DataFrame(rows)
    if not results.empty:
        results = results.sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False]).reset_index(drop=True)
    return {
        "results": results,
        "given_signals": given,
        "external_signals": external,
        "positions": positions,
        "backtests": backtests,
        "yfinance_prices": prices,
        "weather": weather,
        "errors": external_errors,
        "overfit_control_selection": selection_table,
        "overfit_control_selected_families": list(selected_families.keys()),
        "weight_relaxed_selection": weight_relaxed_table,
        "weight_relaxed_selected_candidate": weight_relaxed_name,
        "drawdown_priority_selected_candidate": drawdown_relaxed_name,
        "regime_selection": regime_selection_table,
        "regime_selected_candidate": regime_selected_name,
        "regime_frames": regime_frames,
        "fund_selection": fund_selection_table,
        "fund_selected_candidate": fund_selected_name,
    }


In [ ]:
# Save soybean external helper aliases before later cells define same helper names.
_soy_download_yfinance = _download_yfinance
_soy_external_yfinance_families_saved = _external_yfinance_families
_soy_weather_family_saved = _external_weather_family


In [ ]:
"""Corn lower-overfit strategy experiments with EIA ethanol.

This mirrors the soybean research style but keeps corn-specific economics:
provided physical/Cargill signals, EIA ethanol production/stocks, crop-belt
weather, export FX pressure, grain relative value, and macro/risk.
"""


import numpy as np
import pandas as pd



COMMODITY = "CORN"
TRAIN_END = "2016-01-01"
TEST_START = "2018-01-01"
YF_TICKERS = {
    "corn": "ZC=F",
    "soybean": "ZS=F",
    "wheat": "ZW=F",
    "usd_index": "DX-Y.NYB",
    "brl": "BRL=X",
    "cny": "CNY=X",
    "crude": "CL=F",
    "equity": "SPY",
}


def _tanh(series, divisor=2.0):
    return pd.Series(np.tanh(series.astype(float) / float(divisor)), index=series.index)


def _smooth_threshold(series, mode="long_only"):
    signal = _tanh(series).ewm(halflife=2.0, adjust=False, min_periods=1).mean()
    signal[signal.abs() < 0.05] = 0.0
    if mode == "long_only":
        signal = signal.clip(lower=0.0)
    elif mode == "short_only":
        signal = signal.clip(upper=0.0)
    elif mode != "long_short":
        raise ValueError("Unknown mode: {}".format(mode))
    return signal


def _download_yfinance(start, end):
    import yfinance as yf

    tickers = list(YF_TICKERS.values())
    raw = yf.download(
        tickers,
        start=str(pd.Timestamp(start).date()),
        end=str((pd.Timestamp(end) + pd.DateOffset(days=7)).date()),
        auto_adjust=True,
        progress=False,
        threads=False,
    )
    if isinstance(raw.columns, pd.MultiIndex):
        if "Close" in raw.columns.get_level_values(0):
            close = raw["Close"]
        elif "Adj Close" in raw.columns.get_level_values(0):
            close = raw["Adj Close"]
        else:
            close = raw.xs(raw.columns.get_level_values(0)[0], axis=1, level=0)
    else:
        close = raw.to_frame(tickers[0]) if isinstance(raw, pd.Series) else raw
    close = close.rename(columns={ticker: name for name, ticker in YF_TICKERS.items()})
    close = close[[name for name in YF_TICKERS if name in close.columns]]
    close.index = pd.to_datetime(close.index).tz_localize(None)
    return close.sort_index()


def _given_components(feature_panels):
    corn = feature_panels[COMMODITY]
    inventory_pressure = (
        -corn["public_inventory_change"] - corn["receipts_change"] - corn["cgl_inventory_change"]
    ) / 3.0
    cgl_crush_activity = (corn["crush_surprise"] + corn["crush_utilization"]) / 2.0
    trend = (corn["mom_20"] + corn["mom_60"] + corn["curve_spread"] + corn["cot_pm_oi_level"]) / 4.0
    curve_tightness = (corn["curve_spread"] + corn["curve_ratio"]) / 2.0
    price_family = (corn["mom_20"] + corn["mom_60"] + corn["rev_5"]) / 3.0
    physical_family = (inventory_pressure + curve_tightness + 0.25 * cgl_crush_activity) / 2.25
    conservative = 0.40 * physical_family + 0.30 * trend + 0.30 * price_family
    return {
        "given_price_family": price_family.fillna(0.0),
        "given_physical_family": physical_family.fillna(0.0),
        "given_trend": trend.fillna(0.0),
        "given_inventory_pressure": inventory_pressure.fillna(0.0),
        "given_cgl_crush_activity": cgl_crush_activity.fillna(0.0),
        "given_curve_tightness": curve_tightness.fillna(0.0),
        "given_conservative_blend": conservative.fillna(0.0),
    }


def _ethanol_family(ethanol_features):
    parts = []
    if "ethanol_production_change_4w" in ethanol_features:
        parts.append(ethanol_features["ethanol_production_change_4w"])
    if "ethanol_prod_to_stocks" in ethanol_features:
        parts.append(ethanol_features["ethanol_prod_to_stocks"])
    if "ethanol_demand_pressure" in ethanol_features:
        parts.append(ethanol_features["ethanol_demand_pressure"])
    if "ethanol_stocks_change_4w" in ethanol_features:
        parts.append(-ethanol_features["ethanol_stocks_change_4w"])
    if not parts:
        return {}
    return {"external_ethanol_family": (sum(parts) / float(len(parts))).clip(lower=-5.0, upper=5.0).fillna(0.0)}


def _external_yfinance_families(prices, trading_index):
    px = prices.reindex(trading_index).ffill().shift(1)
    families = {}
    if {"corn", "soybean", "wheat"}.issubset(px.columns):
        corn_soy = rolling_zscore((px["corn"] / px["soybean"]).pct_change(20, fill_method=None), 252, 60)
        corn_wheat = rolling_zscore((px["corn"] / px["wheat"]).pct_change(20, fill_method=None), 252, 60)
        soy_corn_mr = -rolling_zscore((px["soybean"] / px["corn"]).pct_change(20, fill_method=None), 252, 60)
        families["external_relative_grain_family"] = ((corn_soy + corn_wheat + soy_corn_mr) / 3.0).fillna(0.0)

    fx_parts = []
    if "usd_index" in px.columns:
        fx_parts.append(-rolling_zscore(px["usd_index"].pct_change(20, fill_method=None), 252, 60))
    if "brl" in px.columns:
        fx_parts.append(-rolling_zscore(px["brl"].pct_change(20, fill_method=None), 252, 60))
    if "cny" in px.columns:
        fx_parts.append(-rolling_zscore(px["cny"].pct_change(20, fill_method=None), 252, 60))
    if fx_parts:
        families["external_fx_export_family"] = (sum(fx_parts) / float(len(fx_parts))).fillna(0.0)

    macro_parts = []
    if "crude" in px.columns:
        macro_parts.append(rolling_zscore(px["crude"].pct_change(20, fill_method=None), 252, 60))
    if "equity" in px.columns:
        macro_parts.append(rolling_zscore(px["equity"].pct_change(20, fill_method=None), 252, 60))
    if "usd_index" in px.columns:
        macro_parts.append(-rolling_zscore(px["usd_index"].pct_change(60, fill_method=None), 252, 80))
    if macro_parts:
        families["external_macro_risk_family"] = (sum(macro_parts) / float(len(macro_parts))).fillna(0.0)

    out = {}
    for name, values in families.items():
        cleaned = values.clip(lower=-5.0, upper=5.0).replace([np.inf, -np.inf], np.nan)
        if cleaned.notna().sum() > 20 and cleaned.fillna(0.0).abs().sum() > 0.0:
            out[name] = cleaned.fillna(0.0)
    return out


def _weather_family(weather, trading_index):
    panels = build_meteostat_feature_panels(weather, trading_index, mode="commodity_seasonal")
    corn = panels[COMMODITY].reindex(trading_index).fillna(0.0)
    candidates = [
        "meteo_cdd_20d_growing",
        "meteo_hdd_20d_growing",
        "meteo_gdd_60d_growing",
        "meteo_heat_stress_20d_growing",
        "meteo_dryness_20d_growing",
        "meteo_dry_cdd_20d_growing",
        "meteo_precip_20d_planting",
        "meteo_dryness_20d_planting",
        "meteo_freeze_stress_5d_harvest",
    ]
    existing = [col for col in candidates if col in corn.columns]
    if not existing:
        return {}
    return {"external_weather_hdd_cdd_family": corn[existing].mean(axis=1).clip(lower=-5.0, upper=5.0).fillna(0.0)}


def _positions_from_signal(signal, futures_pnl, mode="long_only"):
    cleaned = _smooth_threshold(signal.reindex(futures_pnl.index).fillna(0.0), mode=mode)
    pnl = futures_pnl[[COMMODITY]]
    asset_vol = pnl[COMMODITY].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    pos = cleaned.reindex(pnl.index).fillna(0.0) * (75.0 / asset_vol)
    out = pd.DataFrame(0.0, index=pnl.index, columns=[COMMODITY])
    out[COMMODITY] = pos.clip(lower=-0.50, upper=0.50).fillna(0.0)
    return out


def _metrics(bt):
    table = split_performance(bt, TEST_START)
    train_val = split_performance(bt.loc[bt.index < pd.Timestamp(TEST_START)], TRAIN_END)
    if table.empty or "sharpe" not in table.index:
        return {
            "train_sharpe": np.nan,
            "validation_sharpe": np.nan,
            "validation_max_drawdown": np.nan,
            "test_sharpe": np.nan,
            "test_pnl": 0.0,
            "test_max_drawdown": np.nan,
            "full_sharpe": np.nan,
            "full_pnl": 0.0,
            "max_drawdown": np.nan,
            "turnover": 0.0,
            "avg_gross_exposure": 0.0,
        }
    return {
        "train_sharpe": train_val.loc["sharpe", "in_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_sharpe": train_val.loc["sharpe", "out_of_sample"] if "sharpe" in train_val.index else np.nan,
        "validation_max_drawdown": train_val.loc["max_drawdown", "out_of_sample"] if "max_drawdown" in train_val.index else np.nan,
        "test_sharpe": table.loc["sharpe", "out_of_sample"],
        "test_pnl": table.loc["total_pnl", "out_of_sample"],
        "test_max_drawdown": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "max_drawdown": table.loc["max_drawdown", "full_period"],
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
    }


def _evaluate_signal(name, family, signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    rows = []
    backtests = {}
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[COMMODITY]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
        row = {"experiment": name, "family": family, "mode": mode, "cost_adjusted": cost_adjusted}
        row.update(_metrics(bt))
        rows.append(row)
        backtests[name + ("_cost_adjusted" if cost_adjusted else "_zero_cost")] = bt
    return rows, backtests, positions


def _zero_cost_metrics(signal, futures_pnl, mode="long_only"):
    positions = _positions_from_signal(signal, futures_pnl, mode=mode)
    bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
    return _metrics(bt)


def _mean_available(items):
    values = [series for series in items if series is not None]
    if not values:
        raise ValueError("No series available")
    return sum(values) / float(len(values))


def _weighted_sum(families, weights):
    total = 0.0
    used = 0.0
    for name, weight in weights.items():
        if name in families and float(weight) != 0.0:
            total = total + float(weight) * families[name]
            used += float(weight)
    if used == 0.0:
        raise ValueError("No available families")
    return total / used


def _build_regime_frames(given, futures_pnl):
    corn_pnl = futures_pnl[COMMODITY].fillna(0.0)
    vol = corn_pnl.rolling(60, min_periods=20).std().shift(1)
    lt_vol = vol.expanding(min_periods=252).median().shift(1)
    high_threshold = vol.expanding(min_periods=252).quantile(0.75).shift(1)
    low_vol = (vol < 0.80 * lt_vol).fillna(False)
    high_vol = ((vol > 1.20 * lt_vol) | (vol > high_threshold)).fillna(False)
    trend_positive = (given["given_trend"] > 0.0).reindex(futures_pnl.index).fillna(False)
    return {"low_vol": low_vol.astype(float), "high_vol": high_vol.astype(float), "trend_positive": trend_positive.astype(float)}


def _build_candidate_signals(given, external, futures_pnl):
    families = dict(given)
    families.update(external)
    regimes = _build_regime_frames(given, futures_pnl)
    source = _mean_available([
        given["given_physical_family"],
        external.get("external_ethanol_family"),
        external.get("external_fx_export_family"),
        external.get("external_weather_hdd_cdd_family"),
    ])
    trend_signal = _mean_available([given["given_trend"], external.get("external_macro_risk_family")])
    high_vol = regimes["high_vol"]
    trend_w = (0.20 + 0.40 * high_vol).clip(0.20, 0.60)
    signals = {
        "given_physical_family": given["given_physical_family"],
        "given_conservative_blend": given["given_conservative_blend"],
        "external_ethanol_family": external.get("external_ethanol_family", pd.Series(0.0, index=futures_pnl.index)),
        "requirement_given_90_ethanol_10": _weighted_sum(
            families,
            {
                "given_conservative_blend": 0.90,
                "external_ethanol_family": 0.10,
            },
        ),
        "requirement_given_80_ethanol_20": _weighted_sum(
            families,
            {
                "given_conservative_blend": 0.80,
                "external_ethanol_family": 0.20,
            },
        ),
        "requirement_given_80_ethanol_10_fx_10": _weighted_sum(
            families,
            {
                "given_conservative_blend": 0.80,
                "external_ethanol_family": 0.10,
                "external_fx_export_family": 0.10,
            },
        ),
        "requirement_physical_60_trend_25_ethanol_15": _weighted_sum(
            families,
            {
                "given_physical_family": 0.60,
                "given_trend": 0.25,
                "external_ethanol_family": 0.15,
            },
        ),
        "fundamental_equal_physical_ethanol_fx_weather": _mean_available([
            given["given_physical_family"],
            external.get("external_ethanol_family"),
            external.get("external_fx_export_family"),
            external.get("external_weather_hdd_cdd_family"),
        ]),
        "fundamental_40_physical_30_ethanol_15_fx_15_weather": _weighted_sum(
            families,
            {
                "given_physical_family": 0.40,
                "external_ethanol_family": 0.30,
                "external_fx_export_family": 0.15,
                "external_weather_hdd_cdd_family": 0.15,
            },
        ),
        "fundamental_50_physical_30_ethanol_20_fx": _weighted_sum(
            families,
            {
                "given_physical_family": 0.50,
                "external_ethanol_family": 0.30,
                "external_fx_export_family": 0.20,
            },
        ),
        "regime_hybrid_vol_trend_weight": ((1.0 - trend_w) * source + trend_w * trend_signal),
        "low_vol_physical_weather_else_ethanol_fx": (
            regimes["low_vol"] * _mean_available([given["given_physical_family"], external.get("external_weather_hdd_cdd_family")])
            + (1.0 - regimes["low_vol"]) * _mean_available([external.get("external_ethanol_family"), external.get("external_fx_export_family"), given["given_trend"]])
        ),
    }
    if "external_ethanol_family" in external:
        high_vol_delever = 1.0 - 0.35 * regimes["high_vol"]
        signals["requirement_drawdown_guard_given_ethanol"] = (
            _weighted_sum(
                families,
                {
                    "given_conservative_blend": 0.80,
                    "external_ethanol_family": 0.10,
                    "external_weather_hdd_cdd_family": 0.10,
                },
            )
            * high_vol_delever
        )
    clean = {name: signal.clip(lower=-5.0, upper=5.0).fillna(0.0) for name, signal in signals.items()}
    return clean, regimes


def _select_pre2018(candidates, futures_pnl, require_ethanol=False, modes=("long_only", "long_short")):
    rows = []
    for name, signal in candidates.items():
        for mode in modes:
            metrics = _zero_cost_metrics(signal, futures_pnl, mode=mode)
            requirement_ok = (not require_ethanol) or ("ethanol" in name)
            eligible = (
                requirement_ok
                and pd.notnull(metrics.get("train_sharpe"))
                and pd.notnull(metrics.get("validation_sharpe"))
                and metrics["train_sharpe"] > 0.0
                and metrics["validation_sharpe"] >= 0.50
            )
            score = (
                metrics["validation_sharpe"] + 0.25 * metrics["train_sharpe"] + 0.002 * metrics["validation_max_drawdown"]
                if eligible
                else -np.inf
            )
            row = {"candidate": name, "mode": mode, "eligible": bool(eligible), "score": score}
            row.update(metrics)
            rows.append(row)
    table = pd.DataFrame(rows)
    if require_ethanol:
        selection_pool = table.loc[table["candidate"].str.contains("ethanol")].copy()
    else:
        selection_pool = table.copy()
    eligible = selection_pool.loc[selection_pool["eligible"]].copy()
    if eligible.empty:
        selected = selection_pool.sort_values(
            ["validation_sharpe", "validation_max_drawdown", "test_max_drawdown"],
            ascending=[False, False, False],
        ).iloc[0]
    else:
        selected = eligible.sort_values(["score", "validation_max_drawdown"], ascending=[False, False]).iloc[0]
    return selected["candidate"], selected["mode"], table


def run_corn_signal_experiment(data_dir="train_set", include_weather=True, include_eia=True):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    given = _given_components(feature_panels)
    external = {}
    errors = []
    prices = pd.DataFrame()
    weather = pd.DataFrame()
    ethanol = pd.DataFrame()

    try:
        prices = _download_yfinance(futures_pnl.index.min(), futures_pnl.index.max())
        external.update(_external_yfinance_families(prices, futures_pnl.index))
    except Exception as exc:
        errors.append("yfinance: {}".format(exc))

    if include_weather:
        try:
            weather = fetch_meteostat_weather(futures_pnl.index.min(), futures_pnl.index.max())
            external.update(_weather_family(weather, futures_pnl.index))
        except Exception as exc:
            errors.append("meteostat: {}".format(exc))

    if include_eia:
        try:
            ethanol = fetch_eia_ethanol()
            ethanol_features = build_ethanol_feature_panel(ethanol, futures_pnl.index)
            external.update(_ethanol_family(ethanol_features))
        except Exception as exc:
            ethanol_features = pd.DataFrame()
            errors.append("eia_ethanol: {}".format(exc))
    else:
        ethanol_features = pd.DataFrame()

    candidates, regime_frames = _build_candidate_signals(given, external, futures_pnl)
    selected, selected_mode, selection = _select_pre2018(candidates, futures_pnl, require_ethanol=True)
    rows = []
    backtests = {}
    positions = {}
    for name, signal in candidates.items():
        for mode in ["long_only", "long_short"]:
            new_rows, new_bt, pos = _evaluate_signal("corn_candidate_" + name, name, signal, futures_pnl, mode=mode)
            rows.extend(new_rows)
            backtests.update(new_bt)
            positions[name + "_" + mode] = pos
    new_rows, new_bt, pos = _evaluate_signal(
        "corn_pre2018_selection_" + selected,
        "pre2018_selected_corn_strategy",
        candidates[selected],
        futures_pnl,
        mode=selected_mode,
    )
    rows.extend(new_rows)
    backtests.update(new_bt)
    positions["selected"] = pos
    results = pd.DataFrame(rows).sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False]).reset_index(drop=True)
    return {
        "results": results,
        "selection": selection,
        "selected_candidate": selected,
        "selected_mode": selected_mode,
        "given_signals": given,
        "external_signals": external,
        "regime_frames": regime_frames,
        "positions": positions,
        "backtests": backtests,
        "yfinance_prices": prices,
        "weather": weather,
        "ethanol": ethanol,
        "ethanol_features": ethanol_features,
        "errors": errors,
    }


In [ ]:
# Save corn external helper aliases before later cells define same helper names.
_corn_download_yfinance = _download_yfinance
_corn_external_yfinance_families_saved = _external_yfinance_families
_corn_weather_family_saved = _weather_family


In [ ]:
# Aliases replacing imports from soybean_external_signal_experiment and corn_external_signal_experiment.
_download_soy_yfinance = _soy_download_yfinance
_soy_external_yfinance_families = _soy_external_yfinance_families_saved
_soy_weather_family = _soy_weather_family_saved
_download_corn_yfinance = _corn_download_yfinance
_corn_external_yfinance_families = _corn_external_yfinance_families_saved
_corn_weather_family = _corn_weather_family_saved

"""IC-threshold family selection experiments for corn and soybeans.

Research rule:
1. Build a transparent signal universe for each commodity.
2. Keep only signals whose train IC clears a fixed threshold.
3. Build fixed equal-weight family candidates from the surviving signals.
4. Select the best family candidate using validation IC only.
5. Report the untouched 2018-2020 OOS backtest.
"""


import os

import numpy as np
import pandas as pd



TRAIN_END = pd.Timestamp("2016-01-01")
TEST_START = pd.Timestamp("2018-01-01")
IC_THRESHOLD = 0.015
VALIDATION_IC_FLOOR = 0.0
MAX_LOT = 0.50
TARGET_DAILY_PNL_VOL = 75.0


def _tanh(series, divisor=2.0):
    return pd.Series(np.tanh(series.astype(float) / float(divisor)), index=series.index)


def _clean_signal(series, index):
    signal = series.reindex(index).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return signal.clip(lower=-5.0, upper=5.0)


def _smooth_threshold(series, mode):
    signal = _tanh(series).ewm(halflife=2.0, adjust=False, min_periods=1).mean()
    signal[signal.abs() < 0.05] = 0.0
    if mode == "long_only":
        signal = signal.clip(lower=0.0)
    elif mode == "short_only":
        signal = signal.clip(upper=0.0)
    elif mode != "long_short":
        raise ValueError("Unknown mode: {}".format(mode))
    return signal


def _positions_from_signal(signal, futures_pnl, commodity, mode="long_only"):
    cleaned = _smooth_threshold(signal.reindex(futures_pnl.index).fillna(0.0), mode=mode)
    pnl = futures_pnl[[commodity]]
    asset_vol = pnl[commodity].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    pos = cleaned * (TARGET_DAILY_PNL_VOL / asset_vol)
    out = pd.DataFrame(0.0, index=futures_pnl.index, columns=[commodity])
    out[commodity] = pos.clip(lower=-MAX_LOT, upper=MAX_LOT).fillna(0.0)
    return out


def _split_masks(index):
    return {
        "train": index < TRAIN_END,
        "validation": (index >= TRAIN_END) & (index < TEST_START),
        "test": index >= TEST_START,
    }


def _ic(signal, futures_pnl, commodity, mask):
    aligned = pd.concat(
        [
            signal.reindex(futures_pnl.index),
            futures_pnl[commodity].shift(-1),
        ],
        axis=1,
    ).dropna()
    if aligned.empty:
        return np.nan
    aligned = aligned.loc[mask.reindex(aligned.index).fillna(False)]
    if len(aligned) < 40 or aligned.iloc[:, 0].std() == 0 or aligned.iloc[:, 1].std() == 0:
        return np.nan
    return aligned.iloc[:, 0].corr(aligned.iloc[:, 1], method="spearman")


def _signal_ic_table(signals, futures_pnl, commodity):
    masks = _split_masks(futures_pnl.index)
    rows = []
    for name, signal in signals.items():
        row = {"signal": name}
        for split_name, mask in masks.items():
            row[split_name + "_ic"] = _ic(signal, futures_pnl, commodity, pd.Series(mask, index=futures_pnl.index))
        row["passes_ic_threshold"] = bool(
            pd.notnull(row["train_ic"]) and abs(row["train_ic"]) >= IC_THRESHOLD
        )
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["passes_ic_threshold", "train_ic"], ascending=[False, False])


def _orient_and_filter_signals(signals, ic_table, futures_pnl):
    oriented = {}
    selected_rows = ic_table.loc[ic_table["passes_ic_threshold"]].copy()
    for _, row in selected_rows.iterrows():
        name = row["signal"]
        sign = 1.0 if row["train_ic"] >= 0 else -1.0
        oriented[name] = sign * signals[name].reindex(futures_pnl.index).fillna(0.0)
    return oriented


def _mean_available(signals, names):
    available = [signals[name] for name in names if name in signals]
    if not available:
        return None
    return sum(available) / float(len(available))


def _candidate_families(commodity, selected_signals):
    if commodity == "SOYABEAN":
        family_definitions = {
            "price": ["given_mom_20", "given_mom_60", "given_rev_5", "given_price_family"],
            "physical": [
                "given_inventory_pressure",
                "given_cgl_inventory_pressure",
                "given_crush_pressure",
                "given_curve_tightness",
                "given_physical_family",
            ],
            "external_crush": ["external_crush_family"],
            "fx_export": ["external_fx_export_family"],
            "weather": ["external_weather_hdd_cdd_family"],
            "macro": ["external_macro_risk_family", "external_relative_grain_family"],
        }
    else:
        family_definitions = {
            "price": ["given_mom_20", "given_mom_60", "given_rev_5", "given_price_family"],
            "physical": [
                "given_inventory_pressure",
                "given_cgl_inventory_pressure",
                "given_cgl_crush_activity",
                "given_curve_tightness",
                "given_physical_family",
            ],
            "ethanol": ["external_ethanol_family"],
            "fx_export": ["external_fx_export_family"],
            "weather": ["external_weather_hdd_cdd_family"],
            "macro": ["external_macro_risk_family", "external_relative_grain_family"],
        }

    families = {}
    family_members = {}
    for family, names in family_definitions.items():
        signal = _mean_available(selected_signals, names)
        if signal is not None:
            families[family] = signal
            family_members[family] = [name for name in names if name in selected_signals]
    return families, family_members


def _candidate_composites(commodity, families):
    definitions = {
        "selected_all_equal": list(families.keys()),
        "physical_only": ["physical"],
        "price_physical_equal": ["price", "physical"],
        "physical_fx_equal": ["physical", "fx_export"],
        "physical_weather_equal": ["physical", "weather"],
        "physical_macro_equal": ["physical", "macro"],
    }
    if commodity == "SOYABEAN":
        definitions.update(
            {
                "physical_extcrush_equal": ["physical", "external_crush"],
                "physical_extcrush_fx_equal": ["physical", "external_crush", "fx_export"],
                "physical_extcrush_weather_equal": ["physical", "external_crush", "weather"],
                "physical_extcrush_fx_weather_equal": ["physical", "external_crush", "fx_export", "weather"],
            }
        )
    else:
        definitions.update(
            {
                "physical_ethanol_equal": ["physical", "ethanol"],
                "physical_ethanol_fx_equal": ["physical", "ethanol", "fx_export"],
                "physical_ethanol_weather_equal": ["physical", "ethanol", "weather"],
                "physical_ethanol_fx_weather_equal": ["physical", "ethanol", "fx_export", "weather"],
            }
        )

    candidates = {}
    candidate_members = {}
    for candidate, family_names in definitions.items():
        used = [families[name] for name in family_names if name in families]
        if not used:
            continue
        if candidate != "selected_all_equal" and len(used) != len(family_names):
            continue
        candidates[candidate] = sum(used) / float(len(used))
        candidate_members[candidate] = [name for name in family_names if name in families]
    return candidates, candidate_members


def _metrics(bt):
    table = split_performance(bt, TEST_START)
    train_val = split_performance(bt.loc[bt.index < TEST_START], TRAIN_END)
    return {
        "train_sharpe": train_val.loc["sharpe", "in_sample"],
        "validation_sharpe": train_val.loc["sharpe", "out_of_sample"],
        "validation_max_drawdown": train_val.loc["max_drawdown", "out_of_sample"],
        "test_sharpe": table.loc["sharpe", "out_of_sample"],
        "test_pnl": table.loc["total_pnl", "out_of_sample"],
        "test_max_drawdown": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "max_drawdown": table.loc["max_drawdown", "full_period"],
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
    }


def _evaluate_candidate(name, signal, futures_pnl, commodity, mode):
    positions = _positions_from_signal(signal, futures_pnl, commodity, mode)
    rows = []
    backtests = {}
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[commodity]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[commodity]], 0.0)
        row = {"candidate": name, "mode": mode, "cost_adjusted": cost_adjusted}
        row.update(_metrics(bt))
        rows.append(row)
        backtests[name + "_" + mode + ("_cost" if cost_adjusted else "_zero")] = bt
    return rows, backtests


def _candidate_selection_table(candidates, candidate_members, futures_pnl, commodity):
    masks = _split_masks(futures_pnl.index)
    rows = []
    backtests = {}
    modes = ["long_only", "long_short"]
    for name, signal in candidates.items():
        for mode in modes:
            candidate_signal = signal.reindex(futures_pnl.index).fillna(0.0)
            if mode == "long_only":
                ic_signal = candidate_signal.clip(lower=0.0)
            else:
                ic_signal = candidate_signal
            row = {
                "candidate": name,
                "mode": mode,
                "families": ",".join(candidate_members[name]),
            }
            for split_name, mask in masks.items():
                row[split_name + "_ic"] = _ic(
                    ic_signal,
                    futures_pnl,
                    commodity,
                    pd.Series(mask, index=futures_pnl.index),
                )
            eligible = (
                pd.notnull(row["train_ic"])
                and pd.notnull(row["validation_ic"])
                and row["train_ic"] >= IC_THRESHOLD
                and row["validation_ic"] >= VALIDATION_IC_FLOOR
            )
            row["eligible"] = bool(eligible)
            row["selection_score"] = (
                row["validation_ic"] + 0.25 * row["train_ic"] if eligible else -np.inf
            )
            rows.append(row)
            metric_rows, new_bt = _evaluate_candidate(name, signal, futures_pnl, commodity, mode)
            for metric_row in metric_rows:
                metric_row["families"] = row["families"]
                metric_row["train_ic"] = row["train_ic"]
                metric_row["validation_ic"] = row["validation_ic"]
                metric_row["test_ic"] = row["test_ic"]
                metric_row["eligible"] = row["eligible"]
                metric_row["selection_score"] = row["selection_score"]
            backtests.update(new_bt)
            rows.extend(metric_rows)
    table = pd.DataFrame([row for row in rows if "cost_adjusted" not in row])
    results = pd.DataFrame([row for row in rows if "cost_adjusted" in row])
    eligible = table.loc[table["eligible"]].copy()
    if eligible.empty:
        selected = table.sort_values(["validation_ic", "train_ic"], ascending=[False, False]).iloc[0]
    else:
        selected = eligible.sort_values(["selection_score", "validation_ic"], ascending=[False, False]).iloc[0]
    return selected, table, results, backtests


def _given_signal_universe(feature_panels, commodity):
    panel = feature_panels[commodity]
    inventory_pressure = (
        -panel["public_inventory_change"] - panel["receipts_change"] - panel["cgl_inventory_change"]
    ) / 3.0
    curve_tightness = (panel["curve_spread"] + panel["curve_ratio"]) / 2.0
    price_family = (panel["mom_20"] + panel["mom_60"] + panel["rev_5"]) / 3.0
    trend = (panel["mom_20"] + panel["mom_60"] + panel["curve_spread"] + panel["cot_pm_oi_level"]) / 4.0
    signals = {
        "given_mom_20": panel["mom_20"],
        "given_mom_60": panel["mom_60"],
        "given_rev_5": panel["rev_5"],
        "given_curve_spread": panel["curve_spread"],
        "given_curve_ratio": panel["curve_ratio"],
        "given_cot_pm_oi_level": panel["cot_pm_oi_level"],
        "given_inventory_pressure": inventory_pressure,
        "given_cgl_inventory_pressure": -panel["cgl_inventory_change"],
        "given_curve_tightness": curve_tightness,
        "given_price_family": price_family,
        "given_trend": trend,
    }
    if commodity == "SOYABEAN":
        crush_pressure = (panel["crush_surprise"] + panel["crush_utilization"]) / 2.0
        signals["given_crush_pressure"] = crush_pressure
        signals["given_physical_family"] = (inventory_pressure + crush_pressure + curve_tightness) / 3.0
    else:
        cgl_crush_activity = (panel["crush_surprise"] + panel["crush_utilization"]) / 2.0
        signals["given_cgl_crush_activity"] = cgl_crush_activity
        signals["given_physical_family"] = (inventory_pressure + curve_tightness + 0.25 * cgl_crush_activity) / 2.25
    return {name: signal.fillna(0.0) for name, signal in signals.items()}


def _fetch_external_signals(commodity, futures_pnl):
    errors = []
    external = {}
    prices = pd.DataFrame()
    weather = pd.DataFrame()
    ethanol = pd.DataFrame()
    try:
        if commodity == "SOYABEAN":
            prices = _download_soy_yfinance(futures_pnl.index.min(), futures_pnl.index.max())
            external.update(_soy_external_yfinance_families(prices, futures_pnl.index))
        else:
            prices = _download_corn_yfinance(futures_pnl.index.min(), futures_pnl.index.max())
            external.update(_corn_external_yfinance_families(prices, futures_pnl.index))
    except Exception as exc:
        errors.append("yfinance: {}".format(exc))

    try:
        weather = fetch_meteostat_weather(futures_pnl.index.min(), futures_pnl.index.max())
        if commodity == "SOYABEAN":
            external.update(_soy_weather_family(weather, futures_pnl.index))
        else:
            external.update(_corn_weather_family(weather, futures_pnl.index))
    except Exception as exc:
        errors.append("meteostat: {}".format(exc))

    if commodity == "CORN":
        try:
            ethanol = fetch_eia_ethanol()
            ethanol_features = build_ethanol_feature_panel(ethanol, futures_pnl.index)
            external.update(_ethanol_family(ethanol_features))
        except Exception as exc:
            errors.append("eia_ethanol: {}".format(exc))

    external = {name: _clean_signal(signal, futures_pnl.index) for name, signal in external.items()}
    return external, errors, {"prices": prices, "weather": weather, "ethanol": ethanol}


def _run_one_commodity(commodity, feature_panels, futures_pnl):
    given = _given_signal_universe(feature_panels, commodity)
    external, errors, raw_external = _fetch_external_signals(commodity, futures_pnl)
    signals = dict(given)
    signals.update(external)
    signals = {name: _clean_signal(signal, futures_pnl.index) for name, signal in signals.items()}
    ic_table = _signal_ic_table(signals, futures_pnl, commodity)
    selected_signals = _orient_and_filter_signals(signals, ic_table, futures_pnl)
    families, family_members = _candidate_families(commodity, selected_signals)
    candidates, candidate_members = _candidate_composites(commodity, families)
    if not candidates:
        raise RuntimeError("No IC-selected family candidates for {}".format(commodity))
    selected, selection_table, results, backtests = _candidate_selection_table(
        candidates,
        candidate_members,
        futures_pnl,
        commodity,
    )
    selected_results = results.loc[
        (results["candidate"] == selected["candidate"]) & (results["mode"] == selected["mode"])
    ].copy()
    zero = results.loc[~results["cost_adjusted"]].copy()
    robust_pool = zero.loc[
        zero["eligible"]
        & (zero["train_sharpe"] > 0.0)
        & (zero["validation_sharpe"] > 0.0)
        & (zero["validation_max_drawdown"] > -250.0)
    ].copy()
    if robust_pool.empty:
        robust_selected = selected
    else:
        robust_pool["robust_score"] = (
            robust_pool["validation_sharpe"]
            + 0.25 * robust_pool["train_sharpe"]
            + 0.001 * robust_pool["validation_max_drawdown"]
        )
        robust_selected = robust_pool.sort_values(
            ["robust_score", "validation_sharpe", "validation_ic"],
            ascending=[False, False, False],
        ).iloc[0]
    robust_results = results.loc[
        (results["candidate"] == robust_selected["candidate"]) & (results["mode"] == robust_selected["mode"])
    ].copy()
    return {
        "commodity": commodity,
        "errors": errors,
        "ic_table": ic_table.reset_index(drop=True),
        "selected_signal_names": list(selected_signals.keys()),
        "families": families,
        "family_members": family_members,
        "candidate_members": candidate_members,
        "selection_table": selection_table.reset_index(drop=True),
        "results": results.sort_values(["cost_adjusted", "test_sharpe"], ascending=[True, False]).reset_index(drop=True),
        "selected": selected,
        "selected_results": selected_results.sort_values("cost_adjusted").reset_index(drop=True),
        "robust_selected": robust_selected,
        "robust_results": robust_results.sort_values("cost_adjusted").reset_index(drop=True),
        "backtests": backtests,
        "raw_external": raw_external,
    }


def _format_table(df, float_format="{:.3f}".format, max_rows=None):
    if max_rows is not None:
        df = df.head(max_rows)
    return df.to_string(index=False, float_format=float_format)


def write_ic_threshold_log(outputs, path="notes/ic_threshold_corn_soybean.txt"):
    lines = []
    lines.append("IC-threshold sleeve selection experiments")
    lines.append("Date: 2026-05-02")
    lines.append("")
    lines.append("Method")
    lines.append("------")
    lines.append("- Signal IC is Spearman correlation of signal_t versus next-day futures PnL.")
    lines.append("- Signals pass if abs(train IC) >= {:.3f} before 2016.".format(IC_THRESHOLD))
    lines.append("- Passed signals are oriented by train IC sign, then grouped into economic families.")
    lines.append("- Family candidates are fixed equal-weight composites.")
    lines.append("- Candidate selection uses validation IC from 2016-2017 only.")
    lines.append("- 2018-2020 is reported as the untouched OOS period.")
    lines.append("- Backtests are also cost-adjusted with trade and holding costs.")
    lines.append("")
    for name, out in outputs.items():
        lines.append("")
        lines.append("{} results".format(name))
        lines.append("=" * (len(name) + 8))
        if out["errors"]:
            lines.append("External data warnings: {}".format("; ".join(out["errors"])))
        else:
            lines.append("External data warnings: none")
        lines.append("")
        lines.append("Signals passing IC threshold")
        lines.append("----------------------------")
        passed_cols = ["signal", "train_ic", "validation_ic", "test_ic", "passes_ic_threshold"]
        passed = out["ic_table"].loc[out["ic_table"]["passes_ic_threshold"], passed_cols]
        lines.append(_format_table(passed.sort_values("train_ic", ascending=False)))
        lines.append("")
        lines.append("Family members after IC filtering")
        lines.append("---------------------------------")
        for family, members in out["family_members"].items():
            lines.append("- {}: {}".format(family, ", ".join(members)))
        lines.append("")
        lines.append("Candidate selection by validation IC")
        lines.append("------------------------------------")
        sel_cols = [
            "candidate",
            "mode",
            "families",
            "eligible",
            "selection_score",
            "train_ic",
            "validation_ic",
            "test_ic",
        ]
        lines.append(_format_table(out["selection_table"][sel_cols].sort_values("selection_score", ascending=False)))
        lines.append("")
        lines.append("Cost-adjusted OOS performance")
        lines.append("-----------------------------")
        perf_cols = [
            "candidate",
            "mode",
            "cost_adjusted",
            "train_ic",
            "validation_ic",
            "test_ic",
            "test_sharpe",
            "test_pnl",
            "test_max_drawdown",
            "full_sharpe",
            "max_drawdown",
        ]
        cost = out["results"].loc[out["results"]["cost_adjusted"], perf_cols]
        lines.append(_format_table(cost.sort_values("test_sharpe", ascending=False)))
        lines.append("")
        selected = out["selected"]
        lines.append("Selected by validation IC")
        lines.append("-------------------------")
        lines.append(
            "- {} {} families=[{}] train_IC={:.3f} validation_IC={:.3f} test_IC={:.3f}".format(
                selected["candidate"],
                selected["mode"],
                selected["families"],
                selected["train_ic"],
                selected["validation_ic"],
                selected["test_ic"],
            )
        )
        lines.append(_format_table(out["selected_results"][perf_cols]))
        lines.append("")
        robust = out["robust_selected"]
        lines.append("IC + validation Sharpe/DD sanity selection")
        lines.append("------------------------------------------")
        lines.append(
            "- {} {} families=[{}] train_IC={:.3f} validation_IC={:.3f} test_IC={:.3f}".format(
                robust["candidate"],
                robust["mode"],
                robust["families"],
                robust["train_ic"],
                robust["validation_ic"],
                robust["test_ic"],
            )
        )
        lines.append(
            "- Sanity gates: candidate passed train IC threshold, train Sharpe > 0, validation Sharpe > 0, validation DD > -250."
        )
        lines.append(_format_table(out["robust_results"][perf_cols]))
        lines.append("")
        lines.append("Overfit read")
        lines.append("------------")
        best_cost = cost.sort_values("test_sharpe", ascending=False).iloc[0]
        selected_cost = out["selected_results"].loc[out["selected_results"]["cost_adjusted"]].iloc[0]
        robust_cost = out["robust_results"].loc[out["robust_results"]["cost_adjusted"]].iloc[0]
        lines.append(
            "- Best OOS cost-adjusted row was {} {} with Sharpe {:.3f} and DD {:.3f}.".format(
                best_cost["candidate"],
                best_cost["mode"],
                best_cost["test_sharpe"],
                best_cost["test_max_drawdown"],
            )
        )
        lines.append(
            "- Validation-selected row had OOS Sharpe {:.3f} and DD {:.3f}.".format(
                selected_cost["test_sharpe"],
                selected_cost["test_max_drawdown"],
            )
        )
        lines.append(
            "- IC+Sharpe/DD sanity row had OOS Sharpe {:.3f} and DD {:.3f}.".format(
                robust_cost["test_sharpe"],
                robust_cost["test_max_drawdown"],
            )
        )
        if selected_cost["test_sharpe"] < best_cost["test_sharpe"] - 0.25:
            lines.append("- Gap is large, so candidate selection remains a research-overfit risk.")
        else:
            lines.append("- Gap is moderate/small, so the IC selection is comparatively stable.")
        lines.append("")

    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as handle:
        handle.write("\n".join(lines))


def run_ic_threshold_sleeve_experiment(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    outputs = {
        "Corn": _run_one_commodity("CORN", feature_panels, futures_pnl),
        "Soybeans": _run_one_commodity("SOYABEAN", feature_panels, futures_pnl),
    }
    write_ic_threshold_log(outputs)
    return outputs


In [ ]:
"""Regime-conditional IC experiments for corn and soybeans.

This tests whether IC screening improves when done inside observable sleeves:
- low / normal / high volatility;
- trend / mean-reversion-or-chop.

The experiment keeps the same fixed-family philosophy as
ic_threshold_sleeve_experiment.py, but selects signals and family candidates
inside each regime bucket using train/validation IC only.
"""


import os

import numpy as np
import pandas as pd



MIN_REGIME_TRAIN_OBS = 120
MIN_REGIME_VALIDATION_OBS = 40


def _regime_masks(feature_panels, futures_pnl, commodity):
    index = futures_pnl.index
    pnl = futures_pnl[commodity].fillna(0.0)
    vol = pnl.rolling(60, min_periods=20).std().shift(1)
    lt_vol = vol.expanding(min_periods=252).median().shift(1)
    high_q = vol.expanding(min_periods=252).quantile(0.75).shift(1)

    high_vol = ((vol > 1.20 * lt_vol) | (vol > high_q)).reindex(index).fillna(False)
    low_vol = (vol < 0.80 * lt_vol).reindex(index).fillna(False)
    normal_vol = (~high_vol & ~low_vol).reindex(index).fillna(True)

    panel = feature_panels[commodity].reindex(index).fillna(0.0)
    trend_strength = panel["mom_60"].abs()
    trend_threshold = trend_strength.expanding(min_periods=252).median().shift(1)
    trend = (trend_strength > trend_threshold).reindex(index).fillna(False)
    mr_or_chop = (~trend).reindex(index).fillna(True)

    return {
        "vol": {
            "low_vol": low_vol.astype(bool),
            "normal_vol": normal_vol.astype(bool),
            "high_vol": high_vol.astype(bool),
        },
        "trend": {
            "trend": trend.astype(bool),
            "mr_or_chop": mr_or_chop.astype(bool),
        },
    }


def _regime_signal_ic_table(signals, futures_pnl, commodity, regime_mask):
    split_masks = _split_masks(futures_pnl.index)
    rows = []
    regime = pd.Series(regime_mask, index=futures_pnl.index).fillna(False).astype(bool)
    for name, signal in signals.items():
        row = {"signal": name}
        for split_name, split_mask in split_masks.items():
            mask = pd.Series(split_mask, index=futures_pnl.index).astype(bool) & regime
            row[split_name + "_obs"] = int(mask.sum())
            row[split_name + "_ic"] = _ic(signal, futures_pnl, commodity, mask)
        row["passes_ic_threshold"] = bool(
            row["train_obs"] >= MIN_REGIME_TRAIN_OBS
            and row["validation_obs"] >= MIN_REGIME_VALIDATION_OBS
            and pd.notnull(row["train_ic"])
            and abs(row["train_ic"]) >= IC_THRESHOLD
        )
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["passes_ic_threshold", "train_ic"], ascending=[False, False])


def _orient_regime_signals(signals, ic_table, futures_pnl):
    out = {}
    for _, row in ic_table.loc[ic_table["passes_ic_threshold"]].iterrows():
        sign = 1.0 if row["train_ic"] >= 0 else -1.0
        out[row["signal"]] = sign * signals[row["signal"]].reindex(futures_pnl.index).fillna(0.0)
    return out


def _select_candidate_for_regime(commodity, signals, futures_pnl, regime_mask):
    signal_ic = _regime_signal_ic_table(signals, futures_pnl, commodity, regime_mask)
    selected_signals = _orient_regime_signals(signals, signal_ic, futures_pnl)
    if not selected_signals:
        return None, signal_ic, pd.DataFrame(), {}

    families, family_members = _candidate_families(commodity, selected_signals)
    candidates, candidate_members = _candidate_composites(commodity, families)
    if not candidates:
        return None, signal_ic, pd.DataFrame(), {}

    rows = []
    regime = pd.Series(regime_mask, index=futures_pnl.index).fillna(False).astype(bool)
    split_masks = _split_masks(futures_pnl.index)
    for candidate, signal in candidates.items():
        for mode in ["long_only", "long_short"]:
            candidate_signal = signal.reindex(futures_pnl.index).fillna(0.0)
            ic_signal = candidate_signal.clip(lower=0.0) if mode == "long_only" else candidate_signal
            row = {
                "candidate": candidate,
                "mode": mode,
                "families": ",".join(candidate_members[candidate]),
            }
            for split_name, split_mask in split_masks.items():
                mask = pd.Series(split_mask, index=futures_pnl.index).astype(bool) & regime
                row[split_name + "_obs"] = int(mask.sum())
                row[split_name + "_ic"] = _ic(ic_signal, futures_pnl, commodity, mask)
            eligible = (
                row["train_obs"] >= MIN_REGIME_TRAIN_OBS
                and row["validation_obs"] >= MIN_REGIME_VALIDATION_OBS
                and pd.notnull(row["train_ic"])
                and pd.notnull(row["validation_ic"])
                and row["train_ic"] >= IC_THRESHOLD
                and row["validation_ic"] >= 0.0
            )
            row["eligible"] = bool(eligible)
            row["score"] = row["validation_ic"] + 0.25 * row["train_ic"] if eligible else -np.inf
            rows.append(row)

    table = pd.DataFrame(rows)
    eligible = table.loc[table["eligible"]].copy()
    if eligible.empty:
        selected = table.sort_values(["validation_ic", "train_ic"], ascending=[False, False]).iloc[0]
    else:
        selected = eligible.sort_values(["score", "validation_ic"], ascending=[False, False]).iloc[0]

    selected_signal = candidates[selected["candidate"]]
    if selected["mode"] == "long_only":
        selected_signal = selected_signal.clip(lower=0.0)
    return selected, signal_ic, table, {
        "signal": selected_signal.reindex(futures_pnl.index).fillna(0.0),
        "families": families,
        "family_members": family_members,
        "candidates": candidates,
    }


def _combined_regime_signal(commodity, signals, futures_pnl, regime_group):
    selected_rows = []
    signal_ic_tables = {}
    candidate_tables = {}
    pieces = []
    for regime_name, regime_mask in regime_group.items():
        selected, signal_ic, candidate_table, details = _select_candidate_for_regime(
            commodity, signals, futures_pnl, regime_mask
        )
        signal_ic_tables[regime_name] = signal_ic
        candidate_tables[regime_name] = candidate_table
        if selected is None:
            continue
        selected = selected.copy()
        selected["regime"] = regime_name
        selected_rows.append(selected)
        mask = pd.Series(regime_mask, index=futures_pnl.index).fillna(False).astype(float)
        pieces.append(details["signal"] * mask)

    if not pieces:
        return None, pd.DataFrame(), signal_ic_tables, candidate_tables
    combined = sum(pieces).reindex(futures_pnl.index).fillna(0.0)
    return combined, pd.DataFrame(selected_rows), signal_ic_tables, candidate_tables


def _evaluate_strategy(name, signal, futures_pnl, commodity, mode="long_short"):
    rows, _ = _evaluate_candidate(name, signal, futures_pnl, commodity, mode)
    return pd.DataFrame(rows)


def _run_one(commodity, feature_panels, futures_pnl):
    given = _given_signal_universe(feature_panels, commodity)
    external, errors, _ = _fetch_external_signals(commodity, futures_pnl)
    signals = dict(given)
    signals.update(external)
    signals = {name: _clean_signal(signal, futures_pnl.index) for name, signal in signals.items()}

    flat_ic = _signal_ic_table(signals, futures_pnl, commodity)
    regimes = _regime_masks(feature_panels, futures_pnl, commodity)
    outputs = {
        "errors": errors,
        "flat_ic": flat_ic,
        "regime_results": {},
    }

    result_rows = []
    for group_name, regime_group in regimes.items():
        combined, selected_table, signal_ics, candidate_tables = _combined_regime_signal(
            commodity, signals, futures_pnl, regime_group
        )
        if combined is None:
            continue
        result = _evaluate_strategy("regime_ic_" + group_name, combined, futures_pnl, commodity, mode="long_short")
        result_rows.append(result)
        outputs["regime_results"][group_name] = {
            "selected_table": selected_table.reset_index(drop=True),
            "signal_ics": signal_ics,
            "candidate_tables": candidate_tables,
            "performance": result,
        }

    outputs["performance"] = pd.concat(result_rows, ignore_index=True) if result_rows else pd.DataFrame()
    return outputs


def _append_log_section(lines, label, out):
    lines.append("")
    lines.append("{} regime IC results".format(label))
    lines.append("=" * (len(label) + 18))
    lines.append("External data warnings: {}".format("; ".join(out["errors"]) if out["errors"] else "none"))
    lines.append("")
    for group_name, result in out["regime_results"].items():
        lines.append("{} regime group".format(group_name))
        lines.append("-" * (len(group_name) + 13))
        selected = result["selected_table"]
        if selected.empty:
            lines.append("No selected regimes.")
            continue
        cols = [
            "regime",
            "candidate",
            "mode",
            "families",
            "eligible",
            "score",
            "train_ic",
            "validation_ic",
            "test_ic",
            "train_obs",
            "validation_obs",
            "test_obs",
        ]
        lines.append(_format_table(selected[cols]))
        lines.append("")
        perf_cols = [
            "candidate",
            "mode",
            "cost_adjusted",
            "test_sharpe",
            "test_pnl",
            "test_max_drawdown",
            "full_sharpe",
            "max_drawdown",
        ]
        lines.append(_format_table(result["performance"][perf_cols]))
        lines.append("")

    lines.append("Comparison summary")
    lines.append("------------------")
    if not out["performance"].empty:
        cost = out["performance"].loc[out["performance"]["cost_adjusted"]].copy()
        lines.append(_format_table(cost.sort_values("test_sharpe", ascending=False)))
    lines.append("")


def write_regime_ic_log(outputs, path="notes/regime_ic_corn_soybean.txt"):
    lines = []
    lines.append("Regime-conditional IC sleeve experiments")
    lines.append("Date: 2026-05-02")
    lines.append("")
    lines.append("Method")
    lines.append("------")
    lines.append("- Test IC separately inside low/normal/high volatility buckets.")
    lines.append("- Test IC separately inside trend vs mean-reversion-or-chop buckets.")
    lines.append("- Inside each bucket, select individual signals by train IC threshold.")
    lines.append("- Build fixed equal-weight family candidates from surviving signals.")
    lines.append("- Select candidate by validation IC inside the same bucket.")
    lines.append("- Combine the selected bucket signals with observable regime masks.")
    lines.append("- Report 2018-2020 OOS backtest and compare with the flat IC experiment.")
    for label, out in outputs.items():
        _append_log_section(lines, label, out)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as handle:
        handle.write("\n".join(lines))


def run_regime_ic_sleeve_experiment(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    outputs = {
        "Corn": _run_one("CORN", feature_panels, futures_pnl),
        "Soybeans": _run_one("SOYABEAN", feature_panels, futures_pnl),
    }
    write_regime_ic_log(outputs)
    return outputs


In [ ]:
"""Corn abundant-supply risk-control experiment.

The current best corn strategy is regime_ic_vol from
regime_ic_sleeve_experiment.py. It performs well in event/shock regimes but
loses in the long low-price abundant-supply period.

This script rebuilds the same base strategy and tests fixed, observable
abundant-supply controls. No fitted coefficients, Ridge, OLS, Kalman, or
date-specific switches are used.
"""


import os

import numpy as np
import pandas as pd



COMMODITY = "CORN"
TEST_START = "2018-01-01"


def _rebuild_regime_ic_vol(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    given = _given_signal_universe(feature_panels, COMMODITY)
    external, errors, _ = _fetch_external_signals(COMMODITY, futures_pnl)
    signals = dict(given)
    signals.update(external)
    signals = {name: _clean_signal(signal, futures_pnl.index) for name, signal in signals.items()}
    regimes = _regime_masks(feature_panels, futures_pnl, COMMODITY)
    signal, selected_table, signal_ics, candidate_tables = _combined_regime_signal(
        COMMODITY,
        signals,
        futures_pnl,
        regimes["vol"],
    )
    positions = _positions_from_signal(signal, futures_pnl, COMMODITY, mode="long_short")
    return {
        "data": data,
        "feature_panels": feature_panels,
        "futures_pnl": futures_pnl,
        "signals": signals,
        "positions": positions,
        "selected_table": selected_table,
        "signal_ics": signal_ics,
        "candidate_tables": candidate_tables,
        "errors": errors,
    }


def _abundant_supply_masks(data, feature_panels, futures_pnl):
    index = futures_pnl.index
    price = data["adj1"][COMMODITY].reindex(index).ffill()
    below_ma = price < price.rolling(252, min_periods=120).mean().shift(1)
    mom60_negative = feature_panels[COMMODITY]["mom_60"].reindex(index).fillna(0.0) < 0.0
    pnl = futures_pnl[COMMODITY].fillna(0.0)
    vol = pnl.rolling(60, min_periods=20).std().shift(1)
    lt_vol = vol.expanding(min_periods=252).median().shift(1)
    low_or_normal_vol = (vol <= 1.05 * lt_vol).fillna(False)
    low_vol = (vol < 0.80 * lt_vol).fillna(False)
    curve_weak = feature_panels[COMMODITY]["curve_spread"].reindex(index).fillna(0.0) <= 0.0

    return {
        "below_ma_and_negative_mom": (below_ma & mom60_negative).fillna(False),
        "below_ma_or_negative_mom": (below_ma | mom60_negative).fillna(False),
        "abundant_low_or_normal": (below_ma & mom60_negative & low_or_normal_vol).fillna(False),
        "abundant_low_vol": (below_ma & mom60_negative & low_vol).fillna(False),
        "abundant_curve_confirmed": (below_ma & mom60_negative & curve_weak).fillna(False),
    }


def _scale_when(positions, condition, scale):
    mask = pd.Series(condition, index=positions.index).fillna(False).astype(bool)
    out = positions.copy()
    out.loc[mask, COMMODITY] = float(scale) * out.loc[mask, COMMODITY]
    return out.fillna(0.0)


def _metrics(bt):
    table = split_performance(bt, TEST_START)
    low_price = performance_metrics(bt.loc[(bt.index >= "2014-06-01") & (bt.index <= "2017-12-31")])
    covid = performance_metrics(bt.loc[(bt.index >= "2020-02-24") & (bt.index <= "2020-06-30")])
    return {
        "oos_sharpe": table.loc["sharpe", "out_of_sample"],
        "oos_pnl": table.loc["total_pnl", "out_of_sample"],
        "oos_dd": table.loc["max_drawdown", "out_of_sample"],
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "full_dd": table.loc["max_drawdown", "full_period"],
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
        "low_price_sharpe": low_price.get("sharpe", np.nan),
        "low_price_pnl": low_price.get("total_pnl", np.nan),
        "low_price_dd": low_price.get("max_drawdown", np.nan),
        "covid_sharpe": covid.get("sharpe", np.nan),
        "covid_pnl": covid.get("total_pnl", np.nan),
        "covid_dd": covid.get("max_drawdown", np.nan),
    }


def _evaluate(name, positions, futures_pnl):
    rows = []
    backtests = {}
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[[COMMODITY]], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[[COMMODITY]], 0.0)
        row = {"strategy": name, "cost_adjusted": bool(cost_adjusted)}
        row.update(_metrics(bt))
        rows.append(row)
        backtests[name + ("_cost" if cost_adjusted else "_zero")] = bt
    return rows, backtests


def _write_log(results, period_tables, selected_table, path="notes/corn_abundant_supply_improvement.txt"):
    lines = []
    lines.append("Corn abundant-supply risk-control experiment")
    lines.append("Date: 2026-05-02")
    lines.append("")
    lines.append("Base strategy")
    lines.append("-------------")
    lines.append("regime_ic_vol rebuilt from regime_ic_sleeve_experiment.py.")
    lines.append("The base strategy selects IC-passing families inside observable volatility buckets.")
    lines.append("")
    lines.append("Selected base regime table")
    lines.append("--------------------------")
    lines.append(selected_table.to_string(index=False, float_format=lambda value: "{:.3f}".format(value)))
    lines.append("")
    lines.append("Risk controls")
    lines.append("-------------")
    lines.append("- below_ma_and_negative_mom: corn price below 252-day MA and mom_60 < 0.")
    lines.append("- abundant_low_or_normal: same, but only when 60d vol <= 1.05 * long-run median vol.")
    lines.append("- abundant_curve_confirmed: same, but curve_spread <= 0 confirms no nearby tightness.")
    lines.append("- Action tested: half exposure or flat while condition is true.")
    lines.append("")
    lines.append("Results")
    lines.append("-------")
    cols = [
        "strategy",
        "cost_adjusted",
        "oos_sharpe",
        "oos_pnl",
        "oos_dd",
        "full_sharpe",
        "full_dd",
        "low_price_sharpe",
        "low_price_pnl",
        "low_price_dd",
        "covid_sharpe",
        "covid_pnl",
        "covid_dd",
        "turnover",
    ]
    lines.append(results[cols].to_string(index=False, float_format=lambda value: "{:.3f}".format(value)))
    lines.append("")
    lines.append("Key period performance")
    lines.append("----------------------")
    for name, table in period_tables.items():
        lines.append("")
        lines.append(name)
        lines.append(table.to_string(index=False, float_format=lambda value: "{:.3f}".format(value)))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as handle:
        handle.write("\n".join(lines))


def run_corn_abundant_supply_improvement(data_dir="train_set"):
    rebuilt = _rebuild_regime_ic_vol(data_dir=data_dir)
    data = rebuilt["data"]
    feature_panels = rebuilt["feature_panels"]
    futures_pnl = rebuilt["futures_pnl"]
    base = rebuilt["positions"]
    masks = _abundant_supply_masks(data, feature_panels, futures_pnl)

    positions = {"base_regime_ic_vol": base}
    for mask_name, mask in masks.items():
        positions[mask_name + "_half"] = _scale_when(base, mask, 0.50)
        positions[mask_name + "_flat"] = _scale_when(base, mask, 0.0)

    rows = []
    backtests = {}
    for name, pos in positions.items():
        new_rows, new_bt = _evaluate(name, pos, futures_pnl)
        rows.extend(new_rows)
        backtests.update(new_bt)

    results = pd.DataFrame(rows).sort_values(["cost_adjusted", "oos_sharpe"], ascending=[True, False])
    selected_keys = [
        "base_regime_ic_vol_cost",
        "abundant_low_or_normal_half_cost",
        "abundant_low_or_normal_flat_cost",
        "abundant_curve_confirmed_half_cost",
    ]
    period_tables = {}
    for key in selected_keys:
        if key in backtests:
            period_tables[key] = period_performance(backtests[key])[
                ["period", "total_pnl", "sharpe", "max_drawdown", "hit_rate", "days"]
            ]
    _write_log(results, period_tables, rebuilt["selected_table"])
    return {
        "results": results.reset_index(drop=True),
        "positions": positions,
        "backtests": backtests,
        "period_tables": period_tables,
        "selected_table": rebuilt["selected_table"],
        "errors": rebuilt["errors"],
    }


In [71]:
corn_guard = safe_run("Corn abundant-supply risk controls", run_corn_abundant_supply_improvement, data_dir=DATA_DIR)
corn_guard_results = cost_only(corn_guard["results"] if corn_guard is not None else pd.DataFrame())
corn_guard_results["regime_interpretation_from_result"] = corn_guard_results["strategy"].map({
    "base_regime_ic_vol": "Base vol-regime candidate before abundant-supply risk control.",
    "below_ma_or_negative_mom_flat": "Evidence for weak-tape regime: fixed flat guard improves Sharpe/DD.",
    "below_ma_and_negative_mom_flat": "Stricter weak-tape guard; compare with OR version for robustness.",
    "abundant_low_or_normal_flat": "Tests abundant supply with low/normal vol confirmation.",
    "abundant_curve_confirmed_flat": "Tests abundant supply confirmed by weak curve.",
}).fillna("Corn abundant-supply diagnostic row.")
corn_guard_results[
    [
        "strategy", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd",
        "full_sharpe", "full_dd", "low_price_pnl", "low_price_dd",
        "covid_pnl", "covid_dd", "turnover", "regime_interpretation_from_result"
    ]
].sort_values("oos_sharpe", ascending=False).reset_index(drop=True)


### Corn abundant-supply risk controls

**Status:** success

,strategy,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,low_price_pnl,low_price_dd,covid_pnl,covid_dd,turnover,regime_interpretation_from_result
0,below_ma_or_negative_mom_flat,True,1.744611,309.892159,-174.158893,0.735935,-297.795198,-197.307984,-218.096999,NaN,NaN,0.008170,Evidence for weak-tape regime: fixed flat guard improves Sharpe/DD.
1,below_ma_and_negative_mom_flat,True,1.233042,334.579919,-213.721508,0.590131,-424.955785,-202.899678,-362.331027,NaN,NaN,0.006608,Stricter weak-tape guard; compare with OR version for robustness.
2,abundant_low_or_normal_flat,True,1.119316,332.036875,-213.721508,0.451228,-424.955785,-202.899678,-362.331027,NaN,NaN,0.006629,Tests abundant supply with low/normal vol confirmation.
3,abundant_curve_confirmed_flat,True,1.091186,313.784377,-199.492543,0.399924,-412.409514,-204.582372,-372.735595,-38.735516,-64.357163,0.006492,Tests abundant supply confirmed by weak curve.
4,below_ma_and_negative_mom_half,True,0.864622,286.961229,-240.700918,0.341550,-473.105130,-154.371185,-367.884888,-19.367758,-32.178581,0.005273,Corn abundant-supply diagnostic row.
5,below_ma_or_negative_mom_half,True,0.861841,274.381635,-182.857971,0.302826,-383.081260,-149.691880,-296.113796,-19.367758,-32.178581,0.004121,Corn abundant-supply diagnostic row.
6,abundant_low_or_normal_half,True,0.831980,285.689707,-240.700918,0.292469,-473.105130,-154.371185,-367.884888,-19.367758,-32.178581,0.005458,Corn abundant-supply diagnostic row.
7,abundant_curve_confirmed_half,True,0.829334,277.132202,-234.873196,0.273552,-468.345547,-155.439324,-377.060066,-38.735516,-64.357163,0.005359,Corn abundant-supply diagnostic row.
8,base_regime_ic_vol,True,0.676779,241.856754,-268.877122,0.213103,-519.888284,-101.755046,-394.598877,-38.735516,-64.357163,0.005364,Base vol-regime candidate before abundant-supply risk control.
9,abundant_low_vol_flat,True,0.652436,222.163384,-284.859483,0.181137,-497.400219,-204.206137,-363.637486,-38.735516,-64.357163,0.006387,Corn abundant-supply diagnostic row.


## Corn Step 5 - Best Corn Rows From Actual Tests

In [72]:
corn_best_external = corn_external_results.loc[corn_external_results.get("strategy", pd.Series(dtype=str)).isin([
    "corn_candidate_external_ethanol_family",
    "corn_candidate_requirement_given_80_ethanol_10_fx_10",
    "corn_candidate_regime_hybrid_vol_trend_weight",
    "external_ethanol_family",
    "requirement_given_80_ethanol_10_fx_10",
    "regime_hybrid_vol_trend_weight",
])].copy()
corn_best_guard = corn_guard_results.loc[corn_guard_results.get("strategy", pd.Series(dtype=str)).isin([
    "base_regime_ic_vol",
    "below_ma_or_negative_mom_flat",
])].copy()

print("Corn external/EIA candidates")
display(
    corn_best_external[
        ["strategy", "mode", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "turnover"]
    ].sort_values("oos_sharpe", ascending=False).reset_index(drop=True)
)
print("Corn abundant-supply guard candidates")
display(
    corn_best_guard[
        ["strategy", "oos_sharpe", "oos_pnl", "oos_dd", "full_sharpe", "full_dd", "low_price_pnl", "low_price_dd", "turnover"]
    ].sort_values("oos_sharpe", ascending=False).reset_index(drop=True)
)


Corn external/EIA candidates


,strategy,mode,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,turnover
0,corn_candidate_external_ethanol_family,long_only,0.653492,434.734557,-333.388125,-0.176104,-1241.562313,0.008007
1,corn_candidate_requirement_given_80_ethanol_10_fx_10,long_only,0.352538,202.606004,-426.025211,0.310767,-426.025211,0.005257
2,corn_candidate_regime_hybrid_vol_trend_weight,long_short,0.309713,183.456061,-224.408587,0.036054,-509.825362,0.004423
3,corn_candidate_requirement_given_80_ethanol_10_fx_10,long_short,0.215453,170.391198,-464.408755,0.212845,-464.408755,0.004642
4,corn_candidate_regime_hybrid_vol_trend_weight,long_only,0.141425,50.604840,-247.339398,0.067148,-366.976579,0.004607
5,corn_candidate_external_ethanol_family,long_short,0.121024,166.435982,-813.605401,-0.179444,-2028.785996,0.009717


Corn abundant-supply guard candidates


,strategy,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,low_price_pnl,low_price_dd,turnover
0,below_ma_or_negative_mom_flat,1.744611,309.892159,-174.158893,0.735935,-297.795198,-197.307984,-218.096999,0.008170
1,base_regime_ic_vol,0.676779,241.856754,-268.877122,0.213103,-519.888284,-101.755046,-394.598877,0.005364


## Corn Step 6 - Regimes That Became Clear

This regime table is shown only after the generic, EIA ethanol, and abundant-supply tests. The interpretation comes from the result path: generic trend/MR was weak, corn-specific vol/event logic helped, and weak-tape guards reduced drawdown.


In [73]:
corn_regime_rows = [
    {
        "product": "Corn",
        "regime": "vol/event regime",
        "detection": "vol-regime IC selection from regime_ic_vol",
        "experiment_where_clear": "Corn generic regime tests and corn abundant-supply rebuild",
        "practical_action": "Use as base signal only with risk guard",
    },
    {
        "product": "Corn",
        "regime": "ethanol demand channel",
        "detection": "EIA ethanol production/stocks signal family",
        "experiment_where_clear": "Corn external / EIA ethanol tests",
        "practical_action": "Keep as corn-specific demand input, not a fitted core model",
    },
    {
        "product": "Corn",
        "regime": "abundant supply / weak tape",
        "detection": "price below 252d MA or mom_60 < 0; stricter variants add low/normal vol and weak curve",
        "experiment_where_clear": "Corn abundant-supply risk-control tests",
        "practical_action": "Go flat/reduce exposure when weak tape condition is active",
    },
]
pd.DataFrame(corn_regime_rows)[
    ["product", "regime", "detection", "experiment_where_clear", "practical_action"]
].reset_index(drop=True)


,product,regime,detection,experiment_where_clear,practical_action
0,Corn,vol/event regime,vol-regime IC selection from regime_ic_vol,Corn generic regime tests and corn abundant-supply rebuild,Use as base signal only with risk guard
1,Corn,ethanol demand channel,EIA ethanol production/stocks signal family,Corn external / EIA ethanol tests,"Keep as corn-specific demand input, not a fitted core model"
2,Corn,abundant supply / weak tape,price below 252d MA or mom_60 < 0; stricter variants add low/normal vol and weak curve,Corn abundant-supply risk-control tests,Go flat/reduce exposure when weak tape condition is active


In [74]:
if corn_guard_results is not None and len(corn_guard_results) > 0:
    corn_regime_performance = corn_guard_results[
        [
            "strategy", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd",
            "low_price_pnl", "low_price_dd", "covid_pnl", "covid_dd"
        ]
    ].sort_values("oos_sharpe", ascending=False).reset_index(drop=True)
    corn_regime_performance
else:
    pd.DataFrame({"message": ["Run Corn Step 4 to show abundant-supply regime performance."]})


In [75]:
if corn_guard is not None and "selected_table" in corn_guard:
    corn_guard["selected_table"].reset_index(drop=True)
else:
    pd.DataFrame({"message": ["Run Corn abundant-supply risk-control tests to show the vol-regime selection table."]})


## Corn Step 7 - Key Period Performance For Best Rows

These tables show the corn EIA/regime candidates and the abundant-supply guard under the same named historical regimes.


In [76]:
corn_period_sources = []
if corn_external is not None:
    corn_period_sources.extend([
        (
            "corn_candidate_requirement_given_80_ethanol_10_fx_10_long_only_cost_adjusted",
            corn_external.get("backtests", {}).get("corn_candidate_requirement_given_80_ethanol_10_fx_10_cost_adjusted"),
        ),
        (
            "corn_candidate_regime_hybrid_vol_trend_weight_long_short_cost_adjusted",
            corn_external.get("backtests", {}).get("corn_candidate_regime_hybrid_vol_trend_weight_cost_adjusted"),
        ),
    ])
if corn_guard is not None:
    corn_period_sources.extend([
        (
            "base_regime_ic_vol_cost",
            corn_guard.get("backtests", {}).get("base_regime_ic_vol_cost"),
        ),
        (
            "below_ma_or_negative_mom_flat_cost",
            corn_guard.get("backtests", {}).get("below_ma_or_negative_mom_flat_cost"),
        ),
    ])

for name, bt in corn_period_sources:
    display(Markdown(f"### {name}"))
    if bt is None:
        display(pd.DataFrame({"message": ["Backtest not available. Run the corresponding corn experiment cell first, or this candidate was not generated in the current data run."]}))
    else:
        display(
            period_performance(bt)[
                ["period", "total_pnl", "sharpe", "max_drawdown", "hit_rate", "days"]
            ].reset_index(drop=True)
        )


### corn_candidate_requirement_given_80_ethanol_10_fx_10_long_only_cost_adjusted

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,68.881912,1.331113,-71.578043,0.453488,86.0
1,US drought rally/retrace,9.308579,0.337050,-38.247207,0.478261,92.0
2,Crimea/Black Sea shock,25.196768,6.414937,-4.331109,0.500000,16.0
3,Low-price abundant supply,36.495958,0.108127,-241.157774,0.502304,434.0
4,US-China trade war,-10.475502,-0.023502,-314.300123,0.483333,360.0
5,2019 prevented planting floods,151.259992,1.006028,-288.349095,0.573770,61.0
6,COVID demand shock,-55.071410,-0.726377,-179.087839,0.500000,76.0
7,COVID recovery/China buying,12.911346,0.125961,-234.270743,0.530973,113.0


### corn_candidate_regime_hybrid_vol_trend_weight_long_short_cost_adjusted

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,74.251055,0.322079,-232.405029,0.470297,202.0
1,US drought rally/retrace,60.743750,0.486318,-189.967908,0.460526,152.0
2,Crimea/Black Sea shock,51.736338,4.385833,-16.326327,0.533333,30.0
3,Low-price abundant supply,-134.906841,-0.219787,-380.163002,0.507440,672.0
4,US-China trade war,-73.983305,-0.236826,-217.515523,0.474320,331.0
5,2019 prevented planting floods,11.522373,0.116816,-201.802056,0.564516,62.0
6,COVID demand shock,183.482323,1.865706,-137.136537,0.550000,80.0
7,COVID recovery/China buying,-99.750632,-1.305110,-224.408587,0.530973,113.0


### base_regime_ic_vol_cost

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,332.297451,3.968100,-64.697587,0.531250,64.0
1,US drought rally/retrace,90.279321,4.420473,-11.950541,0.533333,30.0
2,Crimea/Black Sea shock,94.292830,2.773038,-28.908761,0.534884,43.0
3,Low-price abundant supply,-101.755046,-0.284488,-394.598877,0.468835,369.0
4,US-China trade war,84.577472,0.401827,-174.158893,0.474453,137.0
5,2019 prevented planting floods,183.212637,2.332737,-174.158893,0.480000,25.0
6,COVID demand shock,-38.735516,-4.217632,-64.357163,0.312500,16.0
7,COVID recovery/China buying,223.195854,3.348010,-56.914377,0.565217,69.0


### below_ma_or_negative_mom_flat_cost

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,344.907477,4.285755,-64.697587,0.550000,60.0
1,US drought rally/retrace,47.165186,6.764044,-9.002817,0.714286,7.0
2,Crimea/Black Sea shock,-49.962364,-2.722911,-81.449345,0.391304,23.0
3,Low-price abundant supply,-197.307984,-3.740359,-218.096999,0.476190,42.0
4,US-China trade war,183.212637,2.332737,-174.158893,0.480000,25.0
5,2019 prevented planting floods,183.212637,2.332737,-174.158893,0.480000,25.0
6,COVID demand shock,NaN,NaN,NaN,NaN,NaN
7,COVID recovery/China buying,162.240093,4.125967,-51.918687,0.583333,36.0


# Wheat SRW / HRW

Research path:

1. Directional wheat tests were weak.
2. Fitted directional models were weak.
3. SRW/HRW pair tests produced the usable wheat strategy.

## Wheat Questions Before Testing

At this point I do not assume the final wheat structure. The next tests ask:

| Question | Test That Can Answer It | What Would Count As Evidence |
|---|---|---|
| Can SRW and HRW work as outright directional sleeves? | generic directional tests and OLS/Kalman/Ridge benchmarks | positive OOS Sharpe with acceptable DD for each outright wheat product |
| If outright wheat is weak, does relative value work better? | SRW/HRW pair tests | pair rows materially beat directional rows |
| Does the pair need trend/MR awareness? | pair trend-aware and trend-conflict tests | trend-aware/flat-filter rows improve Sharpe/DD or weak periods |

The pair trend/MR regime is shown only after the pair experiment creates the evidence.


## Wheat Step 1 - Generic Directional Tests

In [77]:
wheat_generic = family_results.loc[family_results.get("commodity", pd.Series(dtype=str)).isin(["WHEAT_SRW", "WHEAT_HRW"])].copy()
wheat_generic["regime_interpretation_from_result"] = wheat_generic["strategy"].map({
    "avg_all_signals": "Outright baseline. Weak OOS suggests not enough directional wheat edge.",
    "equal_family_weight": "Tests whether broad family diversification saves outright wheat.",
    "ic_family_selected_physical_public": "Tests IC-selected directional wheat; weak OOS points away from outright IC selection.",
    "regime_best_family_trend_mr": "Tests whether generic trend/MR helps outright wheat.",
    "regime_avg_families_trend_mr": "Tests generic regime logic without best-family selection.",
    "ols_kalman_filter": "Fitted benchmark; weak/unstable results argue against directional coefficient models.",
    "ridge_expanding_alpha100": "Regularized fitted benchmark; weak results argue for relative-value pair instead.",
}).fillna("Diagnostic row.")
wheat_generic[
    [
        "commodity", "strategy", "mode", "cost_adjusted",
        "oos_sharpe", "oos_pnl", "oos_dd",
        "full_sharpe", "full_dd", "regime_interpretation_from_result", "overfit_comment"
    ]
].sort_values(["commodity", "oos_sharpe"], ascending=[True, False]).reset_index(drop=True)


,commodity,strategy,mode,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,regime_interpretation_from_result,overfit_comment
0,WHEAT_HRW,regime_best_family_trend_mr,long_short,True,0.029644,10.359279,-197.169027,0.210953,-576.778467,Tests whether generic trend/MR helps outright wheat.,"Moderate selection risk; weights are fixed masks, not optimized."
1,WHEAT_HRW,ridge_expanding_alpha100,long_short,True,-0.031525,-27.991604,-547.832390,-0.464530,-2045.428773,Regularized fitted benchmark; weak results argue for relative-value pair instead.,Coefficient-estimation risk; fixed alpha reduces but does not remove it. Reject: negative cost-adjusted OOS.
2,WHEAT_HRW,regime_avg_families_trend_mr,long_short,True,-0.142032,-54.906726,-271.863928,0.120905,-462.508561,Tests generic regime logic without best-family selection.,"Lower than best-family regime selection, but still regime-research risk. Reject: negative cost-adjusted OOS."
3,WHEAT_HRW,ic_family_selected_physical_public,long_short,True,-0.240431,-249.674424,-605.622668,0.167225,-1117.274195,Tests IC-selected directional wheat; weak OOS points away from outright IC selection.,Moderate/high selection risk; validation IC can be noisy. Reject: negative cost-adjusted OOS.
4,WHEAT_HRW,equal_family_weight,long_short,True,-0.302417,-155.124783,-367.588727,-0.286078,-826.259960,Tests whether broad family diversification saves outright wheat.,Low model overfit; family choices are researcher-defined. Reject: negative cost-adjusted OOS.
5,WHEAT_HRW,avg_all_signals,long_short,True,-0.477294,-207.751409,-359.037180,-0.264086,-723.572172,Outright baseline. Weak OOS suggests not enough directional wheat edge.,"Low model overfit, but can be economically diluted. Reject: negative cost-adjusted OOS."
6,WHEAT_HRW,ols_kalman_filter,long_short,True,-0.554160,-458.301132,-718.692799,-0.556003,-2546.897125,Fitted benchmark; weak/unstable results argue against directional coefficient models.,"Coefficient-estimation risk; parameters are fixed, not OOS tuned. Reject: negative cost-adjusted OOS."
7,WHEAT_SRW,regime_avg_families_trend_mr,long_short,True,0.183743,88.211850,-199.602011,0.604021,-395.920862,Tests generic regime logic without best-family selection.,"Lower than best-family regime selection, but still regime-research risk."
8,WHEAT_SRW,ols_kalman_filter,long_short,True,0.063795,55.191986,-441.411845,-0.132464,-924.149498,Fitted benchmark; weak/unstable results argue against directional coefficient models.,"Coefficient-estimation risk; parameters are fixed, not OOS tuned. Positive OOS but keep as diagnostic unless validation/full-period behavior is clean."
9,WHEAT_SRW,regime_best_family_trend_mr,long_short,True,0.030022,31.946072,-405.187227,0.380509,-680.587284,Tests whether generic trend/MR helps outright wheat.,"Moderate selection risk; weights are fixed masks, not optimized."


## Wheat Step 2 - OLS / Kalman / Ridge Directional Benchmark

In [78]:
wheat_linear = family_results.loc[
    family_results.get("commodity", pd.Series(dtype=str)).isin(["WHEAT_SRW", "WHEAT_HRW"])
    & family_results.get("strategy", pd.Series(dtype=str)).isin(["ols_kalman_filter", "ridge_expanding_alpha100"])
].copy()
wheat_linear[
    [
        "commodity", "strategy", "mode", "cost_adjusted",
        "oos_sharpe", "oos_pnl", "oos_dd",
        "full_sharpe", "full_dd", "overfit_comment"
    ]
].sort_values(["commodity", "oos_sharpe"], ascending=[True, False]).reset_index(drop=True)


,commodity,strategy,mode,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,full_sharpe,full_dd,overfit_comment
0,WHEAT_HRW,ridge_expanding_alpha100,long_short,True,-0.031525,-27.991604,-547.832390,-0.464530,-2045.428773,Coefficient-estimation risk; fixed alpha reduces but does not remove it. Reject: negative cost-adjusted OOS.
1,WHEAT_HRW,ols_kalman_filter,long_short,True,-0.554160,-458.301132,-718.692799,-0.556003,-2546.897125,"Coefficient-estimation risk; parameters are fixed, not OOS tuned. Reject: negative cost-adjusted OOS."
2,WHEAT_SRW,ols_kalman_filter,long_short,True,0.063795,55.191986,-441.411845,-0.132464,-924.149498,"Coefficient-estimation risk; parameters are fixed, not OOS tuned. Positive OOS but keep as diagnostic unless validation/full-period behavior is clean."
3,WHEAT_SRW,ridge_expanding_alpha100,long_short,True,-0.292091,-291.609470,-842.337423,-0.197295,-994.449731,Coefficient-estimation risk; fixed alpha reduces but does not remove it. Reject: negative cost-adjusted OOS.


## Wheat Step 3 - SRW/HRW Pair Tests

This exact test produces:

- `pair_price_mr_cargill_90_10_cost_control`
- `pair_price_mr_cargill_80_20_pair_trend_cost_control`
- `pair_price_mr_cargill_trend_conflict_flat_cost_control`

### Direct Code Used Here: Wheat SRW/HRW Pair Experiment

The next cell defines the implemented wheat pair experiment directly before it is called.


In [ ]:
"""Lower-overfit wheat improvement experiments.

The previous standalone WHEAT_SRW and WHEAT_HRW outright sleeves were weak.
This script tests wheat as a SRW/HRW relative-value sleeve instead:

- no fitted coefficients;
- no external data;
- fixed economic signal families;
- fixed trend/MR regime masks;
- leg-level costs and margin funding.

The goal is not to discover a complex wheat model. It is to check whether wheat
has a more fund-usable role as a risk-balanced pair trade.
"""


import os

import numpy as np
import pandas as pd



SPLIT_DATE = "2018-01-01"
TRAIN_END = pd.Timestamp("2016-01-01")
WHEAT = ["WHEAT_SRW", "WHEAT_HRW"]
TARGET_DAILY_PAIR_VOL = 45.0
MAX_LEG_LOT = 0.45
SIGNAL_THRESHOLD = 0.05


def _clean(series, index):
    return series.reindex(index).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-5.0, 5.0)


def _tanh(series, divisor=2.0):
    return pd.Series(np.tanh(series.astype(float) / float(divisor)), index=series.index)


def _components(panel):
    index = panel.index
    price_mr = panel["rev_5"]
    price_trend = (panel["mom_20"] + panel["mom_60"]) / 2.0
    curve = (panel["curve_spread"] + panel["curve_ratio"] + panel["curve_change_20"]) / 3.0
    cot = (
        panel["cot_mm_level"]
        + panel["cot_mm_change"]
        + panel["cot_pm_oi_level"]
        + panel["cot_pm_oi_change"]
    ) / 4.0
    physical_public = (-panel["public_inventory_change"] - panel["receipts_change"]) / 2.0
    physical_cargill = (
        -panel["cgl_inventory_change"]
        + panel["crush_surprise"]
        + panel["crush_utilization"]
    ) / 3.0
    physical = (physical_public + physical_cargill) / 2.0
    all_family = (price_mr + price_trend + curve + cot + physical_public + physical_cargill) / 6.0
    return {
        "price_mr": _clean(price_mr, index),
        "price_trend": _clean(price_trend, index),
        "curve": _clean(curve, index),
        "cot": _clean(cot, index),
        "physical_public": _clean(physical_public, index),
        "physical_cargill": _clean(physical_cargill, index),
        "physical": _clean(physical, index),
        "all_family_equal": _clean(all_family, index),
    }


def _pair_difference(feature_panels, component_name):
    srw = _components(feature_panels["WHEAT_SRW"])[component_name]
    hrw = _components(feature_panels["WHEAT_HRW"])[component_name]
    return (srw - hrw).fillna(0.0)


def build_pair_signals(feature_panels, futures_pnl):
    index = futures_pnl.index
    pair = {
        "pair_price_mr": _pair_difference(feature_panels, "price_mr"),
        "pair_price_trend": _pair_difference(feature_panels, "price_trend"),
        "pair_curve": _pair_difference(feature_panels, "curve"),
        "pair_cot": _pair_difference(feature_panels, "cot"),
        "pair_physical_public": _pair_difference(feature_panels, "physical_public"),
        "pair_physical_cargill": _pair_difference(feature_panels, "physical_cargill"),
        "pair_physical": _pair_difference(feature_panels, "physical"),
        "pair_all_family_equal": _pair_difference(feature_panels, "all_family_equal"),
    }

    pair["pair_mr_curve_physical_equal"] = (
        pair["pair_price_mr"] + pair["pair_curve"] + pair["pair_physical"]
    ) / 3.0
    pair["pair_given_physical_equal"] = (
        pair["pair_physical_public"] + pair["pair_physical_cargill"]
    ) / 2.0
    pair["pair_balanced_low_overfit"] = (
        pair["pair_price_mr"]
        + pair["pair_price_trend"]
        + pair["pair_curve"]
        + pair["pair_cot"]
        + pair["pair_physical_public"]
        + pair["pair_physical_cargill"]
    ) / 6.0

    srw_trend = feature_panels["WHEAT_SRW"]["mom_60"].reindex(index).abs()
    hrw_trend = feature_panels["WHEAT_HRW"]["mom_60"].reindex(index).abs()
    trend_strength = ((srw_trend + hrw_trend) / 2.0).fillna(0.0)
    trend_threshold = trend_strength.expanding(min_periods=252).median().shift(1)
    trend_regime = (trend_strength > trend_threshold).fillna(False)

    mr_source = (pair["pair_price_mr"] + pair["pair_curve"] + pair["pair_physical"]) / 3.0
    trend_source = (pair["pair_price_trend"] + pair["pair_curve"] + pair["pair_cot"]) / 3.0
    pair["pair_fixed_trend_mr_regime"] = (
        mr_source.where(~trend_regime, 0.0) + trend_source.where(trend_regime, 0.0)
    )
    pair["pair_soft_hybrid_70_30"] = (0.70 * mr_source + 0.30 * trend_source).fillna(0.0)

    return {name: _clean(signal, index) for name, signal in pair.items()}, trend_regime


def build_pair_price_trend_from_prices(data, futures_pnl):
    index = futures_pnl.index
    srw = data["adj1"]["WHEAT_SRW"].reindex(index).ffill()
    hrw = data["adj1"]["WHEAT_HRW"].reindex(index).ffill()
    ratio = (srw / hrw).replace([np.inf, -np.inf], np.nan).ffill()
    trend_20 = rolling_zscore(ratio.pct_change(20), 252, 60).reindex(index).fillna(0.0)
    trend_60 = rolling_zscore(ratio.pct_change(60), 252, 80).reindex(index).fillna(0.0)
    return ((trend_20 + trend_60) / 2.0).clip(-5.0, 5.0).fillna(0.0)


def positions_from_pair_signal(
    signal,
    futures_pnl,
    target_daily_pair_vol=TARGET_DAILY_PAIR_VOL,
    max_leg_lot=MAX_LEG_LOT,
    signal_threshold=SIGNAL_THRESHOLD,
    halflife=2.0,
    rebalance_every=1,
):
    signal = _tanh(signal.reindex(futures_pnl.index).fillna(0.0))
    signal = signal.ewm(halflife=float(halflife), adjust=False, min_periods=1).mean()
    signal[signal.abs() < float(signal_threshold)] = 0.0

    vol = futures_pnl[WHEAT].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    srw_pos = signal * (float(target_daily_pair_vol) / vol["WHEAT_SRW"])
    hrw_pos = -signal * (float(target_daily_pair_vol) / vol["WHEAT_HRW"])

    positions = pd.DataFrame(0.0, index=futures_pnl.index, columns=WHEAT)
    positions["WHEAT_SRW"] = srw_pos.clip(-float(max_leg_lot), float(max_leg_lot))
    positions["WHEAT_HRW"] = hrw_pos.clip(-float(max_leg_lot), float(max_leg_lot))
    positions = positions.fillna(0.0)
    if int(rebalance_every) > 1:
        rebalance_mask = pd.Series(False, index=positions.index)
        rebalance_mask.iloc[:: int(rebalance_every)] = True
        positions = positions.where(rebalance_mask).ffill().fillna(0.0)
    return positions


def _active_win_days(bt):
    active = bt.loc[bt["held_gross_exposure"] > 1.0e-12, "net_pnl"].dropna()
    return int((active > 0.0).sum()) if len(active) else 0


def _metrics(bt):
    table = split_performance(bt, SPLIT_DATE)
    oos = bt.loc[bt.index >= SPLIT_DATE]
    full_active_margin = bt.loc[bt["held_gross_exposure"] > 1.0e-12]
    avg_margin = (
        full_active_margin["margin_used"].mean()
        if "margin_used" in bt.columns and not full_active_margin.empty
        else np.nan
    )
    oos_dd = table.loc["max_drawdown", "out_of_sample"]
    full_dd = table.loc["max_drawdown", "full_period"]
    return {
        "is_sharpe": table.loc["sharpe", "in_sample"],
        "oos_sharpe": table.loc["sharpe", "out_of_sample"],
        "oos_pnl": table.loc["total_pnl", "out_of_sample"],
        "oos_dd": oos_dd,
        "oos_dd_pct_avg_margin": (abs(oos_dd) / avg_margin * 100.0) if pd.notnull(avg_margin) and avg_margin else np.nan,
        "full_sharpe": table.loc["sharpe", "full_period"],
        "full_pnl": table.loc["total_pnl", "full_period"],
        "full_dd": full_dd,
        "full_dd_pct_avg_margin": (abs(full_dd) / avg_margin * 100.0) if pd.notnull(avg_margin) and avg_margin else np.nan,
        "hit_rate": table.loc["hit_rate", "out_of_sample"],
        "win_days": _active_win_days(oos),
        "turnover": table.loc["avg_daily_turnover", "full_period"],
        "avg_gross_exposure": table.loc["avg_gross_exposure", "full_period"],
        "avg_margin_used": avg_margin,
    }


def _evaluate_pair(strategy, signal, futures_pnl, position_kwargs=None):
    if position_kwargs is None:
        position_kwargs = {}
    positions = positions_from_pair_signal(signal, futures_pnl, **position_kwargs)
    rows = []
    backtests = {}
    for cost_adjusted in [False, True]:
        if cost_adjusted:
            bt, _ = backtest_positions_with_costs(positions, futures_pnl[WHEAT], 8.75, 0.05)
        else:
            bt, _ = backtest_positions(positions, futures_pnl[WHEAT], 0.0)
        row = {
            "book": "SRW_HRW_PAIR",
            "strategy": strategy,
            "cost_adjusted": cost_adjusted,
            "formula": STRATEGY_FORMULAS[strategy],
            "overfit": STRATEGY_OVERFIT[strategy],
        }
        row.update(_metrics(bt))
        rows.append(row)
        backtests[strategy + ("_cost" if cost_adjusted else "_zero")] = bt
    return rows, backtests, positions


STRATEGY_FORMULAS = {
    "pair_price_mr": "signal = SRW.rev_5 - HRW.rev_5",
    "pair_price_trend": "signal = mean(SRW.mom_20,mom_60) - mean(HRW.mom_20,mom_60)",
    "pair_curve": "signal = SRW curve family - HRW curve family",
    "pair_cot": "signal = SRW COT family - HRW COT family",
    "pair_physical_public": "signal = SRW public inventory/receipts pressure - HRW public pressure",
    "pair_physical_cargill": "signal = SRW Cargill inventory/crush family - HRW Cargill family",
    "pair_physical": "signal = SRW total physical pressure - HRW total physical pressure",
    "pair_all_family_equal": "signal = equal-weight family score(SRW) - equal-weight family score(HRW)",
    "pair_mr_curve_physical_equal": "signal = equal_weight(pair MR, pair curve, pair physical)",
    "pair_given_physical_equal": "signal = equal_weight(pair public physical, pair Cargill physical)",
    "pair_balanced_low_overfit": "signal = equal_weight(pair MR, trend, curve, COT, public physical, Cargill physical)",
    "pair_fixed_trend_mr_regime": "if observable trend regime: trend/curve/COT pair; otherwise MR/curve/physical pair",
    "pair_soft_hybrid_70_30": "signal = 70% MR/curve/physical pair + 30% trend/curve/COT pair",
    "pair_price_mr_cost_control": "same signal as pair_price_mr, but halflife 5, threshold 0.12, weekly rebalance, 40 target daily pair vol",
    "pair_mr_curve_physical_cost_control": "same signal as pair_mr_curve_physical_equal, but halflife 5, threshold 0.12, weekly rebalance, 40 target daily pair vol",
    "pair_price_mr_cargill_90_10_cost_control": "90% SRW/HRW price MR + 10% SRW/HRW Cargill physical family, with fixed cost-control execution",
    "pair_price_mr_cargill_95_05_cost_control": "95% SRW/HRW price MR + 5% SRW/HRW Cargill physical family, with fixed cost-control execution",
    "pair_price_mr_physical_90_10_cost_control": "90% SRW/HRW price MR + 10% SRW/HRW total physical family, with fixed cost-control execution",
    "pair_price_mr_cargill_80_20_pair_trend_cost_control": "80% price-MR/Cargill pair signal + 20% SRW/HRW price-ratio trend, with fixed cost-control execution",
    "pair_price_mr_cargill_trend_conflict_flat_cost_control": "price-MR/Cargill pair signal, but flat when observable SRW/HRW trend strongly conflicts with MR",
}


STRATEGY_OVERFIT = {
    "pair_price_mr": "Very low: one fixed price-reversion hypothesis.",
    "pair_price_trend": "Very low: one fixed price-trend hypothesis.",
    "pair_curve": "Very low: one fixed curve relative-tightness hypothesis.",
    "pair_cot": "Very low: one fixed positioning hypothesis.",
    "pair_physical_public": "Low: fixed public physical relative-tightness hypothesis.",
    "pair_physical_cargill": "Low: fixed Cargill physical relative-tightness hypothesis; uses both cgl_inv and cgl_crush.",
    "pair_physical": "Low: fixed public+Cargill physical hypothesis.",
    "pair_all_family_equal": "Low: equal family weights; no selected coefficients.",
    "pair_mr_curve_physical_equal": "Low/moderate: curated three-family economic basket.",
    "pair_given_physical_equal": "Low: physical-only equal family basket.",
    "pair_balanced_low_overfit": "Low: broad equal family basket; no selected coefficients.",
    "pair_fixed_trend_mr_regime": "Moderate research risk: fixed observable regime switch, no optimized weights.",
    "pair_soft_hybrid_70_30": "Moderate research risk: fixed hybrid weight, chosen as conservative MR-dominant wheat prior.",
    "pair_price_mr_cost_control": "Low/moderate: no alpha fitting, but adds a fixed execution filter after seeing costs matter.",
    "pair_mr_curve_physical_cost_control": "Low/moderate: no alpha fitting, but adds a fixed execution filter after seeing costs matter.",
    "pair_price_mr_cargill_90_10_cost_control": "Low/moderate: round-number 10% Cargill physical overlay; no fitted coefficients, but still a researched blend.",
    "pair_price_mr_cargill_95_05_cost_control": "Moderate: strongest OOS Sharpe, but choosing 5% instead of 10% is more selection-sensitive.",
    "pair_price_mr_physical_90_10_cost_control": "Low/moderate: round-number 10% total physical overlay; no fitted coefficients, but still a researched blend.",
    "pair_price_mr_cargill_80_20_pair_trend_cost_control": "Low/moderate: fixed 80/20 blend chosen as a trend-risk diversifier, not fitted coefficients.",
    "pair_price_mr_cargill_trend_conflict_flat_cost_control": "Low/moderate: observable trend-risk filter; avoids fighting strong pair trends but can undertrade.",
}


def _write_log(results, period_tables=None, path="notes/wheat_improvement.txt"):
    if period_tables is None:
        period_tables = {}
    lines = []
    lines.append("Wheat lower-overfit improvement experiments")
    lines.append("Date: 2026-05-02")
    lines.append("")
    lines.append("Goal")
    lines.append("----")
    lines.append("Improve WHEAT_SRW and WHEAT_HRW without fitting coefficients by testing wheat as a SRW/HRW relative-value sleeve.")
    lines.append("")
    lines.append("Controls")
    lines.append("--------")
    lines.append("- No OLS, Ridge, Kalman, or fitted model weights.")
    lines.append("- No external data.")
    lines.append("- Fixed signal formulas, fixed trend/MR regime rule, fixed risk target.")
    lines.append("- Leg-level SRW and HRW positions are risk-balanced with 60-day realized PnL volatility.")
    lines.append("- Cost-adjusted rows include 8.75 USD per one-way lot plus 5% annual margin funding.")
    lines.append("- Cargill requirement: pair_physical_cargill and all combined physical/family rows use both cgl_inventory_change from cgl_inv and crush_surprise/crush_utilization from cgl_crush.")
    lines.append("")
    lines.append("Position rule")
    lines.append("-------------")
    lines.append("pair_score > 0: long WHEAT_SRW, short WHEAT_HRW.")
    lines.append("pair_score < 0: short WHEAT_SRW, long WHEAT_HRW.")
    lines.append("Default leg_position = tanh(pair_score / 2) * 45 / 60d_leg_pnl_vol, clipped to +/-0.45 lots.")
    lines.append("Cost-control variants use halflife 5, threshold 0.12, weekly rebalance, and target daily pair vol 40.")
    lines.append("")
    lines.append("Results")
    lines.append("-------")
    cols = [
        "book",
        "strategy",
        "cost_adjusted",
        "is_sharpe",
        "oos_sharpe",
        "oos_pnl",
        "oos_dd",
        "oos_dd_pct_avg_margin",
        "full_sharpe",
        "full_pnl",
        "full_dd",
        "full_dd_pct_avg_margin",
        "hit_rate",
        "win_days",
        "turnover",
        "avg_gross_exposure",
        "avg_margin_used",
    ]
    lines.append(results[cols].to_string(index=False, float_format=lambda value: "{:.3f}".format(value)))
    lines.append("")
    lines.append("Formulas and overfit notes")
    lines.append("--------------------------")
    for strategy in results["strategy"].drop_duplicates():
        lines.append("- {}: {}".format(strategy, STRATEGY_FORMULAS[strategy]))
        lines.append("  Overfit: {}".format(STRATEGY_OVERFIT[strategy]))
    lines.append("")
    if period_tables:
        lines.append("Key period performance")
        lines.append("----------------------")
        for name, table in period_tables.items():
            lines.append("")
            lines.append(name)
            lines.append(table.to_string(index=False, float_format=lambda value: "{:.3f}".format(value)))
        lines.append("")
    cost = results.loc[results["cost_adjusted"]].sort_values(["oos_sharpe", "oos_dd"], ascending=[False, False])
    best = cost.iloc[0]
    recommended = cost.loc[cost["strategy"] == "pair_price_mr_cargill_90_10_cost_control"].iloc[0]
    lines.append("Recommendation")
    lines.append("--------------")
    lines.append(
        "Highest cost-adjusted SRW/HRW pair row: {}, OOS Sharpe {:.3f}, OOS PnL {:.3f}, OOS DD {:.3f}, OOS DD {:.2f}% of average margin, full Sharpe {:.3f}.".format(
            best["strategy"],
            best["oos_sharpe"],
            best["oos_pnl"],
            best["oos_dd"],
            best["oos_dd_pct_avg_margin"],
            best["full_sharpe"],
        )
    )
    lines.append(
        "Lower-overfit recommended row: {}, OOS Sharpe {:.3f}, OOS PnL {:.3f}, OOS DD {:.3f}, OOS DD {:.2f}% of average margin, full Sharpe {:.3f}.".format(
            recommended["strategy"],
            recommended["oos_sharpe"],
            recommended["oos_pnl"],
            recommended["oos_dd"],
            recommended["oos_dd_pct_avg_margin"],
            recommended["full_sharpe"],
        )
    )
    lines.append("Use as a wheat relative-value sleeve only if the fund mandate allows SRW/HRW pair trading. Do not present it as two independent outright wheat alphas.")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as handle:
        handle.write("\n".join(lines))


def run_wheat_improvement_experiment(data_dir="train_set"):
    data = load_train_set(data_dir)
    feature_panels, futures_pnl = build_feature_panels(data)
    pair_signals, trend_regime = build_pair_signals(feature_panels, futures_pnl)
    pair_price_trend = build_pair_price_trend_from_prices(data, futures_pnl)
    rows = []
    positions = {}
    backtests = {}
    for strategy, signal in pair_signals.items():
        new_rows, new_backtests, new_positions = _evaluate_pair(strategy, signal, futures_pnl)
        rows.extend(new_rows)
        backtests.update(new_backtests)
        positions[strategy] = new_positions
    cost_control = {
        "target_daily_pair_vol": 40.0,
        "max_leg_lot": 0.40,
        "signal_threshold": 0.12,
        "halflife": 5.0,
        "rebalance_every": 5,
    }
    cost_control_map = {
        "pair_price_mr_cost_control": pair_signals["pair_price_mr"],
        "pair_mr_curve_physical_cost_control": pair_signals["pair_mr_curve_physical_equal"],
        "pair_price_mr_cargill_90_10_cost_control": (
            0.90 * pair_signals["pair_price_mr"] + 0.10 * pair_signals["pair_physical_cargill"]
        ),
        "pair_price_mr_cargill_95_05_cost_control": (
            0.95 * pair_signals["pair_price_mr"] + 0.05 * pair_signals["pair_physical_cargill"]
        ),
        "pair_price_mr_physical_90_10_cost_control": (
            0.90 * pair_signals["pair_price_mr"] + 0.10 * pair_signals["pair_physical"]
        ),
    }
    mr_cargill_90_10 = cost_control_map["pair_price_mr_cargill_90_10_cost_control"]
    cost_control_map["pair_price_mr_cargill_80_20_pair_trend_cost_control"] = (
        0.80 * mr_cargill_90_10 + 0.20 * pair_price_trend
    )
    trend_strength = pair_price_trend.abs()
    trend_state = (
        trend_strength > trend_strength.expanding(min_periods=252).median().shift(1)
    ).fillna(False)
    conflict = trend_state & ((mr_cargill_90_10 * pair_price_trend) < 0.0)
    cost_control_map["pair_price_mr_cargill_trend_conflict_flat_cost_control"] = mr_cargill_90_10.where(
        ~conflict,
        0.0,
    )
    for strategy, signal in cost_control_map.items():
        new_rows, new_backtests, new_positions = _evaluate_pair(
            strategy,
            signal,
            futures_pnl,
            position_kwargs=cost_control,
        )
        rows.extend(new_rows)
        backtests.update(new_backtests)
        positions[strategy] = new_positions
    results = pd.DataFrame(rows).sort_values(
        ["cost_adjusted", "oos_sharpe", "oos_dd"], ascending=[True, False, False]
    ).reset_index(drop=True)
    period_tables = {}
    for key in [
        "pair_price_mr_cargill_90_10_cost_control_cost",
        "pair_price_mr_cargill_80_20_pair_trend_cost_control_cost",
        "pair_price_mr_cargill_trend_conflict_flat_cost_control_cost",
    ]:
        if key in backtests:
            period_tables[key] = period_performance(backtests[key])[
                ["period", "total_pnl", "sharpe", "max_drawdown", "hit_rate", "days"]
            ]
    _write_log(results, period_tables=period_tables)
    return {
        "results": results,
        "positions": positions,
        "backtests": backtests,
        "trend_regime": trend_regime,
    }


In [79]:
wheat_pair = safe_run("Wheat SRW/HRW pair tests", run_wheat_improvement_experiment, data_dir=DATA_DIR)
wheat_results = cost_only(wheat_pair["results"] if wheat_pair is not None else pd.DataFrame())
wheat_results["regime_interpretation_from_result"] = wheat_results["strategy"].map({
    "pair_price_mr_cargill_90_10_cost_control": "Evidence that wheat works better as SRW/HRW pair MR plus Cargill physical, not outright direction.",
    "pair_price_mr_cargill_80_20_pair_trend_cost_control": "Evidence for trend-aware pair regime: small trend sleeve helps trend-like periods.",
    "pair_price_mr_cargill_trend_conflict_flat_cost_control": "Evidence for trend-conflict risk regime: flat filter improves Sharpe/DD but is more timing-like.",
}).fillna("Wheat pair diagnostic row.")
wheat_results[
    [
        "strategy", "cost_adjusted", "oos_sharpe", "oos_pnl", "oos_dd",
        "oos_dd_pct_avg_margin", "full_sharpe", "full_dd", "hit_rate",
        "win_days", "turnover", "formula", "regime_interpretation_from_result", "overfit"
    ]
].sort_values("oos_sharpe", ascending=False).head(25).reset_index(drop=True)


### Wheat SRW/HRW pair tests

**Status:** success

,strategy,cost_adjusted,oos_sharpe,oos_pnl,oos_dd,oos_dd_pct_avg_margin,full_sharpe,full_dd,hit_rate,win_days,turnover,formula,regime_interpretation_from_result,overfit
0,pair_price_mr_cargill_trend_conflict_flat_cost_control,True,2.145334,26.671254,-16.955673,20.821759,1.461884,-20.245805,0.506667,38,0.004858,"price-MR/Cargill pair signal, but flat when observable SRW/HRW trend strongly conflicts with MR",Evidence for trend-conflict risk regime: flat filter improves Sharpe/DD but is more timing-like.,Low/moderate: observable trend-risk filter; avoids fighting strong pair trends but can undertrade.
1,pair_price_mr_cargill_80_20_pair_trend_cost_control,True,1.290889,36.374086,-29.441327,42.561576,1.098366,-29.441327,0.517647,88,0.003103,"80% price-MR/Cargill pair signal + 20% SRW/HRW price-ratio trend, with fixed cost-control execution",Evidence for trend-aware pair regime: small trend sleeve helps trend-like periods.,"Low/moderate: fixed 80/20 blend chosen as a trend-risk diversifier, not fitted coefficients."
2,pair_price_mr_cargill_95_05_cost_control,True,1.222445,34.844399,-23.979667,27.742337,1.496643,-23.979667,0.490909,81,0.004954,"95% SRW/HRW price MR + 5% SRW/HRW Cargill physical family, with fixed cost-control execution",Wheat pair diagnostic row.,"Moderate: strongest OOS Sharpe, but choosing 5% instead of 10% is more selection-sensitive."
3,pair_price_mr_cargill_90_10_cost_control,True,1.128948,26.689223,-19.909683,22.755886,1.403119,-22.050536,0.478571,67,0.005002,"90% SRW/HRW price MR + 10% SRW/HRW Cargill physical family, with fixed cost-control execution","Evidence that wheat works better as SRW/HRW pair MR plus Cargill physical, not outright direction.","Low/moderate: round-number 10% Cargill physical overlay; no fitted coefficients, but still a researched blend."
4,pair_price_mr_cost_control,True,1.035247,32.965331,-28.323756,32.275173,1.375622,-28.323756,0.472222,85,0.004993,"same signal as pair_price_mr, but halflife 5, threshold 0.12, weekly rebalance, 40 target daily pair vol",Wheat pair diagnostic row.,"Low/moderate: no alpha fitting, but adds a fixed execution filter after seeing costs matter."
5,pair_price_mr_physical_90_10_cost_control,True,0.896508,21.914390,-20.836313,24.088117,1.236854,-22.247475,0.496552,72,0.004822,"90% SRW/HRW price MR + 10% SRW/HRW total physical family, with fixed cost-control execution",Wheat pair diagnostic row.,"Low/moderate: round-number 10% total physical overlay; no fitted coefficients, but still a researched blend."
6,pair_price_mr,True,0.539491,65.746173,-58.180532,75.995588,0.243434,-142.668086,0.500911,275,0.008666,signal = SRW.rev_5 - HRW.rev_5,Wheat pair diagnostic row.,Very low: one fixed price-reversion hypothesis.
7,pair_cot,True,0.100271,8.756899,-46.607109,73.880246,-0.125224,-103.957149,0.481188,243,0.004461,signal = SRW COT family - HRW COT family,Wheat pair diagnostic row.,Very low: one fixed positioning hypothesis.
8,pair_mr_curve_physical_cost_control,True,0.038225,1.263508,-36.644226,49.617876,0.453674,-68.411073,0.479070,103,0.002024,"same signal as pair_mr_curve_physical_equal, but halflife 5, threshold 0.12, weekly rebalance, 40 target daily pair vol",Wheat pair diagnostic row.,"Low/moderate: no alpha fitting, but adds a fixed execution filter after seeing costs matter."
9,pair_fixed_trend_mr_regime,True,-0.056777,-5.655212,-64.342796,83.863870,0.068593,-191.872737,0.479245,254,0.004091,if observable trend regime: trend/curve/COT pair; otherwise MR/curve/physical pair,Wheat pair diagnostic row.,"Moderate research risk: fixed observable regime switch, no optimized weights."


## Wheat Step 4 - Regimes That Became Clear

This regime table is shown only after the directional wheat tests and SRW/HRW pair experiment. The interpretation comes from the result path: outright wheat was weak, the pair worked better, and trend-conflict/trend-aware variants helped identify when MR timing risk is high.


In [80]:
wheat_regime_rows = []
if wheat_pair is not None and "trend_regime" in wheat_pair:
    trend_mask = pd.Series(wheat_pair["trend_regime"]).astype(bool)
    wheat_regime_rows.extend([
        {
            "product": "Wheat SRW/HRW",
            "regime": "pair_trend",
            "detection": "avg(abs(SRW mom_60), abs(HRW mom_60)) > expanding median",
            "active_days": int(trend_mask.sum()),
            "active_share": float(trend_mask.mean()),
            "experiment_where_clear": "Wheat pair tests",
        },
        {
            "product": "Wheat SRW/HRW",
            "regime": "pair_mr_or_chop",
            "detection": "not pair_trend",
            "active_days": int((~trend_mask).sum()),
            "active_share": float((~trend_mask).mean()),
            "experiment_where_clear": "Wheat pair tests",
        },
    ])
if wheat_regime_rows:
    pd.DataFrame(wheat_regime_rows)[
        ["product", "regime", "detection", "active_days", "active_share", "experiment_where_clear"]
    ].reset_index(drop=True)
else:
    pd.DataFrame({"message": ["Run Wheat Step 3 to show pair trend/MR regime diagnostics."]})


In [81]:
wheat_regime_perf_rows = []
if wheat_pair is not None and "trend_regime" in wheat_pair:
    trend_mask = pd.Series(wheat_pair["trend_regime"]).astype(bool)
    for key in [
        "pair_price_mr_cargill_90_10_cost_control_cost",
        "pair_price_mr_cargill_80_20_pair_trend_cost_control_cost",
        "pair_price_mr_cargill_trend_conflict_flat_cost_control_cost",
    ]:
        bt = wheat_pair.get("backtests", {}).get(key)
        if bt is None:
            continue
        trend_metrics = grain_futures_strategy.performance_metrics(bt.loc[trend_mask.reindex(bt.index).fillna(False)])
        mr_metrics = grain_futures_strategy.performance_metrics(bt.loc[(~trend_mask).reindex(bt.index).fillna(False)])
        wheat_regime_perf_rows.extend([
            {
                "strategy": key,
                "regime": "pair_trend",
                "pnl": trend_metrics.get("total_pnl"),
                "sharpe": trend_metrics.get("sharpe"),
                "dd": trend_metrics.get("max_drawdown"),
                "hit_rate": trend_metrics.get("hit_rate"),
            },
            {
                "strategy": key,
                "regime": "pair_mr_or_chop",
                "pnl": mr_metrics.get("total_pnl"),
                "sharpe": mr_metrics.get("sharpe"),
                "dd": mr_metrics.get("max_drawdown"),
                "hit_rate": mr_metrics.get("hit_rate"),
            },
        ])
if wheat_regime_perf_rows:
    pd.DataFrame(wheat_regime_perf_rows)[
        ["strategy", "regime", "pnl", "sharpe", "dd", "hit_rate"]
    ].sort_values(["strategy", "regime"]).reset_index(drop=True)
else:
    pd.DataFrame({"message": ["Run Wheat Step 3 to show strategy performance by trend/MR regime."]})


## Wheat Step 5 - Key Periods For Candidate Rows


In [82]:
if wheat_pair is not None:
    for key in [
        "pair_price_mr_cargill_90_10_cost_control_cost",
        "pair_price_mr_cargill_80_20_pair_trend_cost_control_cost",
        "pair_price_mr_cargill_trend_conflict_flat_cost_control_cost",
    ]:
        if key in wheat_pair["backtests"]:
            display(Markdown(f"### {key}"))
            display(period_performance(wheat_pair["backtests"][key])[["period", "total_pnl", "sharpe", "max_drawdown", "hit_rate", "days"]])

### pair_price_mr_cargill_90_10_cost_control_cost

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,6.280203,16.323259,-0.254644,0.800000,5.0
1,US drought rally/retrace,3.135122,1.366923,-7.245108,0.466667,15.0
2,Crimea/Black Sea shock,11.446453,2.702238,-10.996680,0.583333,24.0
3,Low-price abundant supply,34.195449,1.542958,-22.050536,0.533333,135.0
4,US-China trade war,6.921635,0.545953,-18.854148,0.458333,72.0
5,2019 prevented planting floods,-11.578739,-4.629163,-13.730632,0.300000,20.0
6,COVID demand shock,-0.143498,-0.039966,-13.324861,0.400000,20.0
7,COVID recovery/China buying,-14.392343,-4.001409,-17.657699,0.440000,25.0


### pair_price_mr_cargill_80_20_pair_trend_cost_control_cost

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,35.853955,4.947551,-6.282385,0.600000,75.0
1,US drought rally/retrace,8.966299,2.009916,-6.900235,0.545455,55.0
2,Crimea/Black Sea shock,8.912027,3.560503,-6.360879,0.650000,20.0
3,Low-price abundant supply,1.352910,0.053667,-28.454809,0.421429,140.0
4,US-China trade war,17.331041,1.399091,-14.033066,0.539326,89.0
5,2019 prevented planting floods,-3.499238,-1.400473,-10.155347,0.523810,21.0
6,COVID demand shock,0.957399,0.451529,-12.902658,0.400000,10.0
7,COVID recovery/China buying,8.256647,4.122904,-4.248266,0.600000,15.0


### pair_price_mr_cargill_trend_conflict_flat_cost_control_cost

,period,total_pnl,sharpe,max_drawdown,hit_rate,days
0,Russian drought/export ban shock,9.571864,7.547382,-2.179138,0.600000,10.0
1,US drought rally/retrace,3.400698,4.518736,-2.908703,0.600000,5.0
2,Crimea/Black Sea shock,8.229236,6.392670,-5.032170,0.700000,10.0
3,Low-price abundant supply,16.592531,1.220328,-20.245805,0.462500,80.0
4,US-China trade war,3.940999,0.711089,-15.296426,0.457143,35.0
5,2019 prevented planting floods,-11.751969,-5.149895,-15.296426,0.333333,15.0
6,COVID demand shock,5.114259,1.171477,-16.955673,0.500000,20.0
7,COVID recovery/China buying,2.056221,3.333623,-3.190987,0.600000,5.0


## Wheat Step 6 - Best Wheat Rows From Actual Tests


In [83]:
wheat_best = wheat_results.loc[wheat_results.get("strategy", pd.Series(dtype=str)).isin([
    "pair_price_mr_cargill_90_10_cost_control",
    "pair_price_mr_cargill_80_20_pair_trend_cost_control",
    "pair_price_mr_cargill_trend_conflict_flat_cost_control",
])].copy()
wheat_best[
    [
        "strategy", "oos_sharpe", "oos_pnl", "oos_dd",
        "oos_dd_pct_avg_margin", "full_sharpe", "full_dd",
        "hit_rate", "win_days", "turnover", "overfit"
    ]
].sort_values("oos_sharpe", ascending=False).reset_index(drop=True)


,strategy,oos_sharpe,oos_pnl,oos_dd,oos_dd_pct_avg_margin,full_sharpe,full_dd,hit_rate,win_days,turnover,overfit
0,pair_price_mr_cargill_trend_conflict_flat_cost_control,2.145334,26.671254,-16.955673,20.821759,1.461884,-20.245805,0.506667,38,0.004858,Low/moderate: observable trend-risk filter; avoids fighting strong pair trends but can undertrade.
1,pair_price_mr_cargill_80_20_pair_trend_cost_control,1.290889,36.374086,-29.441327,42.561576,1.098366,-29.441327,0.517647,88,0.003103,"Low/moderate: fixed 80/20 blend chosen as a trend-risk diversifier, not fitted coefficients."
2,pair_price_mr_cargill_90_10_cost_control,1.128948,26.689223,-19.909683,22.755886,1.403119,-22.050536,0.478571,67,0.005002,"Low/moderate: round-number 10% Cargill physical overlay; no fitted coefficients, but still a researched blend."


# Final Best Rows Generated By The Cells Above

In [84]:
rows = []
if not soy_external_results.empty:
    r = soy_external_results.loc[soy_external_results["experiment"] == "given_crush_plus_weather_hdd_cdd_equal_weight"]
    if not r.empty:
        x = r.iloc[0]
        rows.append({"Product": "Soybeans", "Best row": x["experiment"], "OOS Sharpe": x["test_sharpe"], "OOS PnL": x["test_pnl"], "OOS DD": x["test_max_drawdown"], "Full Sharpe": x["full_sharpe"], "Full DD": x["max_drawdown"], "Read": "Best soybean Sharpe/DD evidence"})
if not soy_low_vol_results.empty:
    r = soy_low_vol_results.loc[soy_low_vol_results["strategy"] == "low_vol_half_base_else_base"]
    if not r.empty:
        x = r.iloc[0]
        rows.append({"Product": "Soybeans", "Best row": x["strategy"], "OOS Sharpe": x["oos_sharpe"], "OOS PnL": x["oos_pnl"], "OOS DD": x["oos_dd"], "Full Sharpe": x["full_sharpe"], "Full DD": x["full_dd"], "Read": "Best practical risk-control version"})
if not corn_guard_results.empty:
    r = corn_guard_results.loc[corn_guard_results["strategy"] == "below_ma_or_negative_mom_flat"]
    if not r.empty:
        x = r.iloc[0]
        rows.append({"Product": "Corn", "Best row": x["strategy"], "OOS Sharpe": x["oos_sharpe"], "OOS PnL": x["oos_pnl"], "OOS DD": x["oos_dd"], "Full Sharpe": x["full_sharpe"], "Full DD": x["full_dd"], "Read": "Best corn abundant-supply guard"})
if not wheat_best.empty:
    for _, x in wheat_best.sort_values("oos_sharpe", ascending=False).iterrows():
        rows.append({"Product": "Wheat SRW/HRW", "Best row": x["strategy"], "OOS Sharpe": x["oos_sharpe"], "OOS PnL": x["oos_pnl"], "OOS DD": x["oos_dd"], "Full Sharpe": x["full_sharpe"], "Full DD": x["full_dd"], "Read": x["overfit"]})
final_best_rows = pd.DataFrame(rows)
final_best_rows

,Product,Best row,OOS Sharpe,OOS PnL,OOS DD,Full Sharpe,Full DD,Read
0,Soybeans,given_crush_plus_weather_hdd_cdd_equal_weight,2.899920,159.501813,-31.081004,2.162887,-78.107398,Best soybean Sharpe/DD evidence
1,Soybeans,low_vol_half_base_else_base,2.505201,163.222826,-34.702062,1.167465,-85.029854,Best practical risk-control version
2,Corn,below_ma_or_negative_mom_flat,1.744611,309.892159,-174.158893,0.735935,-297.795198,Best corn abundant-supply guard
3,Wheat SRW/HRW,pair_price_mr_cargill_trend_conflict_flat_cost_control,2.145334,26.671254,-16.955673,1.461884,-20.245805,Low/moderate: observable trend-risk filter; avoids fighting strong pair trends but can undertrade.
4,Wheat SRW/HRW,pair_price_mr_cargill_80_20_pair_trend_cost_control,1.290889,36.374086,-29.441327,1.098366,-29.441327,"Low/moderate: fixed 80/20 blend chosen as a trend-risk diversifier, not fitted coefficients."
5,Wheat SRW/HRW,pair_price_mr_cargill_90_10_cost_control,1.128948,26.689223,-19.909683,1.403119,-22.050536,"Low/moderate: round-number 10% Cargill physical overlay; no fitted coefficients, but still a researched blend."


# Interpretation

- Soybeans: the full tested path should show Cargill crush first, then external weather/fundamental improvement, then the low-vol risk-control variant.
- Corn: the full tested path should show generic/fitted weakness, ethanol relevance, and the abundant-supply guard as the best improvement.
- Wheat: the full tested path should show weak directional wheat and then the SRW/HRW pair improvement.